# Ozon Tech Contest — FINAL Kaggle Pipeline

**Источники данных разделены явно.** `Kopt Data Private` даёт готовые leakage-safe `snapshot_YYYYMMDD.parquet`, а `Kopt Anchor All` — готовые adaptive anchors на те же cutoff. В Kaggle runtime ни признаки, ни anchors не пересчитываются.

Текущий training contract — три временных фолда: **Nov / Dec / Jan = 0.25 / 0.35 / 0.40**. October не validation, January не outer holdout. Финальное обучение использует пять размеченных snapshot: Sep/Oct/Nov/Dec/Jan; прогноз строится на `2026-02-13`.

Активный baseline — четыре fixed-parameter CatBoost/XGBoost модели. Обе LightGBM specifications сохранены в `MODEL_SPECS` только для provenance и исключены из CV, final fit, stochastic bags, Ridge, hurdle и submission loops.

Пайплайн:

1. читает snapshots только из `Kopt Data Private`, не затрагивая `Kopt Data Private 2`;
2. читает и по `user_id` выравнивает anchors только из `Kopt Anchor All`;
3. строго проверяет schema, user universe, targets, frozen features и `segment_logspace`;
4. прогоняет fixed-parameter temporal CV и получает OOF;
5. переобучает активные модели на всех пяти labelled snapshots;
6. сохраняет base/additional submissions, diagnostics, Error Analysis v2, calibrated hurdle и hierarchical blends.


## 0. Что подключить в Kaggle Input

Обязательно:

- **`Kopt Data Private`** (exact slug `kopt-data-private`) — единственный источник `snapshot_YYYYMMDD.parquet` с готовыми признаками;
- **`Kopt Anchor All`** (exact slug `kopt-anchor-all`) — готовые adaptive anchors на те же cutoff;
- dataset **`samples`** с `sample_submit.csv`.

`Kopt Data Private 2` может оставаться подключённым, но notebook его не рассматривает при поиске snapshot. `full_train_ozon` не читается: признаки и anchors не пересчитываются. `colleagues_submissions` используется только внешним submission/blending-разделом.

Kaggle может смонтировать dataset как `/kaggle/input/<slug>`, `/kaggle/input/datasets/<owner>/<slug>` или под непрозрачным физическим именем. Notebook сначала ищет exact slug, затем безопасно переходит к полному файловому контракту; для snapshots совместный `sample_submit.csv` даёт приоритет нужному источнику над `Kopt Data Private 2`.


In [84]:
from __future__ import annotations

import gc
import hashlib
import itertools
import json
import math
import os
import platform
import random
import subprocess
import sys
import time
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Mapping, Sequence

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 180)
pd.set_option("display.max_rows", 250)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print({
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "working_dir": str(Path.cwd()),
})


{'python': '3.12.13', 'platform': 'Linux-6.12.90+-x86_64-with-glibc2.35', 'working_dir': '/kaggle/working'}


## 1. Единственная конфигурация запуска


In [85]:
# ============================================================
# INPUT / OUTPUT
# ============================================================
KAGGLE_INPUT_ROOT = Path("/kaggle/input")
SNAPSHOT_DATASET_SLUG = "kopt-data-private"
ANCHOR_DATASET_SLUG = "kopt-anchor-all"
SNAPSHOT_DIR_OVERRIDE = None   # explicit directory containing snapshot_*.parquet
ANCHOR_DIR_OVERRIDE = None     # explicit directory containing anchor_segment_adaptive_*.parquet
SAMPLE_SUBMISSION_OVERRIDE = None

OUTPUT_DIR = Path("/kaggle/working/ozon_final")
ANALYSIS_DIR = OUTPUT_DIR / "analysis"
MODEL_DIR = OUTPUT_DIR / "models"
BASE_SUBMISSION_DIR = OUTPUT_DIR / "base_submissions"
BLEND_SUBMISSION_DIR = OUTPUT_DIR / "blend_submissions"
for _d in [OUTPUT_DIR, ANALYSIS_DIR, MODEL_DIR, BASE_SUBMISSION_DIR, BLEND_SUBMISSION_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

# ============================================================
# SNAPSHOT CONTRACT
# ============================================================
INDEX_COL = "__snapshot_index_user_id__"
USER_COL = "user_id"
TARGET_COL = "target_30d"
ANCHOR_COL = "gmv_30d"
SNAPSHOT_COL = "__snapshot_cutoff__"
PREDICTION_CUTOFF = "2026-02-13"

CV_FOLDS = [
    {
        "name": "nov_2025",
        "train_cutoffs": ["2025-09-16", "2025-10-16"],
        "valid_cutoff": "2025-11-15",
        "weight": 0.25,
    },
    {
        "name": "dec_2025",
        "train_cutoffs": ["2025-09-16", "2025-10-16", "2025-11-15"],
        "valid_cutoff": "2025-12-15",
        "weight": 0.35,
    },
    {
        "name": "jan_2026",
        "train_cutoffs": ["2025-09-16", "2025-10-16", "2025-11-15", "2025-12-15"],
        "valid_cutoff": "2026-01-14",
        "weight": 0.40,
    },
]

FINAL_TRAIN_CUTOFFS = [
    "2025-09-16",
    "2025-10-16",
    "2025-11-15",
    "2025-12-15",
    "2026-01-14",
]
ALL_CUTOFFS = [*FINAL_TRAIN_CUTOFFS, PREDICTION_CUTOFF]
REQUIRED_SNAPSHOT_FILENAMES = [
    f"snapshot_{pd.Timestamp(x).strftime('%Y%m%d')}.parquet" for x in ALL_CUTOFFS
]

ADAPTIVE_ANCHOR_COLUMNS = [
    "anchor_segment_adaptive",
    "anchor_volatility_adaptive",
    "anchor_segment_logspace_log",
    "anchor_segment_logspace",
    "anchor_personal_hard",
    "anchor_frequency_monetary_adaptive",
]
ADAPTIVE_ANCHOR_BY_TARGET_MODE = {
    "segment_adaptive": "anchor_segment_adaptive",
    "volatility_adaptive": "anchor_volatility_adaptive",
    # Use the raw anchor and apply log1p exactly once. The *_log column is
    # validated against it, not passed through log1p a second time.
    "segment_logspace": "anchor_segment_logspace",
    "personal_hard": "anchor_personal_hard",
}
REQUIRED_ANCHOR_FILENAMES = [
    f"anchor_segment_adaptive_{pd.Timestamp(x).strftime('%Y%m%d')}.parquet"
    for x in ALL_CUTOFFS
]

# ============================================================
# TRAINING — exact Optuna runner defaults
# ============================================================
USE_GPU = True
ALLOW_GPU_FALLBACK = True
CV_MAX_ROUNDS = {
    "xgboost": 12_000,
    "catboost": 15_000,
}
EARLY_STOPPING_ROUNDS = 50

# None => recover from exact fixed-parameter CV as weighted average of fold best_iteration,
# the same rule used by the supplied Optuna runner.
FIXED_FINAL_ITERATIONS = {
    "xgboost_log_level": None,
    "catboost_log_level": None,
    "catboost_log_diff": None,
    "xgboost_log_diff": None,
}

# Technical smoke only. Final Kaggle run: None.
SMOKE_USERS = None
SMOKE_MAX_ROUNDS = 300

RUN_CV = True
RUN_ERROR_ANALYSIS = True
RUN_FINAL_FIT = True
RUN_BLEND_LAB = True
SAVE_MODELS = True

# ============================================================
# BLEND
# ============================================================
MAX_BLEND_WEIGHT = 0.70
BLEND_RIDGE = 0.01
ALPHA_CLIP = 0.12
SEGMENT_SHRINKAGE = 2000.0
SEGMENT_CLIP = 0.12
FINAL_COMPLEXITY_TIE_TOLERANCE = 0.00005
SAVE_TOP_N_BLEND_SUBMISSIONS = 20

assert [x["name"] for x in CV_FOLDS] == ["nov_2025", "dec_2025", "jan_2026"]
assert np.isclose(sum(x["weight"] for x in CV_FOLDS), 1.0)
assert FINAL_TRAIN_CUTOFFS[-1] == "2026-01-14"
assert PREDICTION_CUTOFF == "2026-02-13"


## 2. Frozen feature lists и fixed hyperparameters


In [86]:
MODEL_SPECS = {'xgboost_log_level': {'name': 'xgboost_log_level',
                       'family': 'xgboost',
                       'target_mode': 'log_level',
                       'features': ['gmv_search_sum_365d',
                                    'search_only_share_active_90d',
                                    'to_ord_block_rate_180_364',
                                    'search_to_ord_block_rate_0_6',
                                    'gmv_cat_block_rate_0_6',
                                    'purchase_days_block_rate_0_6',
                                    'ph_last3_to_median_gmv',
                                    'hazard_carted_after_order_days',
                                    'searches_share30_of_90',
                                    'order_per_search_180d',
                                    'purchase_days_decay30_hl90',
                                    'log1p_decay_to_ord_half_life_60d',
                                    'square_log1p_decay_to_ord_half_life_120d',
                                    'square_log1p_decay_to_ord_half_life_240d',
                                    'order_calendar_day_probability_365d',
                                    'percentile_rank_decay_gmv_half_life_30d',
                                    'hazard_overdue_order_zscore',
                                    'purchase_days_decay30_hl60',
                                    'square_log1p_monthly_orders_mean',
                                    'gmv_frequency_interaction_30d',
                                    'square_log1p_forecast_weekday_profile_gmv_projection',
                                    'percentile_rank_gmv_sum_365d',
                                    'percentile_rank_monthly_gmv_mean',
                                    'searches_sum_7d',
                                    'ph_purchase_days',
                                    'to_cart_sum_14d',
                                    'percentile_rank_decay_gmv_half_life_7d',
                                    'row_days_7d',
                                    'extra_search_sum_5d',
                                    'days_since_last_search_cart',
                                    'searches_share30_of_365',
                                    'days_since_last_search',
                                    'to_ord_block_180_364',
                                    'bayesian_active_day_purchase_probability_365d',
                                    'to_ord_decay30_hl90',
                                    'ph_gmv_cv',
                                    'percentile_rank_to_ord_sum_30d',
                                    'active_days_share30_of_90',
                                    'hol_gmv_log',
                                    'cyc_med_pre7c',
                                    'gmv_block_rate_0_29',
                                    'search_days_block_rate_0_29',
                                    'cutoff_is_weekend',
                                    'hol_cart_uplift',
                                    'active_days_observed_rate_365d'],
                       'params_sampled': {'colsample_bylevel': 0.6441,
                                          'colsample_bynode': 0.6542,
                                          'colsample_bytree': 0.6582,
                                          'gamma': 4.7043,
                                          'learning_rate': 0.008663,
                                          'max_bin': 768,
                                          'max_depth': 6,
                                          'min_child_weight': 5.9805,
                                          'reg_alpha': 0.0000512,
                                          'reg_lambda': 15.3166,
                                          'subsample': 0.5571},
                       'selection_metric': 1.714366,
                       'optuna_metric': 1.711975,
                       'maturity': 'trusted'},
 'catboost_log_level': {'name': 'catboost_log_level',
                        'family': 'catboost',
                        'target_mode': 'log_level',
                        'features': ['cutoff_dayofyear_cos',
                                     'gmv_search_sum_365d',
                                     'any_cart_no_order_day_share_active_90d',
                                     'search_only_share_active_90d',
                                     'cart_per_search_180d',
                                     'order_per_search_180d',
                                     'purchase_days_block_90_179',
                                     'gmv_per_purchase_day_7d',
                                     'gmv_decay30_hl90',
                                     'search_to_cart_block_rate_0_6',
                                     'cyc_cur_search',
                                     'square_log1p_decay_to_ord_half_life_60d',
                                     'square_log1p_gmv_sum_14d',
                                     'log1p_order_gap_mean_days',
                                     'cyc_completed',
                                     'square_log1p_decay_to_ord_half_life_14d',
                                     'purchase_days_calendar_rate_365d',
                                     'days_since_last_order',
                                     'pct_rank_gmv_decay30_hl60',
                                     'search_days_7d__denis',
                                     'cyc_cart_excess',
                                     'row_days_7d',
                                     'cutoff_is_weekend',
                                     'searches_share30_of_180',
                                     'searches_sum_14d',
                                     'searches_calendar_day_rate_14d',
                                     'ph_gmv_cv',
                                     'bayesian_search_to_order_conversion_365d',
                                     'to_ord_share30_of_365',
                                     'prior_freq90_money180',
                                     'percentile_rank_gmv_sum_365d',
                                     'percentile_rank_lifetime_gmv',
                                     'gmv_per_order_30d__denis'],
                        'params_sampled': {'bagging_temperature': 1.3558,
                                           'border_count': 128,
                                           'depth': 6,
                                           'l2_leaf_reg': 113.0073,
                                           'leaf_estimation_iterations': 5,
                                           'learning_rate': 0.005811,
                                           'max_ctr_complexity': 1,
                                           'one_hot_max_size': 58,
                                           'random_strength': 6.1237},
                        'selection_metric': 1.714631,
                        'optuna_metric': 1.7136130789089152,
                        'maturity': 'trusted'},
 'catboost_log_diff': {'name': 'catboost_log_diff',
                       'family': 'catboost',
                       'target_mode': 'log_diff',
                       'features': ['gmv_share_minus_order_share_365d',
                                    'gmv_share_minus_order_share_180d',
                                    'gmv_channel_balance_365d',
                                    'any_cart_no_order_day_share_active_365d',
                                    'search_to_ord_decay30_hl60',
                                    'catalog_order_per_catalog_cart_180d',
                                    'search_gmv_share_180d',
                                    'ph_gmv_median',
                                    'ph_gmv_per_item_mean',
                                    'both_channel_jaccard_365d',
                                    'gmv_maximum_90d',
                                    'gmv_share30_of_365',
                                    'weekday_5_gmv',
                                    'gmv_per_purchase_day_30d',
                                    'days_since_last_purchase',
                                    'gmv_share30_of_90',
                                    'pct_rank_gmv_90d',
                                    'search_order_days_365d',
                                    'prior_last30',
                                    'days_since_last_order',
                                    'prior_avg90',
                                    'cutoff_month_sin',
                                    'purchase_days_share30_of_180',
                                    'purchase_days_calendar_rate_365d',
                                    'hazard_expected_orders_30d',
                                    'gmv_decay30_hl90',
                                    'purchase_days_share30_of_90',
                                    'search_sum_7d',
                                    'cyc_search_progress',
                                    'log1p_decay_to_ord_half_life_240d',
                                    'search_days_7d__denis',
                                    'prior_last30_recency_tau90',
                                    'search_order_per_search_cart_30d',
                                    'hazard_activity_recency_share',
                                    'to_ord_block_rate_0_29',
                                    'days_since_last_cart',
                                    'gmv_search_share30_of_180',
                                    'purchase_days_block_rate_0_29',
                                    'searches_block_rate_0_6',
                                    'search_to_ord_block_180_364',
                                    'decay_acceleration_searches_30d_120d',
                                    'recent_monthly_order_acceleration',
                                    'ph_gap_cv',
                                    'cyc_search_excess',
                                    'order_per_search_365d',
                                    'ph_gmv_mean',
                                    'searches_share30_of_90',
                                    'catalog_cart_share_7d',
                                    'active_days_decay30_hl90',
                                    'user_yoy_current_gmv_ytd',
                                    'search_order_per_search_cart_90d',
                                    'search_to_ord_block_rate_0_6',
                                    'cutoff_days_since_start',
                                    'ph_due_ratio_last_gap'],
                       'params_sampled': {'bagging_temperature': 0.7322,
                                          'border_count': 128,
                                          'depth': 5,
                                          'l2_leaf_reg': 41.4081,
                                          'leaf_estimation_iterations': 2,
                                          'learning_rate': 0.039550,
                                          'max_ctr_complexity': 1,
                                          'one_hot_max_size': 87,
                                          'random_strength': 22.1491},
                       'selection_metric': 1.715168,
                       'optuna_metric': 1.71383,
                       'maturity': 'trusted'},
 'lightgbm_log_level': {'name': 'lightgbm_log_level',
                        'family': 'lightgbm',
                        'target_mode': 'log_level',
                        'features': ['gmv_search_sum_365d',
                                     'gmv_per_purchase_day_7d',
                                     'purchase_days_block_90_179',
                                     'search_to_ord_block_rate_0_6',
                                     'to_ord_block_rate_180_364',
                                     'searches_share30_of_180',
                                     'search_only_share_active_90d',
                                     'gmv_cat_block_rate_0_6',
                                     'searches_sum_14d',
                                     'cart_per_search_7d',
                                     'gmv_decay30_hl90',
                                     'purchase_days_decay30_hl60',
                                     'to_ord_decay30_hl30',
                                     'square_log1p_decay_gmv_half_life_120d',
                                     'hazard_overdue_order_ratio',
                                     'square_log1p_active_gap_mean_days',
                                     'gmv_decay30_hl60',
                                     'gmv_acceleration_60d_to_180d',
                                     'gmv_frequency_interaction_14d',
                                     'percentile_rank_gmv_sum_30d',
                                     'percentile_rank_active_days_90d',
                                     'pct_rank_gmv_180d',
                                     'extra_search_sum_5d',
                                     'square_log1p_decay_to_ord_half_life_240d',
                                     'searches_decay30_hl90',
                                     'ph_gmv_max',
                                     'gmv_active_day_quantile_90_365d',
                                     'search_cart_share_7d',
                                     'to_cart_share30_of_180',
                                     'log1p_order_gap_mean_days',
                                     'ph_gap_median',
                                     'hol_gmv_log',
                                     'ph_last_to_median_gmv',
                                     'hazard_carted_after_order_days',
                                     'days_since_last_any_cart_no_order',
                                     'gmv_mean_active_365d',
                                     'percentile_rank_to_ord_sum_30d',
                                     'decay_acceleration_to_cart_30d_120d',
                                     'percentile_rank_monthly_orders_mean',
                                     'gmv_channel_balance_30d',
                                     'hol_buyer',
                                     'search_to_ord_block_rate_180_364',
                                     'cutoff_is_weekend',
                                     'cyc_med_pre7c',
                                     'cart_days_share30_of_180',
                                     'monthly_orders_coefficient_of_variation',
                                     'search_only_share_active_30d',
                                     'forecast_weekday_profile_gmv_projection',
                                     'pct_rank_to_ord_decay30_hl60'],
                        'params_sampled': {'colsample_bytree': 0.5547,
                                           'extra_trees': True,
                                           'feature_fraction_bynode': 0.8827,
                                           'learning_rate': 0.081387,
                                           'max_bin': 127,
                                           'max_depth': 3,
                                           'min_child_samples': 130,
                                           'min_child_weight': 0.00905,
                                           'min_split_gain': 0.08363,
                                           'path_smooth': 0.000000157,
                                           'reg_alpha': 0.000000017,
                                           'reg_lambda': 86.6899,
                                           'subsample': 0.9904,
                                           'subsample_freq': 1,
                                           'num_leaves_raw': 127},
                        'selection_metric': 1.715356,
                        'optuna_metric': 1.713407,
                        'maturity': 'provisional'},
 'xgboost_log_diff': {'name': 'xgboost_log_diff',
                      'family': 'xgboost',
                      'target_mode': 'log_diff',
                      'features': ['gmv_channel_balance_365d',
                                   'both_channel_jaccard_365d',
                                   'any_cart_no_order_day_30d',
                                   'search_order_share_7d',
                                   'searches_standard_deviation_30d',
                                   'cyc_completion_min',
                                   'gmv_search_sum_365d',
                                   'searches_mean_active_90d',
                                   'to_ord_share30_of_180',
                                   'purchase_days_share30_of_180',
                                   'purchase_days_decay30_hl90',
                                   'search_sum_7d',
                                   'purchase_days_block_90_179',
                                   'log1p_decay_to_ord_half_life_240d',
                                   'cart_days_14d',
                                   'search_order_days_180d',
                                   'ph_gmv_per_item_mean',
                                   'gmv_block_rate_0_29',
                                   'searches_sum_7d',
                                   'cyc_active_progress',
                                   'fresh_search_after_old_purchase',
                                   'forecast_weekday_profile_gmv_projection',
                                   'gmv_maximum_90d',
                                   'extra_search_sum_5d',
                                   'searches_share30_of_365',
                                   'ph_gmv_std',
                                   'order_per_search_30d',
                                   'days_since_last_search_order',
                                   'ph_gap_cv',
                                   'fully_zero_days_30d',
                                   'percentile_rank_decay_gmv_half_life_30d',
                                   'gmv_block_180_364',
                                   'percentile_rank_monthly_gmv_mean',
                                   'row_days_180d',
                                   'ph_gmv_min',
                                   'search_to_cart_block_90_179',
                                   'cutoff_is_weekend',
                                   'fresh_search_cart_no_order_after_old_purchase',
                                   'extra_active_days_10d',
                                   'pct_rank_to_ord_decay30_hl60'],
                      'params_sampled': {'colsample_bylevel': 0.6130,
                                         'colsample_bynode': 0.9325,
                                         'colsample_bytree': 0.8703,
                                         'gamma': 1.6320,
                                         'learning_rate': 0.010300,
                                         'max_bin': 512,
                                         'max_depth': 4,
                                         'min_child_weight': 22.7167,
                                         'reg_alpha': 0.0000136,
                                         'reg_lambda': 111.9384,
                                         'subsample': 0.7788},
                      'selection_metric': 1.716881,
                      'optuna_metric': 1.713494,
                      'maturity': 'trusted'},
 'lightgbm_log_diff': {'name': 'lightgbm_log_diff',
                       'family': 'lightgbm',
                       'target_mode': 'log_diff',
                       'features': ['gmv_share_minus_order_share_365d',
                                    'both_channel_jaccard_365d',
                                    'cyc_completion_min',
                                    'any_cart_no_order_day_30d',
                                    'catalog_only_share_active_180d',
                                    'percentile_rank_lifetime_gmv',
                                    'gmv_share_minus_order_share_180d',
                                    'search_order_share_7d',
                                    'gmv_search_sum_365d',
                                    'gmv_share30_of_365',
                                    'gmv_share30_of_90',
                                    'to_ord_share30_of_180',
                                    'purchase_days_share30_of_180',
                                    'gmv_per_order_180d',
                                    'gmv_block_rate_0_29',
                                    'square_log1p_decay_to_ord_half_life_240d',
                                    'purchase_days_decay30_hl90',
                                    'tkt_vs_market_longterm',
                                    'searches_sum_14d',
                                    'gmv_maximum_90d',
                                    'purchase_days_decay30_hl30',
                                    'searches_share30_of_180',
                                    'ph_gap_std',
                                    'row_days_7d',
                                    'active_days_share30_of_365',
                                    'percentile_rank_gmv_sum_365d',
                                    'ph_gap_cv',
                                    'days_since_last_search_cart',
                                    'extra_cart_days_10d',
                                    'cyc_active_progress',
                                    'monthly_orders_coefficient_of_variation',
                                    'decay_acceleration_to_cart_30d_120d',
                                    'percentile_rank_monthly_orders_mean',
                                    'gmv_active_day_quantile_75_365d',
                                    'search_order_days_180d',
                                    'cyc_med_pre7s',
                                    'gmv_maximum_365d',
                                    'percentile_rank_gmv_sum_30d',
                                    'hazard_overdue_order_zscore',
                                    'search_cart_days_365d',
                                    'square_log1p_active_gap_mean_days',
                                    'to_ord_block_rate_180_364',
                                    'never_active',
                                    'ph_gmv_max',
                                    'fully_zero_days_30d'],
                       'params_sampled': {'colsample_bytree': 0.8896,
                                          'extra_trees': False,
                                          'feature_fraction_bynode': 0.7434,
                                          'learning_rate': 0.084476,
                                          'max_bin': 127,
                                          'max_depth': 3,
                                          'min_child_samples': 220,
                                          'min_child_weight': 1.7711,
                                          'min_split_gain': 0.35945,
                                          'path_smooth': 0.00000161,
                                          'reg_alpha': 0.00000689,
                                          'reg_lambda': 79.4912,
                                          'subsample': 0.9802,
                                          'subsample_freq': 2,
                                          'num_leaves_raw': 31},
                       'selection_metric': 1.71817,
                       'optuna_metric': 1.715655,
                       'maturity': 'provisional'}}

ALL_BASE_MODEL_ORDER = ['xgboost_log_level', 'catboost_log_level', 'catboost_log_diff', 'lightgbm_log_level', 'xgboost_log_diff', 'lightgbm_log_diff']
DISABLED_MODEL_FAMILIES = {'lightgbm'}
MODEL_ORDER = [
    name for name in ALL_BASE_MODEL_ORDER
    if MODEL_SPECS[name]['family'] not in DISABLED_MODEL_FAMILIES
]
# Keep disabled specifications for provenance, but no active loop uses them.
MODEL_SPECS = {name: MODEL_SPECS[name] for name in ALL_BASE_MODEL_ORDER}
MODEL_NAMES = MODEL_ORDER.copy()
assert MODEL_NAMES == ['xgboost_log_level', 'catboost_log_level', 'catboost_log_diff', 'xgboost_log_diff']
assert all(MODEL_SPECS[name]['family'] != 'lightgbm' for name in MODEL_NAMES)

# Manual operational annotation added in this notebook. It is not a field
# emitted by the selection/Optuna artifacts.
MATURITY_PROVENANCE = "manual_not_present_in_source_artifacts"


def unique_keep_order(values):
    return list(dict.fromkeys(values))


def resolve_effective_params(spec: Mapping[str, Any]) -> dict[str, Any]:
    """Turn sampled Optuna params into estimator-only fit parameters."""
    p = dict(spec["params_sampled"])
    if spec["family"] == "lightgbm":
        raw = int(p.pop("num_leaves_raw"))
        depth = int(p["max_depth"])
        p["num_leaves"] = min(raw, 2 ** depth - 1) if depth > 0 else raw
    return p


for _name, _spec in MODEL_SPECS.items():
    # Exact requested names. No __denis -> canonical aliasing.
    _spec["features_raw"] = list(_spec["features"])
    _spec["features"] = unique_keep_order(_spec["features"])
    _spec["duplicates_removed"] = [
        x for i, x in enumerate(_spec["features_raw"]) if x in _spec["features_raw"][:i]
    ]
    _spec["param_metadata"] = {}
    if _spec["family"] == "lightgbm":
        _spec["param_metadata"]["num_leaves_raw"] = int(
            _spec["params_sampled"]["num_leaves_raw"]
        )
    _spec["params"] = resolve_effective_params(_spec)

TRUSTED_MODELS = [n for n in MODEL_NAMES if MODEL_SPECS[n]["maturity"] == "trusted"]
PROVISIONAL_MODELS = [n for n in MODEL_NAMES if MODEL_SPECS[n]["maturity"] == "provisional"]

summary = []
for name in MODEL_NAMES:
    spec = MODEL_SPECS[name]
    summary.append({
        "model": name,
        "family": spec["family"],
        "target_mode": spec["target_mode"],
        "features": len(spec["features"]),
        "selection_metric": spec["selection_metric"],
        "optuna_metric": spec["optuna_metric"],
        "maturity_manual": spec["maturity"],
        "maturity_source": MATURITY_PROVENANCE,
        "duplicates_removed": spec["duplicates_removed"],
        "num_leaves_raw": spec["param_metadata"].get("num_leaves_raw"),
        "num_leaves": spec["params"].get("num_leaves"),
    })
display(pd.DataFrame(summary))
print("Trusted:", TRUSTED_MODELS)
print("Provisional:", PROVISIONAL_MODELS)

assert MODEL_SPECS["lightgbm_log_level"]["params"]["num_leaves"] == 7
assert MODEL_SPECS["lightgbm_log_diff"]["params"]["num_leaves"] == 7
assert all("_num_leaves_raw" not in spec["params"] for spec in MODEL_SPECS.values())
assert MODEL_SPECS["catboost_log_level"]["optuna_metric"] == 1.7136130789089152
assert "search_days_7d__denis" in MODEL_SPECS["catboost_log_level"]["features"]
assert "gmv_per_order_30d__denis" in MODEL_SPECS["catboost_log_level"]["features"]


,model,family,target_mode,features,selection_metric,optuna_metric,maturity_manual,maturity_source,duplicates_removed,num_leaves_raw,num_leaves
0,xgboost_log_level,xgboost,log_level,45,1.714366,1.711975,trusted,manual_not_present_in_source_artifacts,[],None,None
1,catboost_log_level,catboost,log_level,33,1.714631,1.713613,trusted,manual_not_present_in_source_artifacts,[],None,None
2,catboost_log_diff,catboost,log_diff,54,1.715168,1.713830,trusted,manual_not_present_in_source_artifacts,[],None,None
3,xgboost_log_diff,xgboost,log_diff,40,1.716881,1.713494,trusted,manual_not_present_in_source_artifacts,[],None,None


Trusted: ['xgboost_log_level', 'catboost_log_level', 'catboost_log_diff', 'xgboost_log_diff']
Provisional: []


### Зафиксированные особенности конфигурации

- Наборы признаков **frozen**: selector здесь не запускается.
- Имена признаков используются буквально из результатов отбора. В частности, `search_days_7d__denis` и `gmv_per_order_30d__denis` не заменяются на похожие canonical columns.
- LightGBM: `num_leaves_raw` не передаётся estimator напрямую; применяется exact Optuna transform `min(num_leaves_raw, 2**max_depth - 1)`.
- Training loss для всех шести fixed models — RMSE в model space; Optuna objective / reporting — official RMSLE после inverse transform в raw GMV.
- `maturity` (`trusted` / `provisional`) — ручная операционная пометка этого notebook, а не поле артефакта selection/Optuna. Она не используется как автоматическое доказательство качества.
- LightGBM-спецификации сохранены только как provenance и полностью исключены из активного `MODEL_ORDER`; CV/final/stochastic/hurdle loops используют только CatBoost и XGBoost baseline-модели.


## 3. Поиск приватного snapshot cache, sample submission и GPU


In [87]:
def has_cuda() -> bool:
    try:
        out = subprocess.run(
            ["nvidia-smi", "-L"], capture_output=True, text=True,
            timeout=10, check=False,
        )
        return out.returncode == 0 and "GPU" in out.stdout
    except Exception:
        return False


def _exact_dataset_roots(slug: str) -> list[Path]:
    """Find Kaggle mounts whose final directory name exactly equals slug.

    Kaggle may mount an input either as /kaggle/input/<slug> or as
    /kaggle/input/datasets/<owner>/<slug>. Exact-name matching deliberately
    excludes similarly named inputs such as kopt-data-private-2.
    """
    slug_key = str(slug).casefold()
    candidates = [KAGGLE_INPUT_ROOT / slug, Path.cwd() / slug]
    if KAGGLE_INPUT_ROOT.exists():
        candidates.extend(
            path for path in KAGGLE_INPUT_ROOT.rglob("*")
            if path.is_dir() and path.name.casefold() == slug_key
        )
    result = []
    seen = set()
    for path in candidates:
        if not path.is_dir() or path.name.casefold() != slug_key:
            continue
        key = str(path.resolve())
        if key not in seen:
            seen.add(key)
            result.append(path)
    result.sort(key=lambda path: (
        path != KAGGLE_INPUT_ROOT / slug,
        len(path.parts),
        len(str(path)),
    ))
    return result


def _complete_parents(root: Path, required_names, pattern: str) -> list[Path]:
    if not root.is_dir():
        return []
    required = set(required_names)
    parents = {}
    for path in root.rglob(pattern):
        if path.is_file():
            parents.setdefault(path.parent, set()).add(path.name)
    return [
        parent for parent, names in parents.items()
        if required.issubset(names)
    ]


def _discover_contract_parent(
    slug: str, override, label: str, required_names, pattern: str,
    *, prefer_sample: bool = False, forbidden_components=(),
) -> Path:
    if override:
        root = Path(override)
        if not root.exists():
            raise FileNotFoundError(f"{label} override does not exist: {root}")
        candidates = _complete_parents(root, required_names, pattern)
        if not candidates:
            raise FileNotFoundError(
                f"{label} override {root} does not contain the complete file contract"
            )
        return sorted(candidates, key=lambda path: (len(path.parts), len(str(path))))[0]

    # First try human-readable/direct mounts. Some Kaggle runtimes instead use
    # opaque physical mount names, so absence of an exact slug is not an error.
    candidates = []
    for root in _exact_dataset_roots(slug):
        candidates.extend(_complete_parents(root, required_names, pattern))
    used_content_fallback = not candidates
    if used_content_fallback:
        candidates = _complete_parents(KAGGLE_INPUT_ROOT, required_names, pattern)

    candidates = list(dict.fromkeys(candidates))
    if not candidates:
        raise FileNotFoundError(
            f"{label}: в {KAGGLE_INPUT_ROOT} не найден каталог с полным "
            f"файловым контрактом: {', '.join(sorted(required_names))}"
        )

    forbidden = {str(value).casefold() for value in forbidden_components}

    def core_score(path: Path):
        parts = {part.casefold() for part in path.parts}
        return (
            int(prefer_sample and (path / "sample_submit.csv").is_file()),
            int(not bool(parts & forbidden)),
            int(slug.casefold() in parts),
        )

    ranked = sorted(
        candidates,
        key=lambda path: (core_score(path), -len(path.parts), -len(str(path))),
        reverse=True,
    )
    best_core = core_score(ranked[0])
    tied = [path for path in ranked if core_score(path) == best_core]
    if len(tied) > 1:
        raise RuntimeError(
            f"{label}: найдено несколько равнозначных каталогов: "
            f"{[str(path) for path in tied]}. Задайте explicit override."
        )
    selected = ranked[0]
    if used_content_fallback:
        print(
            f"{label}: Kaggle mount name differs from display slug; "
            f"selected by complete file contract: {selected}"
        )
    return selected


def discover_snapshot_dir() -> Path:
    return _discover_contract_parent(
        SNAPSHOT_DATASET_SLUG, SNAPSHOT_DIR_OVERRIDE, "Kopt Data Private snapshots",
        REQUIRED_SNAPSHOT_FILENAMES, "snapshot_*.parquet",
        prefer_sample=True, forbidden_components=("kopt-data-private-2",),
    )


def discover_anchor_dir() -> Path:
    return _discover_contract_parent(
        ANCHOR_DATASET_SLUG, ANCHOR_DIR_OVERRIDE, "Kopt Anchor All",
        REQUIRED_ANCHOR_FILENAMES, "anchor_segment_adaptive_*.parquet",
    )


def discover_sample_submission() -> Path:
    if SAMPLE_SUBMISSION_OVERRIDE:
        path = Path(SAMPLE_SUBMISSION_OVERRIDE)
        if not path.exists():
            raise FileNotFoundError(path)
        return path
    # The supplied screenshot shows a copy next to snapshots. Prefer it;
    # otherwise search only an exact-slug `samples` mount (direct or nested).
    matches = list(SNAPSHOT_DIR.rglob("sample_submit.csv"))
    for root in _exact_dataset_roots("samples"):
        matches.extend(root.rglob("sample_submit.csv"))
    local_sample = Path.cwd() / "sample_submit.csv"
    if local_sample.is_file():
        matches.append(local_sample)
    matches = [path for path in matches if path.is_file()]
    if not matches:
        raise FileNotFoundError(
            "Не найден sample_submit.csv рядом с snapshots или "
            "в dataset с точным slug 'samples'"
        )
    matches = list(dict.fromkeys(matches))
    matches.sort(key=lambda p: (
        SNAPSHOT_DIR not in p.parents and p.parent != SNAPSHOT_DIR,
        len(str(p)),
    ))
    return matches[0]


SNAPSHOT_DIR = discover_snapshot_dir()
ANCHOR_DIR = discover_anchor_dir()
SAMPLE_SUBMISSION_PATH = discover_sample_submission()
GPU_AVAILABLE = has_cuda()

print({
    "snapshot_dataset": SNAPSHOT_DATASET_SLUG,
    "snapshot_dir": str(SNAPSHOT_DIR),
    "anchor_dataset": ANCHOR_DATASET_SLUG,
    "anchor_dir": str(ANCHOR_DIR),
    "sample_submission": str(SAMPLE_SUBMISSION_PATH),
    "gpu_available": GPU_AVAILABLE,
    "gpu_requested": USE_GPU,
})


Kopt Data Private snapshots: Kaggle mount name differs from display slug; selected by complete file contract: /kaggle/input/datasets/denisdugarev/kopt-data-priv
{'snapshot_dataset': 'kopt-data-private', 'snapshot_dir': '/kaggle/input/datasets/denisdugarev/kopt-data-priv', 'anchor_dataset': 'kopt-anchor-all', 'anchor_dir': '/kaggle/input/datasets/vladimirsmirnovve/kopt-anchor-all', 'sample_submission': '/kaggle/input/datasets/denisdugarev/kopt-data-priv/sample_submit.csv', 'gpu_available': True, 'gpu_requested': True}


## 4. Snapshot metadata + sample submission


In [88]:
import pyarrow.parquet as pq

sample_submit = pd.read_csv(SAMPLE_SUBMISSION_PATH)
if USER_COL not in sample_submit.columns:
    raise KeyError(f"sample_submit has no {USER_COL}")
if sample_submit[USER_COL].duplicated().any():
    raise ValueError("sample_submit contains duplicate user_id")
_submission_candidates = [c for c in sample_submit.columns if c != USER_COL]
if len(_submission_candidates) != 1:
    raise ValueError(f"Expected exactly one submission target column, got {_submission_candidates}")
SUBMISSION_TARGET_COL = _submission_candidates[0]


def snapshot_path(cutoff: str) -> Path:
    return SNAPSHOT_DIR / f"snapshot_{pd.Timestamp(cutoff).strftime('%Y%m%d')}.parquet"


def read_snapshot_sidecar(path: Path) -> dict:
    candidates = [
        Path(str(path) + ".meta.json"),
        path.with_suffix(".meta.json"),
    ]
    for p in candidates:
        if p.exists():
            try:
                return json.loads(p.read_text(encoding="utf-8"))
            except Exception as exc:
                print(f"WARNING: cannot parse {p.name}: {exc}")
    return {}


def parquet_info(cutoff: str) -> dict:
    path = snapshot_path(cutoff)
    if not path.exists():
        raise FileNotFoundError(path)
    pf = pq.ParquetFile(path)
    schema = pf.schema_arrow
    types = {name: str(tp) for name, tp in zip(schema.names, schema.types)}
    meta = read_snapshot_sidecar(path)
    return {
        "cutoff": cutoff,
        "path": path,
        "rows": int(pf.metadata.num_rows),
        "columns": list(schema.names),
        "types": types,
        "sidecar": meta,
        "cache_signature": meta.get("cache_signature"),
        "target_known_meta": meta.get("target_known"),
        "categorical_meta": list(meta.get("categorical_features", [])),
    }


SNAPSHOT_INFO = {c: parquet_info(c) for c in ALL_CUTOFFS}
for c, info in SNAPSHOT_INFO.items():
    print(c, info["rows"], len(info["columns"]), info["cache_signature"])

print("sample_submit:", sample_submit.shape, "target_col=", SUBMISSION_TARGET_COL)



def anchor_path(cutoff: str) -> Path:
    return ANCHOR_DIR / f"anchor_segment_adaptive_{pd.Timestamp(cutoff).strftime('%Y%m%d')}.parquet"


def adaptive_anchor_info(cutoff: str) -> dict:
    path = anchor_path(cutoff)
    if not path.exists():
        raise FileNotFoundError(path)
    parquet = pq.ParquetFile(path)
    schema = parquet.schema_arrow
    columns = list(schema.names)
    id_columns = [c for c in [USER_COL, INDEX_COL] if c in columns]
    if len(id_columns) != 1:
        raise KeyError(
            f"{path.name}: expected exactly one id column from "
            f"{[USER_COL, INDEX_COL]}, got {id_columns}"
        )
    missing = sorted(set(ADAPTIVE_ANCHOR_COLUMNS) - set(columns))
    if missing:
        raise KeyError(f"{path.name}: missing adaptive anchors {missing}")
    return {
        "cutoff": cutoff, "path": path,
        "rows": int(parquet.metadata.num_rows),
        "columns": columns, "id_column": id_columns[0],
    }


ANCHOR_INFO = {cutoff: adaptive_anchor_info(cutoff) for cutoff in ALL_CUTOFFS}
for cutoff, info in ANCHOR_INFO.items():
    print("anchor", cutoff, info["rows"], info["id_column"], info["path"].name)


2025-09-16 250000 946 None
2025-10-16 250000 946 None
2025-11-15 250000 946 None
2025-12-15 250000 946 None
2026-01-14 250000 946 None
2026-02-13 250000 946 None
sample_submit: (250000, 2) target_col= predict
anchor 2025-09-16 250000 user_id anchor_segment_adaptive_20250916.parquet
anchor 2025-10-16 250000 user_id anchor_segment_adaptive_20251016.parquet
anchor 2025-11-15 250000 user_id anchor_segment_adaptive_20251115.parquet
anchor 2025-12-15 250000 user_id anchor_segment_adaptive_20251215.parquet
anchor 2026-01-14 250000 user_id anchor_segment_adaptive_20260114.parquet
anchor 2026-02-13 250000 user_id anchor_segment_adaptive_20260213.parquet


## 5. Production snapshot loader — без повторной генерации фичей


In [89]:
NUMERIC_PREFIXES = ("int", "float", "double", "uint")


def _selector_schema_categoricals(info: Mapping[str, Any]) -> list[str]:
    """Exact current nsel/data.py rule: everything not int/float/double/uint is categorical."""
    return [
        name for name in info["columns"]
        if not str(info["types"][name]).startswith(NUMERIC_PREFIXES)
        and name not in {INDEX_COL, TARGET_COL}
    ]


COMMON_COLUMNS = set(SNAPSHOT_INFO[ALL_CUTOFFS[0]]["columns"])
for _c in ALL_CUTOFFS[1:]:
    COMMON_COLUMNS &= set(SNAPSHOT_INFO[_c]["columns"])

UNION_FEATURES = unique_keep_order(
    f for name in MODEL_NAMES for f in MODEL_SPECS[name]["features"]
)

# Authoritative categorical semantics for reproduction come from nsel/data.py schema rule.
CATEGORICAL_FEATURES = set(_selector_schema_categoricals(SNAPSHOT_INFO[ALL_CUTOFFS[0]]))
for _c in ALL_CUTOFFS[1:]:
    current = set(_selector_schema_categoricals(SNAPSHOT_INFO[_c]))
    if current != CATEGORICAL_FEATURES:
        changed = sorted((current ^ CATEGORICAL_FEATURES) & set(UNION_FEATURES))
        if changed:
            raise AssertionError(f"Categorical schema changed across snapshots for frozen features: {changed}")

FROZEN_CATEGORICAL_FEATURES = sorted(set(UNION_FEATURES) & CATEGORICAL_FEATURES)


def load_snapshot(cutoff: str, columns: Sequence[str] | None = None) -> pd.DataFrame:
    """Projected parquet read, sorted exactly like nsel/data.py by structural user id."""
    path = snapshot_path(cutoff)
    available = set(SNAPSHOT_INFO[cutoff]["columns"])
    requested = list(columns) if columns is not None else list(SNAPSHOT_INFO[cutoff]["columns"])
    requested = unique_keep_order(requested)
    # USER_COL is the public alias created from INDEX_COL below. Never read
    # both physical columns and then rename INDEX_COL to USER_COL: that would
    # create duplicate `user_id` columns and DataFrame-valued selection.
    requested = [column for column in requested if column != USER_COL]
    if INDEX_COL not in requested:
        requested = [INDEX_COL, *requested]
    missing = [c for c in requested if c not in available]
    if missing:
        raise KeyError(f"{cutoff}: missing requested columns {missing[:30]}")
    frame = pd.read_parquet(path, columns=requested)
    frame = frame.sort_values(INDEX_COL, kind="stable").reset_index(drop=True)
    if SMOKE_USERS is not None:
        frame = frame.iloc[: int(SMOKE_USERS)].copy()
    frame = frame.rename(columns={INDEX_COL: USER_COL})
    frame[SNAPSHOT_COL] = pd.Timestamp(cutoff)
    return frame


# Analysis-only raw snapshot columns. They are NEVER included in model features unless frozen list says so.
_ANALYSIS_CANDIDATES = {
    "__analysis_days_since_last_purchase": ["days_since_last_purchase", "days_since_last_order"],
    "__analysis_active_days_30d": ["active_days_30d", "active_days_sum_30d"],
    "__analysis_purchase_days_30d": ["purchase_days_30d", "purchase_days_sum_30d"],
    "__analysis_searches_30d": ["searches_30d", "searches_sum_30d", "searches_sum_14d"],
}
ANALYSIS_SOURCE_COLUMNS = {}
for alias, candidates in _ANALYSIS_CANDIDATES.items():
    found = next((c for c in candidates if c in COMMON_COLUMNS), None)
    ANALYSIS_SOURCE_COLUMNS[alias] = found
print("Frozen union:", len(UNION_FEATURES))
print("Frozen categoricals:", FROZEN_CATEGORICAL_FEATURES)
print("Analysis source columns:", ANALYSIS_SOURCE_COLUMNS)



_ADAPTIVE_ANCHOR_CACHE = {}


def load_adaptive_anchors(cutoff: str, columns: Sequence[str] | None = None) -> pd.DataFrame:
    info = ANCHOR_INFO[cutoff]
    requested = list(columns) if columns is not None else list(ADAPTIVE_ANCHOR_COLUMNS)
    requested = unique_keep_order(requested)
    missing = [c for c in requested if c not in info["columns"]]
    if missing:
        raise KeyError(f"{cutoff}: anchor file misses {missing}")
    if cutoff not in _ADAPTIVE_ANCHOR_CACHE:
        frame = pd.read_parquet(
            info["path"], columns=[info["id_column"], *ADAPTIVE_ANCHOR_COLUMNS]
        )
        frame = frame.rename(columns={info["id_column"]: USER_COL})
        if frame[USER_COL].isna().any() or frame[USER_COL].duplicated().any():
            raise AssertionError(f"{cutoff}: invalid/duplicate user ids in adaptive anchors")
        _ADAPTIVE_ANCHOR_CACHE[cutoff] = frame.set_index(USER_COL, drop=False)
    return _ADAPTIVE_ANCHOR_CACHE[cutoff][[USER_COL, *requested]]


def adaptive_anchor_vector(cutoff: str, user_ids, anchor_column: str) -> np.ndarray:
    frame = load_adaptive_anchors(cutoff, [anchor_column])
    requested_index = pd.Index(np.asarray(user_ids), name=USER_COL)
    if requested_index.duplicated().any():
        raise AssertionError(f"{cutoff}: duplicate requested user ids")
    aligned = frame[anchor_column].reindex(requested_index)
    values = pd.to_numeric(aligned, errors="coerce").to_numpy(dtype=np.float64)
    if not np.isfinite(values).all() or np.any(values < 0):
        missing_count = int(np.sum(~np.isfinite(values)))
        raise AssertionError(
            f"{cutoff}/{anchor_column}: invalid or unaligned values; "
            f"nonfinite={missing_count}"
        )
    return values


Frozen union: 141
Frozen categoricals: []
Analysis source columns: {'__analysis_days_since_last_purchase': 'days_since_last_purchase', '__analysis_active_days_30d': 'active_days_30d', '__analysis_purchase_days_30d': 'purchase_days_30d', '__analysis_searches_30d': 'searches_30d'}


In [90]:
# Snapshot cache is immutable input in this notebook.
# No build_unified_features(), no selector, no target re-computation.
# This intentional no-op cell exists only to preserve the final_solution section structure.
print("[PASS] using immutable production snapshot cache; feature generation skipped")


[PASS] using immutable production snapshot cache; feature generation skipped


## 6. Snapshot preflight — до любого обучения


In [91]:
preflight_rows = []
cache_signatures = []
reference_users = None

for cutoff in ALL_CUTOFFS:
    info = SNAPSHOT_INFO[cutoff]
    required = {INDEX_COL, TARGET_COL, ANCHOR_COL, *UNION_FEATURES}
    missing = sorted(required - set(info["columns"]))
    if missing:
        raise KeyError(f"{cutoff}: production cache misses frozen columns: {missing}")

    # Read only structural id + target + anchor for hard checks.
    check_df = load_snapshot(cutoff, [TARGET_COL, ANCHOR_COL])
    users = check_df[USER_COL]
    if users.duplicated().any():
        raise AssertionError(f"{cutoff}: duplicate user_id")
    if reference_users is None:
        reference_users = users.to_numpy(copy=True)
    elif not np.array_equal(reference_users, users.to_numpy(copy=False)):
        raise AssertionError(f"{cutoff}: user universe/order differs after deterministic sort")

    is_prediction = cutoff == PREDICTION_CUTOFF
    target = pd.to_numeric(check_df[TARGET_COL], errors="coerce")
    anchor = pd.to_numeric(check_df[ANCHOR_COL], errors="raise")
    if is_prediction:
        if target.notna().any():
            raise AssertionError("Prediction snapshot must have unknown target_30d (all NaN)")
    else:
        if target.isna().any():
            raise AssertionError(f"{cutoff}: labelled target_30d contains NaN")
        if (target < 0).any():
            raise AssertionError(f"{cutoff}: target_30d contains negative values")
    if anchor.isna().any() or (anchor < 0).any():
        raise AssertionError(f"{cutoff}: invalid gmv_30d anchor")

    if info["target_known_meta"] is not None:
        expected = not is_prediction
        if bool(info["target_known_meta"]) != expected:
            raise AssertionError(
                f"{cutoff}: sidecar target_known={info['target_known_meta']} expected={expected}"
            )
    if info["cache_signature"]:
        cache_signatures.append(info["cache_signature"])

    preflight_rows.append({
        "cutoff": cutoff,
        "rows": len(check_df),
        "columns": len(info["columns"]),
        "target_known": bool(target.notna().all()),
        "anchor_mean": float(anchor.mean()),
        "cache_signature": info["cache_signature"],
    })
    del check_df
    gc.collect()

user_universe = pd.Index(reference_users, name=USER_COL)
if len(sample_submit) != len(user_universe):
    raise AssertionError(
        f"sample_submit rows={len(sample_submit)} != snapshot users={len(user_universe)}"
    )
if set(sample_submit[USER_COL].tolist()) != set(user_universe.tolist()):
    raise AssertionError("sample_submit user universe differs from snapshot cache")
if cache_signatures and len(set(cache_signatures)) != 1:
    raise AssertionError(f"Different cache signatures across snapshots: {set(cache_signatures)}")

PREFLIGHT = pd.DataFrame(preflight_rows)
display(PREFLIGHT)
print("[PASS] exact six-snapshot cache preflight")



# External adaptive-anchor preflight. Alignment is by id, never by row position.
anchor_preflight_rows = []
for cutoff in ALL_CUTOFFS:
    info = ANCHOR_INFO[cutoff]
    frame = load_adaptive_anchors(cutoff)
    anchor_users = pd.Index(frame[USER_COL], name=USER_COL)
    if SMOKE_USERS is None:
        if len(anchor_users) != len(user_universe):
            raise AssertionError(
                f"{cutoff}: anchor rows={len(anchor_users)} != snapshot rows={len(user_universe)}"
            )
        if len(anchor_users.difference(user_universe)) or len(user_universe.difference(anchor_users)):
            raise AssertionError(f"{cutoff}: Kopt Anchor All user universe differs from snapshots")
    else:
        if len(user_universe.difference(anchor_users)):
            raise AssertionError(f"{cutoff}: sampled snapshot users missing from anchors")

    aligned = {
        column: adaptive_anchor_vector(cutoff, user_universe.to_numpy(), column)
        for column in ADAPTIVE_ANCHOR_COLUMNS
    }
    raw_logspace = aligned["anchor_segment_logspace"]
    supplied_logspace = aligned["anchor_segment_logspace_log"]
    if not np.allclose(
        np.log1p(raw_logspace), supplied_logspace,
        rtol=1e-5, atol=1e-6, equal_nan=False,
    ):
        max_gap = float(np.max(np.abs(np.log1p(raw_logspace) - supplied_logspace)))
        raise AssertionError(
            f"{cutoff}: anchor_segment_logspace_log inconsistent with raw; max_gap={max_gap}"
        )
    anchor_preflight_rows.append({
        "cutoff": cutoff, "rows": len(frame),
        "id_column": info["id_column"],
        **{f"mean__{column}": float(values.mean()) for column, values in aligned.items()},
    })
    del frame, aligned
    gc.collect()

ANCHOR_PREFLIGHT = pd.DataFrame(anchor_preflight_rows)
ANCHOR_PREFLIGHT.to_csv(ANALYSIS_DIR / "adaptive_anchor_preflight.csv", index=False)
display(ANCHOR_PREFLIGHT)
print("[PASS] Kopt Anchor All schema, user alignment and logspace consistency")


,cutoff,rows,columns,target_known,anchor_mean,cache_signature
0,2025-09-16,250000,946,True,88.961395,None
1,2025-10-16,250000,946,True,91.203011,None
2,2025-11-15,250000,946,True,91.796371,None
3,2025-12-15,250000,946,True,102.078918,None
4,2026-01-14,250000,946,True,101.425941,None
5,2026-02-13,250000,946,False,84.034042,None


[PASS] exact six-snapshot cache preflight


,cutoff,rows,id_column,mean__anchor_segment_adaptive,mean__anchor_volatility_adaptive,mean__anchor_segment_logspace_log,mean__anchor_segment_logspace,mean__anchor_personal_hard,mean__anchor_frequency_monetary_adaptive
0,2025-09-16,250000,user_id,78.764371,84.122493,1.997941,46.772057,78.753884,76.835182
1,2025-10-16,250000,user_id,82.201580,88.330803,2.075364,49.114784,82.056318,80.608432
2,2025-11-15,250000,user_id,84.060081,90.169235,2.145400,50.329614,84.084602,83.444875
3,2025-12-15,250000,user_id,89.117266,96.134711,2.247845,54.012544,89.627260,87.699514
4,2026-01-14,250000,user_id,92.171877,97.745503,2.314801,55.431347,92.286600,91.142704
5,2026-02-13,250000,user_id,87.584267,90.971765,2.304018,52.934234,85.664429,89.362982


[PASS] Kopt Anchor All schema, user alignment and logspace consistency


## 7. Target adapters и exact RMSLE


In [92]:
def rmsle(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=np.float64).reshape(-1)
    y_pred = np.asarray(y_pred, dtype=np.float64).reshape(-1)
    if len(y_true) != len(y_pred):
        raise ValueError("RMSLE length mismatch")
    if not np.isfinite(y_true).all() or np.any(y_true < 0):
        raise ValueError("RMSLE truth must be finite and non-negative")
    if not np.isfinite(y_pred).all():
        raise ValueError("RMSLE prediction contains NaN/inf")
    y_pred = np.clip(y_pred, 0.0, None)
    return float(np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2)))


def build_model_target(y_raw, anchor, target_mode: str) -> np.ndarray:
    # Exact nsel/model.py contract.
    level = np.log1p(np.clip(np.asarray(y_raw, dtype=np.float64), 0.0, None))
    if target_mode == "log_level":
        return level
    if target_mode == "log_diff":
        return level - np.log1p(np.clip(np.asarray(anchor, dtype=np.float64), 0.0, None))
    raise ValueError(target_mode)


def restore_log_level(prediction, anchor, target_mode: str) -> np.ndarray:
    pred = np.asarray(prediction, dtype=np.float64)
    if target_mode == "log_level":
        return np.clip(pred, 0.0, None)
    if target_mode == "log_diff":
        return np.clip(
            pred + np.log1p(np.clip(np.asarray(anchor, dtype=np.float64), 0.0, None)),
            0.0, None,
        )
    raise ValueError(target_mode)


def inverse_model_target(prediction, anchor, target_mode: str) -> np.ndarray:
    return np.expm1(restore_log_level(prediction, anchor, target_mode))


_y = np.array([0.0, 1.0, 10.0, 1000.0])
_a = np.array([0.0, 5.0, 20.0, 500.0])
for _mode in ("log_level", "log_diff"):
    assert np.allclose(_y, inverse_model_target(build_model_target(_y, _a, _mode), _a, _mode))
print("[PASS] target round-trip log_level/log_diff")


[PASS] target round-trip log_level/log_diff


## 8. Exact ordered Empirical-Bayes target encoding для XGB/LGB


In [93]:
@dataclass(frozen=True)
class EncodingConfig:
    mode: str = "empirical_bayes"
    fixed_smoothing: float = 20.0
    min_count: int = 1
    first_block_value: str = "nan"
    unknown_value: str = "global"
    n_blocks: int = 20
    minimum_variance: float = 1e-9
    maximum_smoothing: float = 10000.0


ENCODING_CONFIG = EncodingConfig()
_CATEGORY_TABLES: dict[str, dict[str, int]] = {}


def _global_mean(target: np.ndarray) -> float:
    return float(np.mean(target)) if len(target) else 0.0


def _category_statistics(codes: np.ndarray, target: np.ndarray, cardinality: int):
    counts = np.bincount(codes, minlength=cardinality).astype(np.float64)
    sums = np.bincount(codes, weights=target, minlength=cardinality)
    squares = np.bincount(codes, weights=target * target, minlength=cardinality)
    return counts, sums, squares


def estimate_smoothing(counts, sums, squares, prior_mean, config=ENCODING_CONFIG) -> float:
    if config.mode == "fixed":
        return float(config.fixed_smoothing)
    present = counts > 0
    if present.sum() < 2:
        return float(config.fixed_smoothing)
    means = np.zeros_like(sums)
    means[present] = sums[present] / counts[present]
    total = counts.sum()
    within = squares.sum() - (sums[present] ** 2 / counts[present]).sum()
    degrees = max(1.0, total - present.sum())
    within_variance = within / degrees
    weights = counts[present] / total
    between_variance = float(np.sum(weights * (means[present] - prior_mean) ** 2))
    harmonic = present.sum() / np.sum(1.0 / counts[present])
    between_variance = between_variance - within_variance / max(1.0, harmonic)
    if between_variance < config.minimum_variance:
        return float(config.maximum_smoothing)
    return float(min(config.maximum_smoothing, within_variance / between_variance))


def _smoothed_values(counts, sums, prior_mean, smoothing, config=ENCODING_CONFIG):
    values = np.full(len(counts), np.nan, dtype=np.float64)
    eligible = counts >= max(1, config.min_count)
    values[eligible] = (sums[eligible] + smoothing * prior_mean) / (counts[eligible] + smoothing)
    if config.unknown_value == "global":
        values[~eligible] = prior_mean
    return values


def _fill_unseen(encoded, prior_mean, config=ENCODING_CONFIG):
    if config.unknown_value == "global":
        return np.where(np.isnan(encoded), prior_mean, encoded)
    return encoded


def encode_column(train_codes, train_target, valid_codes, block_ids, config=ENCODING_CONFIG):
    """Literal transcription of the supplied nsel/encoding.py behavior."""
    cardinality = int(max(train_codes.max(initial=-1), valid_codes.max(initial=-1))) + 1
    prior_mean = _global_mean(train_target)
    order = np.argsort(block_ids, kind="stable")
    blocks = np.array_split(order, max(1, config.n_blocks))

    encoded_train = np.full(len(train_codes), np.nan, dtype=np.float64)
    counts = np.zeros(cardinality, dtype=np.float64)
    sums = np.zeros(cardinality, dtype=np.float64)
    squares = np.zeros(cardinality, dtype=np.float64)

    for position, rows in enumerate(blocks):
        if len(rows) == 0:
            continue
        if position == 0:
            if config.first_block_value == "global":
                encoded_train[rows] = prior_mean
        else:
            smoothing = estimate_smoothing(counts, sums, squares, prior_mean, config)
            values = _smoothed_values(counts, sums, prior_mean, smoothing, config)
            encoded_train[rows] = _fill_unseen(values[train_codes[rows]], prior_mean, config)
        part_counts, part_sums, part_squares = _category_statistics(
            train_codes[rows], train_target[rows], cardinality
        )
        counts += part_counts
        sums += part_sums
        squares += part_squares

    smoothing = estimate_smoothing(counts, sums, squares, prior_mean, config)
    values = _smoothed_values(counts, sums, prior_mean, smoothing, config)
    encoded_valid = _fill_unseen(values[valid_codes], prior_mean, config)
    diagnostics = {
        "prior_mean": prior_mean,
        "smoothing": smoothing,
        "cardinality": cardinality,
        "known_categories": int((counts > 0).sum()),
    }
    return encoded_train, encoded_valid, diagnostics


def build_global_category_table(column: str) -> dict[str, int]:
    """Reproduce nsel/data.py shared string->int dictionary in chronological cutoff order."""
    if column in {USER_COL, INDEX_COL}:
        raise ValueError(f"{column}: structural user id cannot be category-encoded as a feature")
    if column in _CATEGORY_TABLES:
        return _CATEGORY_TABLES[column]
    table: dict[str, int] = {}
    for cutoff in ALL_CUTOFFS:
        part = load_snapshot(cutoff, [column])[[column]]
        series = part[column]
        if isinstance(series, pd.DataFrame):
            raise AssertionError(f"{column}: duplicate projected columns")
        values = series.astype("string").fillna("__nan__").tolist()
        for value in values:
            value = str(value)
            if value not in table:
                table[value] = len(table)
        del part
    _CATEGORY_TABLES[column] = table
    return table


def category_codes(series: pd.Series, column: str) -> np.ndarray:
    table = build_global_category_table(column)
    values = series.astype("string").fillna("__nan__").astype(str)
    # All six cutoffs were included when table was built, so missing mapping is a hard error.
    codes = values.map(table)
    if codes.isna().any():
        raise AssertionError(f"{column}: category mapping miss")
    return codes.to_numpy(dtype=np.int32)


def encode_matrices(X_train, X_valid, target, block_ids, categorical_positions, config=ENCODING_CONFIG):
    diagnostics = {}
    for position in categorical_positions:
        encoded_train, encoded_valid, info = encode_column(
            X_train[:, position].astype(np.int64), target,
            X_valid[:, position].astype(np.int64), block_ids, config,
        )
        X_train[:, position] = encoded_train.astype(np.float32)
        X_valid[:, position] = encoded_valid.astype(np.float32)
        diagnostics[str(position)] = info
    return X_train, X_valid, diagnostics


# Safety sanity: first of 20 ordered chunks is NaN and validation never supplies y.
_tr = np.array([0, 1] * 20, dtype=np.int64)
_va = np.array([0, 1], dtype=np.int64)
_y = np.linspace(-1, 1, len(_tr))
_blocks = np.repeat(np.arange(2), len(_tr) // 2)
_enc_tr, _enc_va, _diag = encode_column(_tr, _y, _va, _blocks)
assert np.isnan(_enc_tr[:2]).all()  # first array_split chunk has two rows here
assert np.isfinite(_enc_va).all()
print("[PASS] exact 20-block EB encoder sanity")


[PASS] exact 20-block EB encoder sanity


## 9. Exact fixed-model adapters: CatBoost / LightGBM / XGBoost


In [94]:
def _round_cap(family: str) -> int:
    cap = int(CV_MAX_ROUNDS[family])
    if SMOKE_USERS is not None:
        cap = min(cap, int(SMOKE_MAX_ROUNDS))
    return cap


def _base_params(
    spec: Mapping[str, Any], *, final_rounds: int | None = None,
    random_seed: int | None = None,
) -> dict[str, Any]:
    family = spec["family"]
    estimator_seed = RANDOM_SEED if random_seed is None else int(random_seed)
    sampled = {k: v for k, v in spec["params"].items() if not k.startswith("_")}
    rounds = int(final_rounds or _round_cap(family))

    if family == "xgboost":
        p = {
            "booster": "gbtree",
            "objective": "reg:squarederror",
            "eval_metric": "rmse",
            "n_estimators": rounds,
            **sampled,
            "grow_policy": "depthwise",
            "tree_method": "hist",
            "device": "cuda" if (USE_GPU and GPU_AVAILABLE) else "cpu",
            "random_state": estimator_seed,
            "n_jobs": -1,
            "verbosity": 0,
        }
        if final_rounds is None:
            p["early_stopping_rounds"] = EARLY_STOPPING_ROUNDS
        return p

    if family == "lightgbm":
        p = {
            "boosting_type": "gbdt",
            "objective": "regression",
            "metric": "rmse",
            "n_estimators": rounds,
            **sampled,
            "random_state": estimator_seed,
            "n_jobs": -1,
            "verbosity": -1,
        }
        if USE_GPU and GPU_AVAILABLE:
            p.update({"device_type": "gpu", "gpu_use_dp": False})
        else:
            p.update({"device_type": "cpu", "force_col_wise": True})
        return p

    p = {
        "loss_function": "RMSE",
        "eval_metric": "RMSE",
        "iterations": rounds,
        **sampled,
        "bootstrap_type": "Bayesian",
        "grow_policy": "SymmetricTree",
        "has_time": True,
        "random_seed": estimator_seed,
        "allow_writing_files": False,
        "task_type": "GPU" if (USE_GPU and GPU_AVAILABLE) else "CPU",
        "verbose": False,
    }
    if p["task_type"] == "GPU":
        p["devices"] = "0"
    return p


def fit_catboost(
    Xtr, ytr, Xva, yva, cat_positions, spec, final_rounds=None,
    sample_weight=None, random_seed=None,
):
    from catboost import CatBoostRegressor, Pool
    p = _base_params(spec, final_rounds=final_rounds, random_seed=random_seed)
    train_pool = Pool(
        Xtr, ytr, cat_features=list(cat_positions), weight=sample_weight
    )
    valid_pool = None if Xva is None else Pool(Xva, yva, cat_features=list(cat_positions))

    def fit_once(params):
        model = CatBoostRegressor(**params)
        if final_rounds is None:
            model.fit(
                train_pool, eval_set=valid_pool,
                early_stopping_rounds=EARLY_STOPPING_ROUNDS,
                use_best_model=True, verbose=False,
            )
        else:
            model.fit(train_pool, verbose=False)
        return model

    try:
        model = fit_once(p)
    except Exception as exc:
        if not (ALLOW_GPU_FALLBACK and p.get("task_type") == "GPU"):
            raise
        print("[GPU fallback][CatBoost]", type(exc).__name__, str(exc)[:250])
        p = dict(p); p["task_type"] = "CPU"; p.pop("devices", None)
        model = fit_once(p)

    pred = None if valid_pool is None else np.asarray(model.predict(valid_pool), dtype=np.float64)
    if final_rounds is None:
        best_raw = model.get_best_iteration()
        best_round = int(best_raw) + 1 if best_raw is not None and int(best_raw) >= 0 else int(p["iterations"])
    else:
        best_round = int(final_rounds)
    return model, pred, best_round


def fit_lightgbm(
    Xtr, ytr, Xva, yva, spec, final_rounds=None,
    sample_weight=None, random_seed=None,
):
    import lightgbm as lgb
    from lightgbm import LGBMRegressor
    p = _base_params(spec, final_rounds=final_rounds, random_seed=random_seed)
    if random_seed is not None:
        p.update({
            "bagging_seed": int(random_seed),
            "feature_fraction_seed": int(random_seed),
            "data_random_seed": int(random_seed),
        })

    def fit_once(params):
        model = LGBMRegressor(**params)
        if final_rounds is None:
            callbacks = [
                lgb.early_stopping(EARLY_STOPPING_ROUNDS, first_metric_only=True, verbose=False),
                lgb.log_evaluation(period=0),
            ]
            model.fit(
                Xtr, ytr, sample_weight=sample_weight,
                eval_set=[(Xva, yva)], eval_metric="rmse", callbacks=callbacks,
            )
        else:
            model.fit(Xtr, ytr, sample_weight=sample_weight)
        return model

    try:
        model = fit_once(p)
    except Exception as exc:
        if not (ALLOW_GPU_FALLBACK and p.get("device_type") == "gpu"):
            raise
        print("[GPU fallback][LightGBM]", type(exc).__name__, str(exc)[:250])
        p = dict(p); p.pop("device_type", None); p.pop("gpu_use_dp", None); p["force_col_wise"] = True
        model = fit_once(p)

    if final_rounds is None:
        best_round = getattr(model, "best_iteration_", None)
        best_round = int(best_round) if best_round is not None and int(best_round) > 0 else int(p["n_estimators"])
        pred = np.asarray(model.predict(Xva, num_iteration=best_round), dtype=np.float64)
    else:
        best_round = int(final_rounds)
        pred = None if Xva is None else np.asarray(model.predict(Xva, num_iteration=best_round), dtype=np.float64)
    return model, pred, best_round


def fit_xgboost(
    Xtr, ytr, Xva, yva, spec, final_rounds=None,
    sample_weight=None, random_seed=None,
):
    from xgboost import XGBRegressor
    p = _base_params(spec, final_rounds=final_rounds, random_seed=random_seed)

    def fit_once(params):
        model = XGBRegressor(**params)
        if final_rounds is None:
            try:
                model.fit(
                    Xtr, ytr, sample_weight=sample_weight,
                    eval_set=[(Xva, yva)], verbose=False,
                )
            except TypeError as exc:
                if "early_stopping_rounds" not in str(exc):
                    raise
                q = dict(params)
                es = int(q.pop("early_stopping_rounds"))
                model = XGBRegressor(**q)
                model.fit(
                    Xtr, ytr, sample_weight=sample_weight,
                    eval_set=[(Xva, yva)], early_stopping_rounds=es, verbose=False,
                )
        else:
            model.fit(Xtr, ytr, sample_weight=sample_weight, verbose=False)
        return model

    try:
        model = fit_once(p)
    except Exception as exc:
        if not (ALLOW_GPU_FALLBACK and p.get("device") == "cuda"):
            raise
        print("[GPU fallback][XGBoost]", type(exc).__name__, str(exc)[:250])
        p = dict(p); p["device"] = "cpu"
        model = fit_once(p)

    pred = None if Xva is None else np.asarray(model.predict(Xva), dtype=np.float64)
    if final_rounds is None:
        best_raw = getattr(model, "best_iteration", None)
        best_round = int(best_raw) + 1 if best_raw is not None else int(p["n_estimators"])
    else:
        best_round = int(final_rounds)
    return model, pred, best_round


def fit_family(
    family, Xtr, ytr, Xva, yva, cat_positions, spec, final_rounds=None,
    sample_weight=None, random_seed=None,
):
    # Optional arguments are used only by the appended diversity experiments.
    # With both left as None, every active baseline specification is equivalent
    # at the parameter/API level; disabled LightGBM specs never reach dispatch.
    if family == "catboost":
        return fit_catboost(
            Xtr, ytr, Xva, yva, cat_positions, spec, final_rounds,
            sample_weight=sample_weight, random_seed=random_seed,
        )
    if family == "lightgbm":
        return fit_lightgbm(
            Xtr, ytr, Xva, yva, spec, final_rounds,
            sample_weight=sample_weight, random_seed=random_seed,
        )
    if family == "xgboost":
        return fit_xgboost(
            Xtr, ytr, Xva, yva, spec, final_rounds,
            sample_weight=sample_weight, random_seed=random_seed,
        )
    raise ValueError(family)


## 10. Сборка exact model matrices из production snapshots


In [95]:
MODEL_FORBIDDEN_FEATURES = {USER_COL, INDEX_COL, TARGET_COL, SNAPSHOT_COL}


def model_feature_order(spec: Mapping[str, Any]):
    # jobs.py uses a set; this notebook additionally enforces the production
    # rule that identifiers/targets can never become estimator features.
    requested = sorted(set(spec["features"]) - MODEL_FORBIDDEN_FEATURES)
    numeric = [c for c in requested if c not in CATEGORICAL_FEATURES]
    categorical = [c for c in requested if c in CATEGORICAL_FEATURES]
    return numeric, categorical, numeric + categorical


def _load_one_model_snapshot(cutoff: str, numeric: Sequence[str], categorical: Sequence[str]):
    cols = [TARGET_COL, ANCHOR_COL, *numeric, *categorical]
    frame = load_snapshot(cutoff, cols)
    n = len(frame)
    X = np.empty((n, len(numeric) + len(categorical)), dtype=np.float32)
    col = 0
    for name in numeric:
        values = pd.to_numeric(frame[name], errors="coerce").to_numpy(dtype=np.float32)
        X[:, col] = np.where(np.isfinite(values), values, np.nan)
        col += 1
    for name in categorical:
        X[:, col] = category_codes(frame[name], name).astype(np.float32)
        col += 1
    return {
        "X": X,
        "user_id": frame[USER_COL].to_numpy(copy=False),
        "y_raw": pd.to_numeric(frame[TARGET_COL], errors="coerce").to_numpy(dtype=np.float64),
        "anchor": pd.to_numeric(frame[ANCHOR_COL], errors="raise").to_numpy(dtype=np.float64),
    }


def materialize_model_pair(train_cutoffs: Sequence[str], valid_cutoff: str, spec: Mapping[str, Any]):
    numeric, categorical, columns = model_feature_order(spec)
    train_parts = [_load_one_model_snapshot(c, numeric, categorical) for c in train_cutoffs]
    valid = _load_one_model_snapshot(valid_cutoff, numeric, categorical)

    X_train = np.concatenate([x["X"] for x in train_parts], axis=0)
    y_train_raw = np.concatenate([x["y_raw"] for x in train_parts])
    anchor_train = np.concatenate([x["anchor"] for x in train_parts])
    block_ids = np.concatenate([
        np.full(len(x["y_raw"]), i, dtype=np.int32) for i, x in enumerate(train_parts)
    ])
    train_user_ids = np.concatenate([x["user_id"] for x in train_parts])

    y_train = build_model_target(y_train_raw, anchor_train, spec["target_mode"])
    y_valid = build_model_target(valid["y_raw"], valid["anchor"], spec["target_mode"])
    categorical_positions = list(range(len(numeric), len(columns)))

    encoding_diag = {}
    if spec["family"] != "catboost" and categorical_positions:
        X_train, valid["X"], encoding_diag = encode_matrices(
            X_train, valid["X"], y_train, block_ids, categorical_positions, ENCODING_CONFIG
        )
        categorical_positions = []

    return {
        "X_train": X_train,
        "y_train": y_train,
        "y_train_raw": y_train_raw,
        "anchor_train": anchor_train,
        "train_user_ids": train_user_ids,
        "block_ids": block_ids,
        "X_valid": valid["X"],
        "y_valid": y_valid,
        "y_valid_raw": valid["y_raw"],
        "anchor_valid": valid["anchor"],
        "valid_user_ids": valid["user_id"],
        "columns": columns,
        "categorical_positions": categorical_positions,
        "categorical_names": categorical,
        "encoding": encoding_diag,
    }


def load_analysis_snapshot(cutoff: str) -> pd.DataFrame:
    raw_cols = unique_keep_order([
        TARGET_COL, ANCHOR_COL,
        *[c for c in ANALYSIS_SOURCE_COLUMNS.values() if c is not None],
    ])
    frame = load_snapshot(cutoff, raw_cols)
    out = frame[[USER_COL, SNAPSHOT_COL, TARGET_COL, ANCHOR_COL]].copy()
    for alias, source in ANALYSIS_SOURCE_COLUMNS.items():
        if source is None:
            out[alias] = np.nan
        else:
            out[alias] = pd.to_numeric(frame[source], errors="coerce")
    return out


def weighted_fold_metric(fold_metrics: pd.DataFrame, metric_col: str = "rmsle") -> float:
    return float(np.sum(fold_metrics["weight"].to_numpy() * fold_metrics[metric_col].to_numpy()))


def recommended_iterations_from_folds(fold_df: pd.DataFrame) -> int:
    # Exact supplied Optuna runner rule: weighted mean of best_iteration, then round.
    return max(1, int(round(float(np.sum(fold_df["weight"] * fold_df["best_round"])))))


## 11. Fixed-parameter temporal CV — 6 моделей


In [96]:
OOF_BY_MODEL = {}
CV_ROWS = []
ITERATION_ROWS = []
SMOOTHING_ROWS = []

if RUN_CV:
    for model_name in MODEL_NAMES:
        spec = MODEL_SPECS[model_name]
        print("\n" + "=" * 90)
        print("CV:", model_name)
        print("=" * 90)
        model_oof_parts = []

        for fold in CV_FOLDS:
            started = time.perf_counter()
            matrices = materialize_model_pair(fold["train_cutoffs"], fold["valid_cutoff"], spec)

            model, pred_model, best_round = fit_family(
                spec["family"],
                matrices["X_train"], matrices["y_train"],
                matrices["X_valid"], matrices["y_valid"],
                matrices["categorical_positions"], spec,
                final_rounds=None,
            )

            pred_raw = inverse_model_target(
                pred_model, matrices["anchor_valid"], spec["target_mode"]
            )
            truth = matrices["y_valid_raw"]
            fold_rmsle = rmsle(truth, pred_raw)
            zero = truth <= 0
            zero_score = rmsle(truth[zero], pred_raw[zero]) if zero.any() else np.nan
            pos_score = rmsle(truth[~zero], pred_raw[~zero]) if (~zero).any() else np.nan
            elapsed = time.perf_counter() - started

            CV_ROWS.append({
                "model": model_name,
                "family": spec["family"],
                "target_mode": spec["target_mode"],
                "fold": fold["name"],
                "weight": fold["weight"],
                "rmsle": fold_rmsle,
                "rmsle_zero": zero_score,
                "rmsle_positive": pos_score,
                "best_round": best_round,
                "seconds": elapsed,
            })
            ITERATION_ROWS.append({
                "model": model_name,
                "fold": fold["name"],
                "weight": fold["weight"],
                "best_round": best_round,
                "hit_cap": best_round >= int(_round_cap(spec["family"]) * 0.98),
            })
            for pos, info in matrices["encoding"].items():
                SMOOTHING_ROWS.append({
                    "model": model_name,
                    "fold": fold["name"],
                    "position": int(pos),
                    "feature": matrices["columns"][int(pos)],
                    "smoothing": info["smoothing"],
                    "cardinality": info["cardinality"],
                    "prior": info["prior_mean"],
                })

            oof = load_analysis_snapshot(fold["valid_cutoff"])
            if not np.array_equal(oof[USER_COL].to_numpy(), matrices["valid_user_ids"]):
                raise AssertionError(f"{model_name}/{fold['name']}: OOF user alignment mismatch")
            oof["fold"] = fold["name"]
            oof[model_name] = pred_raw
            model_oof_parts.append(oof)

            print(
                f"{fold['name']:9s} rmsle={fold_rmsle:.6f} "
                f"best_round={best_round:6d} time={elapsed/60:.1f}m"
            )
            del matrices, model, pred_model, pred_raw, oof
            gc.collect()

        OOF_BY_MODEL[model_name] = pd.concat(model_oof_parts, ignore_index=True)

CV_METRICS = pd.DataFrame(CV_ROWS)
ITERATION_TABLE = pd.DataFrame(ITERATION_ROWS)
SMOOTHING_TABLE = pd.DataFrame(SMOOTHING_ROWS)

if RUN_CV:
    CV_METRICS.to_csv(ANALYSIS_DIR / "base_fold_metrics.csv", index=False)
    ITERATION_TABLE.to_csv(ANALYSIS_DIR / "base_best_iterations.csv", index=False)
    if not SMOOTHING_TABLE.empty:
        SMOOTHING_TABLE.to_csv(ANALYSIS_DIR / "target_encoding_smoothing.csv", index=False)

    _summary_rows = []
    for model_name in MODEL_NAMES:
        spec = MODEL_SPECS[model_name]
        g = CV_METRICS.loc[CV_METRICS["model"].eq(model_name)].copy()
        _summary_rows.append({
            "model": model_name,
            "family": spec["family"],
            "target_mode": spec["target_mode"],
            "weighted_rmsle": weighted_fold_metric(g, "rmsle"),
            "weighted_zero": float(np.nansum(g["weight"] * g["rmsle_zero"])),
            "weighted_positive": float(np.nansum(g["weight"] * g["rmsle_positive"])),
            "recommended_iterations": recommended_iterations_from_folds(g),
            "hit_round_cap": bool(g["best_round"].ge(int(_round_cap(spec["family"]) * 0.98)).any()),
        })
    BASE_SUMMARY = pd.DataFrame(_summary_rows).sort_values("weighted_rmsle").reset_index(drop=True)
    display(BASE_SUMMARY)
    BASE_SUMMARY.to_csv(ANALYSIS_DIR / "base_model_ranking.csv", index=False)
    if BASE_SUMMARY["hit_round_cap"].any():
        print("WARNING: at least one model hit the Optuna round cap; its final iteration estimate is censored.")


CV: xgboost_log_level
nov_2025  rmsle=1.733297 best_round=  1422 time=0.3m
dec_2025  rmsle=1.736723 best_round=  1789 time=0.4m
jan_2026  rmsle=1.677414 best_round=  3231 time=0.7m

CV: catboost_log_level
nov_2025  rmsle=1.734646 best_round=  7748 time=0.9m
dec_2025  rmsle=1.738614 best_round=  7444 time=1.1m
jan_2026  rmsle=1.678974 best_round=  8542 time=1.5m

CV: catboost_log_diff
nov_2025  rmsle=1.733975 best_round=  1741 time=0.2m
dec_2025  rmsle=1.741185 best_round=  1919 time=0.3m
jan_2026  rmsle=1.680042 best_round=  1518 time=0.3m

CV: xgboost_log_diff
nov_2025  rmsle=1.734767 best_round=  3667 time=0.3m
dec_2025  rmsle=1.738642 best_round=  5208 time=0.6m
jan_2026  rmsle=1.678587 best_round=  7449 time=0.9m


,model,family,target_mode,weighted_rmsle,weighted_zero,weighted_positive,recommended_iterations,hit_round_cap
0,xgboost_log_level,xgboost,log_level,1.712143,1.694137,1.723263,2274,False
1,xgboost_log_diff,xgboost,log_diff,1.713651,1.700899,1.721047,5719,False
2,catboost_log_level,catboost,log_level,1.713766,1.700200,1.721355,7959,False
3,catboost_log_diff,catboost,log_diff,1.714925,1.725209,1.703764,1714,False


## 12. Объединение OOF и консистентность


In [97]:
def merge_oof_models(oof_by_model: Mapping[str, pd.DataFrame]) -> pd.DataFrame:
    keys = ["fold", SNAPSHOT_COL, USER_COL]
    base_cols = [
        *keys, TARGET_COL, ANCHOR_COL,
        "__analysis_days_since_last_purchase",
        "__analysis_active_days_30d",
        "__analysis_purchase_days_30d",
        "__analysis_searches_30d",
    ]
    merged = None
    for name in MODEL_NAMES:
        frame = oof_by_model[name]
        cur = frame[base_cols + [name]].copy()
        if merged is None:
            merged = cur
        else:
            merged = merged.merge(cur[keys + [name]], on=keys, how="inner", validate="one_to_one")
    if merged is None:
        raise RuntimeError("OOF is empty")
    expected = len(user_universe) * len(CV_FOLDS)
    if len(merged) != expected:
        raise AssertionError(f"OOF rows={len(merged)} expected={expected}")
    if merged[MODEL_NAMES].isna().any().any():
        raise AssertionError("NaN in base OOF predictions")
    return merged


if RUN_CV:
    OOF = merge_oof_models(OOF_BY_MODEL)
    OOF.to_parquet(ANALYSIS_DIR / "oof_all_models.parquet", index=False)
    print("OOF:", OOF.shape)
    OOF_BY_MODEL.clear()
    gc.collect()


OOF: (750000, 13)


## 13. Full-train fit → active CatBoost/XGBoost base submissions


In [98]:
FINAL_PREDICTIONS = None
FINAL_MODEL_PATHS = {}
FINAL_ENCODING_DIAGNOSTICS = {}
FINAL_ITERATIONS = {}


def save_native_model(model, family: str, path_base: Path):
    if family == "catboost":
        path = path_base.with_suffix(".cbm")
        model.save_model(str(path))
    elif family == "lightgbm":
        path = path_base.with_suffix(".txt")
        model.booster_.save_model(str(path))
    elif family == "xgboost":
        path = path_base.with_suffix(".json")
        model.save_model(str(path))
    else:
        return None
    return path


def build_base_submission(user_ids, prediction) -> pd.DataFrame:
    pred_frame = pd.DataFrame({
        USER_COL: np.asarray(user_ids),
        SUBMISSION_TARGET_COL: np.clip(np.asarray(prediction, dtype=np.float64), 0.0, None),
    })
    if pred_frame[USER_COL].duplicated().any():
        raise AssertionError("prediction user ids contain duplicates")
    sub = sample_submit.drop(columns=[SUBMISSION_TARGET_COL], errors="ignore").merge(
        pred_frame, on=USER_COL, how="left", validate="one_to_one", sort=False
    )
    if len(sub) != len(sample_submit) or sub[SUBMISSION_TARGET_COL].isna().any():
        raise AssertionError("submission alignment failure")
    return sub


if RUN_FINAL_FIT:
    test_base = load_analysis_snapshot(PREDICTION_CUTOFF)
    FINAL_PREDICTIONS = test_base.copy()

    for model_name in MODEL_NAMES:
        spec = MODEL_SPECS[model_name]
        print("\n" + "=" * 90)
        print("FINAL FIT:", model_name)
        print("=" * 90)

        matrices = materialize_model_pair(FINAL_TRAIN_CUTOFFS, PREDICTION_CUTOFF, spec)
        fixed = FIXED_FINAL_ITERATIONS.get(model_name)
        if fixed is None:
            fold_rows = CV_METRICS.loc[CV_METRICS["model"].eq(model_name)]
            if fold_rows.empty:
                raise RuntimeError(
                    f"No CV iteration diagnostics for {model_name}; run CV or set FIXED_FINAL_ITERATIONS"
                )
            final_rounds = recommended_iterations_from_folds(fold_rows)
        else:
            final_rounds = int(fixed)

        model, _, used_rounds = fit_family(
            spec["family"],
            matrices["X_train"], matrices["y_train"],
            None, None,
            matrices["categorical_positions"], spec,
            final_rounds=final_rounds,
        )

        if spec["family"] == "catboost":
            from catboost import Pool
            pred_model = model.predict(Pool(matrices["X_valid"], cat_features=matrices["categorical_positions"]))
        elif spec["family"] == "lightgbm":
            pred_model = model.predict(matrices["X_valid"], num_iteration=used_rounds)
        else:
            pred_model = model.predict(matrices["X_valid"])

        pred_raw = inverse_model_target(pred_model, matrices["anchor_valid"], spec["target_mode"])
        if not np.isfinite(pred_raw).all() or np.any(pred_raw < 0):
            raise FloatingPointError(f"{model_name}: invalid final prediction")

        pred_frame = pd.DataFrame({USER_COL: matrices["valid_user_ids"], model_name: pred_raw})
        FINAL_PREDICTIONS = FINAL_PREDICTIONS.merge(
            pred_frame, on=USER_COL, how="left", validate="one_to_one", sort=False
        )
        FINAL_ENCODING_DIAGNOSTICS[model_name] = matrices["encoding"]
        FINAL_ITERATIONS[model_name] = int(used_rounds)

        if SAVE_MODELS:
            saved = save_native_model(model, spec["family"], MODEL_DIR / model_name)
            FINAL_MODEL_PATHS[model_name] = None if saved is None else str(saved)

        submission = build_base_submission(matrices["valid_user_ids"], pred_raw)
        submission.to_csv(BASE_SUBMISSION_DIR / f"submission_{model_name}.csv", index=False)

        print({
            "model": model_name,
            "final_rounds": used_rounds,
            "pred_mean": float(np.mean(pred_raw)),
            "pred_p99": float(np.quantile(pred_raw, 0.99)),
        })
        del matrices, model, pred_model, pred_raw, pred_frame
        gc.collect()

    FINAL_PREDICTIONS.to_parquet(ANALYSIS_DIR / "final_base_predictions.parquet", index=False)



FINAL FIT: xgboost_log_level
{'model': 'xgboost_log_level', 'final_rounds': 2274, 'pred_mean': 39.29521664783644, 'pred_p99': 464.95162364154885}

FINAL FIT: catboost_log_level
{'model': 'catboost_log_level', 'final_rounds': 7959, 'pred_mean': 41.07169050218371, 'pred_p99': 482.489986343525}

FINAL FIT: catboost_log_diff
{'model': 'catboost_log_diff', 'final_rounds': 1714, 'pred_mean': 35.19878571069391, 'pred_p99': 417.4242382778435}

FINAL FIT: xgboost_log_diff
{'model': 'xgboost_log_diff', 'final_rounds': 5719, 'pred_mean': 38.96664047924997, 'pred_p99': 462.0197054931646}


## 14. Base model ranking


In [99]:
def fold_weight_map():
    return {f["name"]: float(f["weight"]) for f in CV_FOLDS}

FOLD_WEIGHT_MAP = fold_weight_map()


def weighted_oof_rmsle(frame: pd.DataFrame, pred) -> float:
    pred = np.asarray(pred, dtype=np.float64)
    scores = []
    for fold in CV_FOLDS:
        mask = frame["fold"].eq(fold["name"]).to_numpy()
        scores.append(fold["weight"] * rmsle(frame.loc[mask, TARGET_COL], pred[mask]))
    return float(sum(scores))


if RUN_CV:
    BASE_RANKING = []
    for name in MODEL_NAMES:
        BASE_RANKING.append({
            "model": name,
            "weighted_rmsle": weighted_oof_rmsle(OOF, OOF[name].to_numpy()),
            "family": MODEL_SPECS[name]["family"],
            "target_mode": MODEL_SPECS[name]["target_mode"],
            "maturity_manual": MODEL_SPECS[name]["maturity"],
            "maturity_source": MATURITY_PROVENANCE,
            "optuna_metric_snapshot": MODEL_SPECS[name]["optuna_metric"],
        })
    BASE_RANKING = pd.DataFrame(BASE_RANKING).sort_values("weighted_rmsle").reset_index(drop=True)
    display(BASE_RANKING)
    BASE_RANKING.to_csv(ANALYSIS_DIR / "base_ranking_recomputed.csv", index=False)


,model,weighted_rmsle,family,target_mode,maturity_manual,maturity_source,optuna_metric_snapshot
0,xgboost_log_level,1.712143,xgboost,log_level,trusted,manual_not_present_in_source_artifacts,1.711975
1,xgboost_log_diff,1.713651,xgboost,log_diff,trusted,manual_not_present_in_source_artifacts,1.713494
2,catboost_log_level,1.713766,catboost,log_level,trusted,manual_not_present_in_source_artifacts,1.713613
3,catboost_log_diff,1.714925,catboost,log_diff,trusted,manual_not_present_in_source_artifacts,1.713830


## 15. Полный Ozon error analysis


In [100]:
def add_analysis_segments(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    out["target_segment"] = np.where(out[TARGET_COL] <= 0, "zero", "positive")

    a = out[ANCHOR_COL].astype(float)
    out["anchor_segment"] = pd.cut(
        a, bins=[-np.inf, 0, 100, 500, 2000, 10000, np.inf],
        labels=["zero", "0_100", "100_500", "500_2k", "2k_10k", "10k_plus"],
    ).astype(str)

    r = out["__analysis_days_since_last_purchase"].astype(float)
    out["recency_segment"] = pd.cut(
        r, bins=[-np.inf, 7, 30, 90, 365, np.inf],
        labels=["0_7", "8_30", "31_90", "91_365", "365_plus"],
    ).astype("string").fillna("unknown_or_never").astype(str)

    act = out["__analysis_active_days_30d"].astype(float)
    out["activity_segment"] = pd.cut(
        act, bins=[-np.inf, 0, 2, 7, 15, np.inf],
        labels=["0", "1_2", "3_7", "8_15", "16_plus"],
    ).astype("string").fillna("unknown").astype(str)

    s = out["__analysis_searches_30d"].astype(float)
    out["search_segment"] = pd.cut(
        s, bins=[-np.inf, 0, 5, 20, 100, np.inf],
        labels=["0", "1_5", "6_20", "21_100", "100_plus"],
    ).astype("string").fillna("unknown").astype(str)
    return out


def segment_error_table(frame: pd.DataFrame, segment_col: str) -> pd.DataFrame:
    rows = []
    for segment, part in frame.groupby(segment_col, dropna=False, observed=False):
        row = {
            "segment_type": segment_col,
            "segment": str(segment),
            "rows": len(part),
            "target_mean": float(part[TARGET_COL].mean()),
            "anchor_mean": float(part[ANCHOR_COL].mean()),
        }
        for name in MODEL_NAMES:
            row[f"rmsle__{name}"] = rmsle(part[TARGET_COL], part[name])
        rows.append(row)
    return pd.DataFrame(rows)


def systematic_bias_table(frame: pd.DataFrame) -> pd.DataFrame:
    truth_log = np.log1p(frame[TARGET_COL].to_numpy(dtype=np.float64))
    rows = []
    for name in MODEL_NAMES:
        pred_log = np.log1p(frame[name].to_numpy(dtype=np.float64))
        resid = pred_log - truth_log
        slope, intercept = np.polyfit(pred_log, truth_log, 1)
        rows.append({
            "model": name,
            "mean_log_bias": float(resid.mean()),
            "median_log_bias": float(np.median(resid)),
            "mean_abs_log_error": float(np.mean(np.abs(resid))),
            "calibration_slope_truth_on_pred": float(slope),
            "calibration_intercept": float(intercept),
        })
    return pd.DataFrame(rows)


def pairwise_win_rate(frame: pd.DataFrame) -> pd.DataFrame:
    truth_log = np.log1p(frame[TARGET_COL].to_numpy(dtype=np.float64))
    errors = {
        n: np.abs(np.log1p(frame[n].to_numpy(dtype=np.float64)) - truth_log)
        for n in MODEL_NAMES
    }
    mat = pd.DataFrame(index=MODEL_NAMES, columns=MODEL_NAMES, dtype=float)
    for a in MODEL_NAMES:
        for b in MODEL_NAMES:
            mat.loc[a, b] = 0.5 if a == b else float(np.mean(errors[a] < errors[b]))
    return mat


if RUN_CV and RUN_ERROR_ANALYSIS:
    OOF_ANALYSIS = add_analysis_segments(OOF)
    for col in ["target_segment", "anchor_segment", "recency_segment", "activity_segment", "search_segment", "fold"]:
        segment_error_table(OOF_ANALYSIS, col).to_csv(ANALYSIS_DIR / f"errors_by_{col}.csv", index=False)

    BIAS_TABLE = systematic_bias_table(OOF_ANALYSIS)
    BIAS_TABLE.to_csv(ANALYSIS_DIR / "systematic_log_bias.csv", index=False)

    PRED_CORR = OOF_ANALYSIS[MODEL_NAMES].corr()
    truth_log = np.log1p(OOF_ANALYSIS[TARGET_COL].to_numpy(dtype=np.float64))
    residual_df = pd.DataFrame({
        n: np.log1p(OOF_ANALYSIS[n].to_numpy(dtype=np.float64)) - truth_log
        for n in MODEL_NAMES
    })
    RESID_CORR = residual_df.corr()
    WIN_RATE = pairwise_win_rate(OOF_ANALYSIS)
    PRED_CORR.to_csv(ANALYSIS_DIR / "prediction_correlations.csv")
    RESID_CORR.to_csv(ANALYSIS_DIR / "residual_correlations.csv")
    WIN_RATE.to_csv(ANALYSIS_DIR / "pairwise_win_rate.csv")

    mode_rows = []
    for mode in ["log_level", "log_diff"]:
        names = [n for n in MODEL_NAMES if MODEL_SPECS[n]["target_mode"] == mode]
        pred = np.expm1(np.mean(np.log1p(OOF_ANALYSIS[names].to_numpy()), axis=1))
        mode_rows.append({"target_mode": mode, "logmean_rmsle": weighted_oof_rmsle(OOF_ANALYSIS, pred)})
    MODE_COMPARISON = pd.DataFrame(mode_rows)
    MODE_COMPARISON.to_csv(ANALYSIS_DIR / "log_level_vs_log_diff.csv", index=False)

    log_preds = np.log1p(OOF_ANALYSIS[MODEL_NAMES].to_numpy(dtype=np.float64))
    HIGH_DISAGREEMENT = OOF_ANALYSIS[[
        "fold", USER_COL, TARGET_COL, ANCHOR_COL,
        "__analysis_days_since_last_purchase",
        "__analysis_active_days_30d",
        "__analysis_searches_30d",
        *MODEL_NAMES,
    ]].copy()
    HIGH_DISAGREEMENT["log_prediction_std"] = log_preds.std(axis=1)
    HIGH_DISAGREEMENT = HIGH_DISAGREEMENT.nlargest(min(20_000, len(HIGH_DISAGREEMENT)), "log_prediction_std")
    HIGH_DISAGREEMENT.to_parquet(ANALYSIS_DIR / "high_disagreement_users.parquet", index=False)

    print("Systematic bias"); display(BIAS_TABLE)
    print("Prediction correlations"); display(PRED_CORR.round(4))
    print("Residual correlations"); display(RESID_CORR.round(4))
    print("Pairwise win-rate"); display(WIN_RATE.round(3))
    print("log_level vs log_diff"); display(MODE_COMPARISON)


Systematic bias


,model,mean_log_bias,median_log_bias,mean_abs_log_error,calibration_slope_truth_on_pred,calibration_intercept
0,xgboost_log_level,0.037767,0.328771,1.357425,0.994528,-0.024653
1,catboost_log_level,0.040942,0.343229,1.361737,0.999926,-0.040764
2,catboost_log_diff,0.075542,0.355177,1.359342,0.985667,-0.040649
3,xgboost_log_diff,0.046472,0.329836,1.358585,0.991958,-0.027129


Prediction correlations


,xgboost_log_level,catboost_log_level,catboost_log_diff,xgboost_log_diff
xgboost_log_level,1.0000,0.9754,0.9704,0.9544
catboost_log_level,0.9754,1.0000,0.9482,0.9263
catboost_log_diff,0.9704,0.9482,1.0000,0.9650
xgboost_log_diff,0.9544,0.9263,0.9650,1.0000


Residual correlations


,xgboost_log_level,catboost_log_level,catboost_log_diff,xgboost_log_diff
xgboost_log_level,1.0000,0.9976,0.9964,0.9965
catboost_log_level,0.9976,1.0000,0.9966,0.9962
catboost_log_diff,0.9964,0.9966,1.0000,0.9957
xgboost_log_diff,0.9965,0.9962,0.9957,1.0000


Pairwise win-rate


,xgboost_log_level,catboost_log_level,catboost_log_diff,xgboost_log_diff
xgboost_log_level,0.500,0.510,0.508,0.506
catboost_log_level,0.490,0.500,0.500,0.482
catboost_log_diff,0.492,0.500,0.500,0.488
xgboost_log_diff,0.494,0.518,0.512,0.500


log_level vs log_diff


,target_mode,logmean_rmsle
0,log_level,1.711901
1,log_diff,1.712346


## 16. Blend Laboratory — базовые функции


In [101]:
from scipy.optimize import minimize

@dataclass
class BlendCandidate:
    name: str
    oof_pred: np.ndarray
    test_pred: np.ndarray | None
    kind: str
    complexity: int
    selection_safe: bool
    metadata: dict


BLENDS = {}


def raw_mean(matrix: np.ndarray, weights=None):
    matrix = np.asarray(matrix, dtype=np.float64)
    if weights is None:
        return matrix.mean(axis=1)
    w = np.asarray(weights, dtype=np.float64)
    w = w / w.sum()
    return matrix @ w


def log_mean(matrix: np.ndarray, weights=None):
    matrix = np.asarray(matrix, dtype=np.float64)
    logs = np.log1p(np.clip(matrix, 0.0, None))
    if weights is None:
        z = logs.mean(axis=1)
    else:
        w = np.asarray(weights, dtype=np.float64)
        w = w / w.sum()
        z = logs @ w
    return np.expm1(z)


def add_blend(name, oof_pred, test_pred, kind, complexity, selection_safe, metadata=None):
    oof_pred = np.clip(np.asarray(oof_pred, dtype=np.float64), 0.0, None)
    if test_pred is not None:
        test_pred = np.clip(np.asarray(test_pred, dtype=np.float64), 0.0, None)
    BLENDS[name] = BlendCandidate(
        name=name,
        oof_pred=oof_pred,
        test_pred=test_pred,
        kind=kind,
        complexity=int(complexity),
        selection_safe=bool(selection_safe),
        metadata=dict(metadata or {}),
    )


def test_matrix(names):
    if FINAL_PREDICTIONS is None:
        return None
    return FINAL_PREDICTIONS[names].to_numpy(dtype=np.float64)


def oof_matrix(names):
    return OOF[names].to_numpy(dtype=np.float64)


## 17. Simple / maturity-aware / subset blends


In [102]:
if RUN_CV and RUN_BLEND_LAB:
    # Predefined all-6 arithmetic and log means.
    add_blend(
        "all_active_arithmetic",
        raw_mean(oof_matrix(MODEL_NAMES)),
        None if FINAL_PREDICTIONS is None else raw_mean(test_matrix(MODEL_NAMES)),
        "arithmetic_mean", 1, True,
        {"models": MODEL_NAMES},
    )
    add_blend(
        "all_active_logmean",
        log_mean(oof_matrix(MODEL_NAMES)),
        None if FINAL_PREDICTIONS is None else log_mean(test_matrix(MODEL_NAMES)),
        "log_mean", 1, True,
        {"models": MODEL_NAMES},
    )

    fixed_groups = {
        "trusted_logmean": TRUSTED_MODELS,
        "log_level_logmean": [n for n in MODEL_NAMES if MODEL_SPECS[n]["target_mode"] == "log_level"],
        "log_diff_logmean": [n for n in MODEL_NAMES if MODEL_SPECS[n]["target_mode"] == "log_diff"],
    }
    # Disabled families may leave no provisional model. Never call min([]).
    if PROVISIONAL_MODELS:
        fixed_groups["provisional_logmean"] = PROVISIONAL_MODELS
        best_provisional_prior = min(
            PROVISIONAL_MODELS, key=lambda n: MODEL_SPECS[n]["optuna_metric"]
        )
        fixed_groups["trusted_plus_best_provisional_logmean"] = [
            *TRUSTED_MODELS, best_provisional_prior,
        ]

    for blend_name, names in fixed_groups.items():
        add_blend(
            blend_name,
            log_mean(oof_matrix(names)),
            None if FINAL_PREDICTIONS is None else log_mean(test_matrix(names)),
            "fixed_group_logmean", 1, True,
            {"models": names},
        )

    # Validation-score weighted (descriptive; weights are fitted to same OOF ranking).
    current_scores = np.array([
        float(BASE_RANKING.set_index("model").loc[n, "weighted_rmsle"])
        for n in MODEL_NAMES
    ])
    inv_w = 1.0 / np.maximum(current_scores, 1e-12)
    inv_w /= inv_w.sum()
    add_blend(
        "validation_inverse_score_logmean",
        log_mean(oof_matrix(MODEL_NAMES), inv_w),
        None if FINAL_PREDICTIONS is None else log_mean(test_matrix(MODEL_NAMES), inv_w),
        "validation_weighted", 2, False,
        {"weights": dict(zip(MODEL_NAMES, inv_w.tolist()))},
    )

    # Optuna-score softmax: based on supplied Optuna snapshot.
    prior_scores = np.array([MODEL_SPECS[n]["optuna_metric"] for n in MODEL_NAMES], dtype=float)
    temperature = 0.002
    logits = -(prior_scores - prior_scores.min()) / temperature
    opt_w = np.exp(logits - logits.max())
    opt_w /= opt_w.sum()
    add_blend(
        "optuna_softmax_logmean",
        log_mean(oof_matrix(MODEL_NAMES), opt_w),
        None if FINAL_PREDICTIONS is None else log_mean(test_matrix(MODEL_NAMES), opt_w),
        "optuna_weighted", 2, True,
        {"temperature": temperature, "weights": dict(zip(MODEL_NAMES, opt_w.tolist()))},
    )

    # Top-N and best-per-family are intentionally marked exploratory because membership
    # is selected on the same OOF being reported.
    ordered = BASE_RANKING["model"].tolist()
    for k in range(2, min(5, len(ordered)) + 1):
        names = ordered[:k]
        add_blend(
            f"top{k}_logmean",
            log_mean(oof_matrix(names)),
            None if FINAL_PREDICTIONS is None else log_mean(test_matrix(names)),
            "top_n", k, False, {"models": names},
        )

    best_family = []
    active_families = sorted({MODEL_SPECS[n]["family"] for n in MODEL_NAMES})
    for family in active_families:
        family_names = [n for n in ordered if MODEL_SPECS[n]["family"] == family]
        if family_names:
            best_family.append(family_names[0])
    add_blend(
        "best_per_family_logmean",
        log_mean(oof_matrix(best_family)),
        None if FINAL_PREDICTIONS is None else log_mean(test_matrix(best_family)),
        "best_per_family", 3, False, {"models": best_family},
    )

    # Exhaustive subset lab: diagnostic/optimistic model-selection table.
    subset_rows = []
    for r in range(2, len(MODEL_NAMES) + 1):
        for names_tuple in itertools.combinations(MODEL_NAMES, r):
            names = list(names_tuple)
            for space in ["raw", "log"]:
                pred = raw_mean(oof_matrix(names)) if space == "raw" else log_mean(oof_matrix(names))
                score = weighted_oof_rmsle(OOF, pred)
                subset_rows.append({
                    "models": "|".join(names),
                    "n_models": len(names),
                    "space": space,
                    "weighted_rmsle": score,
                })
    SUBSET_TABLE = pd.DataFrame(subset_rows).sort_values("weighted_rmsle")
    SUBSET_TABLE.to_csv(ANALYSIS_DIR / "blend_subset_search_exploratory.csv", index=False)
    print("Best exploratory subsets")
    display(SUBSET_TABLE.head(20))


Best exploratory subsets


,models,n_models,space,weighted_rmsle
21,xgboost_log_level|catboost_log_level|catboost_...,4,log,1.711274
17,xgboost_log_level|catboost_log_diff|xgboost_lo...,3,log,1.711286
15,xgboost_log_level|catboost_log_level|xgboost_l...,3,log,1.711325
5,xgboost_log_level|xgboost_log_diff,2,log,1.711385
20,xgboost_log_level|catboost_log_level|catboost_...,4,raw,1.711474
14,xgboost_log_level|catboost_log_level|xgboost_l...,3,raw,1.711478
16,xgboost_log_level|catboost_log_diff|xgboost_lo...,3,raw,1.711483
4,xgboost_log_level|xgboost_log_diff,2,raw,1.711503
13,xgboost_log_level|catboost_log_level|catboost_...,3,log,1.711696
19,catboost_log_level|catboost_log_diff|xgboost_l...,3,log,1.711834


## 18. Cross-fitted optimized weights: raw-space и log-space


In [103]:
def score_on_rows(frame: pd.DataFrame, pred: np.ndarray) -> float:
    # Renormalize temporal weights over folds actually present in frame.
    present = list(pd.unique(frame["fold"]))
    weights = {f: FOLD_WEIGHT_MAP[f] for f in present}
    denom = sum(weights.values())
    total = 0.0
    for f in present:
        mask = frame["fold"].eq(f).to_numpy()
        total += (weights[f] / denom) * rmsle(frame.loc[mask, TARGET_COL], pred[mask])
    return float(total)


def fit_blend_weights(frame: pd.DataFrame, names: Sequence[str], space: str):
    P = frame[list(names)].to_numpy(dtype=np.float64)
    y = frame[TARGET_COL].to_numpy(dtype=np.float64)
    n = len(names)

    def make_pred(w):
        return raw_mean(P, w) if space == "raw" else log_mean(P, w)

    def objective(w):
        pred = make_pred(w)
        base = score_on_rows(frame, pred)
        ridge = BLEND_RIDGE * float(np.sum((w - 1.0 / n) ** 2))
        return base + ridge

    starts = [
        np.full(n, 1.0 / n),
    ]
    # score-prior start
    indiv = np.array([score_on_rows(frame, P[:, j]) for j in range(n)])
    sw = np.exp(-(indiv - indiv.min()) / 0.003)
    sw /= sw.sum()
    starts.append(sw)

    bounds = [(0.0, MAX_BLEND_WEIGHT) for _ in range(n)]
    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]
    best = None
    for start in starts:
        res = minimize(
            objective, start, method="SLSQP",
            bounds=bounds, constraints=constraints,
            options={"maxiter": 500, "ftol": 1e-10, "disp": False},
        )
        if res.success and (best is None or res.fun < best.fun):
            best = res
    if best is None:
        return np.full(n, 1.0 / n)
    w = np.clip(best.x, 0.0, None)
    w /= w.sum()
    return w


def crossfit_optimized_blend(names: Sequence[str], space: str):
    oof_pred = np.zeros(len(OOF), dtype=np.float64)
    fold_weights = {}

    for fold in CV_FOLDS:
        hold = OOF["fold"].eq(fold["name"]).to_numpy()
        train_part = OOF.loc[~hold].reset_index(drop=True)
        w = fit_blend_weights(train_part, names, space)
        P_hold = OOF.loc[hold, list(names)].to_numpy(dtype=np.float64)
        oof_pred[hold] = raw_mean(P_hold, w) if space == "raw" else log_mean(P_hold, w)
        fold_weights[fold["name"]] = dict(zip(names, w.tolist()))

    full_w = fit_blend_weights(OOF.reset_index(drop=True), names, space)
    test_pred = None
    if FINAL_PREDICTIONS is not None:
        P_test = test_matrix(names)
        test_pred = raw_mean(P_test, full_w) if space == "raw" else log_mean(P_test, full_w)

    meta = {
        "models": list(names),
        "space": space,
        "crossfit_weights": fold_weights,
        "full_oof_weights_for_test": dict(zip(names, full_w.tolist())),
    }
    return oof_pred, test_pred, meta


if RUN_CV and RUN_BLEND_LAB:
    optimized_specs = {
        "all_active": MODEL_NAMES,
        "trusted": TRUSTED_MODELS,
    }
    for group_name, names in optimized_specs.items():
        for space in ["raw", "log"]:
            oof_pred, test_pred, meta = crossfit_optimized_blend(names, space)
            add_blend(
                f"cf_opt_{space}_{group_name}",
                oof_pred, test_pred,
                "crossfit_optimized_weights",
                4, True, meta,
            )


## 19. Cross-fitted global alpha correction


In [104]:
def temporal_row_weights(frame: pd.DataFrame) -> np.ndarray:
    w = np.zeros(len(frame), dtype=np.float64)
    present = list(pd.unique(frame["fold"]))
    denom = sum(FOLD_WEIGHT_MAP[f] for f in present)
    for f in present:
        mask = frame["fold"].eq(f).to_numpy()
        n = int(mask.sum())
        if n:
            w[mask] = (FOLD_WEIGHT_MAP[f] / denom) / n
    return w


def fit_alpha(frame: pd.DataFrame, pred: np.ndarray) -> float:
    residual = (
        np.log1p(frame[TARGET_COL].to_numpy(dtype=np.float64))
        - np.log1p(np.clip(pred, 0.0, None))
    )
    w = temporal_row_weights(frame)
    alpha = float(np.sum(w * residual) / max(np.sum(w), 1e-15))
    return float(np.clip(alpha, -ALPHA_CLIP, ALPHA_CLIP))


def apply_alpha(pred, alpha):
    return np.expm1(np.clip(np.log1p(np.clip(pred, 0.0, None)) + alpha, 0.0, None))


def crossfit_alpha(base_candidate: BlendCandidate):
    out = np.zeros(len(OOF), dtype=np.float64)
    alphas = {}
    for fold in CV_FOLDS:
        hold = OOF["fold"].eq(fold["name"]).to_numpy()
        alpha = fit_alpha(OOF.loc[~hold], base_candidate.oof_pred[~hold])
        out[hold] = apply_alpha(base_candidate.oof_pred[hold], alpha)
        alphas[fold["name"]] = alpha

    full_alpha = fit_alpha(OOF, base_candidate.oof_pred)
    test = None
    if base_candidate.test_pred is not None:
        test = apply_alpha(base_candidate.test_pred, full_alpha)
    return out, test, {
        "base": base_candidate.name,
        "crossfit_alpha": alphas,
        "full_oof_alpha_for_test": full_alpha,
    }


if RUN_CV and RUN_BLEND_LAB:
    alpha_base = BLENDS["cf_opt_log_trusted"]
    oof_pred, test_pred, meta = crossfit_alpha(alpha_base)
    add_blend(
        "cf_alpha_on_optlog_trusted",
        oof_pred, test_pred,
        "crossfit_global_alpha",
        5, True, meta,
    )


## 20. Ozon segment calibration — cross-fitted


In [105]:
def deterministic_calibration_segment(frame: pd.DataFrame) -> pd.Series:
    anchor = frame[ANCHOR_COL].astype(float)
    anchor_bin = pd.cut(
        anchor,
        bins=[-np.inf, 0, 100, 500, 2000, 10000, np.inf],
        labels=["a0", "a1", "a2", "a3", "a4", "a5"],
    ).astype(str)

    rec = frame["__analysis_days_since_last_purchase"].astype(float)
    rec_bin = pd.cut(
        rec,
        bins=[-np.inf, 7, 30, 90, 365, np.inf],
        labels=["r0", "r1", "r2", "r3", "r4"],
    ).astype("string").fillna("r_never").astype(str)
    return anchor_bin + "|" + rec_bin


def fit_segment_corrections(frame: pd.DataFrame, pred: np.ndarray):
    seg = deterministic_calibration_segment(frame)
    resid = (
        np.log1p(frame[TARGET_COL].to_numpy(dtype=np.float64))
        - np.log1p(np.clip(pred, 0.0, None))
    )
    row_w = temporal_row_weights(frame)
    tmp = pd.DataFrame({
        "segment": seg.to_numpy(),
        "resid": resid,
        "w": row_w,
    })
    tmp["wr"] = tmp["w"] * tmp["resid"]
    grouped = tmp.groupby("segment", observed=False)
    stats = grouped.agg(weight_sum=("w", "sum"), wr_sum=("wr", "sum"), count=("resid", "size"))
    stats["mean"] = stats["wr_sum"] / stats["weight_sum"].replace(0, np.nan)
    shrink = stats["count"] / (stats["count"] + SEGMENT_SHRINKAGE)
    delta = (stats["mean"].fillna(0.0) * shrink).clip(-SEGMENT_CLIP, SEGMENT_CLIP)
    return delta.to_dict()


def apply_segment_corrections(frame: pd.DataFrame, pred: np.ndarray, mapping):
    seg = deterministic_calibration_segment(frame)
    delta = seg.map(mapping).fillna(0.0).to_numpy(dtype=np.float64)
    return np.expm1(
        np.clip(np.log1p(np.clip(pred, 0.0, None)) + delta, 0.0, None)
    )


def crossfit_segment_calibration(base_candidate: BlendCandidate):
    out = np.zeros(len(OOF), dtype=np.float64)
    fold_maps = {}
    for fold in CV_FOLDS:
        hold = OOF["fold"].eq(fold["name"]).to_numpy()
        mapping = fit_segment_corrections(
            OOF.loc[~hold], base_candidate.oof_pred[~hold]
        )
        out[hold] = apply_segment_corrections(
            OOF.loc[hold], base_candidate.oof_pred[hold], mapping
        )
        fold_maps[fold["name"]] = mapping

    full_map = fit_segment_corrections(OOF, base_candidate.oof_pred)
    test = None
    if base_candidate.test_pred is not None and FINAL_PREDICTIONS is not None:
        test = apply_segment_corrections(
            FINAL_PREDICTIONS, base_candidate.test_pred, full_map
        )
    return out, test, {
        "base": base_candidate.name,
        "crossfit_segment_maps": fold_maps,
        "full_oof_segment_map_for_test": full_map,
    }


if RUN_CV and RUN_BLEND_LAB:
    seg_base = BLENDS["cf_alpha_on_optlog_trusted"]
    oof_pred, test_pred, meta = crossfit_segment_calibration(seg_base)
    add_blend(
        "cf_segment_calibrated",
        oof_pred, test_pred,
        "crossfit_segment_calibration",
        6, True, meta,
    )


## 21. Disagreement-gated blend — cross-fitted


In [106]:
def disagreement_score(frame: pd.DataFrame, names: Sequence[str]) -> np.ndarray:
    return np.log1p(frame[list(names)].to_numpy(dtype=np.float64)).std(axis=1)


def gated_prediction(best_pred, diverse_pred, disagreement, threshold, high_weight):
    w = np.where(disagreement >= threshold, high_weight, 1.0 - high_weight)
    # log-space interpolation
    z = (
        (1.0 - w) * np.log1p(np.clip(best_pred, 0.0, None))
        + w * np.log1p(np.clip(diverse_pred, 0.0, None))
    )
    return np.expm1(z)


def choose_gate_on_training(frame: pd.DataFrame, best_name: str, diverse_names: Sequence[str]):
    d = disagreement_score(frame, diverse_names)
    best_pred = frame[best_name].to_numpy(dtype=np.float64)
    diverse_pred = log_mean(frame[list(diverse_names)].to_numpy(dtype=np.float64))

    best = None
    for q in [0.50, 0.65, 0.80, 0.90]:
        threshold = float(np.quantile(d, q))
        for high_weight in [0.60, 0.75, 0.90]:
            pred = gated_prediction(best_pred, diverse_pred, d, threshold, high_weight)
            score = score_on_rows(frame, pred)
            candidate = (score, threshold, high_weight, q)
            if best is None or candidate[0] < best[0]:
                best = candidate
    return {
        "score": best[0],
        "threshold": best[1],
        "high_weight": best[2],
        "quantile": best[3],
    }


def crossfit_disagreement_gate(best_name: str, diverse_names: Sequence[str]):
    out = np.zeros(len(OOF), dtype=np.float64)
    fold_specs = {}

    for fold in CV_FOLDS:
        hold = OOF["fold"].eq(fold["name"]).to_numpy()
        tr = OOF.loc[~hold].reset_index(drop=True)
        va = OOF.loc[hold]
        spec = choose_gate_on_training(tr, best_name, diverse_names)

        d = disagreement_score(va, diverse_names)
        diverse = log_mean(va[list(diverse_names)].to_numpy(dtype=np.float64))
        out[hold] = gated_prediction(
            va[best_name].to_numpy(dtype=np.float64),
            diverse,
            d,
            spec["threshold"],
            spec["high_weight"],
        )
        fold_specs[fold["name"]] = spec

    full_spec = choose_gate_on_training(OOF.reset_index(drop=True), best_name, diverse_names)
    test = None
    if FINAL_PREDICTIONS is not None:
        d = disagreement_score(FINAL_PREDICTIONS, diverse_names)
        diverse = log_mean(FINAL_PREDICTIONS[list(diverse_names)].to_numpy(dtype=np.float64))
        test = gated_prediction(
            FINAL_PREDICTIONS[best_name].to_numpy(dtype=np.float64),
            diverse,
            d,
            full_spec["threshold"],
            full_spec["high_weight"],
        )

    return out, test, {
        "best_model": best_name,
        "diverse_models": list(diverse_names),
        "crossfit_gate_specs": fold_specs,
        "full_oof_gate_for_test": full_spec,
    }


if RUN_CV and RUN_BLEND_LAB:
    # Best base comes from supplied Optuna score snapshot, not current OOF ranking.
    prior_best_base = min(TRUSTED_MODELS, key=lambda n: MODEL_SPECS[n]["optuna_metric"])
    oof_pred, test_pred, meta = crossfit_disagreement_gate(
        prior_best_base, TRUSTED_MODELS
    )
    add_blend(
        "cf_disagreement_gate",
        oof_pred, test_pred,
        "crossfit_disagreement_gate",
        6, True, meta,
    )


## 22. Meta-logmean сильных разнообразных ensembles


In [107]:
if RUN_CV and RUN_BLEND_LAB:
    meta_names = [
        "cf_opt_log_trusted",
        "cf_alpha_on_optlog_trusted",
        "cf_segment_calibrated",
        "cf_disagreement_gate",
    ]
    meta_oof = np.column_stack([BLENDS[n].oof_pred for n in meta_names])
    meta_test = None
    if all(BLENDS[n].test_pred is not None for n in meta_names):
        meta_test = np.column_stack([BLENDS[n].test_pred for n in meta_names])

    add_blend(
        "meta_logmean_cf_ensembles",
        log_mean(meta_oof),
        None if meta_test is None else log_mean(meta_test),
        "meta_logmean",
        7, True,
        {"components": meta_names},
    )


## 23. Blend leaderboard и защита от переобучения


In [108]:
def blend_leaderboard():
    rows = []
    for name, cand in BLENDS.items():
        rows.append({
            "blend": name,
            "weighted_rmsle": weighted_oof_rmsle(OOF, cand.oof_pred),
            "kind": cand.kind,
            "complexity": cand.complexity,
            "selection_safe": cand.selection_safe,
        })
    return pd.DataFrame(rows).sort_values(["weighted_rmsle", "complexity"]).reset_index(drop=True)


def choose_final_safe_blend(table: pd.DataFrame) -> str:
    safe = table.loc[table["selection_safe"]].sort_values(
        ["weighted_rmsle", "complexity"]
    ).reset_index(drop=True)
    if safe.empty:
        raise RuntimeError("No selection-safe blends")

    best_score = float(safe.loc[0, "weighted_rmsle"])
    near = safe.loc[
        safe["weighted_rmsle"] <= best_score + FINAL_COMPLEXITY_TIE_TOLERANCE
    ].sort_values(["complexity", "weighted_rmsle"])
    return str(near.iloc[0]["blend"])


if RUN_CV and RUN_BLEND_LAB:
    BLEND_LEADERBOARD = blend_leaderboard()
    FINAL_BLEND_NAME = choose_final_safe_blend(BLEND_LEADERBOARD)
    print("FINAL_BLEND_NAME:", FINAL_BLEND_NAME)
    display(BLEND_LEADERBOARD.head(30))
    BLEND_LEADERBOARD.to_csv(ANALYSIS_DIR / "blend_leaderboard.csv", index=False)

    # Metadata is JSON-safe after normalization.
    blend_specs_json = {}
    for name, cand in BLENDS.items():
        blend_specs_json[name] = {
            "kind": cand.kind,
            "complexity": cand.complexity,
            "selection_safe": cand.selection_safe,
            "metadata": cand.metadata,
            "weighted_rmsle": weighted_oof_rmsle(OOF, cand.oof_pred),
        }
    (ANALYSIS_DIR / "blend_specs.json").write_text(
        json.dumps(blend_specs_json, ensure_ascii=False, indent=2, default=float),
        encoding="utf-8",
    )


FINAL_BLEND_NAME: optuna_softmax_logmean


,blend,weighted_rmsle,kind,complexity,selection_safe
0,optuna_softmax_logmean,1.711121,optuna_weighted,2,True
1,cf_disagreement_gate,1.711165,crossfit_disagreement_gate,6,True
2,cf_opt_log_all_active,1.711212,crossfit_optimized_weights,4,True
3,cf_opt_log_trusted,1.711212,crossfit_optimized_weights,4,True
4,validation_inverse_score_logmean,1.711274,validation_weighted,2,False
5,all_active_logmean,1.711274,log_mean,1,True
6,trusted_logmean,1.711274,fixed_group_logmean,1,True
7,top4_logmean,1.711274,top_n,4,False
8,top3_logmean,1.711325,top_n,3,False
9,top2_logmean,1.711385,top_n,2,False


## 24. Сохранение blend submissions и FINAL


In [109]:
def build_submission_from_prediction(pred: np.ndarray) -> pd.DataFrame:
    if FINAL_PREDICTIONS is None:
        raise RuntimeError("No final predictions")
    pred_frame = pd.DataFrame({
        USER_COL: FINAL_PREDICTIONS[USER_COL].to_numpy(),
        SUBMISSION_TARGET_COL: np.clip(np.asarray(pred, dtype=np.float64), 0.0, None),
    })
    sub = sample_submit.drop(columns=[SUBMISSION_TARGET_COL], errors="ignore").merge(
        pred_frame, on=USER_COL, how="left", validate="one_to_one"
    )
    if len(sub) != len(sample_submit):
        raise AssertionError("Submission row count changed")
    if sub[SUBMISSION_TARGET_COL].isna().any():
        raise AssertionError("Missing predictions in submission")
    return sub


if RUN_BLEND_LAB and RUN_FINAL_FIT:
    available = BLEND_LEADERBOARD.loc[
        BLEND_LEADERBOARD["blend"].map(lambda n: BLENDS[n].test_pred is not None)
    ].head(SAVE_TOP_N_BLEND_SUBMISSIONS)

    for _, row in available.iterrows():
        name = row["blend"]
        sub = build_submission_from_prediction(BLENDS[name].test_pred)
        sub.to_csv(BLEND_SUBMISSION_DIR / f"submission_{name}.csv", index=False)

    final_candidate = BLENDS[FINAL_BLEND_NAME]
    if final_candidate.test_pred is None:
        raise RuntimeError(f"Final blend {FINAL_BLEND_NAME} has no test prediction")

    final_submission = build_submission_from_prediction(final_candidate.test_pred)
    final_submission.to_csv(OUTPUT_DIR / "submission_FINAL.csv", index=False)
    final_submission.to_csv(OUTPUT_DIR / "submission_best_blend.csv", index=False)

    print("Saved:", OUTPUT_DIR / "submission_FINAL.csv")
    display(final_submission.head())


Saved: /kaggle/working/ozon_final/submission_FINAL.csv


,user_id,predict
0,2,1.691598
1,7,73.574504
2,15,7.031988
3,18,137.063283
4,23,0.365792


## 25. Финальные consistency checks и manifest


In [110]:
CHECKS = []


def check(name, condition, detail=""):
    if not condition:
        raise AssertionError(f"[FAIL] {name}: {detail}")
    CHECKS.append({"check": name, "status": "PASS", "detail": detail})
    print(f"[PASS] {name}" + (f" — {detail}" if detail else ""))


check("validation fold count == 3", len(CV_FOLDS) == 3)
check("fold weights sum == 1", np.isclose(sum(f["weight"] for f in CV_FOLDS), 1.0))
check("February not used in CV", all("2025-02" not in f["valid_cutoff"] for f in CV_FOLDS))
check("October not used as validation", all(f["valid_cutoff"] != "2025-10-16" for f in CV_FOLDS))
check("January is validation, not outer holdout", CV_FOLDS[-1]["valid_cutoff"] == "2026-01-14")
check("prediction cutoff is 2026-02-13", PREDICTION_CUTOFF == "2026-02-13")
check("all six production snapshots found", all(snapshot_path(c).exists() for c in ALL_CUTOFFS))
check("prediction target is unknown", not PREFLIGHT.loc[PREFLIGHT["cutoff"].eq(PREDICTION_CUTOFF), "target_known"].iloc[0])
check(
    "LightGBM fully disabled in active training",
    "lightgbm" in DISABLED_MODEL_FAMILIES
    and all(MODEL_SPECS[name]["family"] != "lightgbm" for name in MODEL_NAMES),
)
check("frozen feature lists unique after stable dedup", all(len(s["features"]) == len(set(s["features"])) for s in MODEL_SPECS.values()))
check("Denis feature names preserved", "search_days_7d__denis" in MODEL_SPECS["catboost_log_level"]["features"])
check("all frozen features exist in every snapshot", set(UNION_FEATURES).issubset(COMMON_COLUMNS))
check("sample and snapshot user universe match", len(sample_submit) == len(user_universe))

if RUN_CV:
    check("all active models have 3 fold scores", CV_METRICS.groupby("model").size().eq(3).all())
    check("all OOF predictions finite", np.isfinite(OOF[MODEL_NAMES].to_numpy()).all())

if RUN_FINAL_FIT:
    check("all active base submissions exist", all((BASE_SUBMISSION_DIR / f"submission_{n}.csv").exists() for n in MODEL_NAMES))
    check("all active final predictions finite", np.isfinite(FINAL_PREDICTIONS[MODEL_NAMES].to_numpy()).all())

if RUN_BLEND_LAB and RUN_FINAL_FIT:
    check("submission_FINAL exists", (OUTPUT_DIR / "submission_FINAL.csv").exists())

pd.DataFrame(CHECKS).to_csv(OUTPUT_DIR / "final_checks.csv", index=False)

manifest = {
    "created_at_utc": pd.Timestamp.utcnow().isoformat(),
    "pipeline": "OZON_FINAL_KAGGLE_SNAPSHOT_CACHE",
    "source_generator": {
        "file": "ozon_feature_selection_pipeline_v4_release.py",
        "line_count": 18958,
        "snapshot_entrypoint": "build_ozon_unified_snapshots_for_selection",
        "entrypoint_line": 16287,
    },
    "snapshot_dir": str(SNAPSHOT_DIR),
    "snapshot_files": {c: str(snapshot_path(c)) for c in ALL_CUTOFFS},
    "cache_signatures": sorted(set(x for x in cache_signatures if x)),
    "snapshot_columns": {c: len(SNAPSHOT_INFO[c]["columns"]) for c in ALL_CUTOFFS},
    "frozen_union_feature_count": len(UNION_FEATURES),
    "frozen_categorical_features": FROZEN_CATEGORICAL_FEATURES,
    "sample_submission_path": str(SAMPLE_SUBMISSION_PATH),
    "cv_folds": CV_FOLDS,
    "final_train_cutoffs": FINAL_TRAIN_CUTOFFS,
    "prediction_cutoff": PREDICTION_CUTOFF,
    "model_specs": MODEL_SPECS,
    "final_iterations": FINAL_ITERATIONS,
    "final_model_paths": FINAL_MODEL_PATHS,
    "trusted_models": TRUSTED_MODELS,
    "provisional_models": PROVISIONAL_MODELS,
    "final_blend": globals().get("FINAL_BLEND_NAME"),
    "checks": CHECKS,
}
(OUTPUT_DIR / "run_manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2, default=str), encoding="utf-8"
)

print("\nArtifacts:")
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(OUTPUT_DIR)}")


[PASS] validation fold count == 3
[PASS] fold weights sum == 1
[PASS] February not used in CV
[PASS] October not used as validation
[PASS] January is validation, not outer holdout
[PASS] prediction cutoff is 2026-02-13
[PASS] all six production snapshots found
[PASS] prediction target is unknown
[PASS] LightGBM fully disabled in active training
[PASS] frozen feature lists unique after stable dedup
[PASS] Denis feature names preserved
[PASS] all frozen features exist in every snapshot
[PASS] sample and snapshot user universe match
[PASS] all active models have 3 fold scores
[PASS] all OOF predictions finite
[PASS] all active base submissions exist
[PASS] all active final predictions finite
[PASS] submission_FINAL exists

Artifacts:
  analysis/adaptive_anchor_preflight.csv
  analysis/base_best_iterations.csv
  analysis/base_fold_metrics.csv
  analysis/base_model_ranking.csv
  analysis/base_ranking_recomputed.csv
  analysis/blend_leaderboard.csv
  analysis/blend_specs.json
  analysis/blen

## 26. Финальный checklist перед leaderboard

После `Run All` должны появиться:

- `base_submissions/submission_<model>.csv` — 6 файлов;
- `analysis/base_fold_metrics.csv`, `analysis/oof_all_models.parquet`, error tables и correlation tables;
- `analysis/blend_leaderboard.csv` и `analysis/blend_specs.json`;
- `blend_submissions/*.csv`;
- `submission_FINAL.csv`;
- `run_manifest.json` с snapshot provenance, effective LightGBM leaves и фактическими final iteration counts.

Перед отправкой на leaderboard проверьте, что `SMOKE_USERS is None`, все шесть models имеют по три CV score, ни одна нужная модель не упёрлась в round cap и `final_checks.csv` содержит только `PASS`.


# 27. Stochastic / diversity ensemble experiments

Этот раздел добавляет эксперименты Дениса **поверх неизменённого baseline**:

1. column bagging для четырёх активных CatBoost/XGBoost frozen specifications;
2. target-weight specialists на `catboost_log_level`;
3. Ridge bagging с train-only encoding, imputation и scaling;
4. двухуровневый hierarchical blend;
5. честную cross-fitted affine calibration уровня и разброса;
6. диагностики, ablation, persistent cache и четыре осмысленных submission.

Новая feature selection и новая Optuna здесь не запускаются. Feature subset одного member фиксируется до CV и остаётся тем же на всех folds и final fit. Главной метрикой считается `cross_fitted_oof`; результаты, выбранные по тому же OOF, явно помечаются как exploratory.


In [111]:
# ============================================================
# STOCHASTIC CONFIG — единственная точка управления расширением
# ============================================================
ENABLE_STOCHASTIC_BAGGING = True
# One additional reproducible bootstrap/column-bag per active base model.
# There is no multi-seed ensemble in the main contour.
STOCHASTIC_N_MEMBERS = 1
STOCHASTIC_FEATURE_FRACTION = 0.80
MIN_STOCHASTIC_FEATURES = 20
STOCHASTIC_SEEDS = [17, 43, 97, 137, 211]

ENABLE_TARGET_WEIGHT_SPECIALISTS = True
SPECIALIST_BASE_MODEL = "catboost_log_level"
ZERO_HEAVY_WEIGHT = 1.5
ZERO_LIGHT_WEIGHT = 0.7
SMALL_PURCHASE_WEIGHT = 1.5
SMALL_PURCHASE_QUANTILE = 0.33
ENABLE_LARGE_PURCHASE_SPECIALIST = False
LARGE_PURCHASE_WEIGHT = 1.3
LARGE_PURCHASE_QUANTILE = 0.80

ENABLE_RIDGE_BAGGING = True
RIDGE_N_MEMBERS = 7
RIDGE_FEATURE_FRACTION = 0.75
RIDGE_SEEDS = [11, 29, 47, 83, 131, 181, 239]
# Alpha фиксируется до оценки итогового Ridge OOF. Маленькая exploratory grid
# оставлена выключенной, чтобы лучший score той же OOF не выдавался за unbiased.
RIDGE_ALPHA = 100.0
RIDGE_EXPLORATORY_ALPHA_GRID = None  # например [1.0, 10.0, 100.0, 1000.0]

ENABLE_HIERARCHICAL_BLEND = True
HIERARCHY_PRIMARY = "A"  # A=fine directions; B=family stochastic directions
ENABLE_AFFINE_CALIBRATION = True
AFFINE_SHIFT_BOUNDS = (-0.15, 0.15)
AFFINE_SPREAD_BOUNDS = (0.85, 1.15)
AFFINE_LAMBDA_SHIFT = 0.02
AFFINE_LAMBDA_SPREAD = 0.02

ENABLE_CACHE = True
STOCHASTIC_CACHE_VERSION = "stochastic_diversity_v1"
SAVE_STOCHASTIC_MODELS = False  # predictions+metadata достаточно для resume

STOCHASTIC_DIR = OUTPUT_DIR / "stochastic_diversity"
STOCHASTIC_CACHE_DIR = STOCHASTIC_DIR / "cache"
STOCHASTIC_SUBMISSION_DIR = STOCHASTIC_DIR / "submissions"
for _d in [STOCHASTIC_DIR, STOCHASTIC_CACHE_DIR, STOCHASTIC_SUBMISSION_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

assert STOCHASTIC_N_MEMBERS <= len(STOCHASTIC_SEEDS)
assert RIDGE_N_MEMBERS <= len(RIDGE_SEEDS)
assert 0 < STOCHASTIC_FEATURE_FRACTION <= 1
assert 0 < RIDGE_FEATURE_FRACTION <= 1
assert HIERARCHY_PRIMARY in {"A", "B"}

# Неприкосновенность baseline проверяется повторно в финальной ячейке.
BASELINE_PREDICTION_HASHES = {
    name: hashlib.sha256(
        np.ascontiguousarray(OOF[name].to_numpy(dtype=np.float64)).tobytes()
    ).hexdigest()
    for name in MODEL_NAMES
}
BASELINE_METRIC_SNAPSHOT = {
    name: weighted_oof_rmsle(OOF, OOF[name].to_numpy(dtype=np.float64))
    for name in MODEL_NAMES
}
BASELINE_BEST_NAME = FINAL_BLEND_NAME
BASELINE_BEST_SCORE = weighted_oof_rmsle(
    OOF, BLENDS[BASELINE_BEST_NAME].oof_pred
)

print({
    "baseline": BASELINE_BEST_NAME,
    "baseline_honest_rmsle": BASELINE_BEST_SCORE,
    "stochastic_models_planned": len(MODEL_NAMES) * STOCHASTIC_N_MEMBERS,
    "ridge_members_planned": RIDGE_N_MEMBERS,
})


{'baseline': 'optuna_softmax_logmean', 'baseline_honest_rmsle': 1.7111213151570999, 'stochastic_models_planned': 4, 'ridge_members_planned': 7}


## 27.1 Frozen stochastic registry и content-addressed cache

Subset строится только из `MODEL_SPECS[base_model]["features"]`. Ключ кэша включает dataset fingerprint, модель, target mode, seed, полный subset, параметры, fold и версию этого раздела. Один прерванный запуск не обнуляет уже завершённые folds.


In [112]:
def _json_hash(value: Any, length: int = 20) -> str:
    payload = json.dumps(value, sort_keys=True, ensure_ascii=False, default=str)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:length]


def _dataset_identifier() -> str:
    # Sidecar/schema signatures were computed by the original preflight.
    payload = {
        "version": STOCHASTIC_CACHE_VERSION,
        "cutoffs": ALL_CUTOFFS,
        "snapshot_info": SNAPSHOT_INFO,
        "cache_signatures": cache_signatures,
    }
    return _json_hash(payload, 24)


STOCHASTIC_DATASET_ID = _dataset_identifier()


def _stable_seed_offset(name: str) -> int:
    return int(hashlib.sha256(name.encode("utf-8")).hexdigest()[:8], 16)


def frozen_feature_subset(
    base_model: str, seed: int, fraction: float, minimum: int,
) -> list[str]:
    original = model_feature_order(MODEL_SPECS[base_model])[2]
    if not original:
        raise ValueError(f"{base_model}: empty frozen feature set")
    desired = int(round(len(original) * fraction))
    size = min(len(original), max(minimum, desired))
    generator = np.random.default_rng(int(seed) + _stable_seed_offset(base_model))
    selected = generator.choice(np.asarray(original, dtype=object), size=size, replace=False)
    return sorted(selected.tolist())


def build_stochastic_registry() -> list[dict[str, Any]]:
    registry = []
    if not ENABLE_STOCHASTIC_BAGGING:
        return registry
    for base_model in MODEL_NAMES:
        spec = MODEL_SPECS[base_model]
        original = model_feature_order(spec)[2]
        for seed in STOCHASTIC_SEEDS[:STOCHASTIC_N_MEMBERS]:
            features = frozen_feature_subset(
                base_model, seed, STOCHASTIC_FEATURE_FRACTION,
                MIN_STOCHASTIC_FEATURES,
            )
            registry.append({
                "run_name": f"{base_model}__bag_s{seed}",
                "kind": "column_bagging",
                "base_model": base_model,
                "family": spec["family"],
                "target_mode": spec["target_mode"],
                "seed": int(seed),
                "feature_fraction": STOCHASTIC_FEATURE_FRACTION,
                "n_original_features": len(original),
                "n_selected_features": len(features),
                "features": features,
                "model_params": resolve_effective_params(spec),
                "weight_mode": "none",
            })
    return registry


def build_specialist_registry() -> list[dict[str, Any]]:
    if not ENABLE_TARGET_WEIGHT_SPECIALISTS:
        return []
    spec = MODEL_SPECS[SPECIALIST_BASE_MODEL]
    base = {
        "kind": "target_specialist",
        "base_model": SPECIALIST_BASE_MODEL,
        "family": spec["family"],
        "target_mode": spec["target_mode"],
        "seed": RANDOM_SEED,
        "feature_fraction": 1.0,
        "n_original_features": len(model_feature_order(spec)[2]),
        "n_selected_features": len(model_feature_order(spec)[2]),
        "features": model_feature_order(spec)[2],
        "model_params": resolve_effective_params(spec),
    }
    definitions = [
        ("catboost_log_level__baseline_control", "baseline_control"),
        ("catboost_log_level__zero_heavy_1p5", "zero_heavy"),
        ("catboost_log_level__zero_light_0p7", "zero_light"),
        ("catboost_log_level__small_purchase_1p5", "small_purchase"),
    ]
    if ENABLE_LARGE_PURCHASE_SPECIALIST:
        definitions.append(("catboost_log_level__large_purchase_1p3", "large_purchase"))
    return [{**base, "run_name": name, "weight_mode": mode} for name, mode in definitions]


STOCHASTIC_REGISTRY = build_stochastic_registry()
SPECIALIST_REGISTRY = build_specialist_registry()
ALL_BOOSTING_RUNS = STOCHASTIC_REGISTRY + [
    run for run in SPECIALIST_REGISTRY if run["weight_mode"] != "baseline_control"
]



def run_specification(run: Mapping[str, Any]) -> dict[str, Any]:
    base = MODEL_SPECS[run["base_model"]]
    return {
        **base,
        "features": list(run["features"]),
        "params": dict(base["params"]),
    }


def run_cache_key(run: Mapping[str, Any], fold: Mapping[str, Any] | None, final: bool) -> str:
    payload = {
        "dataset_id": STOCHASTIC_DATASET_ID,
        "version": STOCHASTIC_CACHE_VERSION,
        "run": {k: run[k] for k in [
            "run_name", "base_model", "family", "target_mode", "seed",
            "features", "model_params", "weight_mode",
        ]},
        "split": {
            "kind": "final" if final else "fold",
            "fold": None if fold is None else fold,
            "train_cutoffs": FINAL_TRAIN_CUTOFFS if final else fold["train_cutoffs"],
            "valid_cutoff": PREDICTION_CUTOFF if final else fold["valid_cutoff"],
        },
    }
    return _json_hash(payload)


def cache_paths(run: Mapping[str, Any], fold: Mapping[str, Any] | None, final: bool):
    split = "final" if final else fold["name"]
    directory = STOCHASTIC_CACHE_DIR / run["run_name"] / split
    return directory / "prediction.npz", directory / "metadata.json"


def load_prediction_cache(run, fold=None, final=False):
    if not ENABLE_CACHE:
        return None
    prediction_path, metadata_path = cache_paths(run, fold, final)
    if not prediction_path.exists() or not metadata_path.exists():
        return None
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    if metadata.get("cache_key") != run_cache_key(run, fold, final):
        return None
    with np.load(prediction_path, allow_pickle=False) as saved:
        return {
            "user_id": saved["user_id"],
            "prediction": saved["prediction"].astype(np.float64),
            "truth": saved["truth"].astype(np.float64),
            "metadata": metadata,
        }


def save_prediction_cache(run, fold, final, user_id, prediction, truth, metadata):
    prediction_path, metadata_path = cache_paths(run, fold, final)
    prediction_path.parent.mkdir(parents=True, exist_ok=True)
    metadata = {
        **metadata,
        "cache_key": run_cache_key(run, fold, final),
        "dataset_id": STOCHASTIC_DATASET_ID,
    }
    temporary = prediction_path.with_suffix(".partial.npz")
    np.savez_compressed(
        temporary,
        user_id=np.asarray(user_id),
        prediction=np.asarray(prediction, dtype=np.float64),
        truth=np.asarray(truth, dtype=np.float64),
    )
    temporary.replace(prediction_path)
    metadata_path.write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2, default=str),
        encoding="utf-8",
    )


STOCHASTIC_RUN_REGISTRY = pd.DataFrame([
    {k: v for k, v in run.items() if k not in {"features", "model_params"}}
    | {
        "features_json": json.dumps(run["features"], ensure_ascii=False),
        "model_params_json": json.dumps(run["model_params"], ensure_ascii=False, default=str),
        "feature_subset_hash": _json_hash(run["features"]),
        "dataset_id": STOCHASTIC_DATASET_ID,
    }
    for run in STOCHASTIC_REGISTRY + SPECIALIST_REGISTRY
])
STOCHASTIC_RUN_REGISTRY.to_csv(
    STOCHASTIC_DIR / "stochastic_run_registry.csv", index=False
)
display(STOCHASTIC_RUN_REGISTRY.head())


,run_name,kind,base_model,family,target_mode,seed,feature_fraction,n_original_features,n_selected_features,weight_mode,features_json,model_params_json,feature_subset_hash,dataset_id
0,xgboost_log_level__bag_s17,column_bagging,xgboost_log_level,xgboost,log_level,17,0.8,45,36,none,"[""active_days_share30_of_90"", ""bayesian_active...","{""colsample_bylevel"": 0.6441, ""colsample_bynod...",5dd4d74df9444dd61176,780cec3b3f8cfd59c8916784
1,catboost_log_level__bag_s17,column_bagging,catboost_log_level,catboost,log_level,17,0.8,33,26,none,"[""any_cart_no_order_day_share_active_90d"", ""ba...","{""bagging_temperature"": 1.3558, ""border_count""...",24f6bbd1fccbb8d241e6,780cec3b3f8cfd59c8916784
2,catboost_log_diff__bag_s17,column_bagging,catboost_log_diff,catboost,log_diff,17,0.8,54,43,none,"[""active_days_decay30_hl90"", ""any_cart_no_orde...","{""bagging_temperature"": 0.7322, ""border_count""...",9d5d6fd18649ace8be42,780cec3b3f8cfd59c8916784
3,xgboost_log_diff__bag_s17,column_bagging,xgboost_log_diff,xgboost,log_diff,17,0.8,40,32,none,"[""any_cart_no_order_day_30d"", ""both_channel_ja...","{""colsample_bylevel"": 0.613, ""colsample_bynode...",d3f6169e05b5ed5e84b4,780cec3b3f8cfd59c8916784
4,catboost_log_level__baseline_control,target_specialist,catboost_log_level,catboost,log_level,42,1.0,33,33,baseline_control,"[""any_cart_no_order_day_share_active_90d"", ""ba...","{""bagging_temperature"": 1.3558, ""border_count""...",68bd948e18298afd3e04,780cec3b3f8cfd59c8916784


## 27.2 Column bagging и target-weight specialists

Для каждого boosting member используются исходные fixed hyperparameters. Дополнительного row bagging нет. `zero/small/large` thresholds вычисляются исключительно на train-части соответствующего fold; baseline-control переиспользует уже рассчитанный исходный CatBoost.


In [113]:
def target_sample_weights(y_raw: np.ndarray, mode: str):
    y_raw = np.asarray(y_raw, dtype=np.float64)
    weights = np.ones(len(y_raw), dtype=np.float32)
    threshold = None
    if mode == "none":
        return None, threshold
    if mode == "zero_heavy":
        weights[y_raw <= 0] = ZERO_HEAVY_WEIGHT
    elif mode == "zero_light":
        weights[y_raw <= 0] = ZERO_LIGHT_WEIGHT
    elif mode == "small_purchase":
        positives = y_raw[y_raw > 0]
        if not len(positives):
            raise ValueError("small_purchase: train contains no positive targets")
        threshold = float(np.quantile(positives, SMALL_PURCHASE_QUANTILE))
        weights[(y_raw > 0) & (y_raw <= threshold)] = SMALL_PURCHASE_WEIGHT
    elif mode == "large_purchase":
        positives = y_raw[y_raw > 0]
        if not len(positives):
            raise ValueError("large_purchase: train contains no positive targets")
        threshold = float(np.quantile(positives, LARGE_PURCHASE_QUANTILE))
        weights[y_raw >= threshold] = LARGE_PURCHASE_WEIGHT
    else:
        raise ValueError(f"unknown weight mode: {mode}")
    return weights, threshold


def _cleanup_accelerators():
    gc.collect()
    try:
        import cupy
        cupy.get_default_memory_pool().free_all_blocks()
    except Exception:
        pass


def fit_boosting_split(run, fold=None, final=False):
    cached = load_prediction_cache(run, fold, final)
    if cached is not None:
        cached["metadata"]["cache_hit"] = True
        return cached

    spec = run_specification(run)
    train_cutoffs = FINAL_TRAIN_CUTOFFS if final else fold["train_cutoffs"]
    valid_cutoff = PREDICTION_CUTOFF if final else fold["valid_cutoff"]
    matrices = materialize_model_pair(train_cutoffs, valid_cutoff, spec)
    sample_weight, threshold = target_sample_weights(
        matrices["y_train_raw"], run["weight_mode"]
    )
    train_positives = matrices["y_train_raw"][matrices["y_train_raw"] > 0]
    audit_small_threshold = float(np.quantile(
        train_positives, SMALL_PURCHASE_QUANTILE
    ))
    audit_large_threshold = float(np.quantile(
        train_positives, LARGE_PURCHASE_QUANTILE
    ))

    if final:
        fold_rows = BOOSTING_FOLD_METRICS.loc[
            BOOSTING_FOLD_METRICS["run_name"].eq(run["run_name"])
        ]
        if fold_rows.empty:
            raise RuntimeError(f"{run['run_name']}: no CV rounds for final fit")
        final_rounds = recommended_iterations_from_folds(fold_rows)
    else:
        final_rounds = None

    started = time.perf_counter()
    model, pred_model, best_round = fit_family(
        spec["family"],
        matrices["X_train"], matrices["y_train"],
        None if final else matrices["X_valid"],
        None if final else matrices["y_valid"],
        matrices["categorical_positions"], spec,
        final_rounds=final_rounds,
        sample_weight=sample_weight,
        random_seed=run["seed"],
    )

    if final:
        if spec["family"] == "catboost":
            from catboost import Pool
            pred_model = model.predict(Pool(
                matrices["X_valid"],
                cat_features=matrices["categorical_positions"],
            ))
        elif spec["family"] == "lightgbm":
            pred_model = model.predict(matrices["X_valid"], num_iteration=best_round)
        else:
            pred_model = model.predict(matrices["X_valid"])

    pred_raw = inverse_model_target(
        pred_model, matrices["anchor_valid"], spec["target_mode"]
    )
    pred_raw = np.clip(np.asarray(pred_raw, dtype=np.float64), 0.0, None)
    if not np.isfinite(pred_raw).all():
        raise FloatingPointError(f"{run['run_name']}: non-finite prediction")

    metadata = {
        "run_name": run["run_name"],
        "fold": "final" if final else fold["name"],
        "best_round": int(best_round),
        "threshold": threshold,
        "audit_small_threshold": audit_small_threshold,
        "audit_large_threshold": audit_large_threshold,
        "seconds": time.perf_counter() - started,
        "cache_hit": False,
    }
    result = {
        "user_id": matrices["valid_user_ids"],
        "prediction": pred_raw,
        "truth": matrices["y_valid_raw"],
        "metadata": metadata,
    }
    save_prediction_cache(
        run, fold, final, result["user_id"], result["prediction"],
        result["truth"], metadata,
    )
    del matrices, model, pred_model, sample_weight
    _cleanup_accelerators()
    return result


BOOSTING_FOLD_ROWS = []
EXTRA_OOF_COLUMNS = {}
EXTRA_FINAL_COLUMNS = {}

if RUN_CV and (ENABLE_STOCHASTIC_BAGGING or ENABLE_TARGET_WEIGHT_SPECIALISTS):
    for run in ALL_BOOSTING_RUNS:
        print("\nDIVERSITY CV:", run["run_name"])
        full_prediction = np.full(len(OOF), np.nan, dtype=np.float64)
        for fold in CV_FOLDS:
            result = fit_boosting_split(run, fold=fold, final=False)
            hold = OOF["fold"].eq(fold["name"]).to_numpy()
            expected_users = OOF.loc[hold, USER_COL].to_numpy()
            if not np.array_equal(expected_users, result["user_id"]):
                raise AssertionError(f"{run['run_name']}/{fold['name']}: user alignment")
            full_prediction[hold] = result["prediction"]
            truth = result["truth"]
            zero = truth <= 0
            small_threshold = result["metadata"]["audit_small_threshold"]
            large_threshold = result["metadata"]["audit_large_threshold"]
            small = (truth > 0) & (truth <= small_threshold)
            large = truth >= large_threshold
            BOOSTING_FOLD_ROWS.append({
                "run_name": run["run_name"],
                "base_model": run["base_model"],
                "kind": run["kind"],
                "weight_mode": run["weight_mode"],
                "fold": fold["name"],
                "fold_weight": fold["weight"],
                "weight": fold["weight"],
                "rmsle": rmsle(truth, result["prediction"]),
                "rmsle_zero": rmsle(truth[zero], result["prediction"][zero]),
                "rmsle_positive": rmsle(truth[~zero], result["prediction"][~zero]),
                "rmsle_small_positive": rmsle(
                    truth[small], result["prediction"][small]
                ),
                "rmsle_large_positive": rmsle(
                    truth[large], result["prediction"][large]
                ),
                "best_round": int(result["metadata"]["best_round"]),
                "threshold": result["metadata"].get("threshold"),
                "audit_small_threshold": small_threshold,
                "audit_large_threshold": large_threshold,
                "seconds": result["metadata"].get("seconds", 0.0),
                "cache_hit": result["metadata"].get("cache_hit", False),
            })
        if not np.isfinite(full_prediction).all():
            raise AssertionError(f"{run['run_name']}: incomplete OOF")
        EXTRA_OOF_COLUMNS[run["run_name"]] = full_prediction

BOOSTING_FOLD_METRICS = pd.DataFrame(BOOSTING_FOLD_ROWS)

# Existing CatBoost is the exact weight=1 control; no redundant retraining.
if ENABLE_TARGET_WEIGHT_SPECIALISTS:
    control_name = "catboost_log_level__baseline_control"
    EXTRA_OOF_COLUMNS[control_name] = OOF[SPECIALIST_BASE_MODEL].to_numpy(dtype=np.float64).copy()
    for fold in CV_FOLDS:
        hold = OOF["fold"].eq(fold["name"]).to_numpy()
        truth = OOF.loc[hold, TARGET_COL].to_numpy(dtype=np.float64)
        pred = EXTRA_OOF_COLUMNS[control_name][hold]
        zero = truth <= 0
        audit_source = next(
            row for row in BOOSTING_FOLD_ROWS
            if row["fold"] == fold["name"]
        )
        small_threshold = audit_source["audit_small_threshold"]
        large_threshold = audit_source["audit_large_threshold"]
        small = (truth > 0) & (truth <= small_threshold)
        large = truth >= large_threshold
        BOOSTING_FOLD_ROWS.append({
            "run_name": control_name, "base_model": SPECIALIST_BASE_MODEL,
            "kind": "target_specialist", "weight_mode": "baseline_control",
            "fold": fold["name"], "fold_weight": fold["weight"],
            "weight": fold["weight"], "rmsle": rmsle(truth, pred),
            "rmsle_zero": rmsle(truth[zero], pred[zero]),
            "rmsle_positive": rmsle(truth[~zero], pred[~zero]),
            "rmsle_small_positive": rmsle(truth[small], pred[small]),
            "rmsle_large_positive": rmsle(truth[large], pred[large]),
            "best_round": int(CV_METRICS.loc[
                CV_METRICS["model"].eq(SPECIALIST_BASE_MODEL)
                & CV_METRICS["fold"].eq(fold["name"]), "best_round"
            ].iloc[0]),
            "threshold": None,
            "audit_small_threshold": small_threshold,
            "audit_large_threshold": large_threshold,
            "seconds": 0.0, "cache_hit": True,
        })
    BOOSTING_FOLD_METRICS = pd.DataFrame(BOOSTING_FOLD_ROWS)

for name, values in EXTRA_OOF_COLUMNS.items():
    OOF[name] = values

if RUN_FINAL_FIT and (ENABLE_STOCHASTIC_BAGGING or ENABLE_TARGET_WEIGHT_SPECIALISTS):
    for run in ALL_BOOSTING_RUNS:
        print("DIVERSITY FINAL:", run["run_name"])
        result = fit_boosting_split(run, final=True)
        if not np.array_equal(FINAL_PREDICTIONS[USER_COL].to_numpy(), result["user_id"]):
            raise AssertionError(f"{run['run_name']}: final user alignment")
        EXTRA_FINAL_COLUMNS[run["run_name"]] = result["prediction"]
    if ENABLE_TARGET_WEIGHT_SPECIALISTS:
        EXTRA_FINAL_COLUMNS["catboost_log_level__baseline_control"] = (
            FINAL_PREDICTIONS[SPECIALIST_BASE_MODEL].to_numpy(dtype=np.float64).copy()
        )
    for name, values in EXTRA_FINAL_COLUMNS.items():
        FINAL_PREDICTIONS[name] = values

TARGET_SPECIALIST_METRICS = BOOSTING_FOLD_METRICS.loc[
    BOOSTING_FOLD_METRICS["kind"].eq("target_specialist")
].reset_index(drop=True)
TARGET_SPECIALIST_METRICS.to_csv(
    STOCHASTIC_DIR / "target_specialist_metrics.csv", index=False
)
display(BOOSTING_FOLD_METRICS.head())



DIVERSITY CV: xgboost_log_level__bag_s17

DIVERSITY CV: catboost_log_level__bag_s17

DIVERSITY CV: catboost_log_diff__bag_s17

DIVERSITY CV: xgboost_log_diff__bag_s17

DIVERSITY CV: catboost_log_level__zero_heavy_1p5

DIVERSITY CV: catboost_log_level__zero_light_0p7

DIVERSITY CV: catboost_log_level__small_purchase_1p5
DIVERSITY FINAL: xgboost_log_level__bag_s17
DIVERSITY FINAL: catboost_log_level__bag_s17
DIVERSITY FINAL: catboost_log_diff__bag_s17
DIVERSITY FINAL: xgboost_log_diff__bag_s17
DIVERSITY FINAL: catboost_log_level__zero_heavy_1p5
DIVERSITY FINAL: catboost_log_level__zero_light_0p7
DIVERSITY FINAL: catboost_log_level__small_purchase_1p5


,run_name,base_model,kind,weight_mode,fold,fold_weight,weight,rmsle,rmsle_zero,rmsle_positive,rmsle_small_positive,rmsle_large_positive,best_round,threshold,audit_small_threshold,audit_large_threshold,seconds,cache_hit
0,xgboost_log_level__bag_s17,xgboost_log_level,column_bagging,none,nov_2025,0.25,0.25,1.734251,1.607876,1.826440,1.334072,2.346621,1465,NaN,36.449963,227.329593,14.192268,True
1,xgboost_log_level__bag_s17,xgboost_log_level,column_bagging,none,dec_2025,0.35,0.35,1.737564,1.715174,1.754740,1.300418,2.252863,1738,NaN,36.818668,230.002386,19.537112,True
2,xgboost_log_level__bag_s17,xgboost_log_level,column_bagging,none,jan_2026,0.40,0.40,1.677806,1.714370,1.646103,1.277739,2.163926,2717,NaN,36.978174,231.918842,32.774581,True
3,catboost_log_level__bag_s17,catboost_log_level,column_bagging,none,nov_2025,0.25,0.25,1.735538,1.605281,1.830383,1.328688,2.360330,7141,NaN,36.449963,227.329593,50.400682,False
4,catboost_log_level__bag_s17,catboost_log_level,column_bagging,none,dec_2025,0.35,0.35,1.739867,1.757320,1.726203,1.296678,2.209730,7371,NaN,36.818668,230.002386,60.976716,False


## 27.3 Ridge bagging direction

Ridge использует stable union frozen features активных CatBoost/XGBoost моделей. Категории проходят существующий fold-local empirical-Bayes encoder, после чего `SimpleImputer` и `StandardScaler` обучаются только на train. Alpha зафиксирован конфигом; это level model для `log1p(raw_target)`.


In [114]:
RIDGE_FEATURE_UNION = unique_keep_order(
    feature
    for model_name in MODEL_NAMES
    for feature in model_feature_order(MODEL_SPECS[model_name])[2]
)
RIDGE_FORBIDDEN_COLUMNS = {
    TARGET_COL, USER_COL, SNAPSHOT_COL, "fold", ANCHOR_COL,
}
RIDGE_FEATURE_UNION = [
    feature for feature in RIDGE_FEATURE_UNION
    if feature not in RIDGE_FORBIDDEN_COLUMNS
]


def ridge_feature_subset(seed: int) -> list[str]:
    size = min(
        len(RIDGE_FEATURE_UNION),
        max(MIN_STOCHASTIC_FEATURES, int(round(
            len(RIDGE_FEATURE_UNION) * RIDGE_FEATURE_FRACTION
        ))),
    )
    generator = np.random.default_rng(int(seed) + 700_001)
    return sorted(generator.choice(
        np.asarray(RIDGE_FEATURE_UNION, dtype=object), size=size, replace=False
    ).tolist())


RIDGE_REGISTRY = [
    {
        "run_name": f"ridge_log_level__bag_s{seed}",
        "kind": "ridge_bagging",
        "base_model": "ridge_log_level",
        "family": "ridge",
        "target_mode": "log_level",
        "seed": int(seed),
        "feature_fraction": RIDGE_FEATURE_FRACTION,
        "n_original_features": len(RIDGE_FEATURE_UNION),
        "n_selected_features": len(ridge_feature_subset(seed)),
        "features": ridge_feature_subset(seed),
        "model_params": {"alpha": RIDGE_ALPHA, "solver": "lsqr"},
        "weight_mode": "none",
    }
    for seed in RIDGE_SEEDS[:RIDGE_N_MEMBERS]
]


def ridge_specification(run):
    # Any non-CatBoost family activates the existing train-only category encoder.
    # Use the truthful family label: this path fits sklearn Ridge, never LightGBM.
    return {
        "name": run["run_name"],
        "family": "ridge",
        "target_mode": "log_level",
        "features": list(run["features"]),
        "params": {},
        "maturity": "exploratory",
    }


def fit_ridge_split(run, fold=None, final=False):
    cached = load_prediction_cache(run, fold, final)
    if cached is not None:
        cached["metadata"]["cache_hit"] = True
        return cached

    from sklearn.impute import SimpleImputer
    from sklearn.linear_model import Ridge
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler

    train_cutoffs = FINAL_TRAIN_CUTOFFS if final else fold["train_cutoffs"]
    valid_cutoff = PREDICTION_CUTOFF if final else fold["valid_cutoff"]
    matrices = materialize_model_pair(
        train_cutoffs, valid_cutoff, ridge_specification(run)
    )
    estimator = Pipeline([
        ("imputer", SimpleImputer(strategy="median", copy=False)),
        ("scaler", StandardScaler(copy=False)),
        ("ridge", Ridge(alpha=RIDGE_ALPHA, solver="lsqr", tol=1e-4)),
    ])
    started = time.perf_counter()
    estimator.fit(matrices["X_train"], matrices["y_train"])
    prediction_log = estimator.predict(matrices["X_valid"])
    prediction = np.clip(np.expm1(prediction_log), 0.0, None)
    if not np.isfinite(prediction).all():
        raise FloatingPointError(f"{run['run_name']}: non-finite Ridge prediction")
    metadata = {
        "run_name": run["run_name"],
        "fold": "final" if final else fold["name"],
        "best_round": 1,
        "alpha": RIDGE_ALPHA,
        "seconds": time.perf_counter() - started,
        "cache_hit": False,
    }
    result = {
        "user_id": matrices["valid_user_ids"],
        "prediction": prediction,
        "truth": matrices["y_valid_raw"],
        "metadata": metadata,
    }
    save_prediction_cache(
        run, fold, final, result["user_id"], prediction,
        result["truth"], metadata,
    )
    del matrices, estimator, prediction_log
    gc.collect()
    return result


RIDGE_FOLD_ROWS = []
if RUN_CV and ENABLE_RIDGE_BAGGING:
    for run in RIDGE_REGISTRY:
        full_prediction = np.full(len(OOF), np.nan, dtype=np.float64)
        print("\nRIDGE CV:", run["run_name"])
        for fold in CV_FOLDS:
            result = fit_ridge_split(run, fold=fold)
            hold = OOF["fold"].eq(fold["name"]).to_numpy()
            if not np.array_equal(OOF.loc[hold, USER_COL].to_numpy(), result["user_id"]):
                raise AssertionError(f"{run['run_name']}/{fold['name']}: alignment")
            full_prediction[hold] = result["prediction"]
            RIDGE_FOLD_ROWS.append({
                "run_name": run["run_name"], "fold": fold["name"],
                "weight": fold["weight"],
                "rmsle": rmsle(result["truth"], result["prediction"]),
                "alpha": RIDGE_ALPHA,
                "seconds": result["metadata"].get("seconds", 0.0),
                "cache_hit": result["metadata"].get("cache_hit", False),
            })
        if not np.isfinite(full_prediction).all():
            raise AssertionError(f"{run['run_name']}: incomplete OOF")
        OOF[run["run_name"]] = full_prediction

RIDGE_FOLD_METRICS = pd.DataFrame(RIDGE_FOLD_ROWS)

if RUN_FINAL_FIT and ENABLE_RIDGE_BAGGING:
    for run in RIDGE_REGISTRY:
        print("RIDGE FINAL:", run["run_name"])
        result = fit_ridge_split(run, final=True)
        if not np.array_equal(FINAL_PREDICTIONS[USER_COL].to_numpy(), result["user_id"]):
            raise AssertionError(f"{run['run_name']}: final alignment")
        FINAL_PREDICTIONS[run["run_name"]] = result["prediction"]

RIDGE_FOLD_METRICS.to_csv(STOCHASTIC_DIR / "ridge_member_metrics.csv", index=False)
display(RIDGE_FOLD_METRICS.head())



RIDGE CV: ridge_log_level__bag_s11

RIDGE CV: ridge_log_level__bag_s29

RIDGE CV: ridge_log_level__bag_s47

RIDGE CV: ridge_log_level__bag_s83

RIDGE CV: ridge_log_level__bag_s131

RIDGE CV: ridge_log_level__bag_s181

RIDGE CV: ridge_log_level__bag_s239
RIDGE FINAL: ridge_log_level__bag_s11
RIDGE FINAL: ridge_log_level__bag_s29
RIDGE FINAL: ridge_log_level__bag_s47
RIDGE FINAL: ridge_log_level__bag_s83
RIDGE FINAL: ridge_log_level__bag_s131
RIDGE FINAL: ridge_log_level__bag_s181
RIDGE FINAL: ridge_log_level__bag_s239


,run_name,fold,weight,rmsle,alpha,seconds,cache_hit
0,ridge_log_level__bag_s11,nov_2025,0.25,1.741981,100.0,10.141424,False
1,ridge_log_level__bag_s11,dec_2025,0.35,1.743970,100.0,15.157799,False
2,ridge_log_level__bag_s11,jan_2026,0.40,1.686047,100.0,23.025020,False
3,ridge_log_level__bag_s29,nov_2025,0.25,1.741783,100.0,9.779519,False
4,ridge_log_level__bag_s29,dec_2025,0.35,1.743449,100.0,14.700913,False


## 27.4 Level 1: semantic directions

Members усредняются в log-space с равными весами. Веса отдельных members не оптимизируются. Реализованы обе заранее определённые иерархии: A — отдельные stochastic directions активных моделей; B — одно stochastic direction на активное семейство.


In [115]:
DIRECTION_MEMBERS = {}
DIRECTION_UNDERLYING_COUNT = {}


def register_direction(name: str, members: Sequence[str]):
    members = list(members)
    if not members:
        return
    missing_oof = [member for member in members if member not in OOF.columns]
    if missing_oof:
        raise KeyError(f"{name}: missing OOF members {missing_oof}")
    OOF[name] = log_mean(OOF[members].to_numpy(dtype=np.float64))
    if FINAL_PREDICTIONS is not None:
        missing_final = [m for m in members if m not in FINAL_PREDICTIONS.columns]
        if missing_final:
            raise KeyError(f"{name}: missing final members {missing_final}")
        FINAL_PREDICTIONS[name] = log_mean(
            FINAL_PREDICTIONS[members].to_numpy(dtype=np.float64)
        )
    DIRECTION_MEMBERS[name] = members
    DIRECTION_UNDERLYING_COUNT[name] = len(members)


LEVEL_MODELS = [name for name in MODEL_NAMES if MODEL_SPECS[name]["target_mode"] == "log_level"]
DIFF_MODELS = [name for name in MODEL_NAMES if MODEL_SPECS[name]["target_mode"] == "log_diff"]
register_direction("BASE_LEVEL", LEVEL_MODELS)
register_direction("BASE_DIFF", DIFF_MODELS)

for base_model in MODEL_NAMES:
    members = [
        run["run_name"] for run in STOCHASTIC_REGISTRY
        if run["base_model"] == base_model
    ]
    register_direction(f"BAG_{base_model.upper()}", members)

specialist_members = [
    run["run_name"] for run in SPECIALIST_REGISTRY
    if run["weight_mode"] not in {"baseline_control"}
]
register_direction("TARGET_WEIGHT_SPECIALISTS", specialist_members)
register_direction("RIDGE", [run["run_name"] for run in RIDGE_REGISTRY])

# Hierarchy B: combine level+diff bags within each active boosting family.
for family in sorted({MODEL_SPECS[name]["family"] for name in MODEL_NAMES}):
    family_directions = [
        f"BAG_{name.upper()}" for name in MODEL_NAMES
        if MODEL_SPECS[name]["family"] == family
        and f"BAG_{name.upper()}" in DIRECTION_MEMBERS
    ]
    register_direction(f"{family.upper()}_STOCHASTIC", family_directions)
    DIRECTION_UNDERLYING_COUNT[f"{family.upper()}_STOCHASTIC"] = sum(
        DIRECTION_UNDERLYING_COUNT[x] for x in family_directions
    )

HIERARCHY_A_DIRECTIONS = [
    name for name in [
        "BASE_LEVEL", "BASE_DIFF",
        *[f"BAG_{model.upper()}" for model in MODEL_NAMES],
        "TARGET_WEIGHT_SPECIALISTS", "RIDGE",
    ]
    if name in DIRECTION_MEMBERS
]
HIERARCHY_B_DIRECTIONS = [
    name for name in [
        "BASE_LEVEL", "BASE_DIFF", "XGBOOST_STOCHASTIC",
        "CATBOOST_STOCHASTIC",
        "TARGET_WEIGHT_SPECIALISTS", "RIDGE",
    ]
    if name in DIRECTION_MEMBERS
]

OOF["CURRENT_BEST_ORIGINAL"] = BLENDS[BASELINE_BEST_NAME].oof_pred
if FINAL_PREDICTIONS is not None:
    FINAL_PREDICTIONS["CURRENT_BEST_ORIGINAL"] = BLENDS[BASELINE_BEST_NAME].test_pred

print("Hierarchy A:", HIERARCHY_A_DIRECTIONS)
print("Hierarchy B:", HIERARCHY_B_DIRECTIONS)


Hierarchy A: ['BASE_LEVEL', 'BASE_DIFF', 'BAG_XGBOOST_LOG_LEVEL', 'BAG_CATBOOST_LOG_LEVEL', 'BAG_CATBOOST_LOG_DIFF', 'BAG_XGBOOST_LOG_DIFF', 'TARGET_WEIGHT_SPECIALISTS', 'RIDGE']
Hierarchy B: ['BASE_LEVEL', 'BASE_DIFF', 'XGBOOST_STOCHASTIC', 'CATBOOST_STOCHASTIC', 'TARGET_WEIGHT_SPECIALISTS', 'RIDGE']


## 27.5 Level 2: honest hierarchical blending

Между directions переиспользуется исходный constrained cross-fitted optimizer: неотрицательные веса, сумма 1, max weight и L2 shrinkage. Equal-weight варианты служат обязательным контролем.


In [116]:
HIERARCHICAL_RESULTS = {}


def add_equal_hierarchy(name: str, directions: Sequence[str]):
    oof_prediction = log_mean(OOF[list(directions)].to_numpy(dtype=np.float64))
    test_prediction = None
    if FINAL_PREDICTIONS is not None:
        test_prediction = log_mean(
            FINAL_PREDICTIONS[list(directions)].to_numpy(dtype=np.float64)
        )
    add_blend(
        name, oof_prediction, test_prediction,
        "equal_hierarchical_directions", complexity=2,
        selection_safe=True,
        metadata={"directions": list(directions), "evaluation_type": "standalone_oof"},
    )
    HIERARCHICAL_RESULTS[name] = BLENDS[name]


def add_optimized_hierarchy(name: str, directions: Sequence[str]):
    oof_prediction, test_prediction, metadata = crossfit_optimized_blend(
        list(directions), "log"
    )
    metadata.update({
        "directions": list(directions),
        "evaluation_type": "cross_fitted_oof",
    })
    add_blend(
        name, oof_prediction, test_prediction,
        "crossfit_hierarchical_directions", complexity=4,
        selection_safe=True, metadata=metadata,
    )
    HIERARCHICAL_RESULTS[name] = BLENDS[name]


if RUN_CV and ENABLE_HIERARCHICAL_BLEND:
    add_equal_hierarchy("stochastic_equal_A", HIERARCHY_A_DIRECTIONS)
    add_equal_hierarchy("stochastic_equal_B", HIERARCHY_B_DIRECTIONS)
    add_optimized_hierarchy("stochastic_hierarchical_A", HIERARCHY_A_DIRECTIONS)
    add_optimized_hierarchy("stochastic_hierarchical_B", HIERARCHY_B_DIRECTIONS)

PRIMARY_EQUAL_NAME = f"stochastic_equal_{HIERARCHY_PRIMARY}"
PRIMARY_HIERARCHICAL_NAME = f"stochastic_hierarchical_{HIERARCHY_PRIMARY}"

pd.DataFrame([
    {
        "candidate": name,
        "weighted_rmsle": weighted_oof_rmsle(OOF, candidate.oof_pred),
        "evaluation_type": candidate.metadata.get("evaluation_type"),
    }
    for name, candidate in HIERARCHICAL_RESULTS.items()
]).sort_values("weighted_rmsle")


,candidate,weighted_rmsle,evaluation_type
0,stochastic_equal_A,1.711512,standalone_oof
2,stochastic_hierarchical_A,1.711526,cross_fitted_oof
3,stochastic_hierarchical_B,1.711617,cross_fitted_oof
1,stochastic_equal_B,1.711683,standalone_oof


## 27.6 Cross-fitted affine log-space calibration

Для каждого held-out temporal fold параметры `a`, `b` и `mu` обучаются только на остальных folds. Значения Polar не используются. Диапазоны не расширяются автоматически; попадание `b` на boundary даёт предупреждение.


In [117]:
AFFINE_REPORT_ROWS = []


def apply_affine_log(prediction, mu: float, shift: float, spread: float):
    z = np.log1p(np.clip(np.asarray(prediction, dtype=np.float64), 0.0, None))
    calibrated = np.expm1(mu + shift + spread * (z - mu))
    return np.clip(calibrated, 0.0, None)


def fit_affine_log(frame: pd.DataFrame, prediction, regularized: bool = True):
    prediction = np.asarray(prediction, dtype=np.float64)
    z = np.log1p(np.clip(prediction, 0.0, None))
    row_weights = temporal_row_weights(frame)
    mu = float(np.sum(row_weights * z) / max(row_weights.sum(), 1e-15))

    def objective(params):
        shift, spread = map(float, params)
        candidate = apply_affine_log(prediction, mu, shift, spread)
        value = score_on_rows(frame, candidate)
        if regularized:
            value += AFFINE_LAMBDA_SHIFT * shift ** 2
            value += AFFINE_LAMBDA_SPREAD * (spread - 1.0) ** 2
        return value

    result = minimize(
        objective, np.array([0.0, 1.0]), method="L-BFGS-B",
        bounds=[AFFINE_SHIFT_BOUNDS, AFFINE_SPREAD_BOUNDS],
        options={"maxiter": 250, "ftol": 1e-12},
    )
    if not result.success:
        raise RuntimeError(f"affine optimization failed: {result.message}")
    return float(result.x[0]), float(result.x[1]), mu


def crossfit_affine(candidate: BlendCandidate, regularized: bool = True):
    output = np.full(len(OOF), np.nan, dtype=np.float64)
    fold_parameters = {}
    mode = "regularized" if regularized else "unregularized"
    for fold in CV_FOLDS:
        hold = OOF["fold"].eq(fold["name"]).to_numpy()
        train = OOF.loc[~hold].reset_index(drop=True)
        shift, spread, mu = fit_affine_log(
            train, candidate.oof_pred[~hold], regularized=regularized
        )
        output[hold] = apply_affine_log(
            candidate.oof_pred[hold], mu, shift, spread
        )
        before = candidate.oof_pred[hold]
        after = output[hold]
        before_score = rmsle(OOF.loc[hold, TARGET_COL], before)
        after_score = rmsle(OOF.loc[hold, TARGET_COL], after)
        z_before = np.log1p(np.clip(before, 0.0, None))
        z_after = np.log1p(np.clip(after, 0.0, None))
        AFFINE_REPORT_ROWS.append({
            "candidate": candidate.name, "mode": mode, "fold": fold["name"],
            "a": shift, "b": spread, "calibration_train_mu": mu,
            "log_std_before": float(z_before.std()),
            "log_std_after": float(z_after.std()),
            "log_mean_before": float(z_before.mean()),
            "log_mean_after": float(z_after.mean()),
            "rmsle_before": before_score, "rmsle_after": after_score,
            "improvement": before_score - after_score,
            "spread_on_boundary": bool(
                np.isclose(spread, AFFINE_SPREAD_BOUNDS[0], atol=1e-4)
                or np.isclose(spread, AFFINE_SPREAD_BOUNDS[1], atol=1e-4)
            ),
        })
        fold_parameters[fold["name"]] = {"a": shift, "b": spread, "mu": mu}

    full_shift, full_spread, full_mu = fit_affine_log(
        OOF, candidate.oof_pred, regularized=regularized
    )
    test = None
    if candidate.test_pred is not None:
        test = apply_affine_log(
            candidate.test_pred, full_mu, full_shift, full_spread
        )
    metadata = {
        "base": candidate.name,
        "evaluation_type": "cross_fitted_oof",
        "regularized": regularized,
        "crossfit_parameters": fold_parameters,
        "full_oof_parameters_for_test": {
            "a": full_shift, "b": full_spread, "mu": full_mu,
        },
    }
    return output, test, metadata


AFFINE_BASES = {
    "current_best_original": BLENDS[BASELINE_BEST_NAME],
    "stochastic_equal": BLENDS[PRIMARY_EQUAL_NAME],
    "stochastic_hierarchical": BLENDS[PRIMARY_HIERARCHICAL_NAME],
}
AFFINE_CANDIDATES = {}

if RUN_CV and ENABLE_AFFINE_CALIBRATION:
    for label, candidate in AFFINE_BASES.items():
        # Unregularized is diagnostic only and is never promoted automatically.
        crossfit_affine(candidate, regularized=False)
        oof_pred, test_pred, metadata = crossfit_affine(candidate, regularized=True)
        name = f"{label}__affine_cf"
        add_blend(
            name, oof_pred, test_pred, "crossfit_affine_log",
            complexity=6, selection_safe=True, metadata=metadata,
        )
        AFFINE_CANDIDATES[label] = BLENDS[name]

AFFINE_CALIBRATION_REPORT = pd.DataFrame(AFFINE_REPORT_ROWS)
AFFINE_CALIBRATION_REPORT.to_csv(
    STOCHASTIC_DIR / "affine_calibration_report.csv", index=False
)
if not AFFINE_CALIBRATION_REPORT.empty and AFFINE_CALIBRATION_REPORT["spread_on_boundary"].any():
    print("WARNING: affine b reached configured boundary; range was not expanded automatically.")
display(AFFINE_CALIBRATION_REPORT)


,candidate,mode,fold,a,b,calibration_train_mu,log_std_before,log_std_after,log_mean_before,log_mean_after,rmsle_before,rmsle_after,improvement,spread_on_boundary
0,optuna_softmax_logmean,unregularized,nov_2025,-0.110683,0.996020,2.433298,1.601852,1.595476,2.345616,2.235283,1.732033,1.740112,-0.008079,False
1,optuna_softmax_logmean,unregularized,dec_2025,-0.073998,0.992248,2.381382,1.594751,1.582388,2.467083,2.392421,1.736008,1.735527,0.000481,False
2,optuna_softmax_logmean,unregularized,jan_2026,0.000790,0.999599,2.416472,1.582509,1.581875,2.403736,2.404531,1.676276,1.676347,-0.000071,False
3,optuna_softmax_logmean,regularized,nov_2025,-0.103615,0.996112,2.433298,1.601852,1.595623,2.345616,2.242343,1.732033,1.739387,-0.007354,False
4,optuna_softmax_logmean,regularized,dec_2025,-0.069273,0.992451,2.381382,1.594751,1.582712,2.467083,2.397162,1.736008,1.735463,0.000545,False
5,optuna_softmax_logmean,regularized,jan_2026,0.000734,0.999611,2.416472,1.582509,1.581894,2.403736,2.404475,1.676276,1.676342,-0.000065,False
6,stochastic_equal_A,unregularized,nov_2025,-0.110687,0.998759,2.433507,1.597311,1.595329,2.340562,2.229990,1.732631,1.741049,-0.008418,False
7,stochastic_equal_A,unregularized,dec_2025,-0.066983,0.996090,2.374418,1.593398,1.587169,2.476856,2.409473,1.736720,1.735731,0.000988,False
8,stochastic_equal_A,unregularized,jan_2026,-0.002796,1.000646,2.420067,1.574512,1.575529,2.395577,2.392766,1.676257,1.676007,0.000249,False
9,stochastic_equal_A,regularized,nov_2025,-0.103627,0.998788,2.433507,1.597311,1.595374,2.340562,2.237048,1.732631,1.740303,-0.007672,False


## 27.7 Diversity diagnostics, feature frequency и ablation

Эта ячейка измеряет не только standalone RMSLE, но и prediction/error correlations, fold stability и честное добавление каждого direction к текущему baseline через двухкомпонентный cross-fitted blend.


In [118]:
def prediction_error_correlation(pred_a, pred_b, truth):
    za = np.log1p(np.clip(np.asarray(pred_a), 0.0, None))
    zb = np.log1p(np.clip(np.asarray(pred_b), 0.0, None))
    zy = np.log1p(np.clip(np.asarray(truth), 0.0, None))
    return float(np.corrcoef(za - zy, zb - zy)[0, 1])


def fold_score_dict(prediction) -> dict[str, float]:
    prediction = np.asarray(prediction, dtype=np.float64)
    return {
        fold["name"]: rmsle(
            OOF.loc[OOF["fold"].eq(fold["name"]), TARGET_COL],
            prediction[OOF["fold"].eq(fold["name"]).to_numpy()],
        )
        for fold in CV_FOLDS
    }


STOCHASTIC_MEMBER_ROWS = []
for run in STOCHASTIC_REGISTRY:
    pred = OOF[run["run_name"]].to_numpy(dtype=np.float64)
    original = OOF[run["base_model"]].to_numpy(dtype=np.float64)
    scores = fold_score_dict(pred)
    STOCHASTIC_MEMBER_ROWS.append({
        "member": run["run_name"], "base_model": run["base_model"],
        "seed": run["seed"], "n_features": run["n_selected_features"],
        "standalone_rmsle": weighted_oof_rmsle(OOF, pred),
        "delta_vs_original": weighted_oof_rmsle(OOF, pred)
            - BASELINE_METRIC_SNAPSHOT[run["base_model"]],
        "prediction_corr_with_original": float(np.corrcoef(pred, original)[0, 1]),
        "error_corr_with_original": prediction_error_correlation(
            pred, original, OOF[TARGET_COL]
        ),
        "mean_prediction": float(pred.mean()),
        "std_log_prediction": float(np.log1p(pred).std()),
        **{f"{fold}_rmsle": score for fold, score in scores.items()},
    })
STOCHASTIC_MEMBER_METRICS = pd.DataFrame(STOCHASTIC_MEMBER_ROWS)
STOCHASTIC_MEMBER_METRICS.to_csv(
    STOCHASTIC_DIR / "stochastic_member_metrics.csv", index=False
)


feature_rows = []
for base_model in MODEL_NAMES:
    runs = [run for run in STOCHASTIC_REGISTRY if run["base_model"] == base_model]
    original = model_feature_order(MODEL_SPECS[base_model])[2]
    for feature in original:
        count = sum(feature in run["features"] for run in runs)
        feature_rows.append({
            "base_model": base_model, "feature": feature,
            "members_selected": count, "n_members": len(runs),
            "fraction_members": count / max(len(runs), 1),
        })
STOCHASTIC_FEATURE_REGISTRY = pd.DataFrame(feature_rows)
STOCHASTIC_FEATURE_REGISTRY.to_csv(
    STOCHASTIC_DIR / "stochastic_feature_registry.csv", index=False
)


correlation_rows = []
direction_rows = []
truth = OOF[TARGET_COL].to_numpy(dtype=np.float64)
for direction, members in DIRECTION_MEMBERS.items():
    pred = OOF[direction].to_numpy(dtype=np.float64)
    arithmetic_pred = raw_mean(OOF[members].to_numpy(dtype=np.float64))
    scores = fold_score_dict(pred)
    pair_error = []
    pair_prediction = []
    for left, right in itertools.combinations(members, 2):
        lp = OOF[left].to_numpy(dtype=np.float64)
        rp = OOF[right].to_numpy(dtype=np.float64)
        pc = float(np.corrcoef(lp, rp)[0, 1])
        ec = prediction_error_correlation(lp, rp, truth)
        pair_prediction.append(pc); pair_error.append(ec)
        correlation_rows.append({
            "direction": direction, "left": left, "right": right,
            "prediction_correlation": pc, "error_correlation": ec,
        })
    direction_rows.append({
        "direction": direction,
        "weighted_rmsle": weighted_oof_rmsle(OOF, pred),
        "log_mean_rmsle": weighted_oof_rmsle(OOF, pred),
        "arithmetic_mean_rmsle": weighted_oof_rmsle(OOF, arithmetic_pred),
        "arithmetic_minus_log": weighted_oof_rmsle(OOF, arithmetic_pred)
            - weighted_oof_rmsle(OOF, pred),
        "preferred_aggregation": "equal_log_mean",
        "worst_fold": max(scores.values()),
        "fold_std": float(np.std(list(scores.values()))),
        "n_underlying_models": DIRECTION_UNDERLYING_COUNT.get(direction, len(members)),
        "mean_pairwise_prediction_corr": float(np.mean(pair_prediction)) if pair_prediction else np.nan,
        "mean_pairwise_error_corr": float(np.mean(pair_error)) if pair_error else np.nan,
        "min_pairwise_error_corr": float(np.min(pair_error)) if pair_error else np.nan,
        "max_pairwise_error_corr": float(np.max(pair_error)) if pair_error else np.nan,
        **{f"{fold}_rmsle": score for fold, score in scores.items()},
    })

STOCHASTIC_ERROR_CORRELATIONS = pd.DataFrame(correlation_rows)
STOCHASTIC_DIRECTION_METRICS = pd.DataFrame(direction_rows)
STOCHASTIC_ERROR_CORRELATIONS.to_csv(
    STOCHASTIC_DIR / "stochastic_error_correlations.csv", index=False
)
STOCHASTIC_DIRECTION_METRICS.to_csv(
    STOCHASTIC_DIR / "stochastic_direction_metrics.csv", index=False
)

# One-row specialist audit: segment profile, correlations and honest addition
# to the current baseline. Fold-local thresholds remain in the companion file.
TARGET_SPECIALIST_SUMMARY_ROWS = []
specialist_fold_table = BOOSTING_FOLD_METRICS.loc[
    BOOSTING_FOLD_METRICS["kind"].eq("target_specialist")
].copy()
specialist_fold_table.to_csv(
    STOCHASTIC_DIR / "target_specialist_fold_metrics.csv", index=False
)
for run in SPECIALIST_REGISTRY:
    name = run["run_name"]
    pred = OOF[name].to_numpy(dtype=np.float64)
    fold_rows = specialist_fold_table.loc[
        specialist_fold_table["run_name"].eq(name)
    ]
    addition_pred, _, _ = crossfit_optimized_blend(
        ["CURRENT_BEST_ORIGINAL", name], "log"
    )
    TARGET_SPECIALIST_SUMMARY_ROWS.append({
        "specialist": name, "weight_mode": run["weight_mode"],
        "weighted_rmsle": weighted_oof_rmsle(OOF, pred),
        "weighted_zero_rmsle": float(np.sum(
            fold_rows["weight"] * fold_rows["rmsle_zero"]
        )),
        "weighted_positive_rmsle": float(np.sum(
            fold_rows["weight"] * fold_rows["rmsle_positive"]
        )),
        "weighted_small_positive_rmsle": float(np.sum(
            fold_rows["weight"] * fold_rows["rmsle_small_positive"]
        )),
        "weighted_large_positive_rmsle": float(np.sum(
            fold_rows["weight"] * fold_rows["rmsle_large_positive"]
        )),
        "prediction_corr_with_base": float(np.corrcoef(
            pred, OOF[SPECIALIST_BASE_MODEL].to_numpy(dtype=np.float64)
        )[0, 1]),
        "error_corr_with_base": prediction_error_correlation(
            pred, OOF[SPECIALIST_BASE_MODEL], truth
        ),
        "honest_addition_delta_vs_current_best": weighted_oof_rmsle(
            OOF, addition_pred
        ) - BASELINE_BEST_SCORE,
        "fold_thresholds_json": json.dumps(dict(zip(
            fold_rows["fold"], fold_rows["threshold"]
        )), default=str),
    })
TARGET_SPECIALIST_METRICS = pd.DataFrame(TARGET_SPECIALIST_SUMMARY_ROWS)
TARGET_SPECIALIST_METRICS.to_csv(
    STOCHASTIC_DIR / "target_specialist_metrics.csv", index=False
)

# Standalone Ridge summaries (fold table remains available in memory as
# RIDGE_FOLD_METRICS); the persisted report is one row per member.
RIDGE_MEMBER_ROWS = []
for run in RIDGE_REGISTRY:
    pred = OOF[run["run_name"]].to_numpy(dtype=np.float64)
    scores = fold_score_dict(pred)
    RIDGE_MEMBER_ROWS.append({
        "member": run["run_name"], "seed": run["seed"],
        "alpha": RIDGE_ALPHA, "n_features": run["n_selected_features"],
        "weighted_rmsle": weighted_oof_rmsle(OOF, pred),
        "prediction_corr_with_baseline": float(np.corrcoef(
            pred, BLENDS[BASELINE_BEST_NAME].oof_pred
        )[0, 1]),
        "error_corr_with_baseline": prediction_error_correlation(
            pred, BLENDS[BASELINE_BEST_NAME].oof_pred, truth
        ),
        **{f"{fold}_rmsle": score for fold, score in scores.items()},
    })
RIDGE_MEMBER_METRICS = pd.DataFrame(RIDGE_MEMBER_ROWS)
RIDGE_MEMBER_METRICS.to_csv(
    STOCHASTIC_DIR / "ridge_member_metrics.csv", index=False
)


def ablation_row(name, prediction, n_directions, effective_models, evaluation_type, stable):
    scores = fold_score_dict(prediction)
    total = weighted_oof_rmsle(OOF, prediction)
    return {
        "candidate": name, "weighted_rmsle": total,
        "delta_vs_baseline": total - BASELINE_BEST_SCORE,
        "Nov RMSLE": scores["nov_2025"],
        "Dec RMSLE": scores["dec_2025"],
        "Jan RMSLE": scores["jan_2026"],
        "worst_fold": max(scores.values()),
        "fold_std": float(np.std(list(scores.values()))),
        "n_directions": n_directions,
        "effective_number_of_underlying_models": effective_models,
        "evaluation_type": evaluation_type,
        "stable / exploratory": stable,
    }


ABLATION_ROWS = [ablation_row(
    "current_best_original", BLENDS[BASELINE_BEST_NAME].oof_pred,
    1, len(MODEL_NAMES), "cross_fitted_oof", "stable",
)]

# Honest held-out weights for baseline + each direction.
for direction in HIERARCHY_A_DIRECTIONS:
    oof_pred, _, metadata = crossfit_optimized_blend(
        ["CURRENT_BEST_ORIGINAL", direction], "log"
    )
    scores = fold_score_dict(oof_pred)
    base_scores = fold_score_dict(BLENDS[BASELINE_BEST_NAME].oof_pred)
    stable = all(scores[f] <= base_scores[f] + 0.005 for f in scores)
    ABLATION_ROWS.append(ablation_row(
        f"+ {direction}", oof_pred, 2,
        len(MODEL_NAMES) + DIRECTION_UNDERLYING_COUNT.get(direction, 1),
        "cross_fitted_oof", "stable" if stable else "exploratory",
    ))

for name in [PRIMARY_EQUAL_NAME, PRIMARY_HIERARCHICAL_NAME]:
    candidate = BLENDS[name]
    ABLATION_ROWS.append(ablation_row(
        name, candidate.oof_pred,
        len(candidate.metadata["directions"]),
        sum(DIRECTION_UNDERLYING_COUNT.get(x, 1) for x in candidate.metadata["directions"]),
        candidate.metadata["evaluation_type"], "stable",
    ))

affine_primary = AFFINE_CANDIDATES["stochastic_hierarchical"]
ABLATION_ROWS.append(ablation_row(
    f"{PRIMARY_HIERARCHICAL_NAME} + affine",
    affine_primary.oof_pred,
    len(BLENDS[PRIMARY_HIERARCHICAL_NAME].metadata["directions"]),
    sum(DIRECTION_UNDERLYING_COUNT.get(x, 1)
        for x in BLENDS[PRIMARY_HIERARCHICAL_NAME].metadata["directions"]),
    "cross_fitted_oof", "stable",
))

STOCHASTIC_ABLATION = pd.DataFrame(ABLATION_ROWS).sort_values("weighted_rmsle")
STOCHASTIC_ABLATION.to_csv(
    STOCHASTIC_DIR / "stochastic_ablation.csv", index=False
)

primary_meta = BLENDS[PRIMARY_HIERARCHICAL_NAME].metadata
full_weights = primary_meta["full_oof_weights_for_test"]
HIERARCHICAL_BLEND_WEIGHTS = pd.DataFrame([
    {
        "direction": direction,
        "weight": full_weights[direction],
        "standalone_rmsle": weighted_oof_rmsle(OOF, OOF[direction]),
        "error_corr_with_baseline": prediction_error_correlation(
            OOF[direction], BLENDS[BASELINE_BEST_NAME].oof_pred, truth
        ),
        "underlying_models": DIRECTION_UNDERLYING_COUNT.get(direction, 1),
    }
    for direction in primary_meta["directions"]
]).sort_values("weight", ascending=False)
HIERARCHICAL_BLEND_WEIGHTS.to_csv(
    STOCHASTIC_DIR / "hierarchical_blend_weights.csv", index=False
)

display(STOCHASTIC_MEMBER_METRICS)
display(STOCHASTIC_DIRECTION_METRICS.sort_values("weighted_rmsle"))
display(STOCHASTIC_ABLATION)
display(HIERARCHICAL_BLEND_WEIGHTS)


,member,base_model,seed,n_features,standalone_rmsle,delta_vs_original,prediction_corr_with_original,error_corr_with_original,mean_prediction,std_log_prediction,nov_2025_rmsle,dec_2025_rmsle,jan_2026_rmsle
0,xgboost_log_level__bag_s17,xgboost_log_level,17,36,1.712832,0.000690,0.998434,0.999290,42.301914,1.595391,1.734251,1.737564,1.677806
1,catboost_log_level__bag_s17,catboost_log_level,17,26,1.714558,0.000792,0.994022,0.999440,42.174372,1.590352,1.735538,1.739867,1.679301
2,catboost_log_diff__bag_s17,catboost_log_diff,17,43,1.715567,0.000642,0.972723,0.997212,41.506988,1.583170,1.735316,1.742880,1.679326
3,xgboost_log_diff__bag_s17,xgboost_log_diff,17,32,1.716088,0.002437,0.997545,0.999225,44.139539,1.600128,1.735252,1.740524,1.682729


,direction,weighted_rmsle,log_mean_rmsle,arithmetic_mean_rmsle,arithmetic_minus_log,preferred_aggregation,worst_fold,fold_std,n_underlying_models,mean_pairwise_prediction_corr,mean_pairwise_error_corr,min_pairwise_error_corr,max_pairwise_error_corr,nov_2025_rmsle,dec_2025_rmsle,jan_2026_rmsle
0,BASE_LEVEL,1.711901,1.711901,1.711993,0.000091,equal_log_mean,1.736677,0.027304,2,0.975372,0.997590,0.997590,0.997590,1.733028,1.736677,1.677018
1,BASE_DIFF,1.712346,1.712346,1.712522,0.000177,equal_log_mean,1.737753,0.027325,2,0.965007,0.995665,0.995665,0.995665,1.732668,1.737753,1.677412
8,CATBOOST_STOCHASTIC,1.712516,1.712516,1.712637,0.000121,equal_log_mean,1.739501,0.028774,2,0.927508,0.994379,0.994379,0.994379,1.733598,1.739501,1.675726
9,XGBOOST_STOCHASTIC,1.712649,1.712649,1.712829,0.000180,equal_log_mean,1.737162,0.026760,2,0.951209,0.996046,0.996046,0.996046,1.733050,1.737162,1.678451
2,BAG_XGBOOST_LOG_LEVEL,1.712832,1.712832,1.712832,0.000000,equal_log_mean,1.737564,0.027423,1,NaN,NaN,NaN,NaN,1.734251,1.737564,1.677806
6,TARGET_WEIGHT_SPECIALISTS,1.714204,1.714204,1.715321,0.001117,equal_log_mean,1.739091,0.027386,3,0.997970,0.995595,0.992549,0.998892,1.735345,1.739091,1.679214
3,BAG_CATBOOST_LOG_LEVEL,1.714558,1.714558,1.714558,0.000000,equal_log_mean,1.739867,0.027588,1,NaN,NaN,NaN,NaN,1.735538,1.739867,1.679301
4,BAG_CATBOOST_LOG_DIFF,1.715567,1.715567,1.715567,0.000000,equal_log_mean,1.742880,0.028345,1,NaN,NaN,NaN,NaN,1.735316,1.742880,1.679326
5,BAG_XGBOOST_LOG_DIFF,1.716088,1.716088,1.716088,0.000000,equal_log_mean,1.740524,0.026091,1,NaN,NaN,NaN,NaN,1.735252,1.740524,1.682729
7,RIDGE,1.721680,1.721680,1.724743,0.003062,equal_log_mean,1.742658,0.023741,7,0.708555,0.993738,0.982860,0.998300,1.740786,1.742658,1.691385


,candidate,weighted_rmsle,delta_vs_baseline,Nov RMSLE,Dec RMSLE,Jan RMSLE,worst_fold,fold_std,n_directions,effective_number_of_underlying_models,evaluation_type,stable / exploratory
5,+ BAG_CATBOOST_LOG_DIFF,1.711063,-0.000058,1.732416,1.737671,1.674436,1.737671,0.028651,2,5,cross_fitted_oof,stable
0,current_best_original,1.711121,0.000000,1.732033,1.736008,1.676276,1.736008,0.027269,1,4,cross_fitted_oof,stable
2,+ BASE_DIFF,1.711353,0.000231,1.732001,1.736457,1.676481,1.736457,0.027283,2,6,cross_fitted_oof,stable
1,+ BASE_LEVEL,1.711362,0.000240,1.732377,1.736183,1.676508,1.736183,0.027279,2,6,cross_fitted_oof,stable
3,+ BAG_XGBOOST_LOG_LEVEL,1.711497,0.000376,1.732689,1.736288,1.676561,1.736288,0.027347,2,5,cross_fitted_oof,stable
9,stochastic_equal_A,1.711512,0.000391,1.732631,1.736720,1.676257,1.736720,0.027589,8,18,standalone_oof,stable
10,stochastic_hierarchical_A,1.711526,0.000404,1.732456,1.736911,1.676232,1.736911,0.027614,8,18,cross_fitted_oof,stable
7,+ TARGET_WEIGHT_SPECIALISTS,1.711796,0.000674,1.732856,1.736689,1.676851,1.736689,0.027350,2,7,cross_fitted_oof,stable
4,+ BAG_CATBOOST_LOG_LEVEL,1.711936,0.000814,1.732924,1.736997,1.676890,1.736997,0.027425,2,5,cross_fitted_oof,stable
6,+ BAG_XGBOOST_LOG_DIFF,1.712171,0.001049,1.732392,1.736840,1.677947,1.736840,0.026775,2,5,cross_fitted_oof,stable


,direction,weight,standalone_rmsle,error_corr_with_baseline,underlying_models
4,BAG_CATBOOST_LOG_DIFF,0.194869,1.715567,0.995670,1
2,BAG_XGBOOST_LOG_LEVEL,0.161495,1.712832,0.999081,1
0,BASE_LEVEL,0.153380,1.711901,0.999692,2
1,BASE_DIFF,0.148387,1.712346,0.999221,2
6,TARGET_WEIGHT_SPECIALISTS,0.098463,1.714204,0.998464,3
5,BAG_XGBOOST_LOG_DIFF,0.093117,1.716088,0.997790,1
7,RIDGE,0.075799,1.721680,0.994197,7
3,BAG_CATBOOST_LOG_LEVEL,0.074491,1.714558,0.998478,1


## 27.8 Submissions, финальные проверки и автоматический summary

Ни один кандидат не выбирается автоматически только по общей OOF. Основные A/B варианты определены конфигом до просмотра результата. Все проверки ниже выполняются перед сохранением manifest.


In [119]:
SUBMISSION_CANDIDATES = {
    "submission_original_best.csv": BLENDS[BASELINE_BEST_NAME],
    "submission_stochastic_equal.csv": BLENDS[PRIMARY_EQUAL_NAME],
    "submission_stochastic_hierarchical.csv": BLENDS[PRIMARY_HIERARCHICAL_NAME],
    "submission_stochastic_hierarchical_affine.csv": AFFINE_CANDIDATES[
        "stochastic_hierarchical"
    ],
}

RECOMMENDED_SUBMISSIONS = []
if RUN_FINAL_FIT:
    for filename, candidate in SUBMISSION_CANDIDATES.items():
        if candidate.test_pred is None:
            raise RuntimeError(f"{filename}: final prediction unavailable")
        submission = build_submission_from_prediction(candidate.test_pred)
        path = STOCHASTIC_SUBMISSION_DIR / filename
        submission.to_csv(path, index=False)
        RECOMMENDED_SUBMISSIONS.append(str(path))


FINAL_STOCHASTIC_CHECKS = []


def stochastic_check(name: str, condition: bool, detail: str = ""):
    FINAL_STOCHASTIC_CHECKS.append({
        "check": name, "ok": bool(condition), "detail": detail,
    })
    if not condition:
        raise AssertionError(f"{name}: {detail}")


keys = ["fold", SNAPSHOT_COL, USER_COL]
stochastic_check("OOF key uniqueness", not OOF[keys].duplicated().any())
stochastic_check("OOF expected rows", len(OOF) == len(user_universe) * len(CV_FOLDS))

new_prediction_columns = [
    run["run_name"] for run in STOCHASTIC_REGISTRY + SPECIALIST_REGISTRY + RIDGE_REGISTRY
    if run["run_name"] in OOF.columns
] + list(DIRECTION_MEMBERS)
stochastic_check("no missing OOF", not OOF[new_prediction_columns].isna().any().any())
stochastic_check("finite OOF", np.isfinite(OOF[new_prediction_columns].to_numpy()).all())
stochastic_check("nonnegative raw OOF", (OOF[new_prediction_columns].to_numpy() >= 0).all())

for run in STOCHASTIC_REGISTRY:
    original = set(model_feature_order(MODEL_SPECS[run["base_model"]])[2])
    stochastic_check(
        f"subset of frozen features: {run['run_name']}",
        set(run["features"]).issubset(original),
    )
    stochastic_check(
        f"target absent: {run['run_name']}",
        not (set(run["features"]) & RIDGE_FORBIDDEN_COLUMNS),
    )
    expected = frozen_feature_subset(
        run["base_model"], run["seed"], run["feature_fraction"],
        MIN_STOCHASTIC_FEATURES,
    )
    stochastic_check(
        f"fixed subset across folds/final: {run['run_name']}",
        run["features"] == expected,
    )

stochastic_check(
    "fixed stochastic seeds",
    [run["seed"] for run in STOCHASTIC_REGISTRY[:STOCHASTIC_N_MEMBERS]]
        == STOCHASTIC_SEEDS[:STOCHASTIC_N_MEMBERS],
)
stochastic_check(
    "blend is cross-fitted",
    BLENDS[PRIMARY_HIERARCHICAL_NAME].metadata["evaluation_type"] == "cross_fitted_oof",
)
stochastic_check(
    "calibration is cross-fitted",
    AFFINE_CANDIDATES["stochastic_hierarchical"].metadata["evaluation_type"]
        == "cross_fitted_oof",
)

for model_name in MODEL_NAMES:
    current_hash = hashlib.sha256(
        np.ascontiguousarray(OOF[model_name].to_numpy(dtype=np.float64)).tobytes()
    ).hexdigest()
    stochastic_check(
        f"baseline predictions unchanged: {model_name}",
        current_hash == BASELINE_PREDICTION_HASHES[model_name],
    )
    current_metric = weighted_oof_rmsle(OOF, OOF[model_name])
    stochastic_check(
        f"baseline metric unchanged: {model_name}",
        np.isclose(current_metric, BASELINE_METRIC_SNAPSHOT[model_name], atol=1e-12),
    )

if RUN_FINAL_FIT:
    stochastic_check(
        "submission order",
        all(
            np.array_equal(
                pd.read_csv(path, usecols=[USER_COL])[USER_COL].to_numpy(),
                sample_submit[USER_COL].to_numpy(),
            )
            for path in RECOMMENDED_SUBMISSIONS
        ),
    )

pd.DataFrame(FINAL_STOCHASTIC_CHECKS).to_csv(
    STOCHASTIC_DIR / "stochastic_final_checks.csv", index=False
)

direction_best = STOCHASTIC_DIRECTION_METRICS.sort_values("weighted_rmsle").iloc[0]
ridge_score = (
    float(STOCHASTIC_DIRECTION_METRICS.loc[
        STOCHASTIC_DIRECTION_METRICS["direction"].eq("RIDGE"), "weighted_rmsle"
    ].iloc[0]) if "RIDGE" in DIRECTION_MEMBERS else np.nan
)
specialist_addition = STOCHASTIC_ABLATION.loc[
    STOCHASTIC_ABLATION["candidate"].eq("+ TARGET_WEIGHT_SPECIALISTS")
]
affine_meta = AFFINE_CANDIDATES["stochastic_hierarchical"].metadata[
    "full_oof_parameters_for_test"
]
hier_score = weighted_oof_rmsle(
    OOF, BLENDS[PRIMARY_HIERARCHICAL_NAME].oof_pred
)
affine_score = weighted_oof_rmsle(
    OOF, AFFINE_CANDIDATES["stochastic_hierarchical"].oof_pred
)

STOCHASTIC_SUMMARY = {
    "Current baseline honest RMSLE": BASELINE_BEST_SCORE,
    "Best stochastic direction": direction_best["direction"],
    "Best direction RMSLE": float(direction_best["weighted_rmsle"]),
    "Best direction addition delta": float(
        STOCHASTIC_ABLATION.loc[
            STOCHASTIC_ABLATION["candidate"].str.startswith("+ "),
            "delta_vs_baseline",
        ].min()
    ),
    "Ridge useful": bool(ridge_score < BASELINE_BEST_SCORE),
    "Target weighting useful": bool(
        len(specialist_addition)
        and specialist_addition["delta_vs_baseline"].iloc[0] < 0
    ),
    "Column bagging useful for": STOCHASTIC_ABLATION.loc[
        STOCHASTIC_ABLATION["candidate"].str.startswith("+ BAG_")
        & STOCHASTIC_ABLATION["delta_vs_baseline"].lt(0),
        "candidate",
    ].str.replace("+ ", "", regex=False).tolist(),
    "Hierarchical blend RMSLE": hier_score,
    "Affine calibration": {
        "a": affine_meta["a"], "b": affine_meta["b"],
        "delta": affine_score - hier_score,
    },
    "Best honest stochastic ensemble": min(hier_score, affine_score),
    "Recommended submissions": RECOMMENDED_SUBMISSIONS,
}

(STOCHASTIC_DIR / "stochastic_manifest.json").write_text(
    json.dumps(STOCHASTIC_SUMMARY, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8",
)

print("\nSTOCHASTIC / DIVERSITY SUMMARY")
for key, value in STOCHASTIC_SUMMARY.items():
    print(f"{key}: {value}")



STOCHASTIC / DIVERSITY SUMMARY
Current baseline honest RMSLE: 1.7111213151570999
Best stochastic direction: BASE_LEVEL
Best direction RMSLE: 1.7119013364889963
Best direction addition delta: -5.7894321904727875e-05
Ridge useful: False
Target weighting useful: False
Column bagging useful for: ['BAG_CATBOOST_LOG_DIFF']
Hierarchical blend RMSLE: 1.7115256209908978
Affine calibration: {'a': -0.06416077045685553, 'b': 0.9966523650362231, 'delta': 0.001256561326231198}
Best honest stochastic ensemble: 1.7115256209908978
Recommended submissions: ['/kaggle/working/ozon_final/stochastic_diversity/submissions/submission_original_best.csv', '/kaggle/working/ozon_final/stochastic_diversity/submissions/submission_stochastic_equal.csv', '/kaggle/working/ozon_final/stochastic_diversity/submissions/submission_stochastic_hierarchical.csv', '/kaggle/working/ozon_final/stochastic_diversity/submissions/submission_stochastic_hierarchical_affine.csv']


# 28. Denis-style calibrated hurdle ensemble

Двухстадийная архитектура строится в log-space:

`p(x) = P(Y > 0 | X)`

`m(x) = E[log1p(Y) | Y > 0, X]`

`h(x) = p(x) * m(x)` и итоговый raw-прогноз равен `expm1(h(x))`.

Текущий unconditional regression blend нельзя подставлять вместо `m(x)`: он уже содержит вклад нулевого сегмента, и повторное умножение на `p(x)` дважды учло бы вероятность покупки. Поэтому magnitude-модели ниже используют ваши неизменные frozen specifications и параметры, но обучаются только на положительных train targets. После этого готовые hurdle-направления честно смешиваются с исходным ансамблем регрессий.

Classifier не использует hard threshold, class weights, `scale_pos_weight`, sampling, SMOTE или случайный StratifiedKFold. Основная функция потерь — LogLoss; вероятности дополнительно проходят leakage-safe temporal Platt calibration.


In [120]:
# ============================================================
# HURDLE CONFIG
# ============================================================
ENABLE_HURDLE_EXPERIMENTS = True

# Denis-style CatBoost probability model. Параметры повторяют его основной
# классификатор; различаются только frozen feature views и фиксированные seeds.
HURDLE_CLASSIFIER_ITERATIONS = 4_000
HURDLE_CLASSIFIER_LEARNING_RATE = 0.03
HURDLE_CLASSIFIER_DEPTH = 8
HURDLE_CLASSIFIER_L2 = 16.0
HURDLE_CLASSIFIER_EARLY_STOPPING = 200
HURDLE_CLASSIFIER_SEED = 42

# Отдельные эксперименты на каждом из ваших frozen selected feature sets.
HURDLE_CLASSIFIER_FEATURE_MODELS = MODEL_NAMES.copy()
ENABLE_HURDLE_CLASSIFIER_COLUMN_BAG = False
HURDLE_CLASSIFIER_BAG_SEEDS = STOCHASTIC_SEEDS[:STOCHASTIC_N_MEMBERS]

# Magnitude ensemble переобучает четыре активные CatBoost/XGBoost regression specifications только
# на покупателях. Существующие hyperparameters и feature sets не меняются.
HURDLE_MAGNITUDE_MODELS = MODEL_NAMES.copy()

# Вероятности усредняются арифметически: это среднее оценок P(Y>0|X).
# Magnitude predictions усредняются в log-space, где задан их target.
HURDLE_PRIMARY_PROBABILITY = "P_DENIS_ALL"
HURDLE_PRIMARY_MAGNITUDE = "M_OWN_POSITIVE_ALL_ACTIVE"

HURDLE_CALIBRATION_EPS = 1e-6
HURDLE_CALIBRATION_BINS = 10
HURDLE_LIFT_TOP_FRACTIONS = (0.01, 0.05, 0.10)
HURDLE_CACHE_VERSION = "denis_hurdle_v1"

HURDLE_DIR = STOCHASTIC_DIR / "hurdle"
HURDLE_CACHE_DIR = HURDLE_DIR / "cache"
HURDLE_SUBMISSION_DIR = HURDLE_DIR / "submissions"
for _directory in [HURDLE_DIR, HURDLE_CACHE_DIR, HURDLE_SUBMISSION_DIR]:
    _directory.mkdir(parents=True, exist_ok=True)

assert SPECIALIST_BASE_MODEL == "catboost_log_level"
assert set(HURDLE_CLASSIFIER_FEATURE_MODELS).issubset(MODEL_NAMES)
assert set(HURDLE_MAGNITUDE_MODELS).issubset(MODEL_NAMES)


## 28.1 Probability ensemble: temporal early stopping and calibration

Для каждого outer fold последний разрешённый train cutoff становится внутренним calibration cutoff. Более ранние cutoffs используются для выбора числа деревьев; Platt calibration обучается на вероятностях внутреннего cutoff. Затем classifier переобучается с фиксированным числом деревьев на всём outer-train и применяется к held-out fold. Held-out labels нигде не участвуют ни в fitting, ни в calibration.


In [121]:
def hurdle_cache_key(kind: str, run: Mapping[str, Any], fold, final: bool) -> str:
    outer_train = FINAL_TRAIN_CUTOFFS if final else fold["train_cutoffs"]
    outer_valid = PREDICTION_CUTOFF if final else fold["valid_cutoff"]
    return _json_hash({
        "version": HURDLE_CACHE_VERSION,
        "dataset_id": STOCHASTIC_DATASET_ID,
        "kind": kind,
        "run": run,
        "outer_train": outer_train,
        "outer_valid": outer_valid,
    })


def hurdle_cache_paths(kind: str, run_name: str, fold, final: bool):
    split = "final" if final else fold["name"]
    directory = HURDLE_CACHE_DIR / kind / run_name / split
    return directory / "prediction.npz", directory / "metadata.json"


def load_hurdle_cache(kind, run, fold=None, final=False):
    if not ENABLE_CACHE:
        return None
    prediction_path, metadata_path = hurdle_cache_paths(
        kind, run["run_name"], fold, final
    )
    if not prediction_path.exists() or not metadata_path.exists():
        return None
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    if metadata.get("cache_key") != hurdle_cache_key(kind, run, fold, final):
        return None
    with np.load(prediction_path, allow_pickle=False) as saved:
        result = {name: saved[name] for name in saved.files}
    result["metadata"] = metadata
    result["metadata"]["cache_hit"] = True
    return result


def save_hurdle_cache(kind, run, fold, final, arrays, metadata):
    prediction_path, metadata_path = hurdle_cache_paths(
        kind, run["run_name"], fold, final
    )
    prediction_path.parent.mkdir(parents=True, exist_ok=True)
    temporary = prediction_path.with_suffix(".partial.npz")
    np.savez_compressed(temporary, **arrays)
    temporary.replace(prediction_path)
    metadata = {
        **metadata,
        "cache_key": hurdle_cache_key(kind, run, fold, final),
        "dataset_id": STOCHASTIC_DATASET_ID,
    }
    metadata_path.write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2, default=str),
        encoding="utf-8",
    )


def _classifier_run(name: str, feature_source: str, features, seed: int, kind: str):
    return {
        "run_name": name,
        "feature_source": feature_source,
        "features": sorted(set(features)),
        "seed": int(seed),
        "kind": kind,
        "params": {
            "iterations": HURDLE_CLASSIFIER_ITERATIONS,
            "learning_rate": HURDLE_CLASSIFIER_LEARNING_RATE,
            "depth": HURDLE_CLASSIFIER_DEPTH,
            "l2_leaf_reg": HURDLE_CLASSIFIER_L2,
            "loss_function": "Logloss",
            "eval_metric": "Logloss",
            "class_weights": None,
        },
    }


HURDLE_CLASSIFIER_RUNS = [
    _classifier_run(
        f"denis_clf__features_{model_name}", model_name,
        model_feature_order(MODEL_SPECS[model_name])[2],
        HURDLE_CLASSIFIER_SEED, "full_feature_view",
    )
    for model_name in HURDLE_CLASSIFIER_FEATURE_MODELS
]


if ENABLE_HURDLE_CLASSIFIER_COLUMN_BAG:
    HURDLE_CLASSIFIER_RUNS.extend([
        _classifier_run(
            f"denis_clf__cat_level_bag_s{run['seed']}",
            "catboost_log_level", run["features"], run["seed"],
            "column_bag",
        )
        for run in STOCHASTIC_REGISTRY
        if run["base_model"] == "catboost_log_level"
        and run["seed"] in HURDLE_CLASSIFIER_BAG_SEEDS
    ])

HURDLE_CLASSIFIER_REGISTRY = pd.DataFrame([
    {
        "classifier": run["run_name"],
        "kind": run["kind"],
        "feature_source": run["feature_source"],
        "seed": run["seed"],
        "n_features": len(run["features"]),
        "feature_subset_hash": _json_hash(run["features"]),
        "features_json": json.dumps(run["features"], ensure_ascii=False),
        "natural_class_balance": True,
        "loss_function": "Logloss",
    }
    for run in HURDLE_CLASSIFIER_RUNS
])
HURDLE_CLASSIFIER_REGISTRY.to_csv(
    HURDLE_DIR / "hurdle_classifier_registry.csv", index=False
)


def classifier_specification(run):
    # CatBoost receives the notebook's existing numerical/category matrix and
    # native categorical positions, but classifier hyperparameters are Denis-style.
    base = MODEL_SPECS["catboost_log_level"]
    return {
        **base,
        "name": run["run_name"],
        "family": "catboost",
        "target_mode": "log_level",
        "features": list(run["features"]),
        "params": {},
    }


def denis_classifier_params(seed: int, iterations: int):
    params = {
        "iterations": int(iterations),
        "learning_rate": HURDLE_CLASSIFIER_LEARNING_RATE,
        "depth": HURDLE_CLASSIFIER_DEPTH,
        "l2_leaf_reg": HURDLE_CLASSIFIER_L2,
        "loss_function": "Logloss",
        "eval_metric": "Logloss",
        "random_seed": int(seed),
        "bootstrap_type": "Bayesian",
        "grow_policy": "SymmetricTree",
        "has_time": True,
        "allow_writing_files": False,
        "verbose": False,
        "task_type": "GPU" if (USE_GPU and GPU_AVAILABLE) else "CPU",
    }
    if params["task_type"] == "GPU":
        params["devices"] = "0"
    # Explicit safety: natural prevalence, no class rebalancing.
    for forbidden in ["class_weights", "auto_class_weights", "scale_pos_weight"]:
        params.pop(forbidden, None)
    return params


def fit_probability_classifier(
    X_train, labels_train, X_valid, labels_valid, categorical_positions,
    seed: int, fixed_iterations: int | None = None,
):
    from catboost import CatBoostClassifier, Pool

    iterations = fixed_iterations or HURDLE_CLASSIFIER_ITERATIONS
    params = denis_classifier_params(seed, iterations)
    train_pool = Pool(
        X_train, labels_train, cat_features=list(categorical_positions)
    )
    valid_pool = None if X_valid is None else Pool(
        X_valid, labels_valid, cat_features=list(categorical_positions)
    )

    def fit_once(local_params):
        model = CatBoostClassifier(**local_params)
        if fixed_iterations is None:
            model.fit(
                train_pool, eval_set=valid_pool,
                early_stopping_rounds=HURDLE_CLASSIFIER_EARLY_STOPPING,
                use_best_model=True, verbose=False,
            )
        else:
            model.fit(train_pool, verbose=False)
        return model

    try:
        model = fit_once(params)
    except Exception as exc:
        if not (ALLOW_GPU_FALLBACK and params.get("task_type") == "GPU"):
            raise
        print("[GPU fallback][Hurdle classifier]", type(exc).__name__, str(exc)[:250])
        params = dict(params)
        params["task_type"] = "CPU"
        params.pop("devices", None)
        model = fit_once(params)

    if fixed_iterations is None:
        best = model.get_best_iteration()
        best_iterations = int(best) + 1 if best is not None and int(best) >= 0 else iterations
    else:
        best_iterations = int(fixed_iterations)
    return model, best_iterations


def fit_platt_calibrator(probability, label):
    from sklearn.linear_model import LogisticRegression
    probability = np.clip(
        np.asarray(probability, dtype=np.float64),
        HURDLE_CALIBRATION_EPS, 1.0 - HURDLE_CALIBRATION_EPS,
    )
    logits = np.log(probability / (1.0 - probability)).reshape(-1, 1)
    calibrator = LogisticRegression(
        penalty="l2", C=1.0, solver="lbfgs", max_iter=500,
        class_weight=None, random_state=HURDLE_CLASSIFIER_SEED,
    )
    calibrator.fit(logits, np.asarray(label, dtype=np.int8))
    return calibrator


def apply_platt_calibrator(calibrator, probability):
    probability = np.clip(
        np.asarray(probability, dtype=np.float64),
        HURDLE_CALIBRATION_EPS, 1.0 - HURDLE_CALIBRATION_EPS,
    )
    logits = np.log(probability / (1.0 - probability)).reshape(-1, 1)
    return np.clip(
        calibrator.predict_proba(logits)[:, 1],
        HURDLE_CALIBRATION_EPS, 1.0 - HURDLE_CALIBRATION_EPS,
    )


def fit_classifier_outer_split(run, fold=None, final=False):
    cached = load_hurdle_cache("classifier", run, fold, final)
    if cached is not None:
        return cached

    outer_train = FINAL_TRAIN_CUTOFFS if final else fold["train_cutoffs"]
    outer_valid = PREDICTION_CUTOFF if final else fold["valid_cutoff"]
    if len(outer_train) < 2:
        raise ValueError("classifier requires at least two temporal train cutoffs")
    inner_train = outer_train[:-1]
    calibration_cutoff = outer_train[-1]
    spec = classifier_specification(run)

    # Stage A: choose iterations and calibrator without outer-held-out labels.
    inner = materialize_model_pair(inner_train, calibration_cutoff, spec)
    inner_labels = (inner["y_train_raw"] > 0).astype(np.int8)
    calibration_labels = (inner["y_valid_raw"] > 0).astype(np.int8)
    model, best_iterations = fit_probability_classifier(
        inner["X_train"], inner_labels,
        inner["X_valid"], calibration_labels,
        inner["categorical_positions"], run["seed"], fixed_iterations=None,
    )
    from catboost import Pool
    calibration_raw = model.predict_proba(Pool(
        inner["X_valid"], cat_features=inner["categorical_positions"]
    ))[:, 1]
    calibrator = fit_platt_calibrator(calibration_raw, calibration_labels)
    del inner, model
    _cleanup_accelerators()

    # Stage B: refit on every allowed outer-train row with fixed iterations.
    outer = materialize_model_pair(outer_train, outer_valid, spec)
    outer_labels = (outer["y_train_raw"] > 0).astype(np.int8)
    model, _ = fit_probability_classifier(
        outer["X_train"], outer_labels, None, None,
        outer["categorical_positions"], run["seed"],
        fixed_iterations=best_iterations,
    )
    raw_probability = model.predict_proba(Pool(
        outer["X_valid"], cat_features=outer["categorical_positions"]
    ))[:, 1]
    probability = apply_platt_calibrator(calibrator, raw_probability)
    labels = (outer["y_valid_raw"] > 0).astype(np.int8)
    arrays = {
        "user_id": np.asarray(outer["valid_user_ids"]),
        "probability": np.asarray(probability, dtype=np.float64),
        "raw_probability": np.asarray(raw_probability, dtype=np.float64),
        "label": labels,
        "truth": np.asarray(outer["y_valid_raw"], dtype=np.float64),
    }
    metadata = {
        "run_name": run["run_name"],
        "fold": "final" if final else fold["name"],
        "inner_train_cutoffs": inner_train,
        "calibration_cutoff": calibration_cutoff,
        "outer_train_cutoffs": outer_train,
        "outer_valid_cutoff": outer_valid,
        "best_iterations": best_iterations,
        "platt_intercept": float(calibrator.intercept_[0]),
        "platt_coefficient": float(calibrator.coef_[0, 0]),
        "natural_train_prevalence": float(outer_labels.mean()),
        "cache_hit": False,
    }
    save_hurdle_cache("classifier", run, fold, final, arrays, metadata)
    del outer, model, calibrator
    _cleanup_accelerators()
    return {**arrays, "metadata": metadata}


In [122]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    average_precision_score, brier_score_loss, log_loss, roc_auc_score,
)


HURDLE_CLASSIFIER_FOLD_ROWS = []
HURDLE_CLASSIFIER_CALIBRATION_ROWS = []
HURDLE_CLASSIFIER_LIFT_ROWS = []
HURDLE_PROBABILITY_OOF = {}
HURDLE_PROBABILITY_FINAL = {}


def probability_diagnostics(run_name, fold_name, label, probability, raw_probability):
    label = np.asarray(label, dtype=np.int8)
    probability = np.asarray(probability, dtype=np.float64)
    raw_probability = np.asarray(raw_probability, dtype=np.float64)
    HURDLE_CLASSIFIER_FOLD_ROWS.append({
        "classifier": run_name, "fold": fold_name,
        "prevalence": float(label.mean()),
        "mean_probability": float(probability.mean()),
        "logloss": log_loss(label, probability, labels=[0, 1]),
        "raw_logloss": log_loss(label, raw_probability, labels=[0, 1]),
        "brier": brier_score_loss(label, probability),
        "raw_brier": brier_score_loss(label, raw_probability),
        "average_precision": average_precision_score(label, probability),
        "roc_auc": roc_auc_score(label, probability),
    })
    observed, predicted = calibration_curve(
        label, probability, n_bins=HURDLE_CALIBRATION_BINS,
        strategy="quantile",
    )
    for index, (mean_predicted, fraction_positive) in enumerate(zip(predicted, observed)):
        HURDLE_CLASSIFIER_CALIBRATION_ROWS.append({
            "classifier": run_name, "fold": fold_name, "bin": index,
            "mean_predicted_probability": float(mean_predicted),
            "observed_positive_fraction": float(fraction_positive),
            "calibration_gap": float(mean_predicted - fraction_positive),
        })
    order = np.argsort(-probability)
    prevalence = max(float(label.mean()), 1e-15)
    for fraction in HURDLE_LIFT_TOP_FRACTIONS:
        count = max(1, int(round(len(label) * fraction)))
        top_rate = float(label[order[:count]].mean())
        HURDLE_CLASSIFIER_LIFT_ROWS.append({
            "classifier": run_name, "fold": fold_name,
            "top_fraction": fraction, "top_count": count,
            "top_positive_rate": top_rate,
            "lift": top_rate / prevalence,
        })


if RUN_CV and ENABLE_HURDLE_EXPERIMENTS:
    for run in HURDLE_CLASSIFIER_RUNS:
        print("\nHURDLE CLASSIFIER CV:", run["run_name"])
        full_probability = np.full(len(OOF), np.nan, dtype=np.float64)
        for fold in CV_FOLDS:
            result = fit_classifier_outer_split(run, fold=fold)
            hold = OOF["fold"].eq(fold["name"]).to_numpy()
            if not np.array_equal(OOF.loc[hold, USER_COL].to_numpy(), result["user_id"]):
                raise AssertionError(f"{run['run_name']}/{fold['name']}: alignment")
            full_probability[hold] = result["probability"]
            probability_diagnostics(
                run["run_name"], fold["name"], result["label"],
                result["probability"], result["raw_probability"],
            )
        if not np.isfinite(full_probability).all():
            raise AssertionError(f"{run['run_name']}: incomplete probability OOF")
        HURDLE_PROBABILITY_OOF[run["run_name"]] = full_probability

if RUN_FINAL_FIT and ENABLE_HURDLE_EXPERIMENTS:
    for run in HURDLE_CLASSIFIER_RUNS:
        print("HURDLE CLASSIFIER FINAL:", run["run_name"])
        result = fit_classifier_outer_split(run, final=True)
        if not np.array_equal(FINAL_PREDICTIONS[USER_COL].to_numpy(), result["user_id"]):
            raise AssertionError(f"{run['run_name']}: final alignment")
        HURDLE_PROBABILITY_FINAL[run["run_name"]] = result["probability"]


def register_probability_direction(name: str, members: Sequence[str]):
    members = list(members)
    if not members:
        return
    HURDLE_PROBABILITY_OOF[name] = np.mean(np.vstack([
        HURDLE_PROBABILITY_OOF[member] for member in members
    ]), axis=0)
    if RUN_FINAL_FIT:
        HURDLE_PROBABILITY_FINAL[name] = np.mean(np.vstack([
            HURDLE_PROBABILITY_FINAL[member] for member in members
        ]), axis=0)


full_view_members = [
    run["run_name"] for run in HURDLE_CLASSIFIER_RUNS
    if run["kind"] == "full_feature_view"
]
bag_members = [
    run["run_name"] for run in HURDLE_CLASSIFIER_RUNS
    if run["kind"] == "column_bag"
]
register_probability_direction("P_DENIS_FEATURE_VIEWS", full_view_members)
register_probability_direction("P_DENIS_CAT_BAG", bag_members)
probability_directions_for_all = [
    name for name in ["P_DENIS_FEATURE_VIEWS", "P_DENIS_CAT_BAG"]
    if name in HURDLE_PROBABILITY_OOF
]
register_probability_direction("P_DENIS_ALL", probability_directions_for_all)

HURDLE_CLASSIFIER_DIRECTION_ROWS = []
for direction in probability_directions_for_all + ["P_DENIS_ALL"]:
    probability = HURDLE_PROBABILITY_OOF[direction]
    for fold in CV_FOLDS:
        hold = OOF["fold"].eq(fold["name"]).to_numpy()
        label = (OOF.loc[hold, TARGET_COL].to_numpy(dtype=np.float64) > 0).astype(np.int8)
        pred = probability[hold]
        HURDLE_CLASSIFIER_DIRECTION_ROWS.append({
            "probability_direction": direction,
            "fold": fold["name"], "weight": fold["weight"],
            "prevalence": float(label.mean()),
            "mean_probability": float(pred.mean()),
            "logloss": log_loss(label, pred, labels=[0, 1]),
            "brier": brier_score_loss(label, pred),
            "average_precision": average_precision_score(label, pred),
            "roc_auc": roc_auc_score(label, pred),
        })

HURDLE_CLASSIFIER_METRICS = pd.DataFrame(HURDLE_CLASSIFIER_FOLD_ROWS)
HURDLE_PROBABILITY_CALIBRATION = pd.DataFrame(HURDLE_CLASSIFIER_CALIBRATION_ROWS)
HURDLE_LIFT_TOPK = pd.DataFrame(HURDLE_CLASSIFIER_LIFT_ROWS)
HURDLE_CLASSIFIER_DIRECTION_METRICS = pd.DataFrame(
    HURDLE_CLASSIFIER_DIRECTION_ROWS
)
HURDLE_CLASSIFIER_METRICS.to_csv(
    HURDLE_DIR / "hurdle_classifier_metrics.csv", index=False
)
HURDLE_PROBABILITY_CALIBRATION.to_csv(
    HURDLE_DIR / "hurdle_probability_calibration.csv", index=False
)
HURDLE_LIFT_TOPK.to_csv(HURDLE_DIR / "hurdle_lift_topk.csv", index=False)
HURDLE_CLASSIFIER_DIRECTION_METRICS.to_csv(
    HURDLE_DIR / "hurdle_classifier_direction_metrics.csv", index=False
)
display(HURDLE_CLASSIFIER_METRICS)
display(HURDLE_CLASSIFIER_DIRECTION_METRICS)



HURDLE CLASSIFIER CV: denis_clf__features_xgboost_log_level

HURDLE CLASSIFIER CV: denis_clf__features_catboost_log_level

HURDLE CLASSIFIER CV: denis_clf__features_catboost_log_diff

HURDLE CLASSIFIER CV: denis_clf__features_xgboost_log_diff
HURDLE CLASSIFIER FINAL: denis_clf__features_xgboost_log_level
HURDLE CLASSIFIER FINAL: denis_clf__features_catboost_log_level
HURDLE CLASSIFIER FINAL: denis_clf__features_catboost_log_diff
HURDLE CLASSIFIER FINAL: denis_clf__features_xgboost_log_diff


,classifier,fold,prevalence,mean_probability,logloss,raw_logloss,brier,raw_brier,average_precision,roc_auc
0,denis_clf__features_xgboost_log_level,nov_2025,0.562684,0.548376,0.473875,0.473752,0.156001,0.155948,0.886348,0.851150
1,denis_clf__features_xgboost_log_level,dec_2025,0.563080,0.586906,0.474826,0.473901,0.156689,0.156285,0.884824,0.851110
2,denis_clf__features_xgboost_log_level,jan_2026,0.540660,0.552378,0.469401,0.470819,0.154441,0.154995,0.879213,0.855936
3,denis_clf__features_catboost_log_level,nov_2025,0.562684,0.546424,0.474696,0.474414,0.156241,0.156137,0.885972,0.850636
4,denis_clf__features_catboost_log_level,dec_2025,0.563080,0.590463,0.475988,0.474674,0.157081,0.156537,0.884591,0.850758
5,denis_clf__features_catboost_log_level,jan_2026,0.540660,0.547508,0.470145,0.471430,0.154601,0.155157,0.878768,0.855512
6,denis_clf__features_catboost_log_diff,nov_2025,0.562684,0.546473,0.474787,0.474605,0.156311,0.156241,0.886091,0.850487
7,denis_clf__features_catboost_log_diff,dec_2025,0.563080,0.594765,0.476444,0.474442,0.157229,0.156445,0.884897,0.851037
8,denis_clf__features_catboost_log_diff,jan_2026,0.540660,0.540939,0.469732,0.470626,0.154469,0.154857,0.879101,0.855741
9,denis_clf__features_xgboost_log_diff,nov_2025,0.562684,0.549250,0.474651,0.474542,0.156201,0.156163,0.886120,0.850596


,probability_direction,fold,weight,prevalence,mean_probability,logloss,brier,average_precision,roc_auc
0,P_DENIS_FEATURE_VIEWS,nov_2025,0.25,0.562684,0.547631,0.473662,0.155892,0.886658,0.851284
1,P_DENIS_FEATURE_VIEWS,dec_2025,0.35,0.563080,0.590361,0.474751,0.156652,0.885323,0.851619
2,P_DENIS_FEATURE_VIEWS,jan_2026,0.40,0.540660,0.547848,0.468789,0.154153,0.879665,0.856399
3,P_DENIS_ALL,nov_2025,0.25,0.562684,0.547631,0.473662,0.155892,0.886658,0.851284
4,P_DENIS_ALL,dec_2025,0.35,0.563080,0.590361,0.474751,0.156652,0.885323,0.851619
5,P_DENIS_ALL,jan_2026,0.40,0.540660,0.547848,0.468789,0.154153,0.879665,0.856399


## 28.2 Conditional magnitude ensemble from your regression specifications

Каждая magnitude-модель сохраняет исходные selected features, target mode и лучшие параметры вашей модели. Отличие ровно одно: preprocessing и fitting выполняются только на `target_30d > 0`. Число деревьев выбирается на внутреннем временном cutoff, а outer held-out fold используется только для оценки.


In [123]:
def materialize_positive_magnitude_pair(
    train_cutoffs: Sequence[str], valid_cutoff: str, spec: Mapping[str, Any],
):
    numeric, categorical, columns = model_feature_order(spec)
    train_parts = [
        _load_one_model_snapshot(cutoff, numeric, categorical)
        for cutoff in train_cutoffs
    ]
    valid = _load_one_model_snapshot(valid_cutoff, numeric, categorical)

    X_all = np.concatenate([part["X"] for part in train_parts], axis=0)
    y_raw_all = np.concatenate([part["y_raw"] for part in train_parts])
    anchor_all = np.concatenate([part["anchor"] for part in train_parts])
    block_all = np.concatenate([
        np.full(len(part["y_raw"]), index, dtype=np.int32)
        for index, part in enumerate(train_parts)
    ])
    buyers = y_raw_all > 0
    if not buyers.any():
        raise ValueError("positive magnitude train has no buyers")

    X_train = X_all[buyers]
    y_train_raw = y_raw_all[buyers]
    anchor_train = anchor_all[buyers]
    block_ids = block_all[buyers]
    y_train = build_model_target(
        y_train_raw, anchor_train, spec["target_mode"]
    )
    categorical_positions = list(range(len(numeric), len(columns)))
    encoding = {}
    if spec["family"] != "catboost" and categorical_positions:
        X_train, valid["X"], encoding = encode_matrices(
            X_train, valid["X"], y_train, block_ids,
            categorical_positions, ENCODING_CONFIG,
        )
        categorical_positions = []

    return {
        "X_train": X_train,
        "y_train": y_train,
        "y_train_raw": y_train_raw,
        "anchor_train": anchor_train,
        "block_ids": block_ids,
        "X_valid": valid["X"],
        "y_valid_raw": valid["y_raw"],
        "anchor_valid": valid["anchor"],
        "valid_user_ids": valid["user_id"],
        "columns": columns,
        "categorical_positions": categorical_positions,
        "encoding": encoding,
    }


HURDLE_MAGNITUDE_RUNS = [
    {
        "run_name": f"positive_magnitude__{model_name}",
        "base_model": model_name,
        "family": MODEL_SPECS[model_name]["family"],
        "target_mode": MODEL_SPECS[model_name]["target_mode"],
        "features": model_feature_order(MODEL_SPECS[model_name])[2],
        "seed": RANDOM_SEED,
        "params": resolve_effective_params(MODEL_SPECS[model_name]),
    }
    for model_name in HURDLE_MAGNITUDE_MODELS
]


def predict_fitted_regressor(model, family, X, categorical_positions, rounds):
    if family == "catboost":
        from catboost import Pool
        return np.asarray(model.predict(Pool(
            X, cat_features=list(categorical_positions)
        )), dtype=np.float64)
    if family == "lightgbm":
        return np.asarray(model.predict(X, num_iteration=rounds), dtype=np.float64)
    return np.asarray(model.predict(X), dtype=np.float64)


def fit_magnitude_outer_split(run, fold=None, final=False):
    cached = load_hurdle_cache("magnitude", run, fold, final)
    if cached is not None:
        return cached
    outer_train = FINAL_TRAIN_CUTOFFS if final else fold["train_cutoffs"]
    outer_valid = PREDICTION_CUTOFF if final else fold["valid_cutoff"]
    inner_train = outer_train[:-1]
    tuning_cutoff = outer_train[-1]
    spec = MODEL_SPECS[run["base_model"]]

    # Inner temporal early stopping on buyers only.
    inner = materialize_positive_magnitude_pair(inner_train, tuning_cutoff, spec)
    inner_valid_buyers = inner["y_valid_raw"] > 0
    inner_valid_target = build_model_target(
        inner["y_valid_raw"][inner_valid_buyers],
        inner["anchor_valid"][inner_valid_buyers], spec["target_mode"],
    )
    model, _, best_round = fit_family(
        spec["family"], inner["X_train"], inner["y_train"],
        inner["X_valid"][inner_valid_buyers], inner_valid_target,
        inner["categorical_positions"], spec,
        final_rounds=None, sample_weight=None, random_seed=run["seed"],
    )
    del inner, model
    _cleanup_accelerators()

    # Refit on all allowed positive rows, then predict every held-out user.
    outer = materialize_positive_magnitude_pair(outer_train, outer_valid, spec)
    model, _, used_rounds = fit_family(
        spec["family"], outer["X_train"], outer["y_train"],
        None, None, outer["categorical_positions"], spec,
        final_rounds=best_round, sample_weight=None, random_seed=run["seed"],
    )
    prediction_model = predict_fitted_regressor(
        model, spec["family"], outer["X_valid"],
        outer["categorical_positions"], used_rounds,
    )
    magnitude_raw = inverse_model_target(
        prediction_model, outer["anchor_valid"], spec["target_mode"]
    )
    magnitude_log = np.log1p(np.clip(magnitude_raw, 0.0, None))
    arrays = {
        "user_id": np.asarray(outer["valid_user_ids"]),
        "magnitude_log": np.asarray(magnitude_log, dtype=np.float64),
        "truth": np.asarray(outer["y_valid_raw"], dtype=np.float64),
    }
    metadata = {
        "run_name": run["run_name"],
        "fold": "final" if final else fold["name"],
        "inner_train_cutoffs": inner_train,
        "tuning_cutoff": tuning_cutoff,
        "outer_train_cutoffs": outer_train,
        "outer_valid_cutoff": outer_valid,
        "best_round": int(best_round),
        "positive_train_only": True,
        "cache_hit": False,
    }
    save_hurdle_cache("magnitude", run, fold, final, arrays, metadata)
    del outer, model, prediction_model, magnitude_raw
    _cleanup_accelerators()
    return {**arrays, "metadata": metadata}


HURDLE_MAGNITUDE_OOF = {}
HURDLE_MAGNITUDE_FINAL = {}
HURDLE_MAGNITUDE_FOLD_ROWS = []

if RUN_CV and ENABLE_HURDLE_EXPERIMENTS:
    for run in HURDLE_MAGNITUDE_RUNS:
        print("\nHURDLE MAGNITUDE CV:", run["run_name"])
        full_magnitude = np.full(len(OOF), np.nan, dtype=np.float64)
        for fold in CV_FOLDS:
            result = fit_magnitude_outer_split(run, fold=fold)
            hold = OOF["fold"].eq(fold["name"]).to_numpy()
            if not np.array_equal(OOF.loc[hold, USER_COL].to_numpy(), result["user_id"]):
                raise AssertionError(f"{run['run_name']}/{fold['name']}: alignment")
            full_magnitude[hold] = result["magnitude_log"]
            positive = result["truth"] > 0
            conditional_prediction = np.expm1(result["magnitude_log"])
            HURDLE_MAGNITUDE_FOLD_ROWS.append({
                "magnitude_model": run["run_name"], "fold": fold["name"],
                "weight": fold["weight"],
                "positive_rmsle": rmsle(
                    result["truth"][positive], conditional_prediction[positive]
                ),
                "best_round": result["metadata"]["best_round"],
                "positive_train_only": True,
            })
        HURDLE_MAGNITUDE_OOF[run["run_name"]] = full_magnitude

if RUN_FINAL_FIT and ENABLE_HURDLE_EXPERIMENTS:
    for run in HURDLE_MAGNITUDE_RUNS:
        print("HURDLE MAGNITUDE FINAL:", run["run_name"])
        result = fit_magnitude_outer_split(run, final=True)
        if not np.array_equal(FINAL_PREDICTIONS[USER_COL].to_numpy(), result["user_id"]):
            raise AssertionError(f"{run['run_name']}: final alignment")
        HURDLE_MAGNITUDE_FINAL[run["run_name"]] = result["magnitude_log"]


def register_magnitude_direction(name: str, members: Sequence[str]):
    members = list(members)
    if not members:
        return
    # Members already predict log1p conditional magnitude.
    HURDLE_MAGNITUDE_OOF[name] = np.mean(np.vstack([
        HURDLE_MAGNITUDE_OOF[member] for member in members
    ]), axis=0)
    if RUN_FINAL_FIT:
        HURDLE_MAGNITUDE_FINAL[name] = np.mean(np.vstack([
            HURDLE_MAGNITUDE_FINAL[member] for member in members
        ]), axis=0)


magnitude_level_members = [
    run["run_name"] for run in HURDLE_MAGNITUDE_RUNS
    if run["target_mode"] == "log_level"
]
magnitude_diff_members = [
    run["run_name"] for run in HURDLE_MAGNITUDE_RUNS
    if run["target_mode"] == "log_diff"
]
register_magnitude_direction("M_OWN_POSITIVE_LEVEL", magnitude_level_members)
register_magnitude_direction("M_OWN_POSITIVE_DIFF", magnitude_diff_members)
register_magnitude_direction(
    "M_OWN_POSITIVE_ALL_ACTIVE",
    [run["run_name"] for run in HURDLE_MAGNITUDE_RUNS],
)

HURDLE_MAGNITUDE_DIRECTION_ROWS = []
for direction in [
    "M_OWN_POSITIVE_LEVEL", "M_OWN_POSITIVE_DIFF", "M_OWN_POSITIVE_ALL_ACTIVE",
]:
    magnitude_log = HURDLE_MAGNITUDE_OOF[direction]
    magnitude_raw = np.expm1(magnitude_log)
    for fold in CV_FOLDS:
        hold = OOF["fold"].eq(fold["name"]).to_numpy()
        truth = OOF.loc[hold, TARGET_COL].to_numpy(dtype=np.float64)
        buyers = truth > 0
        HURDLE_MAGNITUDE_DIRECTION_ROWS.append({
            "magnitude_direction": direction,
            "fold": fold["name"], "weight": fold["weight"],
            "positive_rmsle": rmsle(
                truth[buyers], magnitude_raw[hold][buyers]
            ),
        })

HURDLE_MAGNITUDE_METRICS = pd.DataFrame(HURDLE_MAGNITUDE_FOLD_ROWS)
HURDLE_MAGNITUDE_DIRECTION_METRICS = pd.DataFrame(
    HURDLE_MAGNITUDE_DIRECTION_ROWS
)
HURDLE_MAGNITUDE_METRICS.to_csv(
    HURDLE_DIR / "hurdle_magnitude_metrics.csv", index=False
)
HURDLE_MAGNITUDE_DIRECTION_METRICS.to_csv(
    HURDLE_DIR / "hurdle_magnitude_direction_metrics.csv", index=False
)
display(HURDLE_MAGNITUDE_METRICS)
display(HURDLE_MAGNITUDE_DIRECTION_METRICS)



HURDLE MAGNITUDE CV: positive_magnitude__xgboost_log_level

HURDLE MAGNITUDE CV: positive_magnitude__catboost_log_level

HURDLE MAGNITUDE CV: positive_magnitude__catboost_log_diff

HURDLE MAGNITUDE CV: positive_magnitude__xgboost_log_diff
HURDLE MAGNITUDE FINAL: positive_magnitude__xgboost_log_level
HURDLE MAGNITUDE FINAL: positive_magnitude__catboost_log_level
HURDLE MAGNITUDE FINAL: positive_magnitude__catboost_log_diff


,magnitude_model,fold,weight,positive_rmsle,best_round,positive_train_only
0,positive_magnitude__xgboost_log_level,nov_2025,0.25,1.113154,1142,True
1,positive_magnitude__xgboost_log_level,dec_2025,0.35,1.122688,1271,True
2,positive_magnitude__xgboost_log_level,jan_2026,0.40,1.107136,1414,True
3,positive_magnitude__catboost_log_level,nov_2025,0.25,1.114064,6052,True
4,positive_magnitude__catboost_log_level,dec_2025,0.35,1.122898,6672,True
5,positive_magnitude__catboost_log_level,jan_2026,0.40,1.110191,8371,True
6,positive_magnitude__catboost_log_diff,nov_2025,0.25,1.112606,1148,True
7,positive_magnitude__catboost_log_diff,dec_2025,0.35,1.123317,1483,True
8,positive_magnitude__catboost_log_diff,jan_2026,0.40,1.110239,2084,True
9,positive_magnitude__xgboost_log_diff,nov_2025,0.25,1.113235,3275,True


,magnitude_direction,fold,weight,positive_rmsle
0,M_OWN_POSITIVE_LEVEL,nov_2025,0.25,1.112894
1,M_OWN_POSITIVE_LEVEL,dec_2025,0.35,1.122028
2,M_OWN_POSITIVE_LEVEL,jan_2026,0.40,1.107931
3,M_OWN_POSITIVE_DIFF,nov_2025,0.25,1.111017
4,M_OWN_POSITIVE_DIFF,dec_2025,0.35,1.120644
5,M_OWN_POSITIVE_DIFF,jan_2026,0.40,1.108435
6,M_OWN_POSITIVE_ALL_ACTIVE,nov_2025,0.25,1.110946
7,M_OWN_POSITIVE_ALL_ACTIVE,dec_2025,0.35,1.120089
8,M_OWN_POSITIVE_ALL_ACTIVE,jan_2026,0.40,1.107187


## 28.3 Double models and blending with the original regression ensemble

Вероятности и conditional magnitudes сначала образуют самостоятельные hurdle-кандидаты. Только после их OOF-оценки они смешиваются с текущим лучшим regression ensemble через тот же constrained cross-fitted log-space optimizer.


In [124]:
def hurdle_raw_prediction(probability, magnitude_log):
    probability = np.clip(np.asarray(probability, dtype=np.float64), 0.0, 1.0)
    magnitude_log = np.clip(np.asarray(magnitude_log, dtype=np.float64), 0.0, None)
    return np.clip(np.expm1(probability * magnitude_log), 0.0, None)


HURDLE_PROBABILITY_DIRECTIONS = [
    name for name in [
        "P_DENIS_FEATURE_VIEWS", "P_DENIS_CAT_BAG", "P_DENIS_ALL",
    ]
    if name in HURDLE_PROBABILITY_OOF
]
HURDLE_MAGNITUDE_DIRECTIONS = [
    name for name in [
        "M_OWN_POSITIVE_LEVEL", "M_OWN_POSITIVE_DIFF", "M_OWN_POSITIVE_ALL_ACTIVE",
    ]
    if name in HURDLE_MAGNITUDE_OOF
]

HURDLE_CANDIDATE_COLUMNS = []
HURDLE_CANDIDATE_ROWS = []
HURDLE_BLEND_ROWS = []

for probability_name in HURDLE_PROBABILITY_DIRECTIONS:
    for magnitude_name in HURDLE_MAGNITUDE_DIRECTIONS:
        candidate_name = f"HURDLE__{probability_name}__{magnitude_name}"
        prediction = hurdle_raw_prediction(
            HURDLE_PROBABILITY_OOF[probability_name],
            HURDLE_MAGNITUDE_OOF[magnitude_name],
        )
        OOF[candidate_name] = prediction
        if RUN_FINAL_FIT:
            FINAL_PREDICTIONS[candidate_name] = hurdle_raw_prediction(
                HURDLE_PROBABILITY_FINAL[probability_name],
                HURDLE_MAGNITUDE_FINAL[magnitude_name],
            )
        HURDLE_CANDIDATE_COLUMNS.append(candidate_name)
        scores = fold_score_dict(prediction)
        zero = OOF[TARGET_COL].to_numpy(dtype=np.float64) <= 0
        HURDLE_CANDIDATE_ROWS.append({
            "candidate": candidate_name,
            "probability_direction": probability_name,
            "magnitude_direction": magnitude_name,
            "weighted_rmsle": weighted_oof_rmsle(OOF, prediction),
            "zero_rmsle": rmsle(OOF.loc[zero, TARGET_COL], prediction[zero]),
            "positive_rmsle": rmsle(OOF.loc[~zero, TARGET_COL], prediction[~zero]),
            "worst_fold": max(scores.values()),
            "fold_std": float(np.std(list(scores.values()))),
            **{f"{fold}_rmsle": score for fold, score in scores.items()},
        })

        # Honest two-direction mix: original regression ensemble + hurdle.
        blend_oof, blend_test, blend_meta = crossfit_optimized_blend(
            ["CURRENT_BEST_ORIGINAL", candidate_name], "log"
        )
        blend_name = f"CF_ORIGINAL_PLUS__{candidate_name}"
        add_blend(
            blend_name, blend_oof, blend_test,
            "crossfit_original_plus_hurdle", complexity=5,
            selection_safe=True,
            metadata={
                **blend_meta, "evaluation_type": "cross_fitted_oof",
                "hurdle_candidate": candidate_name,
            },
        )
        HURDLE_BLEND_ROWS.append({
            "blend": blend_name, "hurdle_candidate": candidate_name,
            "hurdle_standalone_rmsle": weighted_oof_rmsle(OOF, prediction),
            "blend_rmsle": weighted_oof_rmsle(OOF, blend_oof),
            "delta_vs_original": weighted_oof_rmsle(OOF, blend_oof)
                - BASELINE_BEST_SCORE,
            "evaluation_type": "cross_fitted_oof",
            "full_oof_hurdle_weight_for_test": blend_meta[
                "full_oof_weights_for_test"
            ][candidate_name],
        })


HURDLE_PRIMARY_COLUMN = (
    f"HURDLE__{HURDLE_PRIMARY_PROBABILITY}__{HURDLE_PRIMARY_MAGNITUDE}"
)
HURDLE_PRIMARY_BLEND = f"CF_ORIGINAL_PLUS__{HURDLE_PRIMARY_COLUMN}"

# Equal direction of all double models is diagnostic; no optimizer over raw
# classifier×magnitude cross-product is fitted.
OOF["HURDLE_ALL_EQUAL"] = log_mean(
    OOF[HURDLE_CANDIDATE_COLUMNS].to_numpy(dtype=np.float64)
)
if RUN_FINAL_FIT:
    FINAL_PREDICTIONS["HURDLE_ALL_EQUAL"] = log_mean(
        FINAL_PREDICTIONS[HURDLE_CANDIDATE_COLUMNS].to_numpy(dtype=np.float64)
    )
all_oof, all_test, all_meta = crossfit_optimized_blend(
    ["CURRENT_BEST_ORIGINAL", "HURDLE_ALL_EQUAL"], "log"
)
add_blend(
    "CF_ORIGINAL_PLUS_HURDLE_ALL_EQUAL", all_oof, all_test,
    "crossfit_original_plus_hurdle_direction", complexity=5,
    selection_safe=True,
    metadata={**all_meta, "evaluation_type": "cross_fitted_oof"},
)

HURDLE_CANDIDATE_METRICS = pd.DataFrame(HURDLE_CANDIDATE_ROWS).sort_values(
    "weighted_rmsle"
)
HURDLE_BLEND_ABLATION = pd.DataFrame(HURDLE_BLEND_ROWS).sort_values(
    "blend_rmsle"
)
HURDLE_CANDIDATE_METRICS.to_csv(
    HURDLE_DIR / "hurdle_candidate_metrics.csv", index=False
)
HURDLE_BLEND_ABLATION.to_csv(
    HURDLE_DIR / "hurdle_blend_ablation.csv", index=False
)

primary_metadata = BLENDS[HURDLE_PRIMARY_BLEND].metadata
HURDLE_BLEND_WEIGHTS = pd.DataFrame([
    {
        "fold": fold,
        **weights,
    }
    for fold, weights in primary_metadata["crossfit_weights"].items()
] + [{
    "fold": "full_oof_for_test",
    **primary_metadata["full_oof_weights_for_test"],
}])
HURDLE_BLEND_WEIGHTS.to_csv(
    HURDLE_DIR / "hurdle_blend_weights.csv", index=False
)

display(HURDLE_CANDIDATE_METRICS)
display(HURDLE_BLEND_ABLATION)
display(HURDLE_BLEND_WEIGHTS)


,candidate,probability_direction,magnitude_direction,weighted_rmsle,zero_rmsle,positive_rmsle,worst_fold,fold_std,nov_2025_rmsle,dec_2025_rmsle,jan_2026_rmsle
2,HURDLE__P_DENIS_FEATURE_VIEWS__M_OWN_POSITIVE_...,P_DENIS_FEATURE_VIEWS,M_OWN_POSITIVE_ALL_ACTIVE,1.710070,1.681439,1.740275,1.738224,0.030447,1.732926,1.738224,1.671150
5,HURDLE__P_DENIS_ALL__M_OWN_POSITIVE_ALL_ACTIVE,P_DENIS_ALL,M_OWN_POSITIVE_ALL_ACTIVE,1.710070,1.681439,1.740275,1.738224,0.030447,1.732926,1.738224,1.671150
3,HURDLE__P_DENIS_ALL__M_OWN_POSITIVE_LEVEL,P_DENIS_ALL,M_OWN_POSITIVE_LEVEL,1.710241,1.674229,1.746229,1.737820,0.030383,1.733738,1.737820,1.671424
0,HURDLE__P_DENIS_FEATURE_VIEWS__M_OWN_POSITIVE_...,P_DENIS_FEATURE_VIEWS,M_OWN_POSITIVE_LEVEL,1.710241,1.674229,1.746229,1.737820,0.030383,1.733738,1.737820,1.671424
1,HURDLE__P_DENIS_FEATURE_VIEWS__M_OWN_POSITIVE_...,P_DENIS_FEATURE_VIEWS,M_OWN_POSITIVE_DIFF,1.710481,1.688902,1.735114,1.739338,0.030578,1.732637,1.739338,1.671383
4,HURDLE__P_DENIS_ALL__M_OWN_POSITIVE_DIFF,P_DENIS_ALL,M_OWN_POSITIVE_DIFF,1.710481,1.688902,1.735114,1.739338,0.030578,1.732637,1.739338,1.671383


,blend,hurdle_candidate,hurdle_standalone_rmsle,blend_rmsle,delta_vs_original,evaluation_type,full_oof_hurdle_weight_for_test
2,CF_ORIGINAL_PLUS__HURDLE__P_DENIS_FEATURE_VIEW...,HURDLE__P_DENIS_FEATURE_VIEWS__M_OWN_POSITIVE_...,1.710070,1.710311,-0.000811,cross_fitted_oof,0.524222
5,CF_ORIGINAL_PLUS__HURDLE__P_DENIS_ALL__M_OWN_P...,HURDLE__P_DENIS_ALL__M_OWN_POSITIVE_ALL_ACTIVE,1.710070,1.710311,-0.000811,cross_fitted_oof,0.524222
3,CF_ORIGINAL_PLUS__HURDLE__P_DENIS_ALL__M_OWN_P...,HURDLE__P_DENIS_ALL__M_OWN_POSITIVE_LEVEL,1.710241,1.710350,-0.000771,cross_fitted_oof,0.520142
0,CF_ORIGINAL_PLUS__HURDLE__P_DENIS_FEATURE_VIEW...,HURDLE__P_DENIS_FEATURE_VIEWS__M_OWN_POSITIVE_...,1.710241,1.710350,-0.000771,cross_fitted_oof,0.520142
1,CF_ORIGINAL_PLUS__HURDLE__P_DENIS_FEATURE_VIEW...,HURDLE__P_DENIS_FEATURE_VIEWS__M_OWN_POSITIVE_...,1.710481,1.710447,-0.000675,cross_fitted_oof,0.514472
4,CF_ORIGINAL_PLUS__HURDLE__P_DENIS_ALL__M_OWN_P...,HURDLE__P_DENIS_ALL__M_OWN_POSITIVE_DIFF,1.710481,1.710447,-0.000675,cross_fitted_oof,0.514472


,fold,CURRENT_BEST_ORIGINAL,HURDLE__P_DENIS_ALL__M_OWN_POSITIVE_ALL_ACTIVE
0,nov_2025,0.461393,0.538607
1,dec_2025,0.435602,0.564398
2,jan_2026,0.539282,0.460718
3,full_oof_for_test,0.475778,0.524222


## 28.4 Hurdle checks, submissions and summary

Основной hurdle и его blend заданы конфигом заранее. Таблица остальных комбинаций остаётся exploratory: выбирать лучшую строку и называть тот же OOF unbiased нельзя.


In [125]:
HURDLE_CHECKS = []


def hurdle_check(name: str, condition: bool, detail: str = ""):
    HURDLE_CHECKS.append({"check": name, "ok": bool(condition), "detail": detail})
    if not condition:
        raise AssertionError(f"{name}: {detail}")


for name, probability in HURDLE_PROBABILITY_OOF.items():
    hurdle_check(f"finite probability: {name}", np.isfinite(probability).all())
    hurdle_check(
        f"probability bounds: {name}",
        bool(np.all((probability > 0) & (probability < 1))),
    )
    hurdle_check(
        f"no hard threshold: {name}",
        np.unique(np.round(probability, 8)).size > 100,
    )

for run in HURDLE_CLASSIFIER_RUNS:
    forbidden_params = {"class_weights", "auto_class_weights", "scale_pos_weight"}
    active_forbidden = {
        key for key in forbidden_params
        if run["params"].get(key) not in {None, False}
    }
    hurdle_check(
        f"natural prevalence: {run['run_name']}", not active_forbidden,
        str(active_forbidden),
    )

for name, magnitude in HURDLE_MAGNITUDE_OOF.items():
    hurdle_check(f"finite magnitude: {name}", np.isfinite(magnitude).all())
    hurdle_check(f"nonnegative magnitude log: {name}", bool(np.all(magnitude >= 0)))

hurdle_check(
    "primary hurdle exists", HURDLE_PRIMARY_COLUMN in OOF.columns,
    HURDLE_PRIMARY_COLUMN,
)
hurdle_check(
    "primary blend is cross-fitted",
    BLENDS[HURDLE_PRIMARY_BLEND].metadata["evaluation_type"] == "cross_fitted_oof",
)
hurdle_check(
    "classifier diagnostics exclude accuracy/F1 as primary metrics",
    not ({"accuracy", "f1"} & set(HURDLE_CLASSIFIER_METRICS.columns)),
)

# The original regression ensemble remains a separate direction and unchanged.
for model_name in MODEL_NAMES:
    current_hash = hashlib.sha256(
        np.ascontiguousarray(OOF[model_name].to_numpy(dtype=np.float64)).tobytes()
    ).hexdigest()
    hurdle_check(
        f"baseline unchanged after hurdle: {model_name}",
        current_hash == BASELINE_PREDICTION_HASHES[model_name],
    )

HURDLE_SUBMISSIONS = {}
if RUN_FINAL_FIT:
    candidates = {
        "submission_hurdle_primary.csv": OOF[HURDLE_PRIMARY_COLUMN],
        "submission_original_plus_hurdle_primary.csv": BLENDS[
            HURDLE_PRIMARY_BLEND
        ].test_pred,
        "submission_hurdle_all_equal.csv": FINAL_PREDICTIONS[
            "HURDLE_ALL_EQUAL"
        ].to_numpy(dtype=np.float64),
        "submission_original_plus_hurdle_all_equal.csv": BLENDS[
            "CF_ORIGINAL_PLUS_HURDLE_ALL_EQUAL"
        ].test_pred,
    }
    # Replace primary OOF placeholder with its final prediction.
    candidates["submission_hurdle_primary.csv"] = FINAL_PREDICTIONS[
        HURDLE_PRIMARY_COLUMN
    ].to_numpy(dtype=np.float64)
    for filename, prediction in candidates.items():
        submission = build_submission_from_prediction(prediction)
        path = HURDLE_SUBMISSION_DIR / filename
        submission.to_csv(path, index=False)
        HURDLE_SUBMISSIONS[filename] = str(path)
        hurdle_check(
            f"submission order: {filename}",
            np.array_equal(submission[USER_COL].to_numpy(), sample_submit[USER_COL].to_numpy()),
        )

pd.DataFrame(HURDLE_CHECKS).to_csv(
    HURDLE_DIR / "hurdle_final_checks.csv", index=False
)

primary_score = weighted_oof_rmsle(OOF, OOF[HURDLE_PRIMARY_COLUMN])
primary_blend_score = weighted_oof_rmsle(
    OOF, BLENDS[HURDLE_PRIMARY_BLEND].oof_pred
)
classifier_summary = (
    HURDLE_CLASSIFIER_METRICS
    .groupby("classifier", as_index=False)
    .agg({
        "logloss": "mean", "brier": "mean",
        "average_precision": "mean", "roc_auc": "mean",
    })
    .sort_values("logloss")
)
HURDLE_SUMMARY = {
    "current_original_regression_rmsle": BASELINE_BEST_SCORE,
    "primary_probability_direction": HURDLE_PRIMARY_PROBABILITY,
    "primary_magnitude_direction": HURDLE_PRIMARY_MAGNITUDE,
    "primary_hurdle_rmsle": primary_score,
    "original_plus_primary_hurdle_rmsle": primary_blend_score,
    "honest_blend_delta": primary_blend_score - BASELINE_BEST_SCORE,
    "best_classifier_by_mean_logloss": classifier_summary.iloc[0]["classifier"],
    "best_exploratory_hurdle": HURDLE_CANDIDATE_METRICS.iloc[0]["candidate"],
    "best_exploratory_original_plus_hurdle": HURDLE_BLEND_ABLATION.iloc[0]["blend"],
    "submissions": HURDLE_SUBMISSIONS,
}
(HURDLE_DIR / "hurdle_manifest.json").write_text(
    json.dumps(HURDLE_SUMMARY, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8",
)

display(classifier_summary)
print("\nDENIS-STYLE HURDLE SUMMARY")
for key, value in HURDLE_SUMMARY.items():
    print(f"{key}: {value}")


,classifier,logloss,brier,average_precision,roc_auc
3,denis_clf__features_xgboost_log_level,0.472701,0.155711,0.883461,0.852732
2,denis_clf__features_xgboost_log_diff,0.473205,0.155851,0.883326,0.852478
1,denis_clf__features_catboost_log_level,0.473610,0.155974,0.883110,0.852302
0,denis_clf__features_catboost_log_diff,0.473655,0.156003,0.883363,0.852422



DENIS-STYLE HURDLE SUMMARY
current_original_regression_rmsle: 1.7111213151570999
primary_probability_direction: P_DENIS_ALL
primary_magnitude_direction: M_OWN_POSITIVE_ALL_ACTIVE
primary_hurdle_rmsle: 1.710070005043962
original_plus_primary_hurdle_rmsle: 1.710310565818276
honest_blend_delta: -0.0008107493388238574
best_classifier_by_mean_logloss: denis_clf__features_xgboost_log_level
best_exploratory_hurdle: HURDLE__P_DENIS_FEATURE_VIEWS__M_OWN_POSITIVE_ALL_ACTIVE
best_exploratory_original_plus_hurdle: CF_ORIGINAL_PLUS__HURDLE__P_DENIS_FEATURE_VIEWS__M_OWN_POSITIVE_ALL_ACTIVE
submissions: {'submission_hurdle_primary.csv': '/kaggle/working/ozon_final/stochastic_diversity/hurdle/submissions/submission_hurdle_primary.csv', 'submission_original_plus_hurdle_primary.csv': '/kaggle/working/ozon_final/stochastic_diversity/hurdle/submissions/submission_original_plus_hurdle_primary.csv', 'submission_hurdle_all_equal.csv': '/kaggle/working/ozon_final/stochastic_diversity/hurdle/submissions/submi

# 29. Error Analysis v2

This block is diagnostic-only and runs after all OOF predictions are available. It does not change features, targets, folds, fitted models or the selected submission. It adds conditional signed/absolute residual correlations, consensus failures, rescue pairs, oracle routing potential, behavioural states, two-dimensional segment matrices, calibration by prediction decile, automatic feature-residual scans, worst-user profiling, shallow diagnostic/routing trees, temporal segment dynamics and January-to-February input drift.

All future-target-dependent fields are used only on held-out OOF rows. Diagnostic trees are intentionally shallow and are not promoted to production routing without a separate temporal validation experiment.


In [126]:
ENABLE_ERROR_ANALYSIS_V2 = True
EA2_DIR = ANALYSIS_DIR / "error_analysis_v2"
EA2_DIR.mkdir(parents=True, exist_ok=True)

EA2_MODEL_NAMES = list(MODEL_NAMES)
for _candidate in [
    "CURRENT_BEST_ORIGINAL",
    globals().get("HURDLE_PRIMARY_COLUMN"),
    globals().get("HURDLE_PRIMARY_BLEND"),
]:
    if _candidate and _candidate in OOF.columns and _candidate not in EA2_MODEL_NAMES:
        EA2_MODEL_NAMES.append(_candidate)

EA2_FEATURE_CANDIDATES = unique_keep_order([
    ANCHOR_COL,
    "days_since_last_purchase", "days_since_last_order", "days_since_last_search",
    "days_since_last_search_cart", "days_since_last_cart", "days_since_last_catalog",
    "active_days_7d", "active_days_30d", "active_days_90d", "active_days_180d",
    "purchase_days_7d", "purchase_days_30d", "purchase_days_90d", "purchase_days_180d",
    "searches_7d", "searches_sum_7d", "searches_sum_14d", "searches_30d", "searches_180d",
    "cart_days_7d", "cart_days_14d", "cart_days_30d", "cart_days_365d",
    "search_order_days_7d", "search_order_days_30d", "search_order_days_180d",
    "gmv_sum_7d", "gmv_sum_14d", "gmv_90d", "gmv_180d", "gmv_sum_365d",
    "gmv_maximum_90d", "gmv_maximum_365d", "gmv_recent_to_prior_ratio",
    "gmv_share30_of_90", "gmv_share30_of_180", "gmv_share30_of_365",
    "gmv_per_order_30d__denis", "gmv_per_order_90d__denis", "gmv_per_order_180d",
    "gmv_per_order_365d", "ph_gmv_mean", "ph_gmv_cv", "ph_gap_cv",
    "order_gap_coefficient_of_variation", "monthly_orders_coefficient_of_variation",
    "search_only_share_active_30d", "search_only_share_active_90d",
    "catalog_only_share_active_90d", "any_cart_no_order_day_30d",
    "search_cart_no_order_day_share_active_30d", "search_cart_no_order_day_share_active_90d",
    "order_per_search_30d", "order_per_search_180d", "cart_per_search_7d",
    "search_to_cart_365d", "hazard_activity_recency_share",
    "forecast_weekday_profile_gmv_projection", "observed_customer_age_days",
])
EA2_FEATURES = [
    c for c in EA2_FEATURE_CANDIDATES
    if c in COMMON_COLUMNS and c not in CATEGORICAL_FEATURES
]


def _ea2_first(frame, candidates, default=np.nan):
    found = next((c for c in candidates if c in frame.columns), None)
    if found is None:
        return pd.Series(default, index=frame.index, dtype=float)
    return pd.to_numeric(frame[found], errors="coerce")


def _ea2_zero_plus_quantiles(values, q=4):
    x = pd.to_numeric(values, errors="coerce")
    result = pd.Series("zero_or_missing", index=x.index, dtype="string")
    positive = x.gt(0) & x.notna()
    if positive.sum() >= q:
        try:
            result.loc[positive] = pd.qcut(
                x.loc[positive], q=q, labels=[f"q{i}" for i in range(1, q + 1)],
                duplicates="drop",
            ).astype("string")
        except ValueError:
            result.loc[positive] = "positive"
    elif positive.any():
        result.loc[positive] = "positive"
    return result.astype(str)


def build_ea2_frame():
    keys = ["fold", SNAPSHOT_COL, USER_COL]
    base = OOF[keys + [TARGET_COL, ANCHOR_COL, *EA2_MODEL_NAMES]].copy()
    parts = []
    for fold in CV_FOLDS:
        raw = load_snapshot(fold["valid_cutoff"], EA2_FEATURES)
        keep = [USER_COL, SNAPSHOT_COL, *[c for c in EA2_FEATURES if c != ANCHOR_COL]]
        raw = raw[keep].copy()
        raw["fold"] = fold["name"]
        parts.append(raw)
    features = pd.concat(parts, ignore_index=True)
    frame = base.merge(features, on=keys, how="left", validate="one_to_one")

    truth_log = np.log1p(frame[TARGET_COL].to_numpy(dtype=np.float64))
    for name in EA2_MODEL_NAMES:
        pred_log = np.log1p(np.clip(frame[name].to_numpy(dtype=np.float64), 0.0, None))
        frame[f"err__{name}"] = pred_log - truth_log
        frame[f"abs_err__{name}"] = np.abs(pred_log - truth_log)

    recency = _ea2_first(frame, ["days_since_last_purchase", "days_since_last_order"])
    activity = _ea2_first(frame, ["active_days_30d", "active_days_7d"], 0).fillna(0)
    searches = _ea2_first(frame, ["searches_30d", "searches_sum_14d", "searches_sum_7d"], 0).fillna(0)
    carts = _ea2_first(frame, ["cart_days_30d", "cart_days_14d", "cart_days_7d"], 0).fillna(0)
    purchases30 = _ea2_first(frame, ["purchase_days_30d", "search_order_days_30d"], 0).fillna(0)
    purchases90 = _ea2_first(frame, ["purchase_days_90d", "purchase_days_180d"], 0).fillna(0)

    state = pd.Series("CATALOG_ONLY", index=frame.index, dtype="string")
    state.loc[activity.le(0) & searches.le(0) & carts.le(0)] = "INACTIVE"
    state.loc[searches.gt(0) & carts.le(0) & purchases30.le(0)] = "SEARCH_ONLY"
    state.loc[carts.gt(0) & purchases30.le(0)] = "CART_NO_ORDER"
    state.loc[searches.gt(0) & carts.gt(0) & purchases30.le(0)] = "SEARCH_CART_NO_ORDER"
    state.loc[recency.gt(90) & purchases90.gt(0)] = "LAPSED_BUYER"
    state.loc[recency.between(31, 90, inclusive="both") & activity.gt(0)] = "REACTIVATING"
    state.loc[recency.le(30) & purchases90.lt(5)] = "RECENT_BUYER"
    state.loc[recency.le(30) & purchases90.ge(5)] = "FREQUENT_BUYER"
    frame["behavior_state"] = state.astype(str)
    frame["target_segment_v2"] = np.where(frame[TARGET_COL].gt(0), "positive", "zero")
    frame["recency_segment_v2"] = pd.cut(
        recency, [-np.inf, 7, 30, 90, 180, 365, np.inf],
        labels=["0_7", "8_30", "31_90", "91_180", "181_365", "365_plus"],
    ).astype("string").fillna("unknown").astype(str)
    frame["frequency_segment_v2"] = pd.cut(
        purchases90, [-np.inf, 0, 1, 3, 7, np.inf],
        labels=["0", "1", "2_3", "4_7", "8_plus"],
    ).astype("string").fillna("unknown").astype(str)
    frame["history_gmv_segment_v2"] = _ea2_zero_plus_quantiles(frame[ANCHOR_COL])
    frame["future_gmv_segment_v2"] = _ea2_zero_plus_quantiles(frame[TARGET_COL])
    return frame


if RUN_CV and RUN_ERROR_ANALYSIS and ENABLE_ERROR_ANALYSIS_V2:
    OOF_EA2 = build_ea2_frame()
    OOF_EA2.to_parquet(EA2_DIR / "oof_error_analysis_v2.parquet", index=False)
    print("EA2 models:", EA2_MODEL_NAMES)
    print("EA2 diagnostic features:", len(EA2_FEATURES), EA2_FEATURES)


EA2 models: ['xgboost_log_level', 'catboost_log_level', 'catboost_log_diff', 'xgboost_log_diff', 'CURRENT_BEST_ORIGINAL', 'HURDLE__P_DENIS_ALL__M_OWN_POSITIVE_ALL_ACTIVE']
EA2 diagnostic features: 57 ['gmv_30d', 'days_since_last_purchase', 'days_since_last_order', 'days_since_last_search', 'days_since_last_search_cart', 'days_since_last_cart', 'days_since_last_catalog', 'active_days_7d', 'active_days_30d', 'active_days_90d', 'active_days_180d', 'purchase_days_7d', 'purchase_days_30d', 'purchase_days_90d', 'purchase_days_180d', 'searches_7d', 'searches_sum_7d', 'searches_sum_14d', 'searches_30d', 'searches_180d', 'cart_days_7d', 'cart_days_14d', 'cart_days_30d', 'cart_days_365d', 'search_order_days_7d', 'search_order_days_30d', 'search_order_days_180d', 'gmv_90d', 'gmv_180d', 'gmv_maximum_90d', 'gmv_maximum_365d', 'gmv_recent_to_prior_ratio', 'gmv_share30_of_90', 'gmv_share30_of_180', 'gmv_share30_of_365', 'gmv_per_order_30d__denis', 'gmv_per_order_90d__denis', 'gmv_per_order_180d', 'gm

In [127]:
from scipy.stats import ks_2samp, wasserstein_distance


def ea2_conditional_correlations(frame):
    rows = []
    groupings = [("global", pd.Series("all", index=frame.index))]
    groupings += [("fold", frame["fold"]), ("behavior_state", frame["behavior_state"])]
    groupings += [("target_segment", frame["target_segment_v2"])]
    for group_type, labels in groupings:
        for segment in pd.Series(labels).dropna().unique():
            idx = pd.Series(labels, index=frame.index).eq(segment).to_numpy()
            if idx.sum() < 30:
                continue
            for a, b in itertools.combinations(EA2_MODEL_NAMES, 2):
                for kind, prefix in [("signed", "err__"), ("absolute", "abs_err__")]:
                    corr = np.corrcoef(
                        frame.loc[idx, prefix + a].to_numpy(dtype=float),
                        frame.loc[idx, prefix + b].to_numpy(dtype=float),
                    )[0, 1]
                    rows.append({
                        "group_type": group_type, "segment": str(segment),
                        "rows": int(idx.sum()), "error_kind": kind,
                        "model_a": a, "model_b": b, "correlation": float(corr),
                    })
    return pd.DataFrame(rows)


def ea2_segment_errors(frame, segment_columns):
    rows = []
    for segment_col in segment_columns:
        for segment, part in frame.groupby(segment_col, observed=False, dropna=False):
            truth_log = np.log1p(part[TARGET_COL].to_numpy(dtype=float))
            for model in EA2_MODEL_NAMES:
                err = np.log1p(np.clip(part[model].to_numpy(dtype=float), 0, None)) - truth_log
                rows.append({
                    "segment_type": segment_col, "segment": str(segment), "rows": len(part),
                    "model": model, "rmsle": float(np.sqrt(np.mean(err ** 2))),
                    "mae_log": float(np.mean(np.abs(err))), "bias_log": float(np.mean(err)),
                    "median_error": float(np.median(err)),
                    "under_rate": float(np.mean(err < 0)), "over_rate": float(np.mean(err > 0)),
                    "p90_abs": float(np.quantile(np.abs(err), .90)),
                    "p95_abs": float(np.quantile(np.abs(err), .95)),
                })
    return pd.DataFrame(rows)


def ea2_consensus_and_rescues(frame):
    abs_matrix = frame[[f"abs_err__{m}" for m in EA2_MODEL_NAMES]].to_numpy(dtype=float)
    signed = frame[[f"err__{m}" for m in EA2_MODEL_NAMES]].to_numpy(dtype=float)
    p90 = np.quantile(abs_matrix, .90, axis=0)
    severe = abs_matrix > p90
    consensus = frame[["fold", USER_COL, TARGET_COL, ANCHOR_COL, "behavior_state"]].copy()
    consensus["models_over_p90"] = severe.sum(axis=1)
    consensus["all_overpredict"] = np.all(signed > 0, axis=1)
    consensus["all_underpredict"] = np.all(signed < 0, axis=1)
    consensus["consensus_failure"] = consensus["models_over_p90"].ge(min(4, len(EA2_MODEL_NAMES)))
    consensus["max_abs_log_error"] = abs_matrix.max(axis=1)

    rescue_rows = []
    for ia, a in enumerate(EA2_MODEL_NAMES):
        a80 = np.quantile(abs_matrix[:, ia], .80)
        for ib, b in enumerate(EA2_MODEL_NAMES):
            if a == b:
                continue
            b50 = np.quantile(abs_matrix[:, ib], .50)
            mask = (abs_matrix[:, ia] > a80) & (abs_matrix[:, ib] < b50)
            rescue_rows.append({
                "failed_model": a, "rescue_model": b, "users": int(mask.sum()),
                "share": float(mask.mean()),
                "failed_mae_log": float(abs_matrix[mask, ia].mean()) if mask.any() else np.nan,
                "rescue_mae_log": float(abs_matrix[mask, ib].mean()) if mask.any() else np.nan,
                "anchor_median": float(frame.loc[mask, ANCHOR_COL].median()) if mask.any() else np.nan,
                "positive_target_rate": float(frame.loc[mask, TARGET_COL].gt(0).mean()) if mask.any() else np.nan,
            })
    return consensus, pd.DataFrame(rescue_rows).sort_values("users", ascending=False)


def ea2_oracle(frame):
    errors = frame[[f"abs_err__{m}" for m in EA2_MODEL_NAMES]].to_numpy(dtype=float)
    winners = errors.argmin(axis=1)
    predictions = frame[EA2_MODEL_NAMES].to_numpy(dtype=float)
    oracle_pred = predictions[np.arange(len(frame)), winners]
    rows = [{
        "fold": "weighted_all", "oracle_rmsle": weighted_oof_rmsle(frame, oracle_pred),
        "best_single_rmsle": min(weighted_oof_rmsle(frame, frame[m]) for m in EA2_MODEL_NAMES),
    }]
    for fold in CV_FOLDS:
        mask = frame["fold"].eq(fold["name"]).to_numpy()
        rows.append({
            "fold": fold["name"],
            "oracle_rmsle": rmsle(frame.loc[mask, TARGET_COL], oracle_pred[mask]),
            "best_single_rmsle": min(rmsle(frame.loc[mask, TARGET_COL], frame.loc[mask, m]) for m in EA2_MODEL_NAMES),
        })
    shares = pd.Series(np.asarray(EA2_MODEL_NAMES, dtype=object)[winners]).value_counts(normalize=True)
    return pd.DataFrame(rows), shares.rename_axis("model").reset_index(name="oracle_win_share")


def ea2_prediction_deciles(frame):
    rows = []
    for model in EA2_MODEL_NAMES:
        for fold, part in frame.groupby("fold"):
            ranked = part.copy()
            ranked["decile"] = pd.qcut(
                ranked[model].rank(method="first"), 10, labels=False, duplicates="drop"
            )
            for decile, g in ranked.groupby("decile"):
                rows.append({
                    "model": model, "fold": fold, "prediction_decile": int(decile),
                    "rows": len(g), "prediction_log_mean": float(np.log1p(g[model]).mean()),
                    "truth_log_mean": float(np.log1p(g[TARGET_COL]).mean()),
                    "prediction_raw_mean": float(g[model].mean()), "truth_raw_mean": float(g[TARGET_COL].mean()),
                })
    return pd.DataFrame(rows)


if RUN_CV and RUN_ERROR_ANALYSIS and ENABLE_ERROR_ANALYSIS_V2:
    EA2_CONDITIONAL_CORR = ea2_conditional_correlations(OOF_EA2)
    EA2_SEGMENT_ERRORS = ea2_segment_errors(OOF_EA2, [
        "fold", "behavior_state", "target_segment_v2", "recency_segment_v2",
        "frequency_segment_v2", "history_gmv_segment_v2", "future_gmv_segment_v2",
    ])
    EA2_CONSENSUS, EA2_RESCUES = ea2_consensus_and_rescues(OOF_EA2)
    EA2_ORACLE, EA2_ORACLE_SHARES = ea2_oracle(OOF_EA2)
    EA2_PREDICTION_DECILES = ea2_prediction_deciles(OOF_EA2)

    EA2_CONDITIONAL_CORR.to_csv(EA2_DIR / "conditional_error_correlations.csv", index=False)
    EA2_SEGMENT_ERRORS.to_csv(EA2_DIR / "segment_error_direction.csv", index=False)
    EA2_CONSENSUS.to_parquet(EA2_DIR / "consensus_failures.parquet", index=False)
    EA2_RESCUES.to_csv(EA2_DIR / "rescue_pairs.csv", index=False)
    EA2_ORACLE.to_csv(EA2_DIR / "oracle_routing_potential.csv", index=False)
    EA2_ORACLE_SHARES.to_csv(EA2_DIR / "oracle_model_win_share.csv", index=False)
    EA2_PREDICTION_DECILES.to_csv(EA2_DIR / "prediction_decile_calibration.csv", index=False)

    display(EA2_ORACLE)
    display(EA2_RESCUES.head(20))


,fold,oracle_rmsle,best_single_rmsle
0,weighted_all,1.624802,1.710070
1,nov_2025,1.652783,1.732033
2,dec_2025,1.647161,1.736008
3,jan_2026,1.587750,1.671150


,failed_model,rescue_model,users,share,failed_mae_log,rescue_mae_log,anchor_median,positive_target_rate
7,catboost_log_level,xgboost_log_diff,12,0.000016,2.488141,0.893790,3610.980957,0.583333
6,catboost_log_level,catboost_log_diff,11,0.000015,2.561474,0.902878,0.066573,0.272727
1,xgboost_log_level,catboost_log_diff,9,0.000012,2.445525,0.955815,0.033068,0.111111
11,catboost_log_diff,catboost_log_level,8,0.000011,2.285710,0.870437,0.032924,1.000000
10,catboost_log_diff,xgboost_log_level,7,0.000009,2.324498,0.885868,0.032540,0.857143
15,xgboost_log_diff,xgboost_log_level,7,0.000009,2.295064,0.692443,294.104126,0.857143
16,xgboost_log_diff,catboost_log_level,7,0.000009,2.304232,0.683029,11209.170898,1.000000
2,xgboost_log_level,xgboost_log_diff,4,0.000005,2.402378,0.732031,3586.073242,0.500000
19,xgboost_log_diff,HURDLE__P_DENIS_ALL__M_OWN_POSITIVE_ALL_ACTIVE,3,0.000004,2.212688,0.744242,11874.480469,1.000000
17,xgboost_log_diff,catboost_log_diff,2,0.000003,2.265383,0.996820,147.052063,0.500000


In [ ]:
def ea2_feature_scan(frame, model):
    err = frame[f"err__{model}"].to_numpy(dtype=float)
    abs_err = np.abs(err)
    rows = []
    for feature in EA2_FEATURES:
        x = pd.to_numeric(frame[feature], errors="coerce").to_numpy(dtype=float)
        valid = np.isfinite(x)
        if valid.sum() < 100 or np.nanstd(x[valid]) <= 0:
            continue
        q20, q80 = np.nanquantile(x[valid], [.20, .80])
        low, high = valid & (x <= q20), valid & (x >= q80)
        rows.append({
            "model": model, "feature": feature, "rows": int(valid.sum()),
            "corr_signed": float(np.corrcoef(x[valid], err[valid])[0, 1]),
            "corr_absolute": float(np.corrcoef(x[valid], abs_err[valid])[0, 1]),
            "top_minus_bottom_signed": float(err[high].mean() - err[low].mean()),
            "top_minus_bottom_absolute": float(abs_err[high].mean() - abs_err[low].mean()),
        })
    return pd.DataFrame(rows)


def ea2_worst_profile(frame, model):
    bad = frame[f"abs_err__{model}"].ge(frame[f"abs_err__{model}"].quantile(.90)).to_numpy()
    rows = []
    for feature in EA2_FEATURES:
        x = pd.to_numeric(frame[feature], errors="coerce").to_numpy(dtype=float)
        xb, xg = x[bad & np.isfinite(x)], x[(~bad) & np.isfinite(x)]
        if len(xb) < 30 or len(xg) < 30:
            continue
        pooled = np.sqrt((np.var(xb) + np.var(xg)) / 2.0)
        rows.append({
            "model": model, "feature": feature,
            "bad_median": float(np.median(xb)), "other_median": float(np.median(xg)),
            "bad_mean": float(np.mean(xb)), "other_mean": float(np.mean(xg)),
            "standardized_mean_difference": float((np.mean(xb) - np.mean(xg)) / max(pooled, 1e-12)),
            "bad_zero_rate": float(np.mean(xb == 0)), "other_zero_rate": float(np.mean(xg == 0)),
            "ks_statistic": float(ks_2samp(xb, xg).statistic),
        })
    return pd.DataFrame(rows).sort_values("ks_statistic", ascending=False)


def ea2_psi(reference, current, bins=10):
    ref = np.asarray(reference, dtype=float); cur = np.asarray(current, dtype=float)
    ref = ref[np.isfinite(ref)]; cur = cur[np.isfinite(cur)]
    if len(ref) < 100 or len(cur) < 100:
        return np.nan
    edges = np.unique(np.quantile(ref, np.linspace(0, 1, bins + 1)))
    if len(edges) < 3:
        return 0.0
    edges[0], edges[-1] = -np.inf, np.inf
    rp = np.histogram(ref, bins=edges)[0] / len(ref)
    cp = np.histogram(cur, bins=edges)[0] / len(cur)
    rp, cp = np.clip(rp, 1e-6, None), np.clip(cp, 1e-6, None)
    return float(np.sum((cp - rp) * np.log(cp / rp)))


def ea2_drift():
    jan = load_snapshot("2026-01-14", EA2_FEATURES)
    feb = load_snapshot(PREDICTION_CUTOFF, EA2_FEATURES)
    rows = []
    for feature in EA2_FEATURES:
        a = pd.to_numeric(jan[feature], errors="coerce").to_numpy(dtype=float)
        b = pd.to_numeric(feb[feature], errors="coerce").to_numpy(dtype=float)
        av, bv = a[np.isfinite(a)], b[np.isfinite(b)]
        rows.append({
            "feature": feature, "jan_missing_rate": float(np.mean(~np.isfinite(a))),
            "feb_missing_rate": float(np.mean(~np.isfinite(b))),
            "jan_mean": float(np.mean(av)) if len(av) else np.nan,
            "feb_mean": float(np.mean(bv)) if len(bv) else np.nan,
            "jan_median": float(np.median(av)) if len(av) else np.nan,
            "feb_median": float(np.median(bv)) if len(bv) else np.nan,
            "psi": ea2_psi(av, bv),
            "ks": float(ks_2samp(av, bv).statistic) if len(av) and len(bv) else np.nan,
            "wasserstein": float(wasserstein_distance(av, bv)) if len(av) and len(bv) else np.nan,
        })
    del jan, feb
    gc.collect()
    return pd.DataFrame(rows).sort_values(["psi", "ks"], ascending=False)


def ea2_diagnostic_trees(frame, baseline_model):
    from sklearn.impute import SimpleImputer
    from sklearn.pipeline import make_pipeline
    from sklearn.tree import DecisionTreeClassifier, export_text

    usable = [c for c in EA2_FEATURES if pd.to_numeric(frame[c], errors="coerce").notna().sum() >= 100]
    X = frame[usable].apply(pd.to_numeric, errors="coerce")
    bad = frame[f"abs_err__{baseline_model}"].ge(
        frame[f"abs_err__{baseline_model}"].quantile(.90)
    ).astype(int)
    min_leaf = max(200, len(frame) // 100)
    bad_tree = make_pipeline(
        SimpleImputer(strategy="median"),
        DecisionTreeClassifier(max_depth=3, min_samples_leaf=min_leaf, random_state=RANDOM_SEED),
    )
    bad_tree.fit(X, bad)
    bad_text = export_text(bad_tree[-1], feature_names=usable)

    level_names = [m for m in MODEL_NAMES if MODEL_SPECS[m]["target_mode"] == "log_level"]
    diff_names = [m for m in MODEL_NAMES if MODEL_SPECS[m]["target_mode"] == "log_diff"]
    level_pred = np.expm1(np.mean(np.log1p(frame[level_names].to_numpy(dtype=float)), axis=1))
    diff_pred = np.expm1(np.mean(np.log1p(frame[diff_names].to_numpy(dtype=float)), axis=1))
    truth_log = np.log1p(frame[TARGET_COL].to_numpy(dtype=float))
    route_target = (
        np.abs(np.log1p(diff_pred) - truth_log) < np.abs(np.log1p(level_pred) - truth_log)
    ).astype(int)
    route_tree = make_pipeline(
        SimpleImputer(strategy="median"),
        DecisionTreeClassifier(max_depth=3, min_samples_leaf=min_leaf, random_state=RANDOM_SEED),
    )
    route_tree.fit(X, route_target)
    route_text = export_text(route_tree[-1], feature_names=usable)
    return bad_text, route_text


if RUN_CV and RUN_ERROR_ANALYSIS and ENABLE_ERROR_ANALYSIS_V2:
    _baseline = "CURRENT_BEST_ORIGINAL" if "CURRENT_BEST_ORIGINAL" in EA2_MODEL_NAMES else BASE_RANKING.iloc[0]["model"]
    EA2_FEATURE_BIAS_SCAN = pd.concat(
        [ea2_feature_scan(OOF_EA2, m) for m in EA2_MODEL_NAMES], ignore_index=True
    )
    EA2_WORST_PROFILE = ea2_worst_profile(OOF_EA2, _baseline)
    EA2_DRIFT = ea2_drift()
    EA2_FEATURE_BIAS_SCAN.to_csv(EA2_DIR / "automatic_feature_bias_scan.csv", index=False)
    EA2_WORST_PROFILE.to_csv(EA2_DIR / "worst_user_profile.csv", index=False)
    EA2_DRIFT.to_csv(EA2_DIR / "jan_to_feb_feature_drift.csv", index=False)

    # Feature -> residual deciles for the strongest automatic scan signals.
    _top_features = (
        EA2_FEATURE_BIAS_SCAN.loc[EA2_FEATURE_BIAS_SCAN["model"].eq(_baseline)]
        .assign(strength=lambda x: x[["corr_signed", "corr_absolute"]].abs().max(axis=1))
        .nlargest(12, "strength")["feature"].tolist()
    )
    _decile_rows = []
    for feature in _top_features:
        tmp = OOF_EA2[[feature, f"err__{_baseline}", f"abs_err__{_baseline}"]].copy()
        tmp["decile"] = pd.qcut(tmp[feature].rank(method="first"), 10, labels=False, duplicates="drop")
        for decile, part in tmp.groupby("decile"):
            _decile_rows.append({
                "feature": feature, "decile": int(decile), "rows": len(part),
                "feature_mean": float(part[feature].mean()),
                "signed_error_mean": float(part[f"err__{_baseline}"].mean()),
                "absolute_error_mean": float(part[f"abs_err__{_baseline}"].mean()),
            })
    EA2_FEATURE_DECILES = pd.DataFrame(_decile_rows)
    EA2_FEATURE_DECILES.to_csv(EA2_DIR / "feature_residual_deciles.csv", index=False)

    bad_tree_text, route_tree_text = ea2_diagnostic_trees(OOF_EA2, _baseline)
    (EA2_DIR / "diagnostic_bad_prediction_tree.txt").write_text(bad_tree_text, encoding="utf-8")
    (EA2_DIR / "diagnostic_logdiff_vs_loglevel_routing_tree.txt").write_text(route_tree_text, encoding="utf-8")

    # Two-dimensional matrices: recency x frequency and history x future.
    _two_d = EA2_SEGMENT_ERRORS.loc[EA2_SEGMENT_ERRORS["model"].eq(_baseline)]
    rec_freq = (
        OOF_EA2.groupby(["recency_segment_v2", "frequency_segment_v2"], observed=False)
        .apply(lambda g: rmsle(g[TARGET_COL], g[_baseline]))
        .rename("rmsle").reset_index()
    )
    rec_freq.pivot(index="recency_segment_v2", columns="frequency_segment_v2", values="rmsle").to_csv(
        EA2_DIR / "matrix_recency_x_frequency.csv"
    )
    transition = pd.crosstab(
        OOF_EA2["history_gmv_segment_v2"], OOF_EA2["future_gmv_segment_v2"], normalize="index"
    )
    transition.to_csv(EA2_DIR / "matrix_history_x_future_transition.csv")

    EA2_MANIFEST = {
        "models": EA2_MODEL_NAMES, "diagnostic_features": EA2_FEATURES,
        "baseline_for_profiling": _baseline,
        "consensus_failure_rows": int(EA2_CONSENSUS["consensus_failure"].sum()),
        "oracle_weighted_rmsle": float(EA2_ORACLE.iloc[0]["oracle_rmsle"]),
        "warning": "diagnostic trees and oracle are upper-bound analyses, not production routing",
    }
    (EA2_DIR / "manifest.json").write_text(
        json.dumps(EA2_MANIFEST, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    display(EA2_WORST_PROFILE.head(20))
    display(EA2_DRIFT.head(20))


,model,feature,bad_median,other_median,bad_mean,other_mean,standardized_mean_difference,bad_zero_rate,other_zero_rate,ks_statistic
55,CURRENT_BEST_ORIGINAL,forecast_weekday_profile_gmv_projection,41.092236,23.948665,74.386804,77.845959,-0.021714,0.102027,0.164132,0.122366
30,CURRENT_BEST_ORIGINAL,gmv_maximum_365d,131.439873,89.463657,273.376525,215.672267,0.106104,0.102493,0.164674,0.120296
28,CURRENT_BEST_ORIGINAL,gmv_180d,265.182297,153.701668,511.394549,534.001924,-0.020143,0.140520,0.206575,0.119871
43,CURRENT_BEST_ORIGINAL,monthly_orders_coefficient_of_variation,0.617456,0.547101,0.611043,0.527083,0.263034,0.105040,0.166370,0.113633
38,CURRENT_BEST_ORIGINAL,gmv_per_order_365d,31.364357,26.101948,47.553992,37.653724,0.131198,0.000000,0.000000,0.113222
39,CURRENT_BEST_ORIGINAL,ph_gmv_mean,45.508743,37.161121,69.710913,53.697492,0.154278,0.000000,0.000000,0.111024
27,CURRENT_BEST_ORIGINAL,gmv_90d,119.724697,68.000580,271.239215,285.543906,-0.021459,0.208880,0.277945,0.108206
36,CURRENT_BEST_ORIGINAL,gmv_per_order_90d__denis,20.543507,15.585402,37.217873,25.933341,0.144756,0.208880,0.277945,0.107182
29,CURRENT_BEST_ORIGINAL,gmv_maximum_90d,56.135487,36.430740,139.816202,112.766700,0.078846,0.208880,0.277945,0.105074
37,CURRENT_BEST_ORIGINAL,gmv_per_order_180d,29.132887,24.358212,46.881772,36.678652,0.134727,0.000000,0.000000,0.104559


,feature,jan_missing_rate,feb_missing_rate,jan_mean,feb_mean,jan_median,feb_median,psi,ks,wasserstein
56,observed_customer_age_days,0.000000,0.000000,336.598240,366.598240,373.000000,403.000000,1.440255,0.754744,30.000000
32,gmv_share30_of_90,0.250048,0.246660,0.351433,0.300830,0.266537,0.201237,0.025139,0.072254,0.050603
34,gmv_share30_of_365,0.137372,0.127884,0.129721,0.101688,0.045139,0.029792,0.024779,0.067996,0.028032
33,gmv_share30_of_180,0.179272,0.170548,0.200971,0.163454,0.098375,0.068628,0.024733,0.067593,0.037518
1,days_since_last_purchase,0.135768,0.123328,40.975604,43.795180,18.000000,18.000000,0.018915,0.043426,3.029346
2,days_since_last_order,0.000000,0.000000,1392.956660,1271.550680,23.000000,24.000000,0.017602,0.028728,123.209948
6,days_since_last_catalog,0.081252,0.069188,51.293704,53.791743,24.000000,25.000000,0.011095,0.033778,2.519061
5,days_since_last_cart,0.020740,0.015752,20.786145,21.392275,7.000000,7.000000,0.010303,0.022284,1.063542
4,days_since_last_search_cart,0.024908,0.019476,22.265751,22.996165,8.000000,8.000000,0.010032,0.024122,1.128471
35,gmv_per_order_30d__denis,0.000000,0.000000,21.293660,18.034195,7.668435,5.987786,0.009461,0.042532,3.259464


In [ ]:
# =============================================================================
# EXTERNAL / COLLEAGUES BLEND LAB
#
# Требует уже рассчитанные:
#   sample_submit
#   USER_COL
#   SUBMISSION_TARGET_COL
#   FINAL_PREDICTIONS
#   BLENDS
#   FINAL_BLEND_NAME
#   OUTPUT_DIR
#
# Использует test-only submissions коллег:
#   63_optimum_MARAT.csv
#   submission_final_calibrated.csv
#   submission_final_raw.csv
#
# ВАЖНО:
#   Эти прогнозы не имеют OOF, поэтому этот блок НЕ использует их
#   для selection-safe выбора по CV. Он только:
#       1) проверяет alignment по user_id;
#       2) смотрит diversity / корреляции;
#       3) строит разные blend candidates;
#       4) пробует Polar-style post-calibration;
#       5) сохраняет готовые submission CSV.
# =============================================================================

from pathlib import Path
import json
import math
import numpy as np
import pandas as pd


# -----------------------------------------------------------------------------
# 0. CONFIG
# -----------------------------------------------------------------------------

COLLEAGUES_DATASET_HINT = "colleagues"

MARAT_FILENAME = "63_optimum_MARAT.csv"
POLAR_CALIBRATED_FILENAME = "submission_final_calibrated.csv"
POLAR_RAW_FILENAME = "submission_final_raw.csv"

EXTERNAL_BLEND_DIR = OUTPUT_DIR / "external_colleague_blends"
EXTERNAL_BLEND_DIR.mkdir(parents=True, exist_ok=True)

# Основная сетка веса ВНЕШНЕГО прогноза.
# Собственный ансамбль получает вес 1 - w.
PAIR_WEIGHTS_LOG = (
    0.05,
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.40,
    0.50,
)

# Для raw-space достаточно более редкой сетки:
# RMSLE естественнее соответствует смешиванию в log1p.
PAIR_WEIGHTS_RAW = (
    0.10,
    0.25,
    0.40,
)

# Own + Marat + Polar calibrated.
# TOTAL_EXTERNAL_WEIGHT = общий вес двух внешних прогнозов.
TRIPLE_TOTAL_EXTERNAL_WEIGHTS = (
    0.15,
    0.30,
    0.45,
)

# Какая доля внешней части принадлежит Marat.
# Остаток принадлежит Polar calibrated.
TRIPLE_MARAT_SHARES = (
    1 / 3,
    0.50,
    2 / 3,
)

# Отдельный эксперимент:
# own + submission_final_raw -> затем Polar post-calibration.
POLAR_RAW_BLEND_WEIGHTS = (
    0.10,
    0.20,
    0.30,
    0.40,
    0.50,
)

# Полярная финальная коррекция из описания коллег.
POLAR_SPREAD_SCALE = 1.0454
POLAR_TARGET_LOG_MEAN = 2.329351

# На каких candidate families дополнительно строить post-calibrated версию.
APPLY_POLAR_CALIBRATION_TO_MARAT_BLENDS = True
APPLY_POLAR_CALIBRATION_TO_POLAR_CALIBRATED_BLENDS = True
APPLY_POLAR_CALIBRATION_TO_TRIPLE_BLENDS = True

# Сохранять все построенные варианты.
SAVE_ALL_EXTERNAL_CANDIDATES = True

# При желании после изучения результатов сюда можно поставить имя кандидата.
# Например:
#
# EXTERNAL_BLEND_CHOSEN = "own090_marat010_log_polarcal"
#
EXTERNAL_BLEND_CHOSEN = None


# -----------------------------------------------------------------------------
# 1. DISCOVER FILES
# -----------------------------------------------------------------------------

def discover_external_submission(filename: str) -> Path:
    matches = []

    for root in [Path("/kaggle/input"), Path.cwd()]:
        if not root.exists():
            continue
        matches.extend(root.rglob(filename))

    matches = [p for p in matches if p.is_file()]

    if not matches:
        raise FileNotFoundError(
            f"Не найден {filename}. "
            f"Добавьте Kaggle dataset с colleagues_submissions."
        )

    hint = COLLEAGUES_DATASET_HINT.lower()

    matches.sort(
        key=lambda p: (
            hint not in str(p).lower(),
            len(str(p)),
        )
    )

    if len(matches) > 1:
        print(
            f"[external] несколько кандидатов для {filename}:",
            [str(x) for x in matches],
        )

    return matches[0]


MARAT_PATH = discover_external_submission(MARAT_FILENAME)
POLAR_CALIBRATED_PATH = discover_external_submission(
    POLAR_CALIBRATED_FILENAME
)
POLAR_RAW_PATH = discover_external_submission(POLAR_RAW_FILENAME)

print("External submissions:")
print("  MARAT            :", MARAT_PATH)
print("  Polar calibrated :", POLAR_CALIBRATED_PATH)
print("  Polar raw        :", POLAR_RAW_PATH)


# -----------------------------------------------------------------------------
# 2. STRICT LOAD / USER ALIGNMENT
# -----------------------------------------------------------------------------

def load_external_prediction(
    path: Path,
    *,
    label: str,
) -> np.ndarray:
    frame = pd.read_csv(path)

    if USER_COL not in frame.columns:
        raise KeyError(
            f"{label}: в {path.name} нет {USER_COL}. "
            "Позиционное объединение запрещено."
        )

    if frame[USER_COL].duplicated().any():
        raise ValueError(
            f"{label}: duplicate {USER_COL}"
        )

    prediction_columns = [
        c for c in frame.columns
        if c != USER_COL
    ]

    if SUBMISSION_TARGET_COL in prediction_columns:
        pred_col = SUBMISSION_TARGET_COL
    elif len(prediction_columns) == 1:
        pred_col = prediction_columns[0]
    else:
        raise ValueError(
            f"{label}: невозможно однозначно определить prediction column. "
            f"Колонки: {frame.columns.tolist()}"
        )

    aligned = (
        sample_submit[[USER_COL]]
        .merge(
            frame[[USER_COL, pred_col]],
            on=USER_COL,
            how="left",
            validate="one_to_one",
            sort=False,
        )
    )

    if len(aligned) != len(sample_submit):
        raise AssertionError(
            f"{label}: row count changed after merge"
        )

    if aligned[pred_col].isna().any():
        missing = int(aligned[pred_col].isna().sum())
        raise AssertionError(
            f"{label}: {missing} пользователей отсутствуют"
        )

    pred = aligned[pred_col].to_numpy(dtype=np.float64)

    if not np.isfinite(pred).all():
        raise ValueError(
            f"{label}: prediction contains NaN/inf"
        )

    if np.any(pred < 0):
        raise ValueError(
            f"{label}: submission содержит отрицательные predictions"
        )

    return pred


PRED_MARAT = load_external_prediction(
    MARAT_PATH,
    label="MARAT",
)

PRED_POLAR_CALIBRATED = load_external_prediction(
    POLAR_CALIBRATED_PATH,
    label="POLAR_CALIBRATED",
)

PRED_POLAR_RAW = load_external_prediction(
    POLAR_RAW_PATH,
    label="POLAR_RAW",
)


# -----------------------------------------------------------------------------
# 3. OUR CURRENT FINAL BLEND
# -----------------------------------------------------------------------------

if FINAL_BLEND_NAME not in BLENDS:
    raise KeyError(
        f"FINAL_BLEND_NAME={FINAL_BLEND_NAME!r} not found in BLENDS"
    )

PRED_OWN = BLENDS[FINAL_BLEND_NAME].test_pred

if PRED_OWN is None:
    raise RuntimeError(
        f"{FINAL_BLEND_NAME} has no test prediction"
    )

PRED_OWN = np.asarray(
    PRED_OWN,
    dtype=np.float64,
)

if len(PRED_OWN) != len(sample_submit):
    raise AssertionError(
        "Own final prediction length mismatch"
    )

if not np.isfinite(PRED_OWN).all():
    raise ValueError(
        "Own final predictions contain NaN/inf"
    )

PRED_OWN = np.clip(
    PRED_OWN,
    0.0,
    None,
)


# -----------------------------------------------------------------------------
# 4. BLENDING HELPERS
# -----------------------------------------------------------------------------

def normalize_weights(weights):
    w = np.asarray(
        weights,
        dtype=np.float64,
    )

    if len(w) == 0:
        raise ValueError("Empty weights")

    if not np.isfinite(w).all():
        raise ValueError("Weights contain NaN/inf")

    if np.any(w < 0):
        raise ValueError("Weights must be non-negative")

    total = float(w.sum())

    if total <= 0:
        raise ValueError("Weights sum must be positive")

    return w / total


def blend_predictions(
    predictions,
    weights,
    *,
    space="log",
):
    """
    space='log':
        sum(w * log1p(pred))
        -> expm1

    space='raw':
        sum(w * pred)
    """

    arrays = [
        np.asarray(x, dtype=np.float64)
        for x in predictions
    ]

    n = len(arrays[0])

    if any(len(x) != n for x in arrays):
        raise ValueError(
            "Prediction length mismatch"
        )

    matrix = np.column_stack(arrays)

    if not np.isfinite(matrix).all():
        raise ValueError(
            "Blend input contains NaN/inf"
        )

    if np.any(matrix < 0):
        raise ValueError(
            "Blend input contains negative predictions"
        )

    w = normalize_weights(weights)

    if space == "log":
        blended_log = (
            np.log1p(matrix) @ w
        )
        result = np.expm1(blended_log)

    elif space == "raw":
        result = matrix @ w

    else:
        raise ValueError(
            "space must be 'log' or 'raw'"
        )

    result = np.clip(
        result,
        0.0,
        None,
    )

    if not np.isfinite(result).all():
        raise ValueError(
            "Blend produced NaN/inf"
        )

    return result


# -----------------------------------------------------------------------------
# 5. POLAR-STYLE POST-CALIBRATION
# -----------------------------------------------------------------------------

def find_shift_for_target_clipped_log_mean(
    centered_log_prediction,
    *,
    target_log_mean,
    iterations=120,
):
    """
    Ищет c:

        mean(max(centered_log_prediction + c, 0))
            == target_log_mean
    """

    x = np.asarray(
        centered_log_prediction,
        dtype=np.float64,
    )

    target = float(target_log_mean)

    if target < 0:
        raise ValueError(
            "target_log_mean must be >= 0"
        )

    def objective(shift):
        return float(
            np.maximum(
                x + shift,
                0.0,
            ).mean()
        )

    # Нижняя граница практически гарантирует 0.
    low = (
        -float(np.max(x))
        - target
        - 10.0
    )

    # Начальная верхняя граница.
    high = (
        float(np.max(np.abs(x)))
        + target
        + 10.0
    )

    # На всякий случай расширяем high.
    for _ in range(30):
        if objective(high) >= target:
            break
        high *= 2.0
    else:
        raise RuntimeError(
            "Cannot bracket Polar calibration shift"
        )

    for _ in range(int(iterations)):
        mid = (low + high) / 2.0

        if objective(mid) < target:
            low = mid
        else:
            high = mid

    return (low + high) / 2.0


def polar_post_calibrate(
    raw_prediction,
    *,
    spread_scale=POLAR_SPREAD_SCALE,
    target_log_mean=POLAR_TARGET_LOG_MEAN,
):
    """
    Реализация описанной коллегами коррекции:

    1.
        p = log1p(raw_prediction)

    2. масштаб разброса:
        p_scaled =
            mean(p)
            + (p - mean(p)) * spread_scale

    3. повторное центрирование:
        centered =
            p_scaled - mean(p_scaled)

    4. ищем shift c:
        mean(max(centered + c, 0))
            = target_log_mean

    5.
        calibrated_log =
            max(centered + c, 0)

    6.
        calibrated_raw =
            exp(calibrated_log) - 1
    """

    raw = np.asarray(
        raw_prediction,
        dtype=np.float64,
    )

    if not np.isfinite(raw).all():
        raise ValueError(
            "Calibration input contains NaN/inf"
        )

    if np.any(raw < 0):
        raise ValueError(
            "Calibration input contains negative values"
        )

    log_prediction = np.log1p(raw)

    original_log_mean = float(
        log_prediction.mean()
    )

    original_log_std = float(
        log_prediction.std()
    )

    scaled = (
        original_log_mean
        + (
            log_prediction
            - original_log_mean
        )
        * float(spread_scale)
    )

    scaled_log_std = float(
        scaled.std()
    )

    centered = (
        scaled
        - scaled.mean()
    )

    shift = find_shift_for_target_clipped_log_mean(
        centered,
        target_log_mean=float(target_log_mean),
    )

    calibrated_log = np.maximum(
        centered + shift,
        0.0,
    )

    actual_target_mean = float(
        calibrated_log.mean()
    )

    with np.errstate(
        over="raise",
        invalid="raise",
    ):
        calibrated_raw = np.expm1(
            calibrated_log
        )

    if not np.isfinite(calibrated_raw).all():
        raise ValueError(
            "Polar calibration produced NaN/inf"
        )

    metadata = {
        "spread_scale": float(spread_scale),
        "target_log_mean": float(target_log_mean),
        "shift": float(shift),

        "original_log_mean": original_log_mean,
        "original_log_std": original_log_std,

        "scaled_log_std": scaled_log_std,

        "final_log_mean": actual_target_mean,
        "final_log_std": float(
            calibrated_log.std()
        ),

        "zero_share": float(
            np.mean(calibrated_raw <= 0)
        ),
    }

    return calibrated_raw, metadata


# -----------------------------------------------------------------------------
# 6. DIAGNOSTICS
# -----------------------------------------------------------------------------

SOURCE_PREDICTIONS = {
    "own_final": PRED_OWN,
    "marat_63": PRED_MARAT,
    "polar_final_calibrated": PRED_POLAR_CALIBRATED,
    "polar_final_raw": PRED_POLAR_RAW,
}


def prediction_stats(
    name,
    prediction,
    *,
    own_prediction=PRED_OWN,
):
    p = np.asarray(
        prediction,
        dtype=np.float64,
    )

    lp = np.log1p(p)
    own_log = np.log1p(
        np.asarray(
            own_prediction,
            dtype=np.float64,
        )
    )

    return {
        "name": name,

        "raw_mean": float(
            p.mean()
        ),
        "raw_median": float(
            np.median(p)
        ),
        "raw_std": float(
            p.std()
        ),

        "zero_share": float(
            np.mean(p <= 0)
        ),

        "log_mean": float(
            lp.mean()
        ),
        "log_std": float(
            lp.std()
        ),

        "log_p01": float(
            np.quantile(lp, 0.01)
        ),
        "log_p10": float(
            np.quantile(lp, 0.10)
        ),
        "log_p50": float(
            np.quantile(lp, 0.50)
        ),
        "log_p90": float(
            np.quantile(lp, 0.90)
        ),
        "log_p99": float(
            np.quantile(lp, 0.99)
        ),

        "pearson_log_vs_own": float(
            np.corrcoef(
                lp,
                own_log,
            )[0, 1]
        ),

        "mean_abs_log_diff_vs_own": float(
            np.mean(
                np.abs(
                    lp
                    - own_log
                )
            )
        ),

        "p95_abs_log_diff_vs_own": float(
            np.quantile(
                np.abs(
                    lp
                    - own_log
                ),
                0.95,
            )
        ),
    }


SOURCE_DIAGNOSTICS = pd.DataFrame(
    [
        prediction_stats(
            name,
            pred,
        )
        for name, pred
        in SOURCE_PREDICTIONS.items()
    ]
)

print("\nSOURCE DIAGNOSTICS")
display(SOURCE_DIAGNOSTICS)

SOURCE_DIAGNOSTICS.to_csv(
    EXTERNAL_BLEND_DIR
    / "source_diagnostics.csv",
    index=False,
)


# Pairwise correlations specifically in competition log-space.
_source_names = list(
    SOURCE_PREDICTIONS
)

_source_log_matrix = np.column_stack(
    [
        np.log1p(
            SOURCE_PREDICTIONS[name]
        )
        for name in _source_names
    ]
)

SOURCE_LOG_CORRELATION = pd.DataFrame(
    np.corrcoef(
        _source_log_matrix,
        rowvar=False,
    ),
    index=_source_names,
    columns=_source_names,
)

print("\nLOG1P PREDICTION CORRELATION")
display(SOURCE_LOG_CORRELATION)

SOURCE_LOG_CORRELATION.to_csv(
    EXTERNAL_BLEND_DIR
    / "source_log_correlation.csv"
)


# -----------------------------------------------------------------------------
# 7. CANDIDATE REGISTRY
# -----------------------------------------------------------------------------

EXTERNAL_BLEND_CANDIDATES = {}
EXTERNAL_BLEND_ROWS = []


def weight_token(value):
    return f"{int(round(float(value) * 1000)):03d}"


def add_external_candidate(
    name,
    prediction,
    *,
    family,
    blend_space,
    weights,
    sources,
    post_calibrated=False,
    calibration_meta=None,
):
    if name in EXTERNAL_BLEND_CANDIDATES:
        raise KeyError(
            f"Duplicate candidate: {name}"
        )

    pred = np.asarray(
        prediction,
        dtype=np.float64,
    )

    if len(pred) != len(sample_submit):
        raise AssertionError(
            f"{name}: length mismatch"
        )

    if not np.isfinite(pred).all():
        raise ValueError(
            f"{name}: NaN/inf"
        )

    if np.any(pred < 0):
        raise ValueError(
            f"{name}: negative prediction"
        )

    EXTERNAL_BLEND_CANDIDATES[name] = pred

    row = prediction_stats(
        name,
        pred,
    )

    row.update({
        "family": family,
        "blend_space": blend_space,
        "sources": "+".join(sources),
        "weights_json": json.dumps(
            {
                source: float(weight)
                for source, weight
                in zip(
                    sources,
                    normalize_weights(weights),
                )
            },
            ensure_ascii=False,
        ),
        "post_calibrated": bool(
            post_calibrated
        ),
        "polar_shift": (
            None
            if calibration_meta is None
            else calibration_meta.get("shift")
        ),
        "polar_final_log_mean": (
            None
            if calibration_meta is None
            else calibration_meta.get(
                "final_log_mean"
            )
        ),
        "polar_final_log_std": (
            None
            if calibration_meta is None
            else calibration_meta.get(
                "final_log_std"
            )
        ),
    })

    EXTERNAL_BLEND_ROWS.append(row)


# Наш baseline тоже регистрируем.
add_external_candidate(
    "own_final_control",
    PRED_OWN,
    family="control",
    blend_space="none",
    weights=[1.0],
    sources=["own"],
)


# -----------------------------------------------------------------------------
# 8. OWN + MARAT
# -----------------------------------------------------------------------------

for w_external in PAIR_WEIGHTS_LOG:
    w_own = 1.0 - w_external

    prediction = blend_predictions(
        [
            PRED_OWN,
            PRED_MARAT,
        ],
        [
            w_own,
            w_external,
        ],
        space="log",
    )

    name = (
        f"own{weight_token(w_own)}"
        f"_marat{weight_token(w_external)}"
        "_log"
    )

    add_external_candidate(
        name,
        prediction,
        family="own_plus_marat",
        blend_space="log",
        weights=[
            w_own,
            w_external,
        ],
        sources=[
            "own",
            "marat",
        ],
    )

    if APPLY_POLAR_CALIBRATION_TO_MARAT_BLENDS:
        calibrated, calibration_meta = (
            polar_post_calibrate(
                prediction
            )
        )

        add_external_candidate(
            name + "_polarcal",
            calibrated,
            family="own_plus_marat",
            blend_space="log",
            weights=[
                w_own,
                w_external,
            ],
            sources=[
                "own",
                "marat",
            ],
            post_calibrated=True,
            calibration_meta=calibration_meta,
        )


# Несколько raw-space вариантов для сравнения.
for w_external in PAIR_WEIGHTS_RAW:
    w_own = 1.0 - w_external

    prediction = blend_predictions(
        [
            PRED_OWN,
            PRED_MARAT,
        ],
        [
            w_own,
            w_external,
        ],
        space="raw",
    )

    name = (
        f"own{weight_token(w_own)}"
        f"_marat{weight_token(w_external)}"
        "_raw"
    )

    add_external_candidate(
        name,
        prediction,
        family="own_plus_marat",
        blend_space="raw",
        weights=[
            w_own,
            w_external,
        ],
        sources=[
            "own",
            "marat",
        ],
    )


# -----------------------------------------------------------------------------
# 9. OWN + POLAR FINAL CALIBRATED
# -----------------------------------------------------------------------------

for w_external in PAIR_WEIGHTS_LOG:
    w_own = 1.0 - w_external

    prediction = blend_predictions(
        [
            PRED_OWN,
            PRED_POLAR_CALIBRATED,
        ],
        [
            w_own,
            w_external,
        ],
        space="log",
    )

    name = (
        f"own{weight_token(w_own)}"
        f"_polarcal{weight_token(w_external)}"
        "_log"
    )

    add_external_candidate(
        name,
        prediction,
        family="own_plus_polar_calibrated",
        blend_space="log",
        weights=[
            w_own,
            w_external,
        ],
        sources=[
            "own",
            "polar_calibrated",
        ],
    )

    if APPLY_POLAR_CALIBRATION_TO_POLAR_CALIBRATED_BLENDS:
        calibrated, calibration_meta = (
            polar_post_calibrate(
                prediction
            )
        )

        add_external_candidate(
            name + "_postpolar",
            calibrated,
            family="own_plus_polar_calibrated",
            blend_space="log",
            weights=[
                w_own,
                w_external,
            ],
            sources=[
                "own",
                "polar_calibrated",
            ],
            post_calibrated=True,
            calibration_meta=calibration_meta,
        )


for w_external in PAIR_WEIGHTS_RAW:
    w_own = 1.0 - w_external

    prediction = blend_predictions(
        [
            PRED_OWN,
            PRED_POLAR_CALIBRATED,
        ],
        [
            w_own,
            w_external,
        ],
        space="raw",
    )

    name = (
        f"own{weight_token(w_own)}"
        f"_polarcal{weight_token(w_external)}"
        "_raw"
    )

    add_external_candidate(
        name,
        prediction,
        family="own_plus_polar_calibrated",
        blend_space="raw",
        weights=[
            w_own,
            w_external,
        ],
        sources=[
            "own",
            "polar_calibrated",
        ],
    )


# -----------------------------------------------------------------------------
# 10. OWN + MARAT + POLAR CALIBRATED
#
# external_total:
#     общий вес коллег
#
# marat_share:
#     доля MARAT внутри external_total
# -----------------------------------------------------------------------------

for external_total in TRIPLE_TOTAL_EXTERNAL_WEIGHTS:
    own_weight = 1.0 - external_total

    for marat_share in TRIPLE_MARAT_SHARES:
        marat_weight = (
            external_total
            * marat_share
        )

        polar_weight = (
            external_total
            - marat_weight
        )

        prediction = blend_predictions(
            [
                PRED_OWN,
                PRED_MARAT,
                PRED_POLAR_CALIBRATED,
            ],
            [
                own_weight,
                marat_weight,
                polar_weight,
            ],
            space="log",
        )

        name = (
            f"own{weight_token(own_weight)}"
            f"_marat{weight_token(marat_weight)}"
            f"_polarcal{weight_token(polar_weight)}"
            "_log"
        )

        add_external_candidate(
            name,
            prediction,
            family="own_plus_marat_plus_polar",
            blend_space="log",
            weights=[
                own_weight,
                marat_weight,
                polar_weight,
            ],
            sources=[
                "own",
                "marat",
                "polar_calibrated",
            ],
        )

        if APPLY_POLAR_CALIBRATION_TO_TRIPLE_BLENDS:
            calibrated, calibration_meta = (
                polar_post_calibrate(
                    prediction
                )
            )

            add_external_candidate(
                name + "_postpolar",
                calibrated,
                family="own_plus_marat_plus_polar",
                blend_space="log",
                weights=[
                    own_weight,
                    marat_weight,
                    polar_weight,
                ],
                sources=[
                    "own",
                    "marat",
                    "polar_calibrated",
                ],
                post_calibrated=True,
                calibration_meta=calibration_meta,
            )


# -----------------------------------------------------------------------------
# 11. SPECIAL EXPERIMENT REQUESTED:
#
# OWN + submission_final_raw
# -> THEN the original Polar calibration
#
# MARAT здесь НЕ участвует.
# -----------------------------------------------------------------------------

for w_polar_raw in POLAR_RAW_BLEND_WEIGHTS:
    w_own = 1.0 - w_polar_raw

    # Основной вариант — log-space.
    raw_mix_log = blend_predictions(
        [
            PRED_OWN,
            PRED_POLAR_RAW,
        ],
        [
            w_own,
            w_polar_raw,
        ],
        space="log",
    )

    base_name = (
        f"own{weight_token(w_own)}"
        f"_polarraw{weight_token(w_polar_raw)}"
        "_log"
    )

    add_external_candidate(
        base_name,
        raw_mix_log,
        family="own_plus_polar_raw",
        blend_space="log",
        weights=[
            w_own,
            w_polar_raw,
        ],
        sources=[
            "own",
            "polar_raw",
        ],
    )

    corrected, correction_meta = (
        polar_post_calibrate(
            raw_mix_log
        )
    )

    add_external_candidate(
        base_name + "_polarcal",
        corrected,
        family="own_plus_polar_raw",
        blend_space="log",
        weights=[
            w_own,
            w_polar_raw,
        ],
        sources=[
            "own",
            "polar_raw",
        ],
        post_calibrated=True,
        calibration_meta=correction_meta,
    )


# И несколько raw-space blend -> Polar calibration.
for w_polar_raw in PAIR_WEIGHTS_RAW:
    w_own = 1.0 - w_polar_raw

    raw_mix = blend_predictions(
        [
            PRED_OWN,
            PRED_POLAR_RAW,
        ],
        [
            w_own,
            w_polar_raw,
        ],
        space="raw",
    )

    base_name = (
        f"own{weight_token(w_own)}"
        f"_polarraw{weight_token(w_polar_raw)}"
        "_raw"
    )

    add_external_candidate(
        base_name,
        raw_mix,
        family="own_plus_polar_raw",
        blend_space="raw",
        weights=[
            w_own,
            w_polar_raw,
        ],
        sources=[
            "own",
            "polar_raw",
        ],
    )

    corrected, correction_meta = (
        polar_post_calibrate(
            raw_mix
        )
    )

    add_external_candidate(
        base_name + "_polarcal",
        corrected,
        family="own_plus_polar_raw",
        blend_space="raw",
        weights=[
            w_own,
            w_polar_raw,
        ],
        sources=[
            "own",
            "polar_raw",
        ],
        post_calibrated=True,
        calibration_meta=correction_meta,
    )


# -----------------------------------------------------------------------------
# 12. CONTROL: POLAR POST-CALIBRATION OF OUR OWN FINAL BLEND
#
# Полезно как контроль:
# помогает ли сама коррекция вообще, ещё до добавления чужой модели.
# -----------------------------------------------------------------------------

OWN_POLAR_CALIBRATED, OWN_POLAR_META = (
    polar_post_calibrate(
        PRED_OWN
    )
)

add_external_candidate(
    "own_final_polarcal_control",
    OWN_POLAR_CALIBRATED,
    family="control",
    blend_space="none",
    weights=[1.0],
    sources=["own"],
    post_calibrated=True,
    calibration_meta=OWN_POLAR_META,
)


# -----------------------------------------------------------------------------
# 13. CATALOG / TEST-ONLY DIAGNOSTICS
#
# Это НЕ leaderboard качества.
# Здесь нет y_test, поэтому RMSLE посчитать нельзя.
# -----------------------------------------------------------------------------

EXTERNAL_BLEND_CATALOG = (
    pd.DataFrame(
        EXTERNAL_BLEND_ROWS
    )
    .sort_values(
        [
            "family",
            "post_calibrated",
            "mean_abs_log_diff_vs_own",
        ]
    )
    .reset_index(drop=True)
)

print(
    "\n"
    + "=" * 100
)
print(
    "EXTERNAL BLEND CATALOG "
    "(TEST-ONLY — НЕ CV LEADERBOARD)"
)
print(
    "=" * 100
)

display(
    EXTERNAL_BLEND_CATALOG
)

EXTERNAL_BLEND_CATALOG.to_csv(
    EXTERNAL_BLEND_DIR
    / "external_blend_catalog.csv",
    index=False,
)


# -----------------------------------------------------------------------------
# 14. HIGH-DISAGREEMENT USERS
#
# Показывает, где коллеги дают наиболее отличающийся от нас прогноз.
# Это diversity diagnostic, НЕ доказательство лучшего качества.
# -----------------------------------------------------------------------------

DISAGREEMENT = pd.DataFrame({
    USER_COL:
        sample_submit[USER_COL].to_numpy(),

    "own":
        PRED_OWN,

    "marat":
        PRED_MARAT,

    "polar_calibrated":
        PRED_POLAR_CALIBRATED,

    "polar_raw":
        PRED_POLAR_RAW,
})

DISAGREEMENT[
    "abs_logdiff_marat_vs_own"
] = np.abs(
    np.log1p(
        DISAGREEMENT["marat"]
    )
    - np.log1p(
        DISAGREEMENT["own"]
    )
)

DISAGREEMENT[
    "abs_logdiff_polarcal_vs_own"
] = np.abs(
    np.log1p(
        DISAGREEMENT["polar_calibrated"]
    )
    - np.log1p(
        DISAGREEMENT["own"]
    )
)

DISAGREEMENT[
    "abs_logdiff_polarraw_vs_own"
] = np.abs(
    np.log1p(
        DISAGREEMENT["polar_raw"]
    )
    - np.log1p(
        DISAGREEMENT["own"]
    )
)

DISAGREEMENT[
    "max_external_disagreement"
] = DISAGREEMENT[
    [
        "abs_logdiff_marat_vs_own",
        "abs_logdiff_polarcal_vs_own",
        "abs_logdiff_polarraw_vs_own",
    ]
].max(axis=1)

DISAGREEMENT = (
    DISAGREEMENT
    .sort_values(
        "max_external_disagreement",
        ascending=False,
    )
    .reset_index(drop=True)
)

DISAGREEMENT.head(
    20_000
).to_parquet(
    EXTERNAL_BLEND_DIR
    / "top_20000_external_disagreement_users.parquet",
    index=False,
)


# -----------------------------------------------------------------------------
# 15. SUBMISSION SAVER
# -----------------------------------------------------------------------------

def build_external_submission(
    prediction,
):
    pred = np.asarray(
        prediction,
        dtype=np.float64,
    )

    if len(pred) != len(
        sample_submit
    ):
        raise AssertionError(
            "External submission length mismatch"
        )

    if (
        not np.isfinite(pred).all()
        or np.any(pred < 0)
    ):
        raise ValueError(
            "Invalid submission prediction"
        )

    result = (
        sample_submit[
            [USER_COL]
        ]
        .copy()
    )

    result[
        SUBMISSION_TARGET_COL
    ] = pred

    return result


if SAVE_ALL_EXTERNAL_CANDIDATES:
    print(
        f"\nSaving "
        f"{len(EXTERNAL_BLEND_CANDIDATES)} "
        f"external blend submissions..."
    )

    for index, (
        name,
        prediction,
    ) in enumerate(
        EXTERNAL_BLEND_CANDIDATES.items(),
        start=1,
    ):
        submission = (
            build_external_submission(
                prediction
            )
        )

        submission.to_csv(
            EXTERNAL_BLEND_DIR
            / f"submission_{name}.csv",
            index=False,
        )

        if (
            index % 10 == 0
            or index
            == len(
                EXTERNAL_BLEND_CANDIDATES
            )
        ):
            print(
                f"  saved "
                f"{index}/"
                f"{len(EXTERNAL_BLEND_CANDIDATES)}"
            )


# -----------------------------------------------------------------------------
# 16. OPTIONAL MANUAL CHOICE
#
# После проверки нескольких разумных кандидатов на Kaggle можно поставить
# имя сюда наверху в EXTERNAL_BLEND_CHOSEN и rerun только эту ячейку.
# -----------------------------------------------------------------------------

if EXTERNAL_BLEND_CHOSEN is not None:
    if (
        EXTERNAL_BLEND_CHOSEN
        not in EXTERNAL_BLEND_CANDIDATES
    ):
        raise KeyError(
            f"Unknown EXTERNAL_BLEND_CHOSEN="
            f"{EXTERNAL_BLEND_CHOSEN}"
        )

    chosen_submission = (
        build_external_submission(
            EXTERNAL_BLEND_CANDIDATES[
                EXTERNAL_BLEND_CHOSEN
            ]
        )
    )

    chosen_path = (
        OUTPUT_DIR
        / "submission_EXTERNAL_CHOSEN.csv"
    )

    chosen_submission.to_csv(
        chosen_path,
        index=False,
    )

    print(
        "Saved chosen external blend:",
        chosen_path,
    )


# -----------------------------------------------------------------------------
# 17. METADATA
# -----------------------------------------------------------------------------

EXTERNAL_BLEND_MANIFEST = {
    "warning": (
        "External colleague submissions are test-only. "
        "No OOF/CV quality estimate exists for their contribution."
    ),

    "own_final_blend":
        FINAL_BLEND_NAME,

    "sources": {
        "marat":
            str(MARAT_PATH),

        "polar_final_calibrated":
            str(POLAR_CALIBRATED_PATH),

        "polar_final_raw":
            str(POLAR_RAW_PATH),
    },

    "polar_post_calibration": {
        "spread_scale":
            POLAR_SPREAD_SCALE,

        "target_log_mean":
            POLAR_TARGET_LOG_MEAN,
    },

    "candidate_count":
        len(
            EXTERNAL_BLEND_CANDIDATES
        ),

    "pair_weights_log":
        list(
            PAIR_WEIGHTS_LOG
        ),

    "pair_weights_raw":
        list(
            PAIR_WEIGHTS_RAW
        ),

    "triple_total_external_weights":
        list(
            TRIPLE_TOTAL_EXTERNAL_WEIGHTS
        ),

    "triple_marat_shares":
        list(
            TRIPLE_MARAT_SHARES
        ),

    "polar_raw_blend_weights":
        list(
            POLAR_RAW_BLEND_WEIGHTS
        ),
}

(
    EXTERNAL_BLEND_DIR
    / "external_blend_manifest.json"
).write_text(
    json.dumps(
        EXTERNAL_BLEND_MANIFEST,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    "\nExternal blend artifacts:",
    EXTERNAL_BLEND_DIR,
)

print(
    "\nIMPORTANT: "
    "external_blend_catalog.csv содержит "
    "distribution/diversity diagnostics, "
    "а НЕ оценку качества."
)

External submissions:
  MARAT            : /kaggle/input/datasets/vladimirsmirnovve/different-submissions/63_optimum_MARAT.csv
  Polar calibrated : /kaggle/input/datasets/vladimirsmirnovve/different-submissions/submission_final_calibrated.csv
  Polar raw        : /kaggle/input/datasets/vladimirsmirnovve/different-submissions/submission_final_raw.csv

SOURCE DIAGNOSTICS


,name,raw_mean,raw_median,raw_std,zero_share,log_mean,log_std,log_p01,log_p10,log_p50,log_p90,log_p99,pearson_log_vs_own,mean_abs_log_diff_vs_own,p95_abs_log_diff_vs_own
0,own_final,38.632168,7.143478,102.913144,0.000000,2.338981,1.568721,0.194042,0.399471,2.097217,4.598071,6.128670,1.000000,0.000000,0.000000
1,marat_63,41.972660,6.982958,123.464610,0.000232,2.329700,1.617166,0.135087,0.350263,2.077309,4.665744,6.249376,0.997544,0.093133,0.240654
2,polar_final_calibrated,43.229980,6.820212,124.722653,0.003692,2.329351,1.635245,0.062299,0.349787,2.056712,4.706848,6.278496,0.997450,0.103429,0.259376
3,polar_final_raw,41.151276,7.496189,109.976411,0.000004,2.400285,1.564419,0.231819,0.506822,2.139618,4.674663,6.178057,0.997447,0.101367,0.243987



LOG1P PREDICTION CORRELATION


,own_final,marat_63,polar_final_calibrated,polar_final_raw
own_final,1.000000,0.997544,0.997450,0.997447
marat_63,0.997544,1.000000,0.997919,0.997915
polar_final_calibrated,0.997450,0.997919,1.000000,0.999998
polar_final_raw,0.997447,0.997915,0.999998,1.000000



EXTERNAL BLEND CATALOG (TEST-ONLY — НЕ CV LEADERBOARD)


,name,raw_mean,raw_median,raw_std,zero_share,log_mean,log_std,log_p01,log_p10,log_p50,log_p90,log_p99,pearson_log_vs_own,mean_abs_log_diff_vs_own,p95_abs_log_diff_vs_own,family,blend_space,sources,weights_json,post_calibrated,polar_shift,polar_final_log_mean,polar_final_log_std
0,own_final_control,38.632168,7.143478,102.913144,0.000000,2.338981,1.568721,0.194042,0.399471,2.097217,4.598071,6.128670,1.000000,0.000000,0.000000,control,none,own,"{""own"": 1.0}",False,NaN,NaN,NaN
1,own_final_polarcal_control,43.216237,6.977258,123.902535,0.000792,2.329351,1.639918,0.087015,0.301771,2.076595,4.690987,6.291075,1.000000,0.061413,0.118212,control,none,own,"{""own"": 1.0}",True,2.329334,2.329351,1.639918
2,own950_marat050_log,38.776925,7.136040,103.721506,0.000000,2.338517,1.570955,0.191803,0.396990,2.096304,4.601766,6.134024,0.999994,0.004657,0.012033,own_plus_marat,log,own+marat,"{""own"": 0.95, ""marat"": 0.05}",False,NaN,NaN,NaN
3,own900_marat100_raw,38.966217,7.129668,104.811305,0.000000,2.338718,1.573351,0.189787,0.394916,2.095520,4.605437,6.138799,0.999974,0.009273,0.023942,own_plus_marat,raw,own+marat,"{""own"": 0.9, ""marat"": 0.1}",False,NaN,NaN,NaN
4,own900_marat100_log,38.923881,7.123783,104.549459,0.000000,2.338053,1.573209,0.189592,0.394631,2.094796,4.604806,6.137475,0.999974,0.009313,0.024065,own_plus_marat,log,own+marat,"{""own"": 0.9, ""marat"": 0.1}",False,NaN,NaN,NaN
5,own850_marat150_log,39.073057,7.113813,105.397461,0.000000,2.337589,1.575484,0.187450,0.392239,2.093568,4.607814,6.142564,0.999942,0.013970,0.036098,own_plus_marat,log,own+marat,"{""own"": 0.85, ""marat"": 0.15}",False,NaN,NaN,NaN
6,own800_marat200_log,39.224476,7.109559,106.266000,0.000000,2.337125,1.577779,0.185082,0.390117,2.093044,4.611185,6.148097,0.999897,0.018627,0.048131,own_plus_marat,log,own+marat,"{""own"": 0.8, ""marat"": 0.2}",False,NaN,NaN,NaN
7,own750_marat250_raw,39.467291,7.110922,107.733059,0.000000,2.338047,1.580369,0.182859,0.388228,2.093212,4.615801,6.156102,0.999841,0.023168,0.059828,own_plus_marat,raw,own+marat,"{""own"": 0.75, ""marat"": 0.25}",False,NaN,NaN,NaN
8,own750_marat250_log,39.378160,7.098801,107.155587,0.000000,2.336661,1.580093,0.182301,0.387580,2.091716,4.614624,6.154012,0.999839,0.023283,0.060164,own_plus_marat,log,own+marat,"{""own"": 0.75, ""marat"": 0.25}",False,NaN,NaN,NaN
9,own700_marat300_log,39.534132,7.090745,108.066762,0.000000,2.336197,1.582428,0.179428,0.385226,2.090721,4.617452,6.158927,0.999769,0.027940,0.072196,own_plus_marat,log,own+marat,"{""own"": 0.7, ""marat"": 0.3}",False,NaN,NaN,NaN



Saving 74 external blend submissions...
  saved 10/74
  saved 20/74
  saved 30/74
  saved 40/74
  saved 50/74
  saved 60/74
  saved 70/74
  saved 74/74

External blend artifacts: /kaggle/working/ozon_final/external_colleague_blends

IMPORTANT: external_blend_catalog.csv содержит distribution/diversity diagnostics, а НЕ оценку качества.


In [ ]:
# =============================================================================
# EQUAL-WEIGHT BLEND OF TOP-5 BASE MODEL SPECIFICATIONS
# =============================================================================

TOP5_MODELS = (
    BASE_RANKING
    .sort_values("weighted_rmsle")
    .head(5)["model"]
    .tolist()
)

print("TOP-5 specifications:")
display(
    BASE_RANKING
    .set_index("model")
    .loc[TOP5_MODELS]
    .reset_index()
)

print("\nEqual weights:")
for model_name in TOP5_MODELS:
    print(f"  {model_name}: {1 / len(TOP5_MODELS):.3f}")


# -------------------------------------------------------------------------
# Equal blend in log1p-space:
#
# log_pred = mean(log1p(pred_i))
# pred     = expm1(log_pred)
# -------------------------------------------------------------------------

top5_matrix = (
    FINAL_PREDICTIONS[TOP5_MODELS]
    .to_numpy(dtype=np.float64)
)

top5_equal_log = np.mean(
    np.log1p(
        np.clip(top5_matrix, 0.0, None)
    ),
    axis=1,
)

top5_equal_pred = np.expm1(top5_equal_log)
top5_equal_pred = np.clip(top5_equal_pred, 0.0, None)

assert len(top5_equal_pred) == len(sample_submit)
assert np.isfinite(top5_equal_pred).all()
assert (top5_equal_pred >= 0).all()


# -------------------------------------------------------------------------
# Save submission
# -------------------------------------------------------------------------

submission_top5_equal = sample_submit[[USER_COL]].copy()

submission_top5_equal[SUBMISSION_TARGET_COL] = top5_equal_pred

TOP5_EQUAL_PATH = (
    OUTPUT_DIR
    / "submission_TOP5_EQUAL_LOGMEAN.csv"
)

submission_top5_equal.to_csv(
    TOP5_EQUAL_PATH,
    index=False,
)

print("\nSaved:")
print(TOP5_EQUAL_PATH)

TOP-5 specifications:


,model,weighted_rmsle,family,target_mode,maturity_manual,maturity_source,optuna_metric_snapshot
0,xgboost_log_level,1.712143,xgboost,log_level,trusted,manual_not_present_in_source_artifacts,1.711975
1,xgboost_log_diff,1.713651,xgboost,log_diff,trusted,manual_not_present_in_source_artifacts,1.713494
2,catboost_log_level,1.713766,catboost,log_level,trusted,manual_not_present_in_source_artifacts,1.713613
3,catboost_log_diff,1.714925,catboost,log_diff,trusted,manual_not_present_in_source_artifacts,1.713830



Equal weights:
  xgboost_log_level: 0.250
  xgboost_log_diff: 0.250
  catboost_log_level: 0.250
  catboost_log_diff: 0.250

Saved:
/kaggle/working/ozon_final/submission_TOP5_EQUAL_LOGMEAN.csv


# 30. Optional adaptive CatBoost/XGBoost — disabled by default

This is the final optional block and runs only after the main pipeline, validation and Error Analysis v2. The five non-CPU adaptive models are **not scheduled by default** so the main run stays within the three-hour budget. Set `ENABLE_EXPORTED_EXACT_MODELS = True` in the next cell only when sufficient runtime remains. `catboost_segment_adaptive` belongs to the dedicated CPU artifact and remains frozen even then.

The exact exported feature subsets, native `best_params_json`, published fold metrics and raw evaluation metadata are embedded for all six additional specifications. With the default switch, the block only validates and exports their registry; it performs no adaptive CV or final fit.

The supplied `nsel/model.py` reveals their target contract: only the literal `log_level` label uses `log1p(Y)`; every other label uses `log1p(Y) - log1p(anchor)` and adds the anchor back during restoration. The four exported labels are therefore implemented explicitly as legacy `log_diff` aliases.

An explicitly named, disabled-by-default `log_level_proxy` remains available as a separate counterfactual experiment. `user_id` from `xgboost_volatility_adaptive` is removed as an identifier/memorization risk. Generic `final_eval_params_json` remains provenance only because it mixes estimator-family keys; training uses the family-native Optuna parameters.


In [ ]:
EXPORTED_MODEL_ROWS = json.loads('[{"name":"xgboost_segment_adaptive","family":"xgboost","target_mode":"segment_adaptive","account":"kirkdlillibridge","stage":"focus","trials_done":356,"selection_metric":1.72278716,"optuna_metric":1.719628887,"best_optuna_trial":102,"features":["search_only_share_active_90d","ph_gmv_mean","cyc_completion_min","search_to_cart_block_rate_0_6","order_per_search_180d","any_cart_no_order_day_30d","search_to_ord_90d","square_log1p_gmv_sum_14d","to_ord_block_rate_180_364","search_cart_items_per_query_7d","catalog_gmv_share_180d","days_since_last_order","btyd_purchase_days","gmv_per_order_180d","search_days_14d","ph_purchase_density","gmv_per_order_365d__denis","purchase_days_decay30_hl60","row_days_7d","searches_sum_14d","extra_search_sum_5d","gmv_per_purchase_day_180d","purchase_days_decay30_hl30","ph_gmv_last3_mean","gmv_search_share30_of_365","days_since_last_search_cart","ph_gap_cv","gmv_search_share30_of_180","gmv_maximum_90d","purchase_days_30d","purchase_days_90d","square_log1p_decay_gmv_half_life_120d","gmv_per_purchase_day_30d","gmv_share30_of_90","forecast_weekday_profile_gmv_projection","gmv_active_day_quantile_90_365d","searches_share30_of_365","percentile_rank_lifetime_gmv","percentile_rank_observed_customer_age_days","cutoff_is_weekend","ph_gmv_per_item_median"],"params_sampled":{"colsample_bylevel":0.6419006378821983,"colsample_bynode":0.6905319805033873,"colsample_bytree":0.6045274925618531,"gamma":0.2734085605627655,"learning_rate":0.013598014141037793,"max_bin":256,"max_depth":6,"min_child_weight":59.88266919032519,"reg_alpha":0.013411411189318698,"reg_lambda":182.41334527146344,"subsample":0.9303217589673217},"final_eval_params_raw":{"learning_rate":0.15,"depth":5,"l2_leaf_reg":3.0,"iterations":8000,"early_stopping_rounds":30,"subsample":0.8,"colsample":0.8,"min_data_in_leaf":20},"published_folds":{"nov_2025":{"rmsle":1.741691885918868,"rmsle_zero":1.6445411917899904,"rmsle_positive":1.813606556275576,"trees":139},"dec_2025":{"rmsle":1.7477386700491395,"rmsle_zero":1.7555845417859948,"rmsle_positive":1.741626339507165,"trees":198},"jan_2026":{"rmsle":1.6891391355404475,"rmsle_zero":1.7401955521937578,"rmsle_positive":1.644517096048151,"trees":368}},"feature_artifact":"[xgboost_segment_adaptive/otbor/selected_features_xgboost_segment_adaptive.json]","params_artifact":"[xgboost_segment_adaptive/best_params.json]"},{"name":"catboost_segment_adaptive","family":"catboost","target_mode":"segment_adaptive","account":"jamesslax","stage":"explore","trials_done":14,"selection_metric":1.720539176,"optuna_metric":1.790836959,"best_optuna_trial":13,"features":["search_only_share_active_90d","ph_gmv_mean","hazard_activity_recency_share","order_per_search_180d","cyc_completion_min","catalog_cart_share_180d","gmv_block_rate_90_179","gmv_share_minus_order_share_365d","search_only_share_active_180d","active_days_share30_of_90","gmv_channel_balance_365d","fully_zero_days_180d","days_since_last_catalog","gmv_maximum_365d","square_log1p_order_gap_mean_days","tkt_percentile","gmv_search_share30_of_90","gmv_per_order_180d","gmv_per_order_365d","prior_last30_recency_tau60","days_since_last_cart_addition","gmv_per_order_90d__denis","days_since_last_order","percentile_rank_searches_sum_30d","ph_gmv_last5_mean","decay_acceleration_searches_3d_14d","search_cart_no_order_day_share_active_90d","purchase_days_180d","never_catalog","gmv_per_purchase_day_30d","gmv_maximum_90d","ph_due_ratio_last_gap","calendar_exposure_365d","search_order_per_search_cart_365d","searches_block_180_364","purchase_days_90d","active_days_observed_rate_30d","cart_days_7d","search_order_days_180d","forecast_weekday_profile_gmv_projection","ph_gmv_last","square_log1p_decay_to_ord_half_life_7d","percentile_rank_to_ord_sum_7d","ph_gmv_min","searches_block_rate_180_364","purchase_days_calendar_rate_lifetime","gmv_recent_to_prior_ratio","purchase_days_calendar_rate_365d"],"params_sampled":{"bagging_temperature":0.04863193286290768,"border_count":254,"depth":8,"l2_leaf_reg":4.706745686730944,"leaf_estimation_iterations":3,"learning_rate":0.02052475543901429,"max_ctr_complexity":1,"one_hot_max_size":64,"random_strength":40.1420949829701},"final_eval_params_raw":{"learning_rate":0.15,"depth":5,"l2_leaf_reg":3.0,"iterations":8000,"early_stopping_rounds":30,"subsample":0.8,"colsample":0.8,"min_data_in_leaf":20},"published_folds":{"nov_2025":{"rmsle":1.740553030298089,"rmsle_zero":1.6034948261150224,"rmsle_positive":1.840037137611316,"trees":502},"dec_2025":{"rmsle":1.743316415039278,"rmsle_zero":1.7586489931753058,"rmsle_positive":1.731325597485889,"trees":509},"jan_2026":{"rmsle":1.6881004323158695,"rmsle_zero":1.7555667272701703,"rmsle_positive":1.6285872075668009,"trees":367}},"feature_artifact":"[catboost_segment_adaptive/otbor/selected_features_catboost_segment_adaptive.json]","params_artifact":"[catboost_segment_adaptive/best_params.json]"},{"name":"catboost_segment_logspace","family":"catboost","target_mode":"segment_logspace","account":"slrflr","stage":"explore","trials_done":165,"selection_metric":1.721456632,"optuna_metric":1.720301706,"best_optuna_trial":101,"features":["cutoff_dayofyear_cos","catalog_only_share_active_180d","prior_last30_recency_tau30","cart_days_share30_of_365","search_gmv_share_180d","search_sum_7d","search_cart_no_order_day_share_active_30d","catalog_order_per_catalog_cart_180d","prior_avg90","gmv_block_rate_90_179","ph_gmv_last5_median","weekday_6_gmv","gmv_per_purchase_day_7d","observed_active_month_count","square_log1p_forecast_weekday_profile_gmv_projection","prior_freq90_money180","cutoff_dom_cos","cutoff_month_sin","gmv_block_rate_0_29","gmv_recent_to_prior_ratio","order_calendar_day_probability_365d","tkt_real_gmv90","pct_rank_purchase_days_180d","days_since_last_order","cart_days_share30_of_90","active_days_share30_of_365","hazard_activity_recency_share","pct_rank_purchase_days_90d","cyc_search_progress","pct_rank_gmv_30d","gmv_share30_of_90","hazard_overdue_activity_ratio","cart_days_14d","days_since_last_search_cart","order_gap_coefficient_of_variation","searches_sum_7d","searches_block_rate_90_179","gmv_search_30d","cyc_med_active","gmv_search_90d","percentile_rank_monthly_orders_mean","ph_last3_to_median_gmv","cart_days_calendar_rate_365d","square_log1p_extra_active_days_21d","ph_purchase_density","to_cart_calendar_day_rate_365d","search_days_observed_rate_365d","cat_days_calendar_rate_365d","cart_per_search_7d"],"params_sampled":{"bagging_temperature":0.24660896452032008,"border_count":254,"depth":7,"l2_leaf_reg":90.30230890865958,"leaf_estimation_iterations":1,"learning_rate":0.005992583084712207,"max_ctr_complexity":2,"one_hot_max_size":31,"random_strength":36.77201542623796},"final_eval_params_raw":{"learning_rate":0.15,"depth":5,"l2_leaf_reg":3.0,"iterations":8000,"early_stopping_rounds":30,"subsample":0.8,"colsample":0.8,"min_data_in_leaf":20},"published_folds":{"nov_2025":{"rmsle":1.7422623192640394,"rmsle_zero":1.618765530820403,"rmsle_positive":1.8325047586330399,"trees":453},"dec_2025":{"rmsle":1.74719650934291,"rmsle_zero":1.7825422527289378,"rmsle_positive":1.71926949150897,"trees":536},"jan_2026":{"rmsle":1.6859306849797815,"rmsle_zero":1.7320959172721224,"rmsle_positive":1.6456919156125152,"trees":646}},"feature_artifact":"[catboost_segment_logspace/otbor/selected_features_catboost_segment_logspace.json]","params_artifact":"[catboost_segment_logspace/best_params.json]"},{"name":"catboost_volatility_adaptive","family":"catboost","target_mode":"volatility_adaptive","account":"randallaament","stage":"focus","trials_done":12,"selection_metric":1.73168208,"optuna_metric":1.730322503,"best_optuna_trial":98,"features":["ph_gmv_per_item_median","searches_share30_of_90","search_order_per_search_cart_30d","cyc_completion_min","catalog_cart_share_180d","catalog_gmv_share_30d","gmv_share_minus_order_share_365d","search_order_per_search_cart_180d","days_since_last_order","search_days_block_rate_0_29","active_days_share30_of_90","gmv_maximum_365d","gmv_mean_active_365d","ph_gap_mean","gmv_share30_of_180","square_log1p_decay_gmv_half_life_120d","order_days_180d","purchase_days_block_rate_0_6","gmv_frequency_interaction_30d","active_days_7d","gmv_search_share30_of_90","last_1_active_events_searches","hazard_activity_recency_share","ph_purchase_density","ph_last_to_median_gmv","gmv_per_purchase_day_180d","search_to_ord_90d","gmv_sum_lag_30_44d","searches_180d","log1p_decay_gmv_half_life_30d","percentile_rank_active_days_90d","to_cart_sum_14d","to_ord_block_rate_180_364","row_days_180d","monthly_cart_additions_mean","active_gap_coefficient_of_variation","percentile_rank_searches_sum_30d","hazard_expected_orders_30d","order_per_search_7d","square_log1p_monthly_orders_mean","percentile_rank_monthly_gmv_mean","pct_rank_to_ord_90d","searches_sum_14d","prior_avg90","percentile_rank_monthly_orders_mean","cyc_cart_progress","searches_block_rate_180_364","gmv_frequency_interaction_60d","to_ord_decay30_hl30"],"params_sampled":{"bagging_temperature":0.38919417479724605,"border_count":128,"depth":7,"l2_leaf_reg":98.28633530509391,"leaf_estimation_iterations":1,"learning_rate":0.012378773706973422,"max_ctr_complexity":3,"one_hot_max_size":60,"random_strength":25.592645820142394},"final_eval_params_raw":{"learning_rate":0.15,"depth":5,"l2_leaf_reg":3.0,"iterations":8000,"early_stopping_rounds":30,"subsample":0.8,"colsample":0.8,"min_data_in_leaf":20},"published_folds":{"nov_2025":{"rmsle":1.7421838305729727,"rmsle_zero":1.6223444727855625,"rmsle_positive":1.8299104801767363,"trees":337},"dec_2025":{"rmsle":1.7544554864460273,"rmsle_zero":1.7965678298348957,"rmsle_positive":1.7210687138569727,"trees":515},"jan_2026":{"rmsle":1.705191755492334,"rmsle_zero":1.7980816528251073,"rmsle_positive":1.622099269433697,"trees":859}},"feature_artifact":"[catboost_volatility_adaptive/otbor/selected_features_catboost_volatility_adaptive.json]","params_artifact":"[catboost_volatility_adaptive/best_params.json]"},{"name":"xgboost_personal_hard","family":"xgboost","target_mode":"personal_hard","account":"�","stage":"explore","trials_done":80,"selection_metric":1.779187563,"optuna_metric":1.773674325,"best_optuna_trial":49,"features":["ph_gmv_per_item_median","any_cart_no_order_day_30d","cyc_completion_min","searches_active_day_quantile_25_365d","to_ord_decay30_hl30","days_since_last_search_order","recency_gap_search_cart_no_order_minus_purchase","catalog_gmv_share_30d","search_to_ord_90d","gmv_channel_balance_365d","never_search_order","days_since_last_order","recency_gap_active_minus_purchase","gmv_maximum_90d","gmv_block_rate_0_29","gmv_per_order_180d","ph_gmv_last3_mean","cart_days_14d","purchase_days_block_rate_90_179","cyc_time_progress","gmv_share30_of_365","search_sum_7d","pct_rank_purchase_freshness","to_ord_block_180_364","search_cart_days_365d","row_days_7d","gmv_per_order_365d","gmv_block_180_364","percentile_rank_monthly_orders_mean","searches_7d","hazard_activity_recency_share","search_days_block_rate_0_6","hazard_expected_orders_30d","gmv_block_rate_90_179","percentile_rank_gmv_sum_365d","gmv_share30_of_90","gmv_per_purchase_day_365d","cyc_age_percentile","purchase_days_90d","gmv_search_block_180_364","ph_purchase_density","ph_last_to_median_gmv","ph_gap_mean","hazard_last_interval_vs_mean","gmv_search_share30_of_90","log1p_decay_gmv_half_life_30d","search_order_items_per_query_30d","cyc_q75_len","active_days_recent_to_prior_ratio","cutoff_is_weekend","cyc_med_pre7c","both_channel_day_365d","searches_365d","searches_standard_deviation_30d"],"params_sampled":{"colsample_bylevel":0.7055795055833681,"colsample_bynode":0.7098400502144211,"colsample_bytree":0.8685342948900794,"gamma":3.271310064336281,"learning_rate":0.0050764232746045275,"max_bin":256,"max_depth":9,"min_child_weight":53.08396991076525,"reg_alpha":9.48506635629361e-07,"reg_lambda":0.4255666519303329,"subsample":0.9096013124353859},"final_eval_params_raw":{"learning_rate":0.15,"depth":5,"l2_leaf_reg":3.0,"iterations":8000,"early_stopping_rounds":30,"subsample":0.8,"colsample":0.8,"min_data_in_leaf":20},"published_folds":{"nov_2025":{"rmsle":1.7978578610952374,"rmsle_zero":1.700432944125548,"rmsle_positive":1.8700742005630002,"trees":267},"dec_2025":{"rmsle":1.802038243637356,"rmsle_zero":1.8124765632722795,"rmsle_positive":1.79389681571004,"trees":406},"jan_2026":{"rmsle":1.7475242813710876,"rmsle_zero":1.8079335008741402,"rmsle_positive":1.6945099123976979,"trees":408}},"feature_artifact":"[xgboost_personal_hard/otbor/selected_features_xgboost_personal_hard.json]","params_artifact":"[xgboost_personal_hard/best_params.json]"},{"name":"xgboost_volatility_adaptive","family":"xgboost","target_mode":"volatility_adaptive","account":"�","stage":"explore","trials_done":114,"selection_metric":1.735041194,"optuna_metric":1.732341939,"best_optuna_trial":73,"features":["ph_gmv_per_item_median","cyc_completion_min","catalog_only_share_active_90d","catalog_cart_share_180d","search_days_block_rate_0_29","any_cart_no_order_day_30d","search_cart_items_per_query_180d","catalog_gmv_share_30d","search_order_share_7d","recency_activity_interaction","search_order_items_per_query_7d","gmv_decay30_hl30","gmv_maximum_90d","purchase_days_block_rate_90_179","recency_gap_active_minus_purchase","gmv_per_order_180d","ph_gmv_last3_mean","search_days_block_rate_0_6","log1p_decay_gmv_half_life_30d","gmv_search_block_90_179","gmv_share30_of_365","square_log1p_monthly_orders_mean","hazard_expected_orders_30d","searches_sum_7d","to_ord_block_90_179","row_days_7d","days_since_last_order","to_ord_share30_of_365","search_order_days_7d","searches_share30_of_180","hol_gmv_log","cart_days_365d","purchase_days_90d","search_order_days_30d","forecast_weekday_profile_gmv_projection","hazard_searched_after_order_days","cyc_search_excess","cyc_completion_max","observed_customer_age_days","catalog_cart_share_7d","ph_gmv_cv","purchase_days_30d","user_id","search_to_cart_365d","searches_standard_deviation_30d"],"params_sampled":{"colsample_bylevel":0.6141973114082185,"colsample_bynode":0.635662021450186,"colsample_bytree":0.7349218008506955,"gamma":2.470912745562301,"learning_rate":0.005377341947609814,"max_bin":512,"max_depth":8,"min_child_weight":37.11353610199843,"reg_alpha":0.28347525584404765,"reg_lambda":0.6915041341868166,"subsample":0.7523199824484587},"final_eval_params_raw":{"learning_rate":0.15,"depth":5,"l2_leaf_reg":3.0,"iterations":8000,"early_stopping_rounds":30,"subsample":0.8,"colsample":0.8,"min_data_in_leaf":20},"published_folds":{"nov_2025":{"rmsle":1.7460867507798012,"rmsle_zero":1.615086049043124,"rmsle_positive":1.8414752140848114,"trees":150},"dec_2025":{"rmsle":1.756215465320517,"rmsle_zero":1.8018703110478007,"rmsle_positive":1.7199549428016847,"trees":216},"jan_2026":{"rmsle":1.7096102326062106,"rmsle_zero":1.8260048286193287,"rmsle_positive":1.6041003239066707,"trees":142}},"feature_artifact":"[xgboost_volatility_adaptive/otbor/selected_features_xgboost_volatility_adaptive.json]","params_artifact":"[xgboost_volatility_adaptive/best_params.json]"}]')

# Exact legacy contract from nsel/model.py:
#   if target_mode == "log_level": log1p(Y)
#   else: log1p(Y) - log1p(anchor)
# and restore_level adds log1p(anchor) for every non-log_level mode.
# Therefore the four exported labels below are aliases of log_diff, not four
# undisclosed formulas. The explicit registry avoids relying on a silent else.
EXPORTED_LEGACY_DIFF_ALIASES = {
    "segment_adaptive", "segment_logspace",
    "volatility_adaptive", "personal_hard",
}


def _legacy_diff_build(y, anchor):
    return build_model_target(y, anchor, "log_diff")


def _legacy_diff_restore(prediction, anchor):
    return inverse_model_target(prediction, anchor, "log_diff")


EXPORTED_TARGET_MODE_HANDLERS = {
    mode: {"build": _legacy_diff_build, "restore": _legacy_diff_restore}
    for mode in EXPORTED_LEGACY_DIFF_ALIASES
}


def register_exported_target_mode(mode, build, restore):
    EXPORTED_TARGET_MODE_HANDLERS[str(mode)] = {"build": build, "restore": restore}


# Only catboost_segment_adaptive came from the dedicated CPU artifact. Keep
# that single specification frozen; the other exported specifications are not
# labelled CPU and remain eligible for the separate exact experiment contour.
FROZEN_EXPORTED_MODELS = {"catboost_segment_adaptive"}
ENABLE_EXPORTED_EXACT_MODELS = False  # optional tail block; enable only if runtime remains
ENABLE_EXPORTED_LOG_LEVEL_PROXIES = False
EXPORTED_SKIP_MISSING_FEATURES = True

if ENABLE_EXPORTED_EXACT_MODELS and ENABLE_EXPORTED_LOG_LEVEL_PROXIES:
    raise ValueError("Exact and log_level proxy modes are mutually exclusive")

EXPORTED_MODEL_SPECS = {}
_export_audit = []
_forbidden = {USER_COL, INDEX_COL, TARGET_COL, SNAPSHOT_COL}

for row in EXPORTED_MODEL_ROWS:
    spec = dict(row)
    source_features = unique_keep_order(spec["features"])
    dropped = [c for c in source_features if c in _forbidden]
    safe_features = [c for c in source_features if c not in _forbidden]
    missing = [c for c in safe_features if c not in COMMON_COLUMNS]
    spec.update({
        "source_features": source_features,
        "features": safe_features,
        "dropped_forbidden_features": dropped,
        "missing_features": missing,
        "maturity": "experimental_export",
        "artifact_origin": (
            "cpu_artifact" if spec["name"] == "catboost_segment_adaptive"
            else "combined_model_export"
        ),
        "execution_frozen": spec["name"] in FROZEN_EXPORTED_MODELS,
    })
    # best_params_json is family-native. final_eval_params_raw is provenance
    # only: that generic dictionary mixes keys from different libraries.
    spec["params"] = resolve_effective_params(spec)
    EXPORTED_MODEL_SPECS[spec["name"]] = spec
    _handler_ready = spec["target_mode"] in EXPORTED_TARGET_MODE_HANDLERS
    _export_audit.append({
        "model": spec["name"],
        "family": spec["family"],
        "target_mode": spec["target_mode"],
        "features_exported": len(source_features),
        "features_safe": len(safe_features),
        "missing_count": len(missing),
        "missing_features": ";".join(missing),
        "dropped_forbidden": ";".join(dropped),
        "selection_metric_export": spec["selection_metric"],
        "optuna_metric_export": spec["optuna_metric"],
        "exact_target_handler": _handler_ready,
        "runnable_exact": _handler_ready and not missing and spec["name"] not in FROZEN_EXPORTED_MODELS,
        "runnable_log_level_proxy": not missing and spec["name"] not in FROZEN_EXPORTED_MODELS,
        "artifact_origin": spec["artifact_origin"],
        "execution_frozen": spec["execution_frozen"],
        "parameter_source": "best_params_json",
    })

EXPORTED_MODEL_AUDIT = pd.DataFrame(_export_audit)
EXPORTED_MODEL_AUDIT.to_csv(ANALYSIS_DIR / "experimental_model_audit.csv", index=False)
display(EXPORTED_MODEL_AUDIT)

# Six baseline specifications remain recorded, while active training excludes
# both LightGBM modes. Exported adaptive specifications never mutate MODEL_ORDER.
assert len(ALL_BASE_MODEL_ORDER) == 6
assert len(MODEL_ORDER) == 4
assert all(MODEL_SPECS[name]["family"] in {"catboost", "xgboost"} for name in MODEL_ORDER)
assert len(EXPORTED_MODEL_SPECS) == 6
assert set(ALL_BASE_MODEL_ORDER).isdisjoint(EXPORTED_MODEL_SPECS)

# user_id appeared in one exported feature list. It is an identifier, not a
# behavioural feature, and is removed before any matrix is materialized.
assert EXPORTED_MODEL_SPECS["xgboost_volatility_adaptive"]["dropped_forbidden_features"] == ["user_id"]
assert USER_COL not in EXPORTED_MODEL_SPECS["xgboost_volatility_adaptive"]["features"]

def _export_build_target(y, anchor, mode):
    if mode in {"log_level", "log_diff"}:
        return build_model_target(y, anchor, mode)
    if mode not in EXPORTED_TARGET_MODE_HANDLERS:
        raise RuntimeError(f"No exact target formula registered for {mode!r}")
    result = np.asarray(EXPORTED_TARGET_MODE_HANDLERS[mode]["build"](y, anchor), dtype=np.float64)
    if result.shape != np.asarray(y).shape or not np.isfinite(result).all():
        raise ValueError(f"Invalid target handler output for {mode}")
    return result


def _export_restore(pred, anchor, mode):
    if mode in {"log_level", "log_diff"}:
        return inverse_model_target(pred, anchor, mode)
    if mode not in EXPORTED_TARGET_MODE_HANDLERS:
        raise RuntimeError(f"No exact restore formula registered for {mode!r}")
    result = np.asarray(EXPORTED_TARGET_MODE_HANDLERS[mode]["restore"](pred, anchor), dtype=np.float64)
    if result.shape != np.asarray(pred).shape or not np.isfinite(result).all():
        raise ValueError(f"Invalid restore handler output for {mode}")
    return np.clip(result, 0.0, None)


_contract_y = np.array([0.0, 10.0, 1000.0])
_contract_anchor = np.array([0.0, 20.0, 500.0])
for _mode in sorted(EXPORTED_LEGACY_DIFF_ALIASES):
    _round_trip = _export_restore(
        _export_build_target(_contract_y, _contract_anchor, _mode),
        _contract_anchor, _mode,
    )
    assert np.allclose(_round_trip, _contract_y), _mode
print("[PASS] exported legacy target aliases reproduce nsel/model.py log-diff contract")


,model,family,target_mode,features_exported,features_safe,missing_count,missing_features,dropped_forbidden,selection_metric_export,optuna_metric_export,exact_target_handler,runnable_exact,runnable_log_level_proxy,artifact_origin,execution_frozen,parameter_source
0,xgboost_segment_adaptive,xgboost,segment_adaptive,41,41,0,,,1.722787,1.719629,True,True,True,combined_model_export,False,best_params_json
1,catboost_segment_adaptive,catboost,segment_adaptive,48,48,0,,,1.720539,1.790837,True,False,False,cpu_artifact,True,best_params_json
2,catboost_segment_logspace,catboost,segment_logspace,49,49,0,,,1.721457,1.720302,True,True,True,combined_model_export,False,best_params_json
3,catboost_volatility_adaptive,catboost,volatility_adaptive,49,49,0,,,1.731682,1.730323,True,True,True,combined_model_export,False,best_params_json
4,xgboost_personal_hard,xgboost,personal_hard,54,54,0,,,1.779188,1.773674,True,True,True,combined_model_export,False,best_params_json
5,xgboost_volatility_adaptive,xgboost,volatility_adaptive,45,44,0,,user_id,1.735041,1.732342,True,True,True,combined_model_export,False,best_params_json


[PASS] exported legacy target aliases reproduce nsel/model.py log-diff contract


In [ ]:
def _load_one_exported_snapshot(cutoff, numeric, categorical, spec):
    result = _load_one_model_snapshot(cutoff, numeric, categorical)
    mode = spec["target_mode"]
    if mode in ADAPTIVE_ANCHOR_BY_TARGET_MODE:
        anchor_column = ADAPTIVE_ANCHOR_BY_TARGET_MODE[mode]
        result["anchor"] = adaptive_anchor_vector(
            cutoff, result["user_id"], anchor_column
        )
        result["anchor_column"] = anchor_column
    else:
        result["anchor_column"] = ANCHOR_COL
    return result


def _materialize_exported_pair(train_cutoffs, valid_cutoff, spec, require_valid_target=True):
    numeric, categorical, columns = model_feature_order(spec)
    train_parts = [
        _load_one_exported_snapshot(c, numeric, categorical, spec)
        for c in train_cutoffs
    ]
    valid = _load_one_exported_snapshot(valid_cutoff, numeric, categorical, spec)
    X_train = np.concatenate([x["X"] for x in train_parts], axis=0)
    y_raw = np.concatenate([x["y_raw"] for x in train_parts])
    anchor = np.concatenate([x["anchor"] for x in train_parts])
    blocks = np.concatenate([
        np.full(len(x["y_raw"]), i, dtype=np.int32) for i, x in enumerate(train_parts)
    ])
    y_train = _export_build_target(y_raw, anchor, spec["target_mode"])
    y_valid = (
        _export_build_target(valid["y_raw"], valid["anchor"], spec["target_mode"])
        if require_valid_target else None
    )
    cat_positions = list(range(len(numeric), len(columns)))
    if spec["family"] != "catboost" and cat_positions:
        X_train, valid["X"], _ = encode_matrices(
            X_train, valid["X"], y_train, blocks, cat_positions, ENCODING_CONFIG
        )
        cat_positions = []
    return X_train, y_train, valid, y_valid, cat_positions


def _export_run_specs():
    result = []
    for source in EXPORTED_MODEL_SPECS.values():
        if source["name"] in FROZEN_EXPORTED_MODELS:
            print("FROZEN", source["name"], "— CPU artifact retained but not scheduled")
            continue
        if source["missing_features"]:
            message = f"{source['name']}: missing {source['missing_features']}"
            if EXPORTED_SKIP_MISSING_FEATURES:
                print("SKIP", message)
                continue
            raise KeyError(message)
        if ENABLE_EXPORTED_EXACT_MODELS:
            if source["target_mode"] not in EXPORTED_TARGET_MODE_HANDLERS:
                raise RuntimeError(
                    f"{source['name']}: formula for {source['target_mode']} was not supplied"
                )
            run = dict(source)
            run["run_name"] = "exported_exact__" + source["name"]
            run["source_target_mode"] = source["target_mode"]
            result.append(run)
        if ENABLE_EXPORTED_LOG_LEVEL_PROXIES:
            run = dict(source)
            run["source_target_mode"] = source["target_mode"]
            run["target_mode"] = "log_level"
            run["run_name"] = "exported_log_level_proxy__" + source["name"]
            result.append(run)
    return result


EXPORTED_RUN_SPECS = _export_run_specs()
EXPORTED_CV_ROWS = []
EXPORTED_OOF = {}

if RUN_CV and EXPORTED_RUN_SPECS:
    for spec in EXPORTED_RUN_SPECS:
        parts = []
        for fold in CV_FOLDS:
            Xtr, ytr, valid, yva, cat_pos = _materialize_exported_pair(
                fold["train_cutoffs"], fold["valid_cutoff"], spec
            )
            model, pred_model, best_round = fit_family(
                spec["family"], Xtr, ytr, valid["X"], yva, cat_pos, spec
            )
            pred = _export_restore(pred_model, valid["anchor"], spec["target_mode"])
            EXPORTED_CV_ROWS.append({
                "model": spec["run_name"], "source_model": spec["name"],
                "source_target_mode": spec["source_target_mode"],
                "effective_target_mode": spec["target_mode"],
                "fold": fold["name"], "weight": fold["weight"],
                "rmsle": rmsle(valid["y_raw"], pred), "best_round": best_round,
            })
            parts.append(pd.DataFrame({
                "fold": fold["name"], SNAPSHOT_COL: pd.Timestamp(fold["valid_cutoff"]),
                USER_COL: valid["user_id"], spec["run_name"]: pred,
            }))
            del Xtr, ytr, valid, yva, model, pred_model, pred
            gc.collect()
        EXPORTED_OOF[spec["run_name"]] = pd.concat(parts, ignore_index=True)

EXPORTED_CV_METRICS = pd.DataFrame(EXPORTED_CV_ROWS)
if not EXPORTED_CV_METRICS.empty:
    EXPORTED_CV_METRICS.to_csv(ANALYSIS_DIR / "experimental_model_cv.csv", index=False)
    for name, frame in EXPORTED_OOF.items():
        OOF = OOF.merge(
            frame, on=["fold", SNAPSHOT_COL, USER_COL],
            how="left", validate="one_to_one",
        )
    EXPORTED_CV_SUMMARY = (
        EXPORTED_CV_METRICS.assign(weighted=lambda x: x["weight"] * x["rmsle"])
        .groupby("model", as_index=False)
        .agg(weighted_rmsle=("weighted", "sum"), mean_best_round=("best_round", "mean"))
        .sort_values("weighted_rmsle")
    )
    display(EXPORTED_CV_SUMMARY)


def _predict_exported_native(model, family, X, cat_positions, rounds):
    if family == "catboost":
        from catboost import Pool
        return model.predict(Pool(X, cat_features=cat_positions))
    if family == "lightgbm":
        return model.predict(X, num_iteration=rounds)
    return model.predict(X)


EXPORTED_FINAL_COLUMNS = []
if RUN_FINAL_FIT and EXPORTED_RUN_SPECS:
    for spec in EXPORTED_RUN_SPECS:
        run_name = spec["run_name"]
        fold_rows = EXPORTED_CV_METRICS.loc[EXPORTED_CV_METRICS["model"].eq(run_name)]
        if fold_rows.empty:
            raise RuntimeError(f"No CV rounds for {run_name}")
        final_rounds = max(1, int(round(np.sum(fold_rows["weight"] * fold_rows["best_round"]))))
        Xtr, ytr, valid, _, cat_pos = _materialize_exported_pair(
            FINAL_TRAIN_CUTOFFS, PREDICTION_CUTOFF, spec,
            require_valid_target=False,
        )
        model, _, used_rounds = fit_family(
            spec["family"], Xtr, ytr, None, None, cat_pos, spec,
            final_rounds=final_rounds,
        )
        pred_model = _predict_exported_native(
            model, spec["family"], valid["X"], cat_pos, used_rounds
        )
        pred = _export_restore(pred_model, valid["anchor"], spec["target_mode"])
        pred_frame = pd.DataFrame({USER_COL: valid["user_id"], run_name: pred})
        FINAL_PREDICTIONS = FINAL_PREDICTIONS.merge(
            pred_frame, on=USER_COL, how="left", validate="one_to_one", sort=False
        )
        build_base_submission(valid["user_id"], pred).to_csv(
            BASE_SUBMISSION_DIR / f"submission_{run_name}.csv", index=False
        )
        if SAVE_MODELS:
            save_native_model(model, spec["family"], MODEL_DIR / run_name)
        EXPORTED_FINAL_COLUMNS.append(run_name)
        del Xtr, ytr, valid, model, pred_model, pred, pred_frame
        gc.collect()

# Once an exported run is explicitly enabled, evaluate it both standalone and
# in an honest two-source cross-fitted blend with the existing regression ensemble.
EXPORTED_BLEND_ROWS = []
if RUN_CV and RUN_FINAL_FIT and EXPORTED_FINAL_COLUMNS:
    for run_name in EXPORTED_FINAL_COLUMNS:
        blend_name = "current_plus__" + run_name
        oof_pred, test_pred, metadata = crossfit_optimized_blend(
            ["CURRENT_BEST_ORIGINAL", run_name], "log"
        )
        metadata.update({
            "evaluation_type": "cross_fitted_oof",
            "source_model": run_name,
            "source_is_proxy": run_name.startswith("exported_log_level_proxy__"),
        })
        add_blend(
            blend_name, oof_pred, test_pred, "exported_plus_current",
            complexity=3, selection_safe=True, metadata=metadata,
        )
        score = weighted_oof_rmsle(OOF, oof_pred)
        EXPORTED_BLEND_ROWS.append({"blend": blend_name, "weighted_rmsle": score})
        build_base_submission(sample_submit[USER_COL], test_pred).to_csv(
            BLEND_SUBMISSION_DIR / f"submission_{blend_name}.csv", index=False
        )
    pd.DataFrame(EXPORTED_BLEND_ROWS).sort_values("weighted_rmsle").to_csv(
        ANALYSIS_DIR / "experimental_model_blends.csv", index=False
    )

print({
    "exact_models_enabled": ENABLE_EXPORTED_EXACT_MODELS,
    "log_level_proxies_enabled": ENABLE_EXPORTED_LOG_LEVEL_PROXIES,
    "frozen_models": sorted(FROZEN_EXPORTED_MODELS),
    "scheduled_runs": [x["run_name"] for x in EXPORTED_RUN_SPECS],
    "target_contract": "nsel/model.py: every non-log_level label is a log_diff alias",
})


FROZEN catboost_segment_adaptive — CPU artifact retained but not scheduled
{'exact_models_enabled': False, 'log_level_proxies_enabled': False, 'frozen_models': ['catboost_segment_adaptive'], 'scheduled_runs': [], 'target_contract': 'nsel/model.py: every non-log_level label is a log_diff alias'}


In [133]:
# =============================================================================
# MEGA BLEND — ВСЕ SUBMISSION-ФАЙЛЫ
# =============================================================================

from pathlib import Path
import hashlib
import numpy as np
import pandas as pd


ROOT = Path("/kaggle/working/ozon_final")
OUT_DIR = ROOT / "mega_blends"
OUT_DIR.mkdir(parents=True, exist_ok=True)


# -----------------------------------------------------------------------------
# 1. Собираем все submission*.csv рекурсивно
# -----------------------------------------------------------------------------

all_paths = sorted(ROOT.rglob("submission*.csv"))

# Не включаем собственные mega-blends, если ячейка запускается повторно
all_paths = [
    p for p in all_paths
    if "mega_blends" not in p.parts
]

print(f"Найдено submission-файлов: {len(all_paths)}")


# -----------------------------------------------------------------------------
# 2. Эталонный user_id
# -----------------------------------------------------------------------------

reference = sample_submit[[USER_COL]].copy()
reference[USER_COL] = reference[USER_COL].astype(str)

n_users = len(reference)


def load_submission(path):
    df = pd.read_csv(path)

    if USER_COL not in df.columns:
        return None, f"нет {USER_COL}"

    # В нормальном случае используем известную target-колонку
    if SUBMISSION_TARGET_COL in df.columns:
        pred_col = SUBMISSION_TARGET_COL
    else:
        candidate_cols = [c for c in df.columns if c != USER_COL]

        if len(candidate_cols) != 1:
            return None, "не удалось однозначно определить prediction column"

        pred_col = candidate_cols[0]

    tmp = df[[USER_COL, pred_col]].copy()
    tmp[USER_COL] = tmp[USER_COL].astype(str)

    if tmp[USER_COL].duplicated().any():
        return None, "duplicate user_id"

    # Жёстко align по user_id, а не по позиции строк
    tmp = reference.merge(
        tmp,
        on=USER_COL,
        how="left",
        validate="one_to_one",
    )

    if len(tmp) != n_users:
        return None, "не совпало число пользователей"

    pred = pd.to_numeric(
        tmp[pred_col],
        errors="coerce",
    ).to_numpy(dtype=np.float64)

    if not np.isfinite(pred).all():
        return None, "NaN/inf"

    if np.any(pred < 0):
        return None, "есть отрицательные prediction"

    return pred, None


# -----------------------------------------------------------------------------
# 3. Загружаем валидные сабмиты
# -----------------------------------------------------------------------------

records = []
skipped = []

for path in all_paths:
    pred, error = load_submission(path)

    if error is not None:
        skipped.append((str(path), error))
        continue

    records.append({
        "path": path,
        "pred": pred,
    })


print(f"Валидных submission-файлов: {len(records)}")

if skipped:
    print(f"Пропущено: {len(skipped)}")
    for path, reason in skipped[:20]:
        print(" SKIP:", path, "|", reason)


if not records:
    raise RuntimeError("Не найдено ни одного валидного submission-файла.")


# -----------------------------------------------------------------------------
# 4. Удаляем ТОЧНЫЕ дубликаты прогнозов
#
# Это важно:
# один и тот же prediction мог быть сохранён под несколькими именами.
# Не хотим случайно дать ему повышенный вес.
# -----------------------------------------------------------------------------

unique_records = []
seen_hashes = {}

for rec in records:
    pred = rec["pred"]

    # округление только для определения практически идентичных файлов
    pred_hash = hashlib.sha256(
        np.round(pred, 10).tobytes()
    ).hexdigest()

    if pred_hash not in seen_hashes:
        seen_hashes[pred_hash] = rec["path"]
        unique_records.append(rec)


print(
    f"Уникальных prediction-векторов: "
    f"{len(unique_records)} / {len(records)}"
)


# -----------------------------------------------------------------------------
# 5. Вариант A:
# РАВНОЕ СРЕДНЕЕ ВСЕХ УНИКАЛЬНЫХ SUBMISSIONS В LOG1P-SPACE
# -----------------------------------------------------------------------------

log_matrix = np.column_stack([
    np.log1p(rec["pred"])
    for rec in unique_records
])

mega_equal_log = log_matrix.mean(axis=1)
mega_equal = np.expm1(mega_equal_log)
mega_equal = np.clip(mega_equal, 0.0, None)


submission_equal = sample_submit.copy()
submission_equal[SUBMISSION_TARGET_COL] = mega_equal

equal_path = OUT_DIR / "submission_MEGA_ALL_UNIQUE_EQUAL_LOG.csv"

submission_equal.to_csv(
    equal_path,
    index=False,
)


# -----------------------------------------------------------------------------
# 6. Разбиваем сабмиты на крупные семьи
# -----------------------------------------------------------------------------

def get_family(path):
    path_str = str(path).lower()
    name = path.name.lower()

    # 74 external / colleague blends
    if "external_colleague_blends" in path_str:
        return "external_blends"

    # Base models
    if "base_submissions" in path_str:
        return "base_models"

    # Обычные blend submissions
    if "blend_submissions" in path_str:
        return "internal_blends"

    # Hurdle
    if "hurdle" in name or "hurdle" in path_str:
        return "hurdle"

    # Stochastic / diversity
    if any(x in path_str for x in [
        "stochastic",
        "ridge",
        "hierarchy",
        "bag",
        "specialist",
    ]):
        return "stochastic"

    # Главные submission-файлы в ROOT
    return "other_final"


family_records = {}

for rec in unique_records:
    family = get_family(rec["path"])
    family_records.setdefault(family, []).append(rec)


print("\nСемьи:")
for family, items in family_records.items():
    print(f"  {family:20s}: {len(items)}")


# -----------------------------------------------------------------------------
# 7. Сначала усредняем ВНУТРИ каждой семьи
#
# Затем каждая семья получает ОДИНАКОВЫЙ вес.
#
# Благодаря этому:
# 74 external blends не получают в 74 раза больший вес,
# чем один hurdle submission.
# -----------------------------------------------------------------------------

family_log_predictions = {}

for family, items in family_records.items():

    family_matrix = np.column_stack([
        np.log1p(rec["pred"])
        for rec in items
    ])

    family_log_predictions[family] = family_matrix.mean(axis=1)


family_matrix = np.column_stack(
    list(family_log_predictions.values())
)

mega_family_log = family_matrix.mean(axis=1)
mega_family = np.expm1(mega_family_log)
mega_family = np.clip(mega_family, 0.0, None)


submission_family = sample_submit.copy()
submission_family[SUBMISSION_TARGET_COL] = mega_family

family_path = OUT_DIR / "submission_MEGA_FAMILY_BALANCED_LOG.csv"

submission_family.to_csv(
    family_path,
    index=False,
)


# -----------------------------------------------------------------------------
# 8. Отчёт — какие файлы реально вошли
# -----------------------------------------------------------------------------

manifest_rows = []

for rec in unique_records:
    manifest_rows.append({
        "family": get_family(rec["path"]),
        "filename": rec["path"].name,
        "path": str(rec["path"]),
    })

manifest = pd.DataFrame(manifest_rows)

manifest_path = OUT_DIR / "mega_blend_manifest.csv"
manifest.to_csv(manifest_path, index=False)


# -----------------------------------------------------------------------------
# 9. Простая диагностика результатов
# -----------------------------------------------------------------------------

def describe_pred(name, pred):
    log_pred = np.log1p(pred)

    return {
        "name": name,
        "mean_raw": float(np.mean(pred)),
        "median_raw": float(np.median(pred)),
        "mean_log1p": float(np.mean(log_pred)),
        "std_log1p": float(np.std(log_pred)),
        "p90": float(np.quantile(pred, 0.90)),
        "p99": float(np.quantile(pred, 0.99)),
        "max": float(np.max(pred)),
    }


diagnostics = pd.DataFrame([
    describe_pred(
        "MEGA_ALL_UNIQUE_EQUAL_LOG",
        mega_equal,
    ),
    describe_pred(
        "MEGA_FAMILY_BALANCED_LOG",
        mega_family,
    ),
])

diagnostics.to_csv(
    OUT_DIR / "mega_blend_diagnostics.csv",
    index=False,
)


print("\n" + "=" * 80)
print("MEGA BLENDS READY")
print("=" * 80)

print("\n1. Все уникальные submissions с равными весами:")
print(equal_path)

print("\n2. Family-balanced — РЕКОМЕНДУЕМЫЙ вариант:")
print(family_path)

print("\nManifest:")
print(manifest_path)

print("\nDiagnostics:")
display(diagnostics)

print("\nКоличество файлов по семьям:")
display(
    manifest.groupby("family")
    .size()
    .rename("n_submissions")
    .sort_values(ascending=False)
    .to_frame()
)

Найдено submission-файлов: 108
Валидных submission-файлов: 108
Уникальных prediction-векторов: 99 / 108

Семьи:
  base_models         : 4
  internal_blends     : 15
  external_blends     : 73
  hurdle              : 4
  stochastic          : 3

MEGA BLENDS READY

1. Все уникальные submissions с равными весами:
/kaggle/working/ozon_final/mega_blends/submission_MEGA_ALL_UNIQUE_EQUAL_LOG.csv

2. Family-balanced — РЕКОМЕНДУЕМЫЙ вариант:
/kaggle/working/ozon_final/mega_blends/submission_MEGA_FAMILY_BALANCED_LOG.csv

Manifest:
/kaggle/working/ozon_final/mega_blends/mega_blend_manifest.csv

Diagnostics:


,name,mean_raw,median_raw,mean_log1p,std_log1p,p90,p99,max
0,MEGA_ALL_UNIQUE_EQUAL_LOG,40.575464,7.00429,2.330908,1.601405,102.704482,491.060670,8246.087341
1,MEGA_FAMILY_BALANCED_LOG,38.474087,6.91162,2.316372,1.579571,98.174168,457.380605,8720.623913



Количество файлов по семьям:


,n_submissions
family,
external_blends,73
internal_blends,15
base_models,4
hurdle,4
stochastic,3


In [135]:
# =============================================================================
# FINAL 50/50 BLEND
#
# 50% = EQUAL LOG-BLEND OF ALL OWN BASE + BAGGING REGRESSORS
# 50% = EQUAL LOG-BLEND OF:
#       - submission_hurdle_seed42.csv
#       - submission_hurdle_seed7.csv
#
# В текущем notebook:
#   4 base regressors
#   + все run_name из STOCHASTIC_REGISTRY
#
# При STOCHASTIC_N_MEMBERS = 1:
#   4 base + 4 bagging = 8 собственных регрессоров
#
# Тогда:
#   каждый own regressor = 50% / 8 = 6.25%
#   Denis seed42        = 25%
#   Denis seed7         = 25%
#
# Всё смешивается в log1p-space.
# =============================================================================

from pathlib import Path
import numpy as np
import pandas as pd


# =============================================================================
# PATHS
# =============================================================================

ROOT = Path("/kaggle/working/ozon_final")
KAGGLE_INPUT = Path("/kaggle/input")

OUT_DIR = ROOT / "final_50_50_all_regressors_denis"
OUT_DIR.mkdir(parents=True, exist_ok=True)


# =============================================================================
# SETTINGS
# =============================================================================

OWN_WEIGHT = 0.50
DENIS_WEIGHT = 0.50

DENIS_FILENAMES = [
    "submission_hurdle_seed42.csv",
    "submission_hurdle_seed7.csv",
]


# =============================================================================
# 1. OWN REGRESSORS
#
# Берём:
#   - все активные BASE models из MODEL_NAMES;
#   - все отдельные stochastic column-bagging models
#     из STOCHASTIC_REGISTRY.
#
# НЕ берём:
#   - готовые blends;
#   - Ridge;
#   - hierarchy;
#   - affine;
#   - hurdle;
#   - target-weight specialists.
#
# То есть именно отдельные base + bagging boosting regressors.
# =============================================================================

base_regressors = list(MODEL_NAMES)

bagging_regressors = [
    run["run_name"]
    for run in STOCHASTIC_REGISTRY
]


own_regressors = (
    base_regressors
    + bagging_regressors
)


# Удаляем возможные повторения имён, сохраняя порядок
own_regressors = list(dict.fromkeys(own_regressors))


print("=" * 90)
print("OWN REGRESSORS USED")
print("=" * 90)

print("\nBASE:")
for name in base_regressors:
    print(" ", name)

print("\nBAGGING:")
for name in bagging_regressors:
    print(" ", name)

print("\nTOTAL OWN REGRESSORS:", len(own_regressors))


# =============================================================================
# 2. CHECK THAT ALL OWN PREDICTIONS EXIST
# =============================================================================

if FINAL_PREDICTIONS is None:
    raise RuntimeError(
        "FINAL_PREDICTIONS is None. "
        "Сначала должен быть выполнен final fit."
    )


missing = [
    name
    for name in own_regressors
    if name not in FINAL_PREDICTIONS.columns
]

if missing:
    raise RuntimeError(
        "В FINAL_PREDICTIONS отсутствуют модели:\n"
        + "\n".join(missing)
    )


# =============================================================================
# 3. STRICT USER ALIGNMENT
# =============================================================================

reference = sample_submit[[USER_COL]].copy()
reference[USER_COL] = reference[USER_COL].astype(str)

own_frame = FINAL_PREDICTIONS[
    [USER_COL] + own_regressors
].copy()

own_frame[USER_COL] = own_frame[USER_COL].astype(str)


if own_frame[USER_COL].duplicated().any():
    raise ValueError(
        "FINAL_PREDICTIONS contains duplicated user_id"
    )


own_frame = reference.merge(
    own_frame,
    on=USER_COL,
    how="left",
    validate="one_to_one",
)


if own_frame[own_regressors].isna().any().any():
    raise ValueError(
        "NaN after aligning FINAL_PREDICTIONS with sample_submit"
    )


own_matrix = own_frame[
    own_regressors
].to_numpy(dtype=np.float64)


if not np.isfinite(own_matrix).all():
    raise ValueError(
        "Own regressors contain NaN/inf"
    )

if (own_matrix < 0).any():
    raise ValueError(
        "Own regressors contain negative predictions"
    )


# =============================================================================
# 4. EQUAL LOG-BLEND OF ALL OWN BASE + BAGGING REGRESSORS
# =============================================================================

own_log_matrix = np.log1p(
    np.clip(
        own_matrix,
        0.0,
        None,
    )
)

own_blend_log = own_log_matrix.mean(axis=1)

own_blend = np.expm1(
    own_blend_log
)

own_blend = np.clip(
    own_blend,
    0.0,
    None,
)


# =============================================================================
# 5. FIND DENIS FILES AUTOMATICALLY
# =============================================================================

DENIS_FILES = []


for filename in DENIS_FILENAMES:

    matches = list(
        KAGGLE_INPUT.rglob(filename)
    )

    if len(matches) == 0:
        raise FileNotFoundError(
            f"\nНе найден файл:\n"
            f"  {filename}\n"
            f"внутри:\n"
            f"  {KAGGLE_INPUT}\n\n"
            f"Проверь, что Denis_2_files добавлен "
            f"в Notebook Inputs."
        )

    if len(matches) > 1:

        print(
            f"\nWARNING: найдено несколько "
            f"{filename}:"
        )

        for match in matches:
            print(" ", match)

        print(
            "Использую первый найденный файл."
        )

    DENIS_FILES.append(
        matches[0]
    )


print("\n" + "=" * 90)
print("DENIS FILES")
print("=" * 90)

for path in DENIS_FILES:
    print(path)


# =============================================================================
# 6. LOAD EXTERNAL SUBMISSION
# =============================================================================

def load_submission(path):

    df = pd.read_csv(path)

    if USER_COL not in df.columns:
        raise ValueError(
            f"{path}: missing {USER_COL}"
        )

    if SUBMISSION_TARGET_COL in df.columns:

        pred_col = SUBMISSION_TARGET_COL

    else:

        candidate_cols = [
            col
            for col in df.columns
            if col != USER_COL
        ]

        if len(candidate_cols) != 1:
            raise ValueError(
                f"{path}: cannot determine "
                f"prediction column.\n"
                f"Columns: {list(df.columns)}"
            )

        pred_col = candidate_cols[0]


    tmp = df[
        [USER_COL, pred_col]
    ].copy()

    tmp[USER_COL] = (
        tmp[USER_COL]
        .astype(str)
    )


    if tmp[USER_COL].duplicated().any():
        raise ValueError(
            f"{path}: duplicated {USER_COL}"
        )


    tmp = reference.merge(
        tmp,
        on=USER_COL,
        how="left",
        validate="one_to_one",
    )


    pred = pd.to_numeric(
        tmp[pred_col],
        errors="coerce",
    ).to_numpy(
        dtype=np.float64
    )


    if not np.isfinite(pred).all():
        raise ValueError(
            f"{path}: NaN/inf after alignment"
        )

    if (pred < 0).any():
        raise ValueError(
            f"{path}: negative predictions"
        )


    return pred


# =============================================================================
# 7. DENIS 50/50 INTERNAL BLEND
# =============================================================================

denis_preds = [
    load_submission(path)
    for path in DENIS_FILES
]


if len(denis_preds) != 2:
    raise RuntimeError(
        f"Expected 2 Denis predictions, "
        f"got {len(denis_preds)}"
    )


denis_log_matrix = np.column_stack([
    np.log1p(pred)
    for pred in denis_preds
])


denis_blend_log = (
    denis_log_matrix.mean(axis=1)
)


denis_blend = np.expm1(
    denis_blend_log
)

denis_blend = np.clip(
    denis_blend,
    0.0,
    None,
)


# =============================================================================
# 8. FINAL 50 / 50
#
# 50% = ALL OWN BASE + BAGGING
# 50% = DENIS seed42 + seed7
# =============================================================================

final_log = (
    OWN_WEIGHT
    * own_blend_log

    +

    DENIS_WEIGHT
    * denis_blend_log
)


final_pred = np.expm1(
    final_log
)

final_pred = np.clip(
    final_pred,
    0.0,
    None,
)


# =============================================================================
# 9. SAVE FINAL SUBMISSION
# =============================================================================

submission = sample_submit.copy()

submission[
    SUBMISSION_TARGET_COL
] = final_pred


OUTPUT_PATH = (
    OUT_DIR
    / "submission_50pct_all_base_bagging_50pct_denis.csv"
)


submission.to_csv(
    OUTPUT_PATH,
    index=False,
)


# =============================================================================
# 10. SAVE COMPONENTS
# =============================================================================

own_submission = (
    sample_submit.copy()
)

own_submission[
    SUBMISSION_TARGET_COL
] = own_blend


OWN_COMPONENT_PATH = (
    OUT_DIR
    / "submission_COMPONENT_all_base_bagging.csv"
)

own_submission.to_csv(
    OWN_COMPONENT_PATH,
    index=False,
)


denis_submission = (
    sample_submit.copy()
)

denis_submission[
    SUBMISSION_TARGET_COL
] = denis_blend


DENIS_COMPONENT_PATH = (
    OUT_DIR
    / "submission_COMPONENT_denis_seed42_seed7.csv"
)

denis_submission.to_csv(
    DENIS_COMPONENT_PATH,
    index=False,
)


# =============================================================================
# 11. WEIGHT REPORT
# =============================================================================

own_individual_weight = (
    OWN_WEIGHT
    / len(own_regressors)
)


weight_rows = []


for name in own_regressors:

    weight_rows.append({
        "prediction": name,
        "family": (
            "bagging"
            if "__bag_" in name
            else "base"
        ),
        "final_weight": (
            own_individual_weight
        ),
    })


weight_rows.extend([
    {
        "prediction":
            "submission_hurdle_seed42.csv",
        "family":
            "denis_hurdle",
        "final_weight":
            DENIS_WEIGHT / 2,
    },
    {
        "prediction":
            "submission_hurdle_seed7.csv",
        "family":
            "denis_hurdle",
        "final_weight":
            DENIS_WEIGHT / 2,
    },
])


weights_df = pd.DataFrame(
    weight_rows
)


# =============================================================================
# 12. DIAGNOSTICS
# =============================================================================

diagnostics = pd.DataFrame([
    {
        "prediction":
            "OWN_ALL_BASE_BAGGING",

        "n_members":
            len(own_regressors),

        "mean_log1p":
            float(
                own_blend_log.mean()
            ),

        "std_log1p":
            float(
                own_blend_log.std()
            ),

        "mean_raw":
            float(
                own_blend.mean()
            ),

        "median_raw":
            float(
                np.median(own_blend)
            ),
    },

    {
        "prediction":
            "DENIS_HURDLE",

        "n_members":
            2,

        "mean_log1p":
            float(
                denis_blend_log.mean()
            ),

        "std_log1p":
            float(
                denis_blend_log.std()
            ),

        "mean_raw":
            float(
                denis_blend.mean()
            ),

        "median_raw":
            float(
                np.median(denis_blend)
            ),
    },

    {
        "prediction":
            "FINAL_50_50",

        "n_members":
            len(own_regressors) + 2,

        "mean_log1p":
            float(
                final_log.mean()
            ),

        "std_log1p":
            float(
                final_log.std()
            ),

        "mean_raw":
            float(
                final_pred.mean()
            ),

        "median_raw":
            float(
                np.median(final_pred)
            ),
    },
])


# =============================================================================
# 13. CORRELATION
# =============================================================================

correlation_df = pd.DataFrame({
    "own_all_base_bagging":
        own_blend_log,

    "denis_seed42":
        np.log1p(
            denis_preds[0]
        ),

    "denis_seed7":
        np.log1p(
            denis_preds[1]
        ),

    "denis_equal":
        denis_blend_log,

    "final_50_50":
        final_log,
}).corr()


# =============================================================================
# 14. SAVE REPORTS
# =============================================================================

weights_df.to_csv(
    OUT_DIR
    / "final_weights.csv",
    index=False,
)

diagnostics.to_csv(
    OUT_DIR
    / "diagnostics.csv",
    index=False,
)

correlation_df.to_csv(
    OUT_DIR
    / "correlations_logspace.csv",
)


# =============================================================================
# 15. FINAL REPORT
# =============================================================================

print("\n" + "=" * 90)
print("FINAL BLEND READY")
print("=" * 90)


print(
    f"\nOWN REGRESSORS: "
    f"{len(own_regressors)}"
)

print(
    f"BASE: "
    f"{len(base_regressors)}"
)

print(
    f"BAGGING: "
    f"{len(bagging_regressors)}"
)


print(
    "\nWeight of EACH own regressor:"
)

print(
    f"{own_individual_weight:.6f} "
    f"= "
    f"{own_individual_weight * 100:.3f}%"
)


print(
    "\nDenis seed42:"
)

print(
    f"{DENIS_WEIGHT / 2:.6f} "
    f"= "
    f"{DENIS_WEIGHT / 2 * 100:.2f}%"
)


print(
    "\nDenis seed7:"
)

print(
    f"{DENIS_WEIGHT / 2:.6f} "
    f"= "
    f"{DENIS_WEIGHT / 2 * 100:.2f}%"
)


print("\nWEIGHTS:")
display(weights_df)


print("\nDIAGNOSTICS:")
display(diagnostics)


print("\nLOG-SPACE CORRELATIONS:")
display(correlation_df)


print("\nFINAL SUBMISSION:")
print(OUTPUT_PATH)


print("\nOWN COMPONENT:")
print(OWN_COMPONENT_PATH)


print("\nDENIS COMPONENT:")
print(DENIS_COMPONENT_PATH)


print("\n" + "=" * 90)
print("DONE")
print("=" * 90)

OWN REGRESSORS USED

BASE:
  xgboost_log_level
  catboost_log_level
  catboost_log_diff
  xgboost_log_diff

BAGGING:
  xgboost_log_level__bag_s17
  catboost_log_level__bag_s17
  catboost_log_diff__bag_s17
  xgboost_log_diff__bag_s17

TOTAL OWN REGRESSORS: 8

DENIS FILES
/kaggle/input/datasets/vladimirsmirnovve/denis-2-files/submission_hurdle_seed42.csv
/kaggle/input/datasets/vladimirsmirnovve/denis-2-files/submission_hurdle_seed7.csv

FINAL BLEND READY

OWN REGRESSORS: 8
BASE: 4
BAGGING: 4

Weight of EACH own regressor:
0.062500 = 6.250%

Denis seed42:
0.250000 = 25.00%

Denis seed7:
0.250000 = 25.00%

WEIGHTS:


,prediction,family,final_weight
0,xgboost_log_level,base,0.0625
1,catboost_log_level,base,0.0625
2,catboost_log_diff,base,0.0625
3,xgboost_log_diff,base,0.0625
4,xgboost_log_level__bag_s17,bagging,0.0625
5,catboost_log_level__bag_s17,bagging,0.0625
6,catboost_log_diff__bag_s17,bagging,0.0625
7,xgboost_log_diff__bag_s17,bagging,0.0625
8,submission_hurdle_seed42.csv,denis_hurdle,0.2500
9,submission_hurdle_seed7.csv,denis_hurdle,0.2500



DIAGNOSTICS:


,prediction,n_members,mean_log1p,std_log1p,mean_raw,median_raw
0,OWN_ALL_BASE_BAGGING,8,2.345141,1.569373,38.755391,7.215338
1,DENIS_HURDLE,2,2.329351,1.579554,38.770151,6.912459
2,FINAL_50_50,10,2.337246,1.573264,38.678822,7.055836



LOG-SPACE CORRELATIONS:


,own_all_base_bagging,denis_seed42,denis_seed7,denis_equal,final_50_50
own_all_base_bagging,1.000000,0.996892,0.996706,0.996953,0.999233
denis_seed42,0.996892,1.000000,0.999382,0.999845,0.999135
denis_seed7,0.996706,0.999382,1.000000,0.999846,0.999042
denis_equal,0.996953,0.999845,0.999846,1.000000,0.999243
final_50_50,0.999233,0.999135,0.999042,0.999243,1.000000



FINAL SUBMISSION:
/kaggle/working/ozon_final/final_50_50_all_regressors_denis/submission_50pct_all_base_bagging_50pct_denis.csv

OWN COMPONENT:
/kaggle/working/ozon_final/final_50_50_all_regressors_denis/submission_COMPONENT_all_base_bagging.csv

DENIS COMPONENT:
/kaggle/working/ozon_final/final_50_50_all_regressors_denis/submission_COMPONENT_denis_seed42_seed7.csv

DONE


In [136]:
# =============================================================================
# FINAL EXPERIMENT
#
# 70% = Denis hurdle:
#       - submission_hurdle_seed7.csv
#       - submission_hurdle_seed42.csv
#       equal weights inside Denis family
#
# 30% = ALL OWN CLEAN BOOSTING REGRESSORS:
#       - all MODEL_NAMES
#       - all STOCHASTIC_REGISTRY entries with kind == "column_bagging"
#
# Then:
# Polar-style post-calibration
#
#   1) each prediction is centered in log1p-space
#   2) hierarchical weighted blend:
#          70% Denis
#          30% Own regressors
#   3) spread *= 1.0454
#   4) re-center
#   5) find c:
#          mean(max(centered + c, 0)) = 2.329351
#   6) expm1()
#
# Saves BOTH:
#   A) normal 70/30 log-blend WITHOUT calibration
#   B) 70/30 blend WITH exact Polar-style calibration
# =============================================================================

from pathlib import Path
import numpy as np
import pandas as pd


# =============================================================================
# 0. CONFIG
# =============================================================================

DENIS_SEED7_PATH = Path(
    "/kaggle/input/datasets/vladimirsmirnovve/"
    "denis-2-files/submission_hurdle_seed7.csv"
)

DENIS_SEED42_PATH = Path(
    "/kaggle/input/datasets/vladimirsmirnovve/"
    "denis-2-files/submission_hurdle_seed42.csv"
)

DENIS_WEIGHT = 0.70
OWN_WEIGHT = 0.30

POLAR_SPREAD_SCALE = 1.0454
POLAR_TARGET_LOG_MEAN = 2.329351

OUT_DIR = (
    Path("/kaggle/working/ozon_final")
    / "denis70_own30_polarcal"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# =============================================================================
# 1. BASIC CHECKS
# =============================================================================

if abs(
    DENIS_WEIGHT
    + OWN_WEIGHT
    - 1.0
) > 1e-12:
    raise ValueError(
        "DENIS_WEIGHT + OWN_WEIGHT must equal 1."
    )

if FINAL_PREDICTIONS is None:
    raise RuntimeError(
        "FINAL_PREDICTIONS is None. "
        "Сначала выполни final-fit и stochastic blocks."
    )


# =============================================================================
# 2. DEFINE ALL OWN CLEAN REGRESSORS
# =============================================================================
#
# BASE:
#   MODEL_NAMES
#
# BAGGING:
#   только настоящий column-bagging/random-subspace boosting.
#
# НЕ включаем:
#   - target specialists
#   - Ridge
#   - Hurdle
#   - hierarchy
#   - affine
#   - готовые blends
# =============================================================================

base_regressors = list(MODEL_NAMES)

bagging_regressors = [
    run["run_name"]
    for run in STOCHASTIC_REGISTRY
    if run.get("kind") == "column_bagging"
]

own_regressors = list(
    dict.fromkeys(
        base_regressors
        + bagging_regressors
    )
)


print("=" * 90)
print("OWN CLEAN REGRESSORS")
print("=" * 90)

print("\nBASE:")
for name in base_regressors:
    print(" ", name)

print("\nCOLUMN BAGGING:")
for name in bagging_regressors:
    print(" ", name)

print(
    "\nTOTAL OWN CLEAN REGRESSORS:",
    len(own_regressors),
)


# =============================================================================
# 3. MAKE SURE ALL OWN PREDICTIONS EXIST
# =============================================================================

missing = [
    name
    for name in own_regressors
    if name not in FINAL_PREDICTIONS.columns
]

if missing:
    raise RuntimeError(
        "Missing own predictions in FINAL_PREDICTIONS:\n"
        + "\n".join(missing)
    )


# =============================================================================
# 4. REFERENCE USER ORDER
# =============================================================================

reference = (
    sample_submit[[USER_COL]]
    .copy()
)

reference[USER_COL] = (
    reference[USER_COL]
    .astype(str)
)

N_USERS = len(reference)


# =============================================================================
# 5. ALIGN OWN FINAL_PREDICTIONS
# =============================================================================

own_frame = FINAL_PREDICTIONS[
    [USER_COL]
    + own_regressors
].copy()

own_frame[USER_COL] = (
    own_frame[USER_COL]
    .astype(str)
)

if own_frame[USER_COL].duplicated().any():
    raise ValueError(
        "FINAL_PREDICTIONS has duplicated user_id."
    )

own_frame = reference.merge(
    own_frame,
    on=USER_COL,
    how="left",
    validate="one_to_one",
)

own_matrix = (
    own_frame[own_regressors]
    .to_numpy(dtype=np.float64)
)

if not np.isfinite(own_matrix).all():
    raise ValueError(
        "Own regressors contain NaN/inf."
    )

if np.any(own_matrix < 0):
    raise ValueError(
        "Own regressors contain negative predictions."
    )


# =============================================================================
# 6. EXTERNAL SUBMISSION LOADER
# =============================================================================

def load_submission(path):

    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"Submission not found:\n{path}"
        )

    df = pd.read_csv(path)

    if USER_COL not in df.columns:
        raise ValueError(
            f"{path}: missing {USER_COL}"
        )

    if SUBMISSION_TARGET_COL in df.columns:

        pred_col = SUBMISSION_TARGET_COL

    else:

        candidate_cols = [
            c
            for c in df.columns
            if c != USER_COL
        ]

        if len(candidate_cols) != 1:
            raise ValueError(
                f"{path}: cannot uniquely determine "
                f"prediction column.\n"
                f"Columns: {list(df.columns)}"
            )

        pred_col = candidate_cols[0]

    tmp = df[
        [USER_COL, pred_col]
    ].copy()

    tmp[USER_COL] = (
        tmp[USER_COL]
        .astype(str)
    )

    if tmp[USER_COL].duplicated().any():
        raise ValueError(
            f"{path}: duplicated {USER_COL}"
        )

    tmp = reference.merge(
        tmp,
        on=USER_COL,
        how="left",
        validate="one_to_one",
    )

    pred = pd.to_numeric(
        tmp[pred_col],
        errors="coerce",
    ).to_numpy(
        dtype=np.float64
    )

    if len(pred) != N_USERS:
        raise ValueError(
            f"{path}: wrong row count"
        )

    if not np.isfinite(pred).all():
        raise ValueError(
            f"{path}: NaN/inf predictions"
        )

    if np.any(pred < 0):
        raise ValueError(
            f"{path}: negative predictions"
        )

    return pred


# =============================================================================
# 7. LOAD DENIS
# =============================================================================

denis_seed7 = load_submission(
    DENIS_SEED7_PATH
)

denis_seed42 = load_submission(
    DENIS_SEED42_PATH
)


print("\n" + "=" * 90)
print("DENIS FILES")
print("=" * 90)

print(DENIS_SEED7_PATH)
print(DENIS_SEED42_PATH)


# =============================================================================
# 8. ORDINARY COMPONENT BLENDS
#
# Это обычные log1p-blends БЕЗ Polar calibration.
# Сохраняем их отдельно для диагностики.
# =============================================================================

# -------------------------------------------------------------------------
# OWN:
# equal log-weight to every clean own regressor
# -------------------------------------------------------------------------

own_log_matrix = np.log1p(
    own_matrix
)

own_log_blend = (
    own_log_matrix.mean(axis=1)
)

own_raw_blend = np.expm1(
    own_log_blend
)

own_raw_blend = np.clip(
    own_raw_blend,
    0.0,
    None,
)


# -------------------------------------------------------------------------
# DENIS:
# seed7 = 50% inside Denis
# seed42 = 50% inside Denis
# -------------------------------------------------------------------------

denis_log_matrix = np.column_stack([
    np.log1p(denis_seed7),
    np.log1p(denis_seed42),
])

denis_log_blend = (
    denis_log_matrix.mean(axis=1)
)

denis_raw_blend = np.expm1(
    denis_log_blend
)

denis_raw_blend = np.clip(
    denis_raw_blend,
    0.0,
    None,
)


# =============================================================================
# 9. STANDARD 70 / 30 LOG-BLEND
#
# БЕЗ какой-либо Polar calibration.
#
# Это контрольный submission:
#
#   70% Denis
#   30% own clean regressors
# =============================================================================

standard_final_log = (
    DENIS_WEIGHT
    * denis_log_blend
    +
    OWN_WEIGHT
    * own_log_blend
)

standard_final_raw = np.expm1(
    standard_final_log
)

standard_final_raw = np.clip(
    standard_final_raw,
    0.0,
    None,
)


# =============================================================================
# 10. POLAR HIERARCHICAL CENTERING
#
# В исходной схеме каждый individual prediction сначала центрируется:
#
#   p_i_centered = p_i - mean(p_i)
#
# Потом выполняется weighted blend.
# =============================================================================

# -------------------------------------------------------------------------
# OWN individual centered logs
# -------------------------------------------------------------------------

own_centered_log_matrix = (
    own_log_matrix
    - own_log_matrix.mean(
        axis=0,
        keepdims=True,
    )
)

# equal weights inside own family
own_centered_direction = (
    own_centered_log_matrix.mean(
        axis=1
    )
)


# -------------------------------------------------------------------------
# DENIS individual centered logs
# -------------------------------------------------------------------------

denis_centered_log_matrix = (
    denis_log_matrix
    - denis_log_matrix.mean(
        axis=0,
        keepdims=True,
    )
)

# seed7 / seed42 = equal weights
denis_centered_direction = (
    denis_centered_log_matrix.mean(
        axis=1
    )
)


# -------------------------------------------------------------------------
# Hierarchical family blend
#
# Denis = 70%
# Own   = 30%
# -------------------------------------------------------------------------

centered_70_30 = (
    DENIS_WEIGHT
    * denis_centered_direction
    +
    OWN_WEIGHT
    * own_centered_direction
)


# Numerical sanity:
print(
    "\nCentered blend mean:",
    centered_70_30.mean(),
)


# =============================================================================
# 11. POLAR SPREAD SCALING
#
# p_scaled =
#     mean(p)
#     + 1.0454 * (p - mean(p))
# =============================================================================

pre_scale_mean = float(
    centered_70_30.mean()
)

scaled = (
    pre_scale_mean
    +
    POLAR_SPREAD_SCALE
    * (
        centered_70_30
        - pre_scale_mean
    )
)


# =============================================================================
# 12. RE-CENTER
# =============================================================================

scaled_centered = (
    scaled
    - scaled.mean()
)


# =============================================================================
# 13. BINARY SEARCH FOR SHIFT
#
# Find c:
#
# mean(max(scaled_centered + c, 0))
#     == 2.329351
# =============================================================================

def find_shift_for_target_clipped_log_mean(
    centered_log_prediction,
    *,
    target_log_mean,
    iterations=120,
):

    x = np.asarray(
        centered_log_prediction,
        dtype=np.float64,
    )

    target = float(
        target_log_mean
    )

    if target < 0:
        raise ValueError(
            "target_log_mean must be >= 0"
        )

    def objective(shift):

        return float(
            np.maximum(
                x + shift,
                0.0,
            ).mean()
        )

    low = (
        -float(np.max(x))
        - target
        - 10.0
    )

    high = (
        float(np.max(np.abs(x)))
        + target
        + 10.0
    )

    # Guarantee bracket
    for _ in range(30):

        if objective(high) >= target:
            break

        high *= 2.0

    else:
        raise RuntimeError(
            "Cannot bracket calibration shift."
        )

    for _ in range(
        int(iterations)
    ):

        mid = (
            low + high
        ) / 2.0

        if objective(mid) < target:

            low = mid

        else:

            high = mid

    return (
        low + high
    ) / 2.0


polar_shift = (
    find_shift_for_target_clipped_log_mean(
        scaled_centered,
        target_log_mean=POLAR_TARGET_LOG_MEAN,
    )
)


# =============================================================================
# 14. FINAL POLAR-CALIBRATED LOG PREDICTION
# =============================================================================

polar_final_log = np.maximum(
    scaled_centered
    + polar_shift,
    0.0,
)


actual_log_mean = float(
    polar_final_log.mean()
)

if not np.isclose(
    actual_log_mean,
    POLAR_TARGET_LOG_MEAN,
    atol=1e-10,
):
    raise RuntimeError(
        "Polar target log mean was not reached."
    )


# =============================================================================
# 15. BACK TO RAW GMV
# =============================================================================

polar_final_raw = np.expm1(
    polar_final_log
)

polar_final_raw = np.clip(
    polar_final_raw,
    0.0,
    None,
)

if not np.isfinite(
    polar_final_raw
).all():
    raise ValueError(
        "Calibrated prediction contains NaN/inf."
    )


# =============================================================================
# 16. SAVE SUBMISSIONS
# =============================================================================

# -------------------------------------------------------------------------
# A. Uncalibrated 70/30
# -------------------------------------------------------------------------

standard_submission = (
    sample_submit.copy()
)

standard_submission[
    SUBMISSION_TARGET_COL
] = standard_final_raw


STANDARD_PATH = (
    OUT_DIR
    / "submission_Denis70_Own30_LOG_no_calibration.csv"
)

standard_submission.to_csv(
    STANDARD_PATH,
    index=False,
)


# -------------------------------------------------------------------------
# B. Polar calibrated 70/30
# -------------------------------------------------------------------------

polar_submission = (
    sample_submit.copy()
)

polar_submission[
    SUBMISSION_TARGET_COL
] = polar_final_raw


POLAR_PATH = (
    OUT_DIR
    / "submission_Denis70_Own30_POLAR_CALIBRATED.csv"
)

polar_submission.to_csv(
    POLAR_PATH,
    index=False,
)


# =============================================================================
# 17. SAVE COMPONENTS
# =============================================================================

own_submission = (
    sample_submit.copy()
)

own_submission[
    SUBMISSION_TARGET_COL
] = own_raw_blend

OWN_COMPONENT_PATH = (
    OUT_DIR
    / "submission_COMPONENT_own_clean_regressors.csv"
)

own_submission.to_csv(
    OWN_COMPONENT_PATH,
    index=False,
)


denis_submission = (
    sample_submit.copy()
)

denis_submission[
    SUBMISSION_TARGET_COL
] = denis_raw_blend

DENIS_COMPONENT_PATH = (
    OUT_DIR
    / "submission_COMPONENT_denis_seed7_seed42.csv"
)

denis_submission.to_csv(
    DENIS_COMPONENT_PATH,
    index=False,
)


# =============================================================================
# 18. EXACT EFFECTIVE WEIGHTS
# =============================================================================

own_individual_weight = (
    OWN_WEIGHT
    / len(own_regressors)
)

weight_rows = []

for model_name in own_regressors:

    weight_rows.append({
        "prediction":
            model_name,

        "family":
            "own",

        "final_family_weight":
            OWN_WEIGHT,

        "effective_weight":
            own_individual_weight,
    })


weight_rows.extend([
    {
        "prediction":
            "submission_hurdle_seed7.csv",

        "family":
            "denis",

        "final_family_weight":
            DENIS_WEIGHT,

        "effective_weight":
            DENIS_WEIGHT / 2,
    },

    {
        "prediction":
            "submission_hurdle_seed42.csv",

        "family":
            "denis",

        "final_family_weight":
            DENIS_WEIGHT,

        "effective_weight":
            DENIS_WEIGHT / 2,
    },
])


weights_df = pd.DataFrame(
    weight_rows
)

weights_df.to_csv(
    OUT_DIR
    / "effective_weights.csv",
    index=False,
)


# =============================================================================
# 19. DIAGNOSTICS
# =============================================================================

diagnostics = pd.DataFrame([
    {
        "prediction":
            "OWN_CLEAN",

        "mean_log1p":
            float(
                own_log_blend.mean()
            ),

        "std_log1p":
            float(
                own_log_blend.std()
            ),

        "mean_raw":
            float(
                own_raw_blend.mean()
            ),
    },

    {
        "prediction":
            "DENIS_EQUAL",

        "mean_log1p":
            float(
                denis_log_blend.mean()
            ),

        "std_log1p":
            float(
                denis_log_blend.std()
            ),

        "mean_raw":
            float(
                denis_raw_blend.mean()
            ),
    },

    {
        "prediction":
            "70_DENIS_30_OWN_NO_CAL",

        "mean_log1p":
            float(
                standard_final_log.mean()
            ),

        "std_log1p":
            float(
                standard_final_log.std()
            ),

        "mean_raw":
            float(
                standard_final_raw.mean()
            ),
    },

    {
        "prediction":
            "70_DENIS_30_OWN_POLAR_CAL",

        "mean_log1p":
            float(
                polar_final_log.mean()
            ),

        "std_log1p":
            float(
                polar_final_log.std()
            ),

        "mean_raw":
            float(
                polar_final_raw.mean()
            ),
    },
])

diagnostics.to_csv(
    OUT_DIR
    / "diagnostics.csv",
    index=False,
)


# =============================================================================
# 20. CORRELATIONS
# =============================================================================

corr_df = pd.DataFrame({
    "own":
        own_log_blend,

    "denis_seed7":
        np.log1p(
            denis_seed7
        ),

    "denis_seed42":
        np.log1p(
            denis_seed42
        ),

    "denis_equal":
        denis_log_blend,

    "70_30_no_cal":
        standard_final_log,

    "70_30_polar_cal":
        polar_final_log,
}).corr()


corr_df.to_csv(
    OUT_DIR
    / "correlations_logspace.csv",
)


# =============================================================================
# 21. FINAL REPORT
# =============================================================================

print("\n" + "=" * 90)
print("70% DENIS + 30% OWN CLEAN REGRESSORS")
print("=" * 90)

print(
    f"\nOwn clean regressors: "
    f"{len(own_regressors)}"
)

print(
    f"  base: "
    f"{len(base_regressors)}"
)

print(
    f"  column bagging: "
    f"{len(bagging_regressors)}"
)


print("\nEFFECTIVE WEIGHTS:")

print(
    f"Each own clean regressor: "
    f"{own_individual_weight:.6f} "
    f"= {100 * own_individual_weight:.3f}%"
)

print(
    f"Denis seed7: "
    f"{DENIS_WEIGHT / 2:.4f} "
    f"= {100 * DENIS_WEIGHT / 2:.1f}%"
)

print(
    f"Denis seed42: "
    f"{DENIS_WEIGHT / 2:.4f} "
    f"= {100 * DENIS_WEIGHT / 2:.1f}%"
)


print("\nPOLAR CALIBRATION:")

print(
    "spread scale:",
    POLAR_SPREAD_SCALE,
)

print(
    "shift:",
    polar_shift,
)

print(
    "target log mean:",
    POLAR_TARGET_LOG_MEAN,
)

print(
    "actual calibrated log mean:",
    actual_log_mean,
)

print(
    "fraction exactly zero:",
    float(
        np.mean(
            polar_final_log == 0.0
        )
    ),
)


print("\nWEIGHTS:")
display(weights_df)


print("\nDIAGNOSTICS:")
display(diagnostics)


print("\nLOG-SPACE CORRELATIONS:")
display(corr_df)


print("\n" + "=" * 90)
print("FILES")
print("=" * 90)

print(
    "\nCONTROL — 70/30 WITHOUT CALIBRATION:"
)
print(STANDARD_PATH)

print(
    "\nMAIN EXPERIMENT — 70/30 + POLAR CALIBRATION:"
)
print(POLAR_PATH)

print(
    "\nOWN COMPONENT:"
)
print(OWN_COMPONENT_PATH)

print(
    "\nDENIS COMPONENT:"
)
print(DENIS_COMPONENT_PATH)

print("\nDONE")

OWN CLEAN REGRESSORS

BASE:
  xgboost_log_level
  catboost_log_level
  catboost_log_diff
  xgboost_log_diff

COLUMN BAGGING:
  xgboost_log_level__bag_s17
  catboost_log_level__bag_s17
  catboost_log_diff__bag_s17
  xgboost_log_diff__bag_s17

TOTAL OWN CLEAN REGRESSORS: 8

DENIS FILES
/kaggle/input/datasets/vladimirsmirnovve/denis-2-files/submission_hurdle_seed7.csv
/kaggle/input/datasets/vladimirsmirnovve/denis-2-files/submission_hurdle_seed42.csv

Centered blend mean: 1.636544766370207e-14

70% DENIS + 30% OWN CLEAN REGRESSORS

Own clean regressors: 8
  base: 4
  column bagging: 4

EFFECTIVE WEIGHTS:
Each own clean regressor: 0.037500 = 3.750%
Denis seed7: 0.3500 = 35.0%
Denis seed42: 0.3500 = 35.0%

POLAR CALIBRATION:
spread scale: 1.0454
shift: 2.3293498240223354
target log mean: 2.329351
actual calibrated log mean: 2.3293509999999995
fraction exactly zero: 0.00014

WEIGHTS:


,prediction,family,final_family_weight,effective_weight
0,xgboost_log_level,own,0.3,0.0375
1,catboost_log_level,own,0.3,0.0375
2,catboost_log_diff,own,0.3,0.0375
3,xgboost_log_diff,own,0.3,0.0375
4,xgboost_log_level__bag_s17,own,0.3,0.0375
5,catboost_log_level__bag_s17,own,0.3,0.0375
6,catboost_log_diff__bag_s17,own,0.3,0.0375
7,xgboost_log_diff__bag_s17,own,0.3,0.0375
8,submission_hurdle_seed7.csv,denis,0.7,0.3500
9,submission_hurdle_seed42.csv,denis,0.7,0.3500



DIAGNOSTICS:


,prediction,mean_log1p,std_log1p,mean_raw
0,OWN_CLEAN,2.345141,1.569373,38.755391
1,DENIS_EQUAL,2.329351,1.579554,38.770151
2,70_DENIS_30_OWN_NO_CAL,2.334088,1.575493,38.695271
3,70_DENIS_30_OWN_POLAR_CAL,2.329351,1.647019,43.506654



LOG-SPACE CORRELATIONS:


,own,denis_seed7,denis_seed42,denis_equal,70_30_no_cal,70_30_polar_cal
own,1.000000,0.996706,0.996892,0.996953,0.998500,0.998500
denis_seed7,0.996706,1.000000,0.999382,0.999846,0.999546,0.999546
denis_seed42,0.996892,0.999382,1.000000,0.999845,0.999601,0.999601
denis_equal,0.996953,0.999846,0.999845,1.000000,0.999728,0.999728
70_30_no_cal,0.998500,0.999546,0.999601,0.999728,1.000000,1.000000
70_30_polar_cal,0.998500,0.999546,0.999601,0.999728,1.000000,1.000000



FILES

CONTROL — 70/30 WITHOUT CALIBRATION:
/kaggle/working/ozon_final/denis70_own30_polarcal/submission_Denis70_Own30_LOG_no_calibration.csv

MAIN EXPERIMENT — 70/30 + POLAR CALIBRATION:
/kaggle/working/ozon_final/denis70_own30_polarcal/submission_Denis70_Own30_POLAR_CALIBRATED.csv

OWN COMPONENT:
/kaggle/working/ozon_final/denis70_own30_polarcal/submission_COMPONENT_own_clean_regressors.csv

DENIS COMPONENT:
/kaggle/working/ozon_final/denis70_own30_polarcal/submission_COMPONENT_denis_seed7_seed42.csv

DONE


In [139]:
# =============================================================================
# FINAL EXPERIMENT
#
# GROUP A = 70%
#   equal blend of:
#       - ALL OWN DOUBLE MODELS = HURDLE_CANDIDATE_COLUMNS
#       - Denis seed7
#       - Denis seed42
#
# GROUP B = 30%
#   equal blend of:
#       - ALL ORIGINAL REGRESSION BLENDS from BLEND_LEADERBOARD
#
# THEN:
#   Polar-style calibration:
#       1) center individual predictions in log1p-space
#       2) hierarchical 70/30 blend
#       3) spread *= 1.0454
#       4) re-center
#       5) shift c so mean(max(log_pred + c, 0)) = 2.329351
#       6) expm1
#
# IMPORTANT:
#   - no base models directly in GROUP B
#   - no stochastic/hurdle blends in GROUP B
#   - GROUP B is exactly the original regression BLEND_LEADERBOARD snapshot
# =============================================================================

from pathlib import Path
import numpy as np
import pandas as pd


# =============================================================================
# 0. CONFIG
# =============================================================================

GROUP_A_WEIGHT = 0.70
GROUP_B_WEIGHT = 0.30

POLAR_SPREAD_SCALE = 1.0454
POLAR_TARGET_LOG_MEAN = 2.329351

DENIS_SEED7_PATH = Path(
    "/kaggle/input/datasets/vladimirsmirnovve/"
    "denis-2-files/submission_hurdle_seed7.csv"
)

DENIS_SEED42_PATH = Path(
    "/kaggle/input/datasets/vladimirsmirnovve/"
    "denis-2-files/submission_hurdle_seed42.csv"
)

OUT_DIR = (
    Path("/kaggle/working/ozon_final")
    / "doublemodels_denis70_regressionblends30"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# =============================================================================
# 1. BASIC CHECKS
# =============================================================================

if abs(
    GROUP_A_WEIGHT
    + GROUP_B_WEIGHT
    - 1.0
) > 1e-12:
    raise ValueError(
        "GROUP_A_WEIGHT + GROUP_B_WEIGHT must equal 1."
    )

if FINAL_PREDICTIONS is None:
    raise RuntimeError(
        "FINAL_PREDICTIONS is None."
    )

if "HURDLE_CANDIDATE_COLUMNS" not in globals():
    raise RuntimeError(
        "HURDLE_CANDIDATE_COLUMNS not found. "
        "Сначала выполни Hurdle / Double Models block."
    )

if "BLEND_LEADERBOARD" not in globals():
    raise RuntimeError(
        "BLEND_LEADERBOARD not found. "
        "Сначала выполни original regression Blend Lab."
    )


# =============================================================================
# 2. REFERENCE USER ORDER
# =============================================================================

reference = (
    sample_submit[[USER_COL]]
    .copy()
)

reference[USER_COL] = (
    reference[USER_COL]
    .astype(str)
)

N_USERS = len(reference)


# =============================================================================
# 3. HELPER — ALIGN EXTERNAL SUBMISSION
# =============================================================================

def load_submission(path):

    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"File not found:\n{path}"
        )

    df = pd.read_csv(path)

    if USER_COL not in df.columns:
        raise ValueError(
            f"{path}: missing {USER_COL}"
        )

    if SUBMISSION_TARGET_COL in df.columns:

        pred_col = SUBMISSION_TARGET_COL

    else:

        candidate_cols = [
            c
            for c in df.columns
            if c != USER_COL
        ]

        if len(candidate_cols) != 1:
            raise ValueError(
                f"{path}: cannot determine prediction column.\n"
                f"Columns: {list(df.columns)}"
            )

        pred_col = candidate_cols[0]

    tmp = df[
        [USER_COL, pred_col]
    ].copy()

    tmp[USER_COL] = (
        tmp[USER_COL]
        .astype(str)
    )

    if tmp[USER_COL].duplicated().any():
        raise ValueError(
            f"{path}: duplicated {USER_COL}"
        )

    tmp = reference.merge(
        tmp,
        on=USER_COL,
        how="left",
        validate="one_to_one",
    )

    pred = pd.to_numeric(
        tmp[pred_col],
        errors="coerce",
    ).to_numpy(
        dtype=np.float64
    )

    if len(pred) != N_USERS:
        raise ValueError(
            f"{path}: wrong number of rows"
        )

    if not np.isfinite(pred).all():
        raise ValueError(
            f"{path}: NaN/inf predictions"
        )

    if np.any(pred < 0):
        raise ValueError(
            f"{path}: negative predictions"
        )

    return pred


# =============================================================================
# 4. ALIGN FINAL_PREDICTIONS TO SAMPLE SUBMISSION
# =============================================================================

final_frame = FINAL_PREDICTIONS.copy()

final_frame[USER_COL] = (
    final_frame[USER_COL]
    .astype(str)
)

if final_frame[USER_COL].duplicated().any():
    raise ValueError(
        "FINAL_PREDICTIONS contains duplicated user_id."
    )

final_frame = reference.merge(
    final_frame,
    on=USER_COL,
    how="left",
    validate="one_to_one",
)


# =============================================================================
# 5. GROUP A
#
# ALL DOUBLE MODELS + DENIS seed7 + DENIS seed42
#
# Double Models are exactly individual:
#   probability direction × magnitude direction
#
# We intentionally DO NOT include:
#   HURDLE_ALL_EQUAL
# because it is already an aggregate of these models.
# =============================================================================

double_model_names = list(
    HURDLE_CANDIDATE_COLUMNS
)

if not double_model_names:
    raise RuntimeError(
        "HURDLE_CANDIDATE_COLUMNS is empty."
    )


missing_double = [
    name
    for name in double_model_names
    if name not in final_frame.columns
]

if missing_double:
    raise RuntimeError(
        "Missing Double Models:\n"
        + "\n".join(missing_double)
    )


print("=" * 100)
print("GROUP A — DOUBLE MODELS + DENIS")
print("=" * 100)

print("\nDOUBLE MODELS:")

for name in double_model_names:
    print(" ", name)


# -----------------------------------------------------------------------------
# Own Double Models
# -----------------------------------------------------------------------------

double_matrix = final_frame[
    double_model_names
].to_numpy(
    dtype=np.float64
)

if not np.isfinite(
    double_matrix
).all():
    raise ValueError(
        "Double Models contain NaN/inf."
    )

if np.any(
    double_matrix < 0
):
    raise ValueError(
        "Double Models contain negative predictions."
    )


# -----------------------------------------------------------------------------
# Denis external models
# -----------------------------------------------------------------------------

denis_seed7 = load_submission(
    DENIS_SEED7_PATH
)

denis_seed42 = load_submission(
    DENIS_SEED42_PATH
)


print("\nDENIS:")

print(
    " ",
    DENIS_SEED7_PATH,
)

print(
    " ",
    DENIS_SEED42_PATH,
)


# =============================================================================
# 6. BUILD GROUP A MATRIX
#
# Every individual output receives equal weight INSIDE GROUP A.
# =============================================================================

group_a_names = (
    double_model_names
    + [
        "DENIS_HURDLE_SEED7",
        "DENIS_HURDLE_SEED42",
    ]
)


group_a_raw_matrix = np.column_stack([
    double_matrix,
    denis_seed7,
    denis_seed42,
])


if group_a_raw_matrix.shape[1] != len(
    group_a_names
):
    raise RuntimeError(
        "GROUP A matrix/name mismatch."
    )


group_a_log_matrix = np.log1p(
    np.clip(
        group_a_raw_matrix,
        0.0,
        None,
    )
)


# =============================================================================
# 7. GROUP A — ORDINARY EQUAL LOG BLEND
# =============================================================================

group_a_log_blend = (
    group_a_log_matrix.mean(
        axis=1
    )
)

group_a_raw_blend = np.expm1(
    group_a_log_blend
)

group_a_raw_blend = np.clip(
    group_a_raw_blend,
    0.0,
    None,
)


# =============================================================================
# 8. GROUP B
#
# ALL ORIGINAL REGRESSION BLENDS
#
# BLEND_LEADERBOARD was created BEFORE stochastic/hurdle blocks,
# therefore it is the clean snapshot of original regression blends.
# =============================================================================

regression_blend_names = (
    BLEND_LEADERBOARD[
        "blend"
    ]
    .astype(str)
    .tolist()
)


# Only candidates with final/test prediction available.
regression_blend_names = [
    name
    for name in regression_blend_names
    if (
        name in BLENDS
        and
        BLENDS[name].test_pred is not None
    )
]


if not regression_blend_names:
    raise RuntimeError(
        "No regression blends found."
    )


print("\n" + "=" * 100)
print("GROUP B — ALL REGRESSION BLENDS")
print("=" * 100)

for name in regression_blend_names:
    print(" ", name)

print(
    "\nTOTAL REGRESSION BLENDS:",
    len(regression_blend_names),
)


# =============================================================================
# 9. BUILD GROUP B MATRIX
# =============================================================================

group_b_raw_matrix = np.column_stack([
    np.asarray(
        BLENDS[name].test_pred,
        dtype=np.float64,
    )
    for name
    in regression_blend_names
])


if group_b_raw_matrix.shape[0] != N_USERS:

    # BLENDS.test_pred follows FINAL_PREDICTIONS order,
    # while sample submission alignment may differ.
    #
    # Normally these are identical in this notebook.
    raise RuntimeError(
        "Regression blend row count does not match sample submission."
    )


if not np.isfinite(
    group_b_raw_matrix
).all():
    raise ValueError(
        "Regression blends contain NaN/inf."
    )

if np.any(
    group_b_raw_matrix < 0
):
    raise ValueError(
        "Regression blends contain negative predictions."
    )


group_b_log_matrix = np.log1p(
    np.clip(
        group_b_raw_matrix,
        0.0,
        None,
    )
)


# =============================================================================
# 10. GROUP B — ORDINARY EQUAL LOG BLEND
# =============================================================================

group_b_log_blend = (
    group_b_log_matrix.mean(
        axis=1
    )
)

group_b_raw_blend = np.expm1(
    group_b_log_blend
)

group_b_raw_blend = np.clip(
    group_b_raw_blend,
    0.0,
    None,
)


# =============================================================================
# 11. STANDARD 70 / 30 BLEND — NO CALIBRATION
#
# 70% Group A
# 30% Group B
# =============================================================================

standard_final_log = (
    GROUP_A_WEIGHT
    * group_a_log_blend
    +
    GROUP_B_WEIGHT
    * group_b_log_blend
)


standard_final_raw = np.expm1(
    standard_final_log
)

standard_final_raw = np.clip(
    standard_final_raw,
    0.0,
    None,
)


# =============================================================================
# 12. POLAR-STYLE CENTERING
#
# IMPORTANT:
# Every INDIVIDUAL prediction is centered BEFORE averaging.
#
# p_i_centered = p_i - mean(p_i)
# =============================================================================

group_a_centered_matrix = (
    group_a_log_matrix
    -
    group_a_log_matrix.mean(
        axis=0,
        keepdims=True,
    )
)


group_b_centered_matrix = (
    group_b_log_matrix
    -
    group_b_log_matrix.mean(
        axis=0,
        keepdims=True,
    )
)


# Equal weights inside each family
group_a_centered_direction = (
    group_a_centered_matrix.mean(
        axis=1
    )
)

group_b_centered_direction = (
    group_b_centered_matrix.mean(
        axis=1
    )
)


# =============================================================================
# 13. HIERARCHICAL 70 / 30 BLEND
# =============================================================================

centered_70_30 = (
    GROUP_A_WEIGHT
    * group_a_centered_direction
    +
    GROUP_B_WEIGHT
    * group_b_centered_direction
)


print(
    "\nCentered 70/30 mean:",
    float(
        centered_70_30.mean()
    ),
)


# =============================================================================
# 14. SPREAD CORRECTION
#
# p_scaled =
#     mean(p)
#     + 1.0454 * (p - mean(p))
# =============================================================================

blend_mean = float(
    centered_70_30.mean()
)


scaled = (
    blend_mean
    +
    POLAR_SPREAD_SCALE
    * (
        centered_70_30
        - blend_mean
    )
)


# =============================================================================
# 15. RE-CENTER AFTER SPREAD
# =============================================================================

scaled_centered = (
    scaled
    -
    scaled.mean()
)


# =============================================================================
# 16. FIND SHIFT c
#
# mean(max(p_centered + c, 0))
#     = 2.329351
# =============================================================================

def find_shift_for_target_clipped_log_mean(
    centered_log_prediction,
    target_log_mean,
    iterations=120,
):

    x = np.asarray(
        centered_log_prediction,
        dtype=np.float64,
    )

    target = float(
        target_log_mean
    )

    def clipped_mean(
        shift
    ):

        return float(
            np.maximum(
                x + shift,
                0.0,
            ).mean()
        )


    low = (
        -float(np.max(x))
        - target
        - 10.0
    )

    high = (
        float(
            np.max(
                np.abs(x)
            )
        )
        + target
        + 10.0
    )


    for _ in range(30):

        if clipped_mean(
            high
        ) >= target:
            break

        high *= 2.0

    else:
        raise RuntimeError(
            "Could not bracket calibration shift."
        )


    for _ in range(
        int(iterations)
    ):

        mid = (
            low
            + high
        ) / 2.0


        if clipped_mean(
            mid
        ) < target:

            low = mid

        else:

            high = mid


    return (
        low
        + high
    ) / 2.0


polar_shift = (
    find_shift_for_target_clipped_log_mean(
        scaled_centered,
        POLAR_TARGET_LOG_MEAN,
    )
)


# =============================================================================
# 17. FINAL CALIBRATED LOG PREDICTION
# =============================================================================

polar_final_log = np.maximum(
    scaled_centered
    + polar_shift,
    0.0,
)


actual_log_mean = float(
    polar_final_log.mean()
)


if not np.isclose(
    actual_log_mean,
    POLAR_TARGET_LOG_MEAN,
    atol=1e-10,
):

    raise RuntimeError(
        "Target log mean calibration failed."
    )


# =============================================================================
# 18. BACK TO RAW GMV
# =============================================================================

polar_final_raw = np.expm1(
    polar_final_log
)

polar_final_raw = np.clip(
    polar_final_raw,
    0.0,
    None,
)


# =============================================================================
# 19. SAVE — CONTROL WITHOUT CALIBRATION
# =============================================================================

control_submission = (
    sample_submit.copy()
)

control_submission[
    SUBMISSION_TARGET_COL
] = standard_final_raw


CONTROL_PATH = (
    OUT_DIR
    / "submission_DOUBLE_DENIS70_REGBLENDS30_NO_CAL.csv"
)


control_submission.to_csv(
    CONTROL_PATH,
    index=False,
)


# =============================================================================
# 20. SAVE — MAIN POLAR-CALIBRATED SUBMISSION
# =============================================================================

polar_submission = (
    sample_submit.copy()
)

polar_submission[
    SUBMISSION_TARGET_COL
] = polar_final_raw


POLAR_PATH = (
    OUT_DIR
    / "submission_DOUBLE_DENIS70_REGBLENDS30_POLAR_CAL.csv"
)


polar_submission.to_csv(
    POLAR_PATH,
    index=False,
)


# =============================================================================
# 21. SAVE COMPONENT A
# =============================================================================

group_a_submission = (
    sample_submit.copy()
)

group_a_submission[
    SUBMISSION_TARGET_COL
] = group_a_raw_blend


GROUP_A_PATH = (
    OUT_DIR
    / "submission_COMPONENT_DOUBLE_MODELS_PLUS_DENIS.csv"
)


group_a_submission.to_csv(
    GROUP_A_PATH,
    index=False,
)


# =============================================================================
# 22. SAVE COMPONENT B
# =============================================================================

group_b_submission = (
    sample_submit.copy()
)

group_b_submission[
    SUBMISSION_TARGET_COL
] = group_b_raw_blend


GROUP_B_PATH = (
    OUT_DIR
    / "submission_COMPONENT_ALL_REGRESSION_BLENDS.csv"
)


group_b_submission.to_csv(
    GROUP_B_PATH,
    index=False,
)


# =============================================================================
# 23. EFFECTIVE WEIGHTS
# =============================================================================

group_a_individual_weight = (
    GROUP_A_WEIGHT
    /
    len(group_a_names)
)


group_b_individual_weight = (
    GROUP_B_WEIGHT
    /
    len(regression_blend_names)
)


weight_rows = []


for name in group_a_names:

    weight_rows.append({
        "group":
            "A_double_plus_denis",

        "prediction":
            name,

        "group_weight":
            GROUP_A_WEIGHT,

        "effective_weight":
            group_a_individual_weight,
    })


for name in regression_blend_names:

    weight_rows.append({
        "group":
            "B_regression_blends",

        "prediction":
            name,

        "group_weight":
            GROUP_B_WEIGHT,

        "effective_weight":
            group_b_individual_weight,
    })


weights_df = pd.DataFrame(
    weight_rows
)


weights_df.to_csv(
    OUT_DIR
    / "effective_weights.csv",
    index=False,
)


# =============================================================================
# 24. DIAGNOSTICS
# =============================================================================

diagnostics = pd.DataFrame([
    {
        "prediction":
            "GROUP_A_DOUBLE_PLUS_DENIS",

        "n_members":
            len(group_a_names),

        "mean_log1p":
            float(
                group_a_log_blend.mean()
            ),

        "std_log1p":
            float(
                group_a_log_blend.std()
            ),

        "mean_raw":
            float(
                group_a_raw_blend.mean()
            ),
    },

    {
        "prediction":
            "GROUP_B_REGRESSION_BLENDS",

        "n_members":
            len(regression_blend_names),

        "mean_log1p":
            float(
                group_b_log_blend.mean()
            ),

        "std_log1p":
            float(
                group_b_log_blend.std()
            ),

        "mean_raw":
            float(
                group_b_raw_blend.mean()
            ),
    },

    {
        "prediction":
            "70_30_NO_CAL",

        "n_members":
            (
                len(group_a_names)
                +
                len(regression_blend_names)
            ),

        "mean_log1p":
            float(
                standard_final_log.mean()
            ),

        "std_log1p":
            float(
                standard_final_log.std()
            ),

        "mean_raw":
            float(
                standard_final_raw.mean()
            ),
    },

    {
        "prediction":
            "70_30_POLAR_CAL",

        "n_members":
            (
                len(group_a_names)
                +
                len(regression_blend_names)
            ),

        "mean_log1p":
            float(
                polar_final_log.mean()
            ),

        "std_log1p":
            float(
                polar_final_log.std()
            ),

        "mean_raw":
            float(
                polar_final_raw.mean()
            ),
    },
])


diagnostics.to_csv(
    OUT_DIR
    / "diagnostics.csv",
    index=False,
)


# =============================================================================
# 25. CORRELATIONS
# =============================================================================

corr_df = pd.DataFrame({
    "group_A_double_plus_denis":
        group_a_log_blend,

    "group_B_regression_blends":
        group_b_log_blend,

    "70_30_no_cal":
        standard_final_log,

    "70_30_polar_cal":
        polar_final_log,

    "denis_seed7":
        np.log1p(
            denis_seed7
        ),

    "denis_seed42":
        np.log1p(
            denis_seed42
        ),
}).corr()


corr_df.to_csv(
    OUT_DIR
    / "correlations_logspace.csv",
)


# =============================================================================
# 26. FINAL REPORT
# =============================================================================

print("\n" + "=" * 100)
print("FINAL DOUBLE MODELS + DENIS / REGRESSION BLENDS")
print("=" * 100)


print(
    f"\nGROUP A weight: "
    f"{GROUP_A_WEIGHT:.2f}"
)

print(
    f"GROUP A members: "
    f"{len(group_a_names)}"
)

print(
    f"  Double Models: "
    f"{len(double_model_names)}"
)

print(
    "  Denis models: 2"
)

print(
    f"Weight of EACH Group A member: "
    f"{group_a_individual_weight:.6f} "
    f"= {100 * group_a_individual_weight:.3f}%"
)


print(
    f"\nGROUP B weight: "
    f"{GROUP_B_WEIGHT:.2f}"
)

print(
    f"GROUP B regression blends: "
    f"{len(regression_blend_names)}"
)

print(
    f"Weight of EACH regression blend: "
    f"{group_b_individual_weight:.6f} "
    f"= {100 * group_b_individual_weight:.3f}%"
)


print("\nPOLAR CALIBRATION:")

print(
    "  spread scale:",
    POLAR_SPREAD_SCALE,
)

print(
    "  shift:",
    polar_shift,
)

print(
    "  target clipped log mean:",
    POLAR_TARGET_LOG_MEAN,
)

print(
    "  actual clipped log mean:",
    actual_log_mean,
)

print(
    "  fraction exactly zero:",
    float(
        np.mean(
            polar_final_log == 0.0
        )
    ),
)


print("\nWEIGHTS:")
display(weights_df)


print("\nDIAGNOSTICS:")
display(diagnostics)


print("\nCORRELATIONS:")
display(corr_df)


print("\n" + "=" * 100)
print("FILES")
print("=" * 100)


print(
    "\nCONTROL — 70/30 WITHOUT CALIBRATION:"
)
print(CONTROL_PATH)


print(
    "\nMAIN — 70/30 WITH POLAR CALIBRATION:"
)
print(POLAR_PATH)


print(
    "\nGROUP A COMPONENT:"
)
print(GROUP_A_PATH)


print(
    "\nGROUP B COMPONENT:"
)
print(GROUP_B_PATH)


print("\nDONE")

GROUP A — DOUBLE MODELS + DENIS

DOUBLE MODELS:
  HURDLE__P_DENIS_FEATURE_VIEWS__M_OWN_POSITIVE_LEVEL
  HURDLE__P_DENIS_FEATURE_VIEWS__M_OWN_POSITIVE_DIFF
  HURDLE__P_DENIS_FEATURE_VIEWS__M_OWN_POSITIVE_ALL_ACTIVE
  HURDLE__P_DENIS_ALL__M_OWN_POSITIVE_LEVEL
  HURDLE__P_DENIS_ALL__M_OWN_POSITIVE_DIFF
  HURDLE__P_DENIS_ALL__M_OWN_POSITIVE_ALL_ACTIVE

DENIS:
  /kaggle/input/datasets/vladimirsmirnovve/denis-2-files/submission_hurdle_seed7.csv
  /kaggle/input/datasets/vladimirsmirnovve/denis-2-files/submission_hurdle_seed42.csv

GROUP B — ALL REGRESSION BLENDS
  optuna_softmax_logmean
  cf_disagreement_gate
  cf_opt_log_all_active
  cf_opt_log_trusted
  validation_inverse_score_logmean
  all_active_logmean
  trusted_logmean
  top4_logmean
  top3_logmean
  top2_logmean
  cf_opt_raw_all_active
  cf_opt_raw_trusted
  all_active_arithmetic
  log_level_logmean
  best_per_family_logmean
  log_diff_logmean
  meta_logmean_cf_ensembles
  cf_alpha_on_optlog_trusted
  cf_segment_calibrated

TOTAL REGR

,group,prediction,group_weight,effective_weight
0,A_double_plus_denis,HURDLE__P_DENIS_FEATURE_VIEWS__M_OWN_POSITIVE_...,0.7,0.087500
1,A_double_plus_denis,HURDLE__P_DENIS_FEATURE_VIEWS__M_OWN_POSITIVE_...,0.7,0.087500
2,A_double_plus_denis,HURDLE__P_DENIS_FEATURE_VIEWS__M_OWN_POSITIVE_...,0.7,0.087500
3,A_double_plus_denis,HURDLE__P_DENIS_ALL__M_OWN_POSITIVE_LEVEL,0.7,0.087500
4,A_double_plus_denis,HURDLE__P_DENIS_ALL__M_OWN_POSITIVE_DIFF,0.7,0.087500
5,A_double_plus_denis,HURDLE__P_DENIS_ALL__M_OWN_POSITIVE_ALL_ACTIVE,0.7,0.087500
6,A_double_plus_denis,DENIS_HURDLE_SEED7,0.7,0.087500
7,A_double_plus_denis,DENIS_HURDLE_SEED42,0.7,0.087500
8,B_regression_blends,optuna_softmax_logmean,0.3,0.015789
9,B_regression_blends,cf_disagreement_gate,0.3,0.015789



DIAGNOSTICS:


,prediction,n_members,mean_log1p,std_log1p,mean_raw
0,GROUP_A_DOUBLE_PLUS_DENIS,8,2.278751,1.590607,37.799146
1,GROUP_B_REGRESSION_BLENDS,19,2.330822,1.567832,38.240604
2,70_30_NO_CAL,27,2.294372,1.583524,37.915918
3,70_30_POLAR_CAL,27,2.329351,1.655416,44.478764



CORRELATIONS:


,group_A_double_plus_denis,group_B_regression_blends,70_30_no_cal,70_30_polar_cal,denis_seed7,denis_seed42
group_A_double_plus_denis,1.000000,0.999243,0.999933,0.999933,0.998221,0.998387
group_B_regression_blends,0.999243,1.000000,0.999626,0.999626,0.996807,0.997001
70_30_no_cal,0.999933,0.999626,1.000000,1.000000,0.997959,0.998133
70_30_polar_cal,0.999933,0.999626,1.000000,1.000000,0.997959,0.998133
denis_seed7,0.998221,0.996807,0.997959,0.997959,1.000000,0.999382
denis_seed42,0.998387,0.997001,0.998133,0.998133,0.999382,1.000000



FILES

CONTROL — 70/30 WITHOUT CALIBRATION:
/kaggle/working/ozon_final/doublemodels_denis70_regressionblends30/submission_DOUBLE_DENIS70_REGBLENDS30_NO_CAL.csv

MAIN — 70/30 WITH POLAR CALIBRATION:
/kaggle/working/ozon_final/doublemodels_denis70_regressionblends30/submission_DOUBLE_DENIS70_REGBLENDS30_POLAR_CAL.csv

GROUP A COMPONENT:
/kaggle/working/ozon_final/doublemodels_denis70_regressionblends30/submission_COMPONENT_DOUBLE_MODELS_PLUS_DENIS.csv

GROUP B COMPONENT:
/kaggle/working/ozon_final/doublemodels_denis70_regressionblends30/submission_COMPONENT_ALL_REGRESSION_BLENDS.csv

DONE


In [140]:
# =============================================================================
# OZONE — DENIS 80% / OWN CLEAN REGRESSORS 20%
# + GLOBAL CALIBRATION LAB
#
# Denis:
#   seed7  = 40% total
#   seed42 = 40% total
#
# Own clean regressors:
#   20% total, equal log-weights across:
#       - MODEL_NAMES
#       - column-bagging boosting models from STOCHASTIC_REGISTRY
#
# NO:
#   - local weights
#   - segment weights
#   - Ridge
#   - hurdle models
#   - target specialists
#   - ready-made blends
#
# Outputs:
#   1. SAFE 80/20 no calibration
#   2. SAFE-ish 75/25 no calibration
#   3. Polar full
#   4. shift-only
#   5. spread-only, raw mean preserved
#   6. 50% shrink raw -> Polar
#   7. 75% shrink raw -> Polar
#   8. joint 50% shrink of spread + level
# =============================================================================

from pathlib import Path
import hashlib

import numpy as np
import pandas as pd


# =============================================================================
# 0. CONFIG
# =============================================================================

ROOT = Path("/kaggle/working/ozon_final")

OUT_DIR = (
    ROOT
    / "denis80_own20_global_calibration_lab"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


DENIS_SEED7_PATH = Path(
    "/kaggle/input/datasets/vladimirsmirnovve/"
    "denis-2-files/submission_hurdle_seed7.csv"
)

DENIS_SEED42_PATH = Path(
    "/kaggle/input/datasets/vladimirsmirnovve/"
    "denis-2-files/submission_hurdle_seed42.csv"
)


# Main family weights
DENIS_WEIGHT = 0.80
OWN_WEIGHT = 0.20


# Conservative neighboring blend
DENIS_WEIGHT_75 = 0.75
OWN_WEIGHT_25 = 0.25


# Polar calibration parameters
POLAR_SPREAD = 1.0454
POLAR_TARGET_LOG_MEAN = 2.329351


# 50% shrinkage toward Polar
SHRINK_50 = 0.50

# 75% shrinkage toward Polar
SHRINK_75 = 0.75


# Halfway spread:
# 1.0 -> 1.0454
JOINT_SHRINK_SPREAD = (
    1.0
    + 0.50 * (POLAR_SPREAD - 1.0)
)


# =============================================================================
# 1. CHECKS
# =============================================================================

if FINAL_PREDICTIONS is None:
    raise RuntimeError(
        "FINAL_PREDICTIONS is None. "
        "Сначала выполни final fit."
    )

if abs(
    DENIS_WEIGHT
    + OWN_WEIGHT
    - 1.0
) > 1e-12:
    raise ValueError(
        "80/20 weights do not sum to 1."
    )


# =============================================================================
# 2. REFERENCE USER ORDER
# =============================================================================

reference = (
    sample_submit[
        [USER_COL]
    ]
    .copy()
)

reference[USER_COL] = (
    reference[USER_COL]
    .astype(str)
)

N_USERS = len(reference)


# =============================================================================
# 3. FIND ALL OWN CLEAN REGRESSORS
# =============================================================================
#
# Base:
#   MODEL_NAMES
#
# Column bagging:
#   STOCHASTIC_REGISTRY entries that are genuine random-subspace
#   boosting models.
#
# Fallback:
#   if "kind" is absent, __bag_ in run_name is accepted.
# =============================================================================

base_regressors = list(
    MODEL_NAMES
)


bagging_regressors = []

if "STOCHASTIC_REGISTRY" in globals():

    for run in STOCHASTIC_REGISTRY:

        name = run.get(
            "run_name"
        )

        kind = run.get(
            "kind",
            "",
        )

        if not name:
            continue

        if (
            kind == "column_bagging"
            or "__bag_" in name
        ):

            bagging_regressors.append(
                name
            )


# Fallback: look directly in FINAL_PREDICTIONS
if not bagging_regressors:

    bagging_regressors = [
        c
        for c in FINAL_PREDICTIONS.columns
        if "__bag_" in str(c)
    ]


own_regressors = list(
    dict.fromkeys(
        base_regressors
        + bagging_regressors
    )
)


print("=" * 100)
print("OWN CLEAN REGRESSORS")
print("=" * 100)

print("\nBASE:")
for name in base_regressors:
    print(" ", name)

print("\nCOLUMN BAGGING:")
for name in bagging_regressors:
    print(" ", name)

print(
    "\nTOTAL:",
    len(own_regressors),
)


# =============================================================================
# 4. ALIGN OWN PREDICTIONS
# =============================================================================

missing = [
    name
    for name in own_regressors
    if name not in FINAL_PREDICTIONS.columns
]

if missing:

    raise RuntimeError(
        "Missing own predictions:\n"
        + "\n".join(missing)
    )


fp = FINAL_PREDICTIONS[
    [USER_COL]
    + own_regressors
].copy()

fp[USER_COL] = (
    fp[USER_COL]
    .astype(str)
)


if fp[USER_COL].duplicated().any():

    raise ValueError(
        "FINAL_PREDICTIONS contains duplicated user_id."
    )


fp = reference.merge(
    fp,
    on=USER_COL,
    how="left",
    validate="one_to_one",
)


own_matrix = fp[
    own_regressors
].to_numpy(
    dtype=np.float64
)


if not np.isfinite(
    own_matrix
).all():

    raise ValueError(
        "Own predictions contain NaN/inf."
    )


if np.any(
    own_matrix < 0
):

    raise ValueError(
        "Own predictions contain negative values."
    )


# =============================================================================
# 5. REMOVE EXACT DUPLICATE OWN PREDICTIONS
# =============================================================================
#
# Не хотим, чтобы один и тот же prediction,
# случайно сохранённый под двумя именами,
# получил двойной вес.
# =============================================================================

unique_indices = []
unique_names = []

seen = {}


for j, name in enumerate(
    own_regressors
):

    pred = own_matrix[:, j]

    key = hashlib.sha256(
        np.round(
            pred,
            10,
        ).tobytes()
    ).hexdigest()


    if key in seen:

        print(
            "DUPLICATE SKIPPED:",
            name,
            "==",
            seen[key],
        )

        continue


    seen[key] = name

    unique_indices.append(
        j
    )

    unique_names.append(
        name
    )


own_matrix = own_matrix[
    :,
    unique_indices,
]

own_regressors = (
    unique_names
)


print(
    "\nUnique own regressors:",
    len(own_regressors),
)


# =============================================================================
# 6. EXTERNAL SUBMISSION LOADER
# =============================================================================

def load_submission(
    path,
):

    path = Path(path)


    if not path.exists():

        raise FileNotFoundError(
            f"File not found:\n{path}"
        )


    df = pd.read_csv(
        path
    )


    if USER_COL not in df.columns:

        raise ValueError(
            f"{path}: missing {USER_COL}"
        )


    if SUBMISSION_TARGET_COL in df.columns:

        pred_col = (
            SUBMISSION_TARGET_COL
        )

    else:

        candidates = [
            c
            for c in df.columns
            if c != USER_COL
        ]


        if len(candidates) != 1:

            raise ValueError(
                f"{path}: cannot determine "
                f"prediction column.\n"
                f"Columns: {list(df.columns)}"
            )


        pred_col = candidates[0]


    tmp = df[
        [USER_COL, pred_col]
    ].copy()


    tmp[USER_COL] = (
        tmp[USER_COL]
        .astype(str)
    )


    if tmp[USER_COL].duplicated().any():

        raise ValueError(
            f"{path}: duplicated user_id"
        )


    tmp = reference.merge(
        tmp,
        on=USER_COL,
        how="left",
        validate="one_to_one",
    )


    pred = pd.to_numeric(
        tmp[pred_col],
        errors="coerce",
    ).to_numpy(
        dtype=np.float64
    )


    if len(pred) != N_USERS:

        raise ValueError(
            f"{path}: row count mismatch"
        )


    if not np.isfinite(
        pred
    ).all():

        raise ValueError(
            f"{path}: NaN/inf"
        )


    if np.any(
        pred < 0
    ):

        raise ValueError(
            f"{path}: negative predictions"
        )


    return pred


# =============================================================================
# 7. LOAD DENIS
# =============================================================================

denis_seed7 = load_submission(
    DENIS_SEED7_PATH
)

denis_seed42 = load_submission(
    DENIS_SEED42_PATH
)


print("\n" + "=" * 100)
print("DENIS")
print("=" * 100)

print(DENIS_SEED7_PATH)
print(DENIS_SEED42_PATH)


# =============================================================================
# 8. BUILD FAMILY LOG PREDICTIONS
# =============================================================================
#
# RMSLE == RMSE in log1p space,
# поэтому основное смешивание делаем именно там.
# =============================================================================

own_log_matrix = np.log1p(
    own_matrix
)


own_log = (
    own_log_matrix.mean(
        axis=1
    )
)


denis_log_matrix = np.column_stack([
    np.log1p(
        denis_seed7
    ),

    np.log1p(
        denis_seed42
    ),
])


denis_log = (
    denis_log_matrix.mean(
        axis=1
    )
)


# =============================================================================
# 9. SAFE GLOBAL BLENDS — NO CALIBRATION
# =============================================================================

# Requested 80 / 20
safe_80_20_log = (
    DENIS_WEIGHT
    * denis_log
    +
    OWN_WEIGHT
    * own_log
)


# Conservative neighbor: 75 / 25
safe_75_25_log = (
    DENIS_WEIGHT_75
    * denis_log
    +
    OWN_WEIGHT_25
    * own_log
)


# =============================================================================
# 10. CALIBRATION HELPERS
# =============================================================================

def find_shift_for_clipped_log_mean(
    centered,
    target_mean,
    iterations=120,
):

    x = np.asarray(
        centered,
        dtype=np.float64,
    )

    target_mean = float(
        target_mean
    )


    def current_mean(
        shift,
    ):

        return float(
            np.maximum(
                x + shift,
                0.0,
            ).mean()
        )


    low = (
        -float(
            np.max(x)
        )
        - target_mean
        - 10.0
    )


    high = (
        float(
            np.max(
                np.abs(x)
            )
        )
        + target_mean
        + 10.0
    )


    # Guarantee upper bracket
    for _ in range(30):

        if (
            current_mean(high)
            >= target_mean
        ):
            break

        high *= 2.0

    else:

        raise RuntimeError(
            "Cannot bracket calibration shift."
        )


    for _ in range(
        int(iterations)
    ):

        mid = (
            low
            + high
        ) / 2.0


        if (
            current_mean(mid)
            < target_mean
        ):

            low = mid

        else:

            high = mid


    return (
        low
        + high
    ) / 2.0


def global_spread_level_calibration(
    log_prediction,
    *,
    spread,
    target_log_mean,
):

    z = np.asarray(
        log_prediction,
        dtype=np.float64,
    )


    centered = (
        z
        - z.mean()
    )


    shaped = (
        float(spread)
        * centered
    )


    shift = (
        find_shift_for_clipped_log_mean(
            shaped,
            target_log_mean,
        )
    )


    calibrated = np.maximum(
        shaped
        + shift,
        0.0,
    )


    return (
        calibrated,
        float(shift),
    )


# =============================================================================
# 11. VARIANT A — FULL POLAR
#
# Spread = 1.0454
# Mean clipped log = 2.329351
# =============================================================================

polar_full_log, polar_full_shift = (
    global_spread_level_calibration(
        safe_80_20_log,
        spread=POLAR_SPREAD,
        target_log_mean=POLAR_TARGET_LOG_MEAN,
    )
)


# =============================================================================
# 12. VARIANT B — SHIFT ONLY
#
# Spread stays exactly 1.0.
#
# Tests whether previous gain mainly comes from
# correcting overall level rather than dispersion.
# =============================================================================

shift_only_log, shift_only_c = (
    global_spread_level_calibration(
        safe_80_20_log,
        spread=1.0,
        target_log_mean=POLAR_TARGET_LOG_MEAN,
    )
)


# =============================================================================
# 13. VARIANT C — SPREAD ONLY, ORIGINAL LEVEL PRESERVED
#
# Increase spread to 1.0454,
# but force clipped log mean back to the ORIGINAL
# 80/20 log mean.
#
# This isolates the effect of spread.
# =============================================================================

raw_80_20_log_mean = float(
    safe_80_20_log.mean()
)


spread_only_log, spread_only_c = (
    global_spread_level_calibration(
        safe_80_20_log,
        spread=POLAR_SPREAD,
        target_log_mean=raw_80_20_log_mean,
    )
)


# =============================================================================
# 14. VARIANT D — 50% SHRINKAGE TOWARD FULL POLAR
#
# Half raw 80/20
# Half full Polar
#
# Global, rank-stable-ish and much safer than jumping fully.
# =============================================================================

polar_shrink50_log = (
    (1.0 - SHRINK_50)
    * safe_80_20_log
    +
    SHRINK_50
    * polar_full_log
)


polar_shrink50_log = np.maximum(
    polar_shrink50_log,
    0.0,
)


# =============================================================================
# 15. VARIANT E — 75% SHRINKAGE TOWARD FULL POLAR
# =============================================================================

polar_shrink75_log = (
    (1.0 - SHRINK_75)
    * safe_80_20_log
    +
    SHRINK_75
    * polar_full_log
)


polar_shrink75_log = np.maximum(
    polar_shrink75_log,
    0.0,
)


# =============================================================================
# 16. VARIANT F — JOINT SHRINKAGE 50%
#
# Instead of blending final raw/Polar predictions,
# shrink BOTH calibration parameters:
#
# spread:
#   1.0 -> 1.0454 only halfway
#
# target:
#   raw mean -> 2.329351 only halfway
#
# This is the most conservative calibrated variant.
# =============================================================================

joint_target_mean = (
    0.50
    * raw_80_20_log_mean
    +
    0.50
    * POLAR_TARGET_LOG_MEAN
)


joint_shrink50_log, joint_shrink50_c = (
    global_spread_level_calibration(
        safe_80_20_log,
        spread=JOINT_SHRINK_SPREAD,
        target_log_mean=joint_target_mean,
    )
)


# =============================================================================
# 17. OPTIONAL DIFFERENT CONTROL — RAW-SPACE 80/20
#
# Not my primary choice because RMSLE naturally corresponds
# to log1p-space, but useful as one deliberately different forecast.
# =============================================================================

own_raw_mean = (
    own_matrix.mean(
        axis=1
    )
)

denis_raw_mean = (
    np.column_stack([
        denis_seed7,
        denis_seed42,
    ])
    .mean(
        axis=1
    )
)


rawspace_80_20 = (
    DENIS_WEIGHT
    * denis_raw_mean
    +
    OWN_WEIGHT
    * own_raw_mean
)


rawspace_80_20 = np.clip(
    rawspace_80_20,
    0.0,
    None,
)


rawspace_80_20_log = np.log1p(
    rawspace_80_20
)


# =============================================================================
# 18. ALL VARIANTS
# =============================================================================

variants = {
    # SAFEST
    "SAFE_Denis80_Own20_NO_CAL":
        safe_80_20_log,

    # Neighboring safe point
    "SAFE_Denis75_Own25_NO_CAL":
        safe_75_25_log,

    # Full previous-style correction
    "CAL_Denis80_Own20_POLAR_FULL":
        polar_full_log,

    # Ablations
    "CAL_Denis80_Own20_SHIFT_ONLY":
        shift_only_log,

    "CAL_Denis80_Own20_SPREAD_ONLY_KEEP_RAW_LEVEL":
        spread_only_log,

    # Shrinkage variants
    "CAL_Denis80_Own20_POLAR_SHRINK50":
        polar_shrink50_log,

    "CAL_Denis80_Own20_POLAR_SHRINK75":
        polar_shrink75_log,

    "CAL_Denis80_Own20_JOINT_SHRINK50":
        joint_shrink50_log,

    # Deliberately different control
    "EXP_Denis80_Own20_RAWSPACE":
        rawspace_80_20_log,
}


# =============================================================================
# 19. SAVE ALL SUBMISSIONS
# =============================================================================

saved_paths = {}


for name, log_pred in variants.items():

    log_pred = np.asarray(
        log_pred,
        dtype=np.float64,
    )


    if not np.isfinite(
        log_pred
    ).all():

        raise ValueError(
            f"{name}: log prediction contains NaN/inf"
        )


    log_pred = np.maximum(
        log_pred,
        0.0,
    )


    raw_pred = np.expm1(
        log_pred
    )


    raw_pred = np.clip(
        raw_pred,
        0.0,
        None,
    )


    submission = (
        sample_submit.copy()
    )


    submission[
        SUBMISSION_TARGET_COL
    ] = raw_pred


    path = (
        OUT_DIR
        / f"submission_{name}.csv"
    )


    submission.to_csv(
        path,
        index=False,
    )


    saved_paths[
        name
    ] = path


# =============================================================================
# 20. DIAGNOSTICS
# =============================================================================

baseline_log = (
    variants[
        "SAFE_Denis80_Own20_NO_CAL"
    ]
)


diagnostic_rows = []


for name, log_pred in variants.items():

    z = np.maximum(
        np.asarray(
            log_pred,
            dtype=np.float64,
        ),
        0.0,
    )


    raw = np.expm1(
        z
    )


    corr = float(
        np.corrcoef(
            baseline_log,
            z,
        )[0, 1]
    )


    diagnostic_rows.append({
        "variant":
            name,

        "mean_log1p":
            float(
                z.mean()
            ),

        "std_log1p":
            float(
                z.std()
            ),

        "zero_share":
            float(
                np.mean(
                    z == 0.0
                )
            ),

        "mean_raw":
            float(
                raw.mean()
            ),

        "median_raw":
            float(
                np.median(
                    raw
                )
            ),

        "p90_raw":
            float(
                np.quantile(
                    raw,
                    0.90,
                )
            ),

        "p99_raw":
            float(
                np.quantile(
                    raw,
                    0.99,
                )
            ),

        "corr_with_safe80_log":
            corr,

        "file":
            str(
                saved_paths[name]
            ),
    })


diagnostics = pd.DataFrame(
    diagnostic_rows
)


diagnostics.to_csv(
    OUT_DIR
    / "calibration_catalog.csv",
    index=False,
)


# =============================================================================
# 21. EFFECTIVE WEIGHTS
# =============================================================================

own_individual_weight = (
    OWN_WEIGHT
    / len(own_regressors)
)


weights = []


weights.append({
    "prediction":
        "submission_hurdle_seed7.csv",

    "family":
        "Denis",

    "effective_weight_SAFE_80_20":
        DENIS_WEIGHT / 2,
})


weights.append({
    "prediction":
        "submission_hurdle_seed42.csv",

    "family":
        "Denis",

    "effective_weight_SAFE_80_20":
        DENIS_WEIGHT / 2,
})


for name in own_regressors:

    weights.append({
        "prediction":
            name,

        "family":
            "Own clean regressor",

        "effective_weight_SAFE_80_20":
            own_individual_weight,
    })


weights_df = pd.DataFrame(
    weights
)


weights_df.to_csv(
    OUT_DIR
    / "effective_weights_80_20.csv",
    index=False,
)


# =============================================================================
# 22. CALIBRATION PARAMETER REPORT
# =============================================================================

parameter_report = pd.DataFrame([
    {
        "variant":
            "RAW_80_20",

        "spread":
            1.0,

        "target_log_mean":
            raw_80_20_log_mean,

        "shift":
            np.nan,
    },

    {
        "variant":
            "POLAR_FULL",

        "spread":
            POLAR_SPREAD,

        "target_log_mean":
            POLAR_TARGET_LOG_MEAN,

        "shift":
            polar_full_shift,
    },

    {
        "variant":
            "SHIFT_ONLY",

        "spread":
            1.0,

        "target_log_mean":
            POLAR_TARGET_LOG_MEAN,

        "shift":
            shift_only_c,
    },

    {
        "variant":
            "SPREAD_ONLY_KEEP_RAW_LEVEL",

        "spread":
            POLAR_SPREAD,

        "target_log_mean":
            raw_80_20_log_mean,

        "shift":
            spread_only_c,
    },

    {
        "variant":
            "JOINT_SHRINK50",

        "spread":
            JOINT_SHRINK_SPREAD,

        "target_log_mean":
            joint_target_mean,

        "shift":
            joint_shrink50_c,
    },
])


parameter_report.to_csv(
    OUT_DIR
    / "calibration_parameters.csv",
    index=False,
)


# =============================================================================
# 23. FINAL REPORT
# =============================================================================

print("\n" + "=" * 100)
print("DENIS 80 / OWN 20 GLOBAL CALIBRATION LAB READY")
print("=" * 100)


print(
    "\nOwn regressors:",
    len(own_regressors),
)

print(
    "Denis seed7 effective weight:",
    DENIS_WEIGHT / 2,
)

print(
    "Denis seed42 effective weight:",
    DENIS_WEIGHT / 2,
)

print(
    "Each own regressor effective weight:",
    own_individual_weight,
)


print("\nRAW 80/20:")
print(
    "mean log1p =",
    raw_80_20_log_mean,
)


print("\nPOLAR:")
print(
    "spread =",
    POLAR_SPREAD,
)

print(
    "target mean log =",
    POLAR_TARGET_LOG_MEAN,
)

print(
    "full Polar shift =",
    polar_full_shift,
)


print("\nJOINT SHRINK 50:")
print(
    "spread =",
    JOINT_SHRINK_SPREAD,
)

print(
    "target log mean =",
    joint_target_mean,
)

print(
    "shift =",
    joint_shrink50_c,
)


print("\n" + "=" * 100)
print("VARIANTS")
print("=" * 100)

display(
    diagnostics.sort_values(
        "variant"
    )
)


print("\nWEIGHTS:")
display(
    weights_df
)


print("\nPARAMETERS:")
display(
    parameter_report
)


print("\n" + "=" * 100)
print("FILES")
print("=" * 100)

for name, path in saved_paths.items():
    print(
        f"{name:<55} -> {path}"
    )


print("\nDONE")

OWN CLEAN REGRESSORS

BASE:
  xgboost_log_level
  catboost_log_level
  catboost_log_diff
  xgboost_log_diff

COLUMN BAGGING:
  xgboost_log_level__bag_s17
  catboost_log_level__bag_s17
  catboost_log_diff__bag_s17
  xgboost_log_diff__bag_s17

TOTAL: 8

Unique own regressors: 8

DENIS
/kaggle/input/datasets/vladimirsmirnovve/denis-2-files/submission_hurdle_seed7.csv
/kaggle/input/datasets/vladimirsmirnovve/denis-2-files/submission_hurdle_seed42.csv

DENIS 80 / OWN 20 GLOBAL CALIBRATION LAB READY

Own regressors: 8
Denis seed7 effective weight: 0.4
Denis seed42 effective weight: 0.4
Each own regressor effective weight: 0.025

RAW 80/20:
mean log1p = 2.332508992590961

POLAR:
spread = 1.0454
target mean log = 2.329351
full Polar shift = 2.329350251205166

JOINT SHRINK 50:
spread = 1.0227
target log mean = 2.33092999629548
shift = 2.33092999629548

VARIANTS


,variant,mean_log1p,std_log1p,zero_share,mean_raw,median_raw,p90_raw,p99_raw,corr_with_safe80_log,file
7,CAL_Denis80_Own20_JOINT_SHRINK50,2.330930,1.612544,0.000000,41.066551,6.912130,106.272923,491.526120,1.000000,/kaggle/working/ozon_final/denis80_own20_globa...
2,CAL_Denis80_Own20_POLAR_FULL,2.329351,1.648335,0.000104,43.603604,6.853742,111.824577,534.840292,1.000000,/kaggle/working/ozon_final/denis80_own20_globa...
5,CAL_Denis80_Own20_POLAR_SHRINK50,2.330930,1.612543,0.000000,41.066535,6.912127,106.272883,491.525936,1.000000,/kaggle/working/ozon_final/denis80_own20_globa...
6,CAL_Denis80_Own20_POLAR_SHRINK75,2.330140,1.630439,0.000000,42.311125,6.882880,109.013716,512.726816,1.000000,/kaggle/working/ozon_final/denis80_own20_globa...
3,CAL_Denis80_Own20_SHIFT_ONLY,2.329351,1.576752,0.000000,38.588309,6.945814,100.672778,450.285463,1.000000,/kaggle/working/ozon_final/denis80_own20_globa...
4,CAL_Denis80_Own20_SPREAD_ONLY_KEEP_RAW_LEVEL,2.332509,1.648336,0.000076,43.744696,6.878585,112.181469,536.535288,1.000000,/kaggle/working/ozon_final/denis80_own20_globa...
8,EXP_Denis80_Own20_RAWSPACE,2.335495,1.577193,0.000000,38.847471,7.002940,101.267827,452.945098,0.999996,/kaggle/working/ozon_final/denis80_own20_globa...
1,SAFE_Denis75_Own25_NO_CAL,2.333298,1.576111,0.000000,38.703562,6.987726,100.778903,451.874995,0.999992,/kaggle/working/ozon_final/denis80_own20_globa...
0,SAFE_Denis80_Own20_NO_CAL,2.332509,1.576752,0.000000,38.713527,6.970946,100.994367,451.712871,1.000000,/kaggle/working/ozon_final/denis80_own20_globa...



WEIGHTS:


,prediction,family,effective_weight_SAFE_80_20
0,submission_hurdle_seed7.csv,Denis,0.400
1,submission_hurdle_seed42.csv,Denis,0.400
2,xgboost_log_level,Own clean regressor,0.025
3,catboost_log_level,Own clean regressor,0.025
4,catboost_log_diff,Own clean regressor,0.025
5,xgboost_log_diff,Own clean regressor,0.025
6,xgboost_log_level__bag_s17,Own clean regressor,0.025
7,catboost_log_level__bag_s17,Own clean regressor,0.025
8,catboost_log_diff__bag_s17,Own clean regressor,0.025
9,xgboost_log_diff__bag_s17,Own clean regressor,0.025



PARAMETERS:


,variant,spread,target_log_mean,shift
0,RAW_80_20,1.0000,2.332509,NaN
1,POLAR_FULL,1.0454,2.329351,2.329350
2,SHIFT_ONLY,1.0000,2.329351,2.329351
3,SPREAD_ONLY_KEEP_RAW_LEVEL,1.0454,2.332509,2.332509
4,JOINT_SHRINK50,1.0227,2.330930,2.330930



FILES
SAFE_Denis80_Own20_NO_CAL                               -> /kaggle/working/ozon_final/denis80_own20_global_calibration_lab/submission_SAFE_Denis80_Own20_NO_CAL.csv
SAFE_Denis75_Own25_NO_CAL                               -> /kaggle/working/ozon_final/denis80_own20_global_calibration_lab/submission_SAFE_Denis75_Own25_NO_CAL.csv
CAL_Denis80_Own20_POLAR_FULL                            -> /kaggle/working/ozon_final/denis80_own20_global_calibration_lab/submission_CAL_Denis80_Own20_POLAR_FULL.csv
CAL_Denis80_Own20_SHIFT_ONLY                            -> /kaggle/working/ozon_final/denis80_own20_global_calibration_lab/submission_CAL_Denis80_Own20_SHIFT_ONLY.csv
CAL_Denis80_Own20_SPREAD_ONLY_KEEP_RAW_LEVEL            -> /kaggle/working/ozon_final/denis80_own20_global_calibration_lab/submission_CAL_Denis80_Own20_SPREAD_ONLY_KEEP_RAW_LEVEL.csv
CAL_Denis80_Own20_POLAR_SHRINK50                        -> /kaggle/working/ozon_final/denis80_own20_global_calibration_lab/submission_CAL_Denis80_Ow

In [141]:
# =============================================================================
# OZONE — FINAL 3 SUBMISSIONS BEFORE GRAM — CAREFUL V2
#
# We deliberately keep only three evidence-driven variants:
#
# 1) SAFE / NEARBY
#    Denis 75% + Own clean regressors 25%
#    Full Polar-style GLOBAL calibration:
#       spread = 1.0454
#       clipped mean(log1p) = 2.329351
#
#    This changes only the family weight slightly from the already successful
#    Denis70 / Own30 + Polar submission.
#
# 2) AGGRESSIVE WEIGHT
#    Denis 80% + Own clean regressors 20%
#    The SAME full Polar global calibration.
#
#    This tests one clean hypothesis:
#       "Does moving further toward the stronger Denis family keep helping?"
#
# 3) CALIBRATION ABLATION
#    Denis 70% + Own clean regressors 30%
#    LEVEL correction only:
#       spread = 1.0
#       clipped mean(log1p) = 2.329351
#
#    This keeps the already successful 70/30 family weight fixed and tests
#    whether the transferred spread factor 1.0454 is actually useful for
#    OUR mixture.
#
# No arbitrary 50% shrinkage parameter.
# No local weights.
# No segment weights.
# No Ridge.
# No target specialists.
# No ready-made blends.
#
# The 4th submission slot is intentionally reserved for conservative Gram/Ridge
# correction.
# =============================================================================

from pathlib import Path
import hashlib

import numpy as np
import pandas as pd


# =============================================================================
# 0. CONFIG
# =============================================================================

ROOT = Path("/kaggle/working/ozon_final")

OUT_DIR = ROOT / "final_3_before_gram"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DENIS_SEED7_PATH = Path(
    "/kaggle/input/datasets/vladimirsmirnovve/"
    "denis-2-files/submission_hurdle_seed7.csv"
)

DENIS_SEED42_PATH = Path(
    "/kaggle/input/datasets/vladimirsmirnovve/"
    "denis-2-files/submission_hurdle_seed42.csv"
)

DENIS_WEIGHT = 0.80
OWN_WEIGHT = 0.20

SAFE_DENIS_WEIGHT = 0.75
SAFE_OWN_WEIGHT = 0.25

ABLATION_DENIS_WEIGHT = 0.70
ABLATION_OWN_WEIGHT = 0.30

POLAR_SPREAD = 1.0454
POLAR_TARGET_LOG_MEAN = 2.329351


# =============================================================================
# 1. BASIC CHECKS
# =============================================================================

if FINAL_PREDICTIONS is None:
    raise RuntimeError(
        "FINAL_PREDICTIONS is None. "
        "Run final fit / stochastic blocks first."
    )

if abs(DENIS_WEIGHT + OWN_WEIGHT - 1.0) > 1e-12:
    raise ValueError("DENIS_WEIGHT + OWN_WEIGHT must equal 1.")


# =============================================================================
# 2. REFERENCE USER ORDER
# =============================================================================

reference = sample_submit[[USER_COL]].copy()
reference[USER_COL] = reference[USER_COL].astype(str)

N_USERS = len(reference)


# =============================================================================
# 3. OWN CLEAN REGRESSORS
#
# Include:
#   - all active base regressors from MODEL_NAMES
#   - genuine column-bagging boosting regressors
#
# Exclude:
#   - hurdle/double models
#   - Ridge
#   - target specialists
#   - finished blends
# =============================================================================

base_regressors = list(MODEL_NAMES)

bagging_regressors = []

if "STOCHASTIC_REGISTRY" in globals():
    for run in STOCHASTIC_REGISTRY:
        name = run.get("run_name")
        kind = run.get("kind", "")

        if not name:
            continue

        if kind == "column_bagging" or "__bag_" in name:
            bagging_regressors.append(name)

# Safe fallback if registry metadata is unavailable.
if not bagging_regressors:
    bagging_regressors = [
        c
        for c in FINAL_PREDICTIONS.columns
        if "__bag_" in str(c)
    ]

own_regressors = list(
    dict.fromkeys(
        base_regressors + bagging_regressors
    )
)

if not own_regressors:
    raise RuntimeError("No own clean regressors found.")


# =============================================================================
# 4. ALIGN OWN PREDICTIONS
# =============================================================================

missing = [
    name
    for name in own_regressors
    if name not in FINAL_PREDICTIONS.columns
]

if missing:
    raise RuntimeError(
        "Missing own predictions:\n"
        + "\n".join(missing)
    )

fp = FINAL_PREDICTIONS[
    [USER_COL] + own_regressors
].copy()

fp[USER_COL] = fp[USER_COL].astype(str)

if fp[USER_COL].duplicated().any():
    raise ValueError(
        "FINAL_PREDICTIONS contains duplicated user_id."
    )

fp = reference.merge(
    fp,
    on=USER_COL,
    how="left",
    validate="one_to_one",
)

own_matrix = fp[
    own_regressors
].to_numpy(dtype=np.float64)

if not np.isfinite(own_matrix).all():
    raise ValueError(
        "Own predictions contain NaN/inf."
    )

if np.any(own_matrix < 0):
    raise ValueError(
        "Own predictions contain negative values."
    )


# =============================================================================
# 5. REMOVE EXACT / PRACTICALLY EXACT DUPLICATE OWN PREDICTIONS
# =============================================================================

unique_indices = []
unique_names = []
seen_hashes = {}

for j, name in enumerate(own_regressors):
    pred = own_matrix[:, j]

    pred_hash = hashlib.sha256(
        np.round(pred, 10).tobytes()
    ).hexdigest()

    if pred_hash in seen_hashes:
        print(
            "DUPLICATE SKIPPED:",
            name,
            "==",
            seen_hashes[pred_hash],
        )
        continue

    seen_hashes[pred_hash] = name
    unique_indices.append(j)
    unique_names.append(name)

own_matrix = own_matrix[:, unique_indices]
own_regressors = unique_names

if len(own_regressors) == 0:
    raise RuntimeError(
        "No unique own regressors left after deduplication."
    )


# =============================================================================
# 6. EXTERNAL SUBMISSION LOADER
# =============================================================================

def load_submission(path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"File not found:\n{path}"
        )

    df = pd.read_csv(path)

    if USER_COL not in df.columns:
        raise ValueError(
            f"{path}: missing {USER_COL}"
        )

    if SUBMISSION_TARGET_COL in df.columns:
        pred_col = SUBMISSION_TARGET_COL
    else:
        candidates = [
            c
            for c in df.columns
            if c != USER_COL
        ]

        if len(candidates) != 1:
            raise ValueError(
                f"{path}: cannot determine prediction column.\n"
                f"Columns: {list(df.columns)}"
            )

        pred_col = candidates[0]

    tmp = df[
        [USER_COL, pred_col]
    ].copy()

    tmp[USER_COL] = tmp[USER_COL].astype(str)

    if tmp[USER_COL].duplicated().any():
        raise ValueError(
            f"{path}: duplicated {USER_COL}"
        )

    tmp = reference.merge(
        tmp,
        on=USER_COL,
        how="left",
        validate="one_to_one",
    )

    pred = pd.to_numeric(
        tmp[pred_col],
        errors="coerce",
    ).to_numpy(dtype=np.float64)

    if len(pred) != N_USERS:
        raise ValueError(
            f"{path}: row count mismatch"
        )

    if not np.isfinite(pred).all():
        raise ValueError(
            f"{path}: NaN/inf after alignment"
        )

    if np.any(pred < 0):
        raise ValueError(
            f"{path}: negative predictions"
        )

    return pred


# =============================================================================
# 7. LOAD DENIS
# =============================================================================

denis_seed7 = load_submission(
    DENIS_SEED7_PATH
)

denis_seed42 = load_submission(
    DENIS_SEED42_PATH
)


# =============================================================================
# 8. FAMILY LOG PREDICTIONS
#
# RMSLE is RMSE after log1p transformation, so blending is performed
# in log1p-space.
# =============================================================================

own_log_matrix = np.log1p(own_matrix)
own_log = own_log_matrix.mean(axis=1)

denis_log_matrix = np.column_stack([
    np.log1p(denis_seed7),
    np.log1p(denis_seed42),
])

denis_log = denis_log_matrix.mean(axis=1)


# =============================================================================
# 9. GLOBAL FAMILY BLENDS
# =============================================================================

blend_80_20_log = (
    DENIS_WEIGHT * denis_log
    + OWN_WEIGHT * own_log
)

blend_75_25_log = (
    SAFE_DENIS_WEIGHT * denis_log
    + SAFE_OWN_WEIGHT * own_log
)

blend_70_30_log = (
    ABLATION_DENIS_WEIGHT * denis_log
    + ABLATION_OWN_WEIGHT * own_log
)


# =============================================================================
# 10. GLOBAL CALIBRATION HELPERS
# =============================================================================

def find_shift_for_clipped_log_mean(
    centered,
    target_mean,
    iterations=120,
):
    x = np.asarray(
        centered,
        dtype=np.float64,
    )

    target_mean = float(target_mean)

    if target_mean < 0:
        raise ValueError(
            "target_mean must be non-negative."
        )

    def clipped_mean(shift):
        return float(
            np.maximum(
                x + shift,
                0.0,
            ).mean()
        )

    low = (
        -float(np.max(x))
        - target_mean
        - 10.0
    )

    high = (
        float(np.max(np.abs(x)))
        + target_mean
        + 10.0
    )

    for _ in range(30):
        if clipped_mean(high) >= target_mean:
            break
        high *= 2.0
    else:
        raise RuntimeError(
            "Could not bracket calibration shift."
        )

    for _ in range(int(iterations)):
        mid = (low + high) / 2.0

        if clipped_mean(mid) < target_mean:
            low = mid
        else:
            high = mid

    return (low + high) / 2.0


def global_spread_level_calibration(
    log_prediction,
    *,
    spread,
    target_log_mean,
):
    z = np.asarray(
        log_prediction,
        dtype=np.float64,
    )

    centered = z - z.mean()
    shaped = float(spread) * centered

    shift = find_shift_for_clipped_log_mean(
        shaped,
        target_log_mean,
    )

    calibrated = np.maximum(
        shaped + shift,
        0.0,
    )

    return calibrated, float(shift)


# =============================================================================
# 11. SAFE — 75 / 25 + FULL GLOBAL POLAR CALIBRATION
#
# Only a small family-weight move from the already successful 70/30 Polar run.
# No local or segment-specific weights.
# =============================================================================

safe_75_25_polar_log, safe_75_25_polar_shift = (
    global_spread_level_calibration(
        blend_75_25_log,
        spread=POLAR_SPREAD,
        target_log_mean=POLAR_TARGET_LOG_MEAN,
    )
)


# =============================================================================
# 12. BEST BET — 80 / 20 + FULL POLAR CALIBRATION
# =============================================================================

polar_full_log, polar_full_shift = (
    global_spread_level_calibration(
        blend_80_20_log,
        spread=POLAR_SPREAD,
        target_log_mean=POLAR_TARGET_LOG_MEAN,
    )
)


# =============================================================================
# 13. CALIBRATION ABLATION — 70 / 30 + SHIFT ONLY
#
# Keep the already successful 70/30 family weights.
# Remove ONLY the transferred spread expansion.
#
# spread = 1.0
# clipped mean(log1p) = 2.329351
#
# This is a much cleaner test than an arbitrary 50% shrinkage coefficient.
# =============================================================================

shift_only_70_30_log, shift_only_70_30_shift = (
    global_spread_level_calibration(
        blend_70_30_log,
        spread=1.0,
        target_log_mean=POLAR_TARGET_LOG_MEAN,
    )
)


# =============================================================================
# 14. ONLY THREE SUBMISSIONS
#
# Slot 4 is reserved for conservative Gram/Ridge correction.
# =============================================================================

variants = {
    "SAFE_Denis75_Own25_POLAR_FULL":
        safe_75_25_polar_log,

    "AGGRESSIVE_Denis80_Own20_POLAR_FULL":
        polar_full_log,

    "ABLATION_Denis70_Own30_SHIFT_ONLY":
        shift_only_70_30_log,
}


# =============================================================================
# 15. SAVE
# =============================================================================

saved_paths = {}

for name, log_pred in variants.items():
    z = np.asarray(
        log_pred,
        dtype=np.float64,
    )

    if not np.isfinite(z).all():
        raise ValueError(
            f"{name}: NaN/inf in log prediction."
        )

    z = np.maximum(
        z,
        0.0,
    )

    raw_pred = np.expm1(z)
    raw_pred = np.clip(
        raw_pred,
        0.0,
        None,
    )

    submission = sample_submit.copy()
    submission[
        SUBMISSION_TARGET_COL
    ] = raw_pred

    path = (
        OUT_DIR
        / f"submission_{name}.csv"
    )

    submission.to_csv(
        path,
        index=False,
    )

    saved_paths[name] = path


# =============================================================================
# 16. DIAGNOSTICS
# =============================================================================

diagnostic_rows = []

for name, log_pred in variants.items():
    z = np.maximum(
        np.asarray(
            log_pred,
            dtype=np.float64,
        ),
        0.0,
    )

    raw = np.expm1(z)

    diagnostic_rows.append({
        "variant": name,
        "mean_log1p": float(z.mean()),
        "std_log1p": float(z.std()),
        "zero_share": float(np.mean(z == 0.0)),
        "mean_raw": float(raw.mean()),
        "median_raw": float(np.median(raw)),
        "p90_raw": float(np.quantile(raw, 0.90)),
        "p99_raw": float(np.quantile(raw, 0.99)),
        "corr_with_safe75_polar_log": float(
            np.corrcoef(
                safe_75_25_polar_log,
                z,
            )[0, 1]
        ),
        "file": str(saved_paths[name]),
    })

diagnostics = pd.DataFrame(
    diagnostic_rows
)

diagnostics.to_csv(
    OUT_DIR / "diagnostics.csv",
    index=False,
)


# =============================================================================
# 17. FINAL REPORT
# =============================================================================

own_individual_weight = (
    OWN_WEIGHT
    / len(own_regressors)
)

print("\n" + "=" * 100)
print("FINAL 3 SUBMISSIONS BEFORE GRAM — CAREFUL V2")
print("=" * 100)

print(
    f"\nOwn clean regressors: "
    f"{len(own_regressors)}"
)

print(
    f"  Base: "
    f"{len(base_regressors)}"
)

print(
    f"  Column bagging: "
    f"{len(bagging_regressors)}"
)

print("\nSAFE 75/25 FAMILY WEIGHTS:")
print(
    f"  Denis family: {SAFE_DENIS_WEIGHT:.2f}"
)
print(
    f"  Own family:   {SAFE_OWN_WEIGHT:.2f}"
)
print(
    f"  Safe 75/25 Polar shift: {safe_75_25_polar_shift:.9f}"
)

print("\nMAIN 80/20 EFFECTIVE WEIGHTS:")

print(
    f"  Denis seed7:  "
    f"{DENIS_WEIGHT / 2:.4f} "
    f"({100 * DENIS_WEIGHT / 2:.1f}%)"
)

print(
    f"  Denis seed42: "
    f"{DENIS_WEIGHT / 2:.4f} "
    f"({100 * DENIS_WEIGHT / 2:.1f}%)"
)

print(
    f"  Each own regressor: "
    f"{own_individual_weight:.6f} "
    f"({100 * own_individual_weight:.3f}%)"
)

print("\nCALIBRATION:")

print(
    f"  Safe 75/25 full Polar shift: "
    f"{safe_75_25_polar_shift:.9f}"
)

print(
    f"  Aggressive 80/20 full Polar shift: "
    f"{polar_full_shift:.9f}"
)

print(
    f"  Ablation 70/30 shift-only shift: "
    f"{shift_only_70_30_shift:.9f}"
)

print(
    f"  Polar target clipped log mean: "
    f"{POLAR_TARGET_LOG_MEAN:.9f}"
)

print(
    f"  Full Polar spread: "
    f"{POLAR_SPREAD:.6f}"
)

print(
    "  Shift-only spread: 1.000000"
)

print("\nFILES:")

for name, path in saved_paths.items():
    print(
        f"  {name:<50} -> {path}"
    )

print("\nDIAGNOSTICS:")
display(diagnostics)

print("\n4th slot: RESERVED FOR GRAM/RIDGE CORRECTION")
print("=" * 100)



FINAL 3 SUBMISSIONS BEFORE GRAM — CAREFUL V2

Own clean regressors: 8
  Base: 4
  Column bagging: 4

SAFE 75/25 FAMILY WEIGHTS:
  Denis family: 0.75
  Own family:   0.25
  Safe 75/25 Polar shift: 2.329350075

MAIN 80/20 EFFECTIVE WEIGHTS:
  Denis seed7:  0.4000 (40.0%)
  Denis seed42: 0.4000 (40.0%)
  Each own regressor: 0.025000 (2.500%)

CALIBRATION:
  Safe 75/25 full Polar shift: 2.329350075
  Aggressive 80/20 full Polar shift: 2.329350251
  Ablation 70/30 shift-only shift: 2.329351000
  Polar target clipped log mean: 2.329351000
  Full Polar spread: 1.045400
  Shift-only spread: 1.000000

FILES:
  SAFE_Denis75_Own25_POLAR_FULL                      -> /kaggle/working/ozon_final/final_3_before_gram/submission_SAFE_Denis75_Own25_POLAR_FULL.csv
  AGGRESSIVE_Denis80_Own20_POLAR_FULL                -> /kaggle/working/ozon_final/final_3_before_gram/submission_AGGRESSIVE_Denis80_Own20_POLAR_FULL.csv
  ABLATION_Denis70_Own30_SHIFT_ONLY                  -> /kaggle/working/ozon_final/final_3

,variant,mean_log1p,std_log1p,zero_share,mean_raw,median_raw,p90_raw,p99_raw,corr_with_safe75_polar_log,file
0,SAFE_Denis75_Own25_POLAR_FULL,2.329351,1.647665,0.000124,43.554074,6.864531,111.482530,534.598570,1.000000,/kaggle/working/ozon_final/final_3_before_gram...
1,AGGRESSIVE_Denis80_Own20_POLAR_FULL,2.329351,1.648335,0.000104,43.603604,6.853742,111.824577,534.840292,0.999992,/kaggle/working/ozon_final/final_3_before_gram...
2,ABLATION_Denis70_Own30_SHIFT_ONLY,2.329351,1.575493,0.000000,38.507679,6.961688,100.159437,450.080042,0.999992,/kaggle/working/ozon_final/final_3_before_gram...



4th slot: RESERVED FOR GRAM/RIDGE CORRECTION


In [142]:
# =============================================================================
# DENIS 75% / OWN 25% + SHIFT ONLY
#
# Предпоследний submission.
#
# Веса:
#   Denis seed7  = 37.5%
#   Denis seed42 = 37.5%
#   Own clean regressors together = 25%
#
# Calibration:
#   spread = 1.0            <-- НЕТ растяжения 1.0454
#   target clipped log mean = 2.329351
#
# Никаких local / segment weights.
# =============================================================================

from pathlib import Path
import numpy as np
import pandas as pd


# =============================================================================
# 0. CONFIG
# =============================================================================

DENIS_WEIGHT = 0.75
OWN_WEIGHT = 0.25

TARGET_LOG_MEAN = 2.329351

DENIS_SEED7_PATH = Path(
    "/kaggle/input/datasets/vladimirsmirnovve/"
    "denis-2-files/submission_hurdle_seed7.csv"
)

DENIS_SEED42_PATH = Path(
    "/kaggle/input/datasets/vladimirsmirnovve/"
    "denis-2-files/submission_hurdle_seed42.csv"
)

OUT_DIR = Path(
    "/kaggle/working/ozon_final/denis75_own25_shift_only"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

OUT_PATH = (
    OUT_DIR
    / "submission_Denis75_Own25_SHIFT_ONLY.csv"
)


# =============================================================================
# 1. REFERENCE ORDER
# =============================================================================

reference = sample_submit[
    [USER_COL]
].copy()

reference[USER_COL] = (
    reference[USER_COL]
    .astype(str)
)

N_USERS = len(reference)


# =============================================================================
# 2. FIND OWN CLEAN REGRESSORS
#
# Берём:
#   - все MODEL_NAMES
#   - только column-bagging boosting models
#
# НЕ берём:
#   - Hurdle / Double Models
#   - Ridge
#   - target specialists
#   - готовые blends
# =============================================================================

base_regressors = list(
    MODEL_NAMES
)

bagging_regressors = []

if "STOCHASTIC_REGISTRY" in globals():

    for run in STOCHASTIC_REGISTRY:

        name = run.get(
            "run_name"
        )

        kind = run.get(
            "kind",
            "",
        )

        if not name:
            continue

        if (
            kind == "column_bagging"
            or "__bag_" in name
        ):
            bagging_regressors.append(
                name
            )


# fallback
if not bagging_regressors:

    bagging_regressors = [
        c
        for c in FINAL_PREDICTIONS.columns
        if "__bag_" in str(c)
    ]


own_regressors = list(
    dict.fromkeys(
        base_regressors
        + bagging_regressors
    )
)


print("=" * 90)
print("OWN CLEAN REGRESSORS")
print("=" * 90)

for name in own_regressors:
    print(" ", name)

print(
    "\nTOTAL:",
    len(own_regressors),
)


# =============================================================================
# 3. ALIGN OWN PREDICTIONS
# =============================================================================

missing = [
    name
    for name in own_regressors
    if name not in FINAL_PREDICTIONS.columns
]

if missing:
    raise RuntimeError(
        "Missing own predictions:\n"
        + "\n".join(missing)
    )


own_df = FINAL_PREDICTIONS[
    [USER_COL]
    + own_regressors
].copy()

own_df[USER_COL] = (
    own_df[USER_COL]
    .astype(str)
)


if own_df[USER_COL].duplicated().any():
    raise ValueError(
        "Duplicated user_id in FINAL_PREDICTIONS."
    )


own_df = reference.merge(
    own_df,
    on=USER_COL,
    how="left",
    validate="one_to_one",
)


own_matrix = own_df[
    own_regressors
].to_numpy(
    dtype=np.float64
)


if not np.isfinite(
    own_matrix
).all():
    raise ValueError(
        "Own predictions contain NaN/inf."
    )

if np.any(
    own_matrix < 0
):
    raise ValueError(
        "Own predictions contain negative values."
    )


# =============================================================================
# 4. LOAD DENIS
# =============================================================================

def load_submission(path):

    df = pd.read_csv(
        path
    )

    df[USER_COL] = (
        df[USER_COL]
        .astype(str)
    )

    if SUBMISSION_TARGET_COL in df.columns:

        pred_col = (
            SUBMISSION_TARGET_COL
        )

    else:

        candidates = [
            c
            for c in df.columns
            if c != USER_COL
        ]

        if len(candidates) != 1:
            raise ValueError(
                f"Cannot determine prediction column: {path}"
            )

        pred_col = candidates[0]


    df = reference.merge(
        df[
            [USER_COL, pred_col]
        ],
        on=USER_COL,
        how="left",
        validate="one_to_one",
    )


    pred = pd.to_numeric(
        df[pred_col],
        errors="coerce",
    ).to_numpy(
        dtype=np.float64
    )


    if not np.isfinite(
        pred
    ).all():
        raise ValueError(
            f"NaN/inf in {path}"
        )

    if np.any(
        pred < 0
    ):
        raise ValueError(
            f"Negative predictions in {path}"
        )


    return pred


denis_seed7 = load_submission(
    DENIS_SEED7_PATH
)

denis_seed42 = load_submission(
    DENIS_SEED42_PATH
)


# =============================================================================
# 5. BUILD OWN + DENIS FAMILY PREDICTIONS IN LOG1P SPACE
# =============================================================================

own_log = np.log1p(
    own_matrix
).mean(
    axis=1
)


denis_log = np.column_stack([
    np.log1p(
        denis_seed7
    ),
    np.log1p(
        denis_seed42
    ),
]).mean(
    axis=1
)


# =============================================================================
# 6. 75 / 25 BLEND
# =============================================================================

blend_log = (
    DENIS_WEIGHT
    * denis_log
    +
    OWN_WEIGHT
    * own_log
)


print("\nRAW 75/25:")
print(
    "mean_log1p =",
    float(
        blend_log.mean()
    )
)

print(
    "std_log1p  =",
    float(
        blend_log.std()
    )
)


# =============================================================================
# 7. SHIFT-ONLY CALIBRATION
#
# IMPORTANT:
#
# spread = 1.0
#
# То есть:
#
# centered = blend - mean(blend)
#
# НЕТ:
# centered *= 1.0454
#
# Затем ищем shift c:
#
# mean(max(centered + c, 0)) = 2.329351
# =============================================================================

centered = (
    blend_log
    - blend_log.mean()
)


def clipped_mean(
    shift,
):

    return float(
        np.maximum(
            centered + shift,
            0.0,
        ).mean()
    )


low = (
    -float(
        np.max(centered)
    )
    - TARGET_LOG_MEAN
    - 10.0
)

high = (
    float(
        np.max(
            np.abs(centered)
        )
    )
    + TARGET_LOG_MEAN
    + 10.0
)


# bracket
for _ in range(30):

    if (
        clipped_mean(high)
        >= TARGET_LOG_MEAN
    ):
        break

    high *= 2.0

else:
    raise RuntimeError(
        "Could not bracket calibration shift."
    )


# binary search
for _ in range(120):

    mid = (
        low + high
    ) / 2.0

    if (
        clipped_mean(mid)
        < TARGET_LOG_MEAN
    ):
        low = mid

    else:
        high = mid


shift = (
    low + high
) / 2.0


# =============================================================================
# 8. FINAL LOG PREDICTION
# =============================================================================

final_log = np.maximum(
    centered + shift,
    0.0,
)


actual_mean = float(
    final_log.mean()
)


if not np.isclose(
    actual_mean,
    TARGET_LOG_MEAN,
    atol=1e-10,
):
    raise RuntimeError(
        "Target log mean was not reached."
    )


# =============================================================================
# 9. BACK TO RAW TARGET
# =============================================================================

final_pred = np.expm1(
    final_log
)

final_pred = np.clip(
    final_pred,
    0.0,
    None,
)


if not np.isfinite(
    final_pred
).all():
    raise RuntimeError(
        "Final predictions contain NaN/inf."
    )


# =============================================================================
# 10. SAVE SUBMISSION
# =============================================================================

submission = (
    sample_submit.copy()
)

submission[
    SUBMISSION_TARGET_COL
] = final_pred


submission.to_csv(
    OUT_PATH,
    index=False,
)


# =============================================================================
# 11. REPORT
# =============================================================================

print("\n" + "=" * 90)
print("DENIS 75 / OWN 25 — SHIFT ONLY")
print("=" * 90)

print(
    f"\nDenis total weight: "
    f"{DENIS_WEIGHT:.2f}"
)

print(
    f"  seed7:  "
    f"{DENIS_WEIGHT / 2:.3f}"
)

print(
    f"  seed42: "
    f"{DENIS_WEIGHT / 2:.3f}"
)

print(
    f"\nOwn total weight: "
    f"{OWN_WEIGHT:.2f}"
)

print(
    f"Each own regressor: "
    f"{OWN_WEIGHT / len(own_regressors):.6f}"
)

print("\nCALIBRATION:")
print(
    "spread = 1.0"
)

print(
    "target_log_mean =",
    TARGET_LOG_MEAN,
)

print(
    "shift =",
    shift,
)

print(
    "actual_log_mean =",
    actual_mean,
)

print(
    "std_log1p AFTER calibration =",
    float(
        final_log.std()
    )
)

print(
    "zero_share =",
    float(
        np.mean(
            final_log == 0.0
        )
    )
)

print("\nSAVED:")
print(
    OUT_PATH
)

print("\nREADY TO SUBMIT")

OWN CLEAN REGRESSORS
  xgboost_log_level
  catboost_log_level
  catboost_log_diff
  xgboost_log_diff
  xgboost_log_level__bag_s17
  catboost_log_level__bag_s17
  catboost_log_diff__bag_s17
  xgboost_log_diff__bag_s17

TOTAL: 8

RAW 75/25:
mean_log1p = 2.3332984907386596
std_log1p  = 1.5761106391708255

DENIS 75 / OWN 25 — SHIFT ONLY

Denis total weight: 0.75
  seed7:  0.375
  seed42: 0.375

Own total weight: 0.25
Each own regressor: 0.031250

CALIBRATION:
spread = 1.0
target_log_mean = 2.329351
shift = 2.329351
actual_log_mean = 2.329351
std_log1p AFTER calibration = 1.5761106391708255
zero_share = 0.0

SAVED:
/kaggle/working/ozon_final/denis75_own25_shift_only/submission_Denis75_Own25_SHIFT_ONLY.csv

READY TO SUBMIT
